<a href="https://colab.research.google.com/github/silviasd2025-bit/dea_studies/blob/main/ppml/20261006_ppml_claude.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
cooperation_pipeline.py ? G20 and EU-27, one program, shared engine
=====================================================================
Consolidates g20_pipeline.py and eu_pipeline.py into a single file. The
DEA/Malmquist-Luenberger engine, the PPML/FE-OLS estimation family, and
the panel-assembly logic are written ONCE and parametrized by a per-club
CONFIG dict (Section 3) rather than duplicated ? the two clubs differ in
their membership, their live-data connector (World Bank vs Eurostat, which
return different JSON shapes and so need different parsers), and their
fixed geography/language/border facts, not in the underlying model.

SECTIONS
  0. Dependency bootstrap
  1. Shared utilities (zscore, haversine, session, canonicalisation)
  2. DEA / Malmquist-Luenberger energy-security engine (club-agnostic)
  3. CLUB CONFIGS ? G20 and EU-27 membership, connectors, fixed facts
  4. Connectors ? World Bank (G20), Eurostat (EU), Stooq (both)
  5. Loaders for proprietary/coded data, with flagged simulated fallbacks
  6. Panel assembly (club-agnostic, driven by CLUBS[club])
  7. Estimation + reporting (PPML, FE-OLS, robustness ? club-agnostic)
  8. Cross-club comparison (only runs when --club both)
  9. CLI / main / self-tests

USAGE
  pip install pyfixest pandas numpy scipy matplotlib requests
  python cooperation_pipeline.py                    # both clubs, online if reachable
  python cooperation_pipeline.py --club g20          # G20 only
  python cooperation_pipeline.py --club eu           # EU-27 only
  python cooperation_pipeline.py --pooled            # pooled interaction model + Monte Carlo
  python cooperation_pipeline.py --montecarlo        # Monte Carlo validation only
  python cooperation_pipeline.py --templates         # blank data/*.csv for both clubs
  python cooperation_pipeline.py --selftest          # offline unit checks
  python cooperation_pipeline.py --offline           # skip network, use fallbacks
"""
from __future__ import annotations
import sys

# =========================================================================
# SECTION 0 ? DEPENDENCY BOOTSTRAP
# =========================================================================
def _ensure_deps():
    import importlib.util, subprocess
    required = ["numpy", "pandas", "scipy", "matplotlib", "requests", "pyfixest"]
    missing = [p for p in required if importlib.util.find_spec(p) is None]
    if missing:
        print(f"[setup] installing missing packages: {', '.join(missing)}")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
            import importlib as _il; _il.invalidate_caches()
        except Exception as e:
            print(f"[setup] auto-install failed ({e}). Run:  !pip install {' '.join(missing)}")
            raise

_ensure_deps()

import io, math, time, warnings
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from itertools import combinations
from pyfixest.demeaners import LsmrDemeaner
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import linprog
from scipy.stats import chi2, norm

warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount('/content/drive')

# =========================================================================
# SECTION 1 ? SHARED UTILITIES
# =========================================================================
SEED = 20260624
TRY_ONLINE = ("--offline" not in sys.argv)
STRICT_MODE = ("--strict" in sys.argv)


def _enforce_strict_mode(club_key, prov, allowed_simulated):
    """
    Scans this club's provenance log for any variable marked SIMULATED
    (or ABSENT, which is the same problem under a different name) whose
    'variable' name doesn't start with one of allowed_simulated. If any
    are found, stops immediately with a full, itemised missing-data
    report -- naming every offending variable and what real source would
    resolve it -- rather than letting the run continue on a silently
    mixed real/simulated basis.
    """
    offenders = []
    for row in prov:
        var = row.get("variable", "")
        status = row.get("status", "")
        if ("SIMULATED" in status.upper() or "ABSENT" in status.upper()) and not any(
            var.startswith(allowed) for allowed in allowed_simulated
        ):
            offenders.append(row)
    if not offenders:
        return
    lines = [f"--strict mode: {club_key} run STOPPED -- {len(offenders)} required "
            f"non-cooperation/non-FDI variable(s) have no real data:"]
    remedy = {
        "R&D/GDP": "populate via live World Bank/Eurostat access, or extend GERD_PCT (Section 3B)",
        "distance": "should not occur -- distance is FIXED/LIVE by design; report this as a bug",
        "price volatility": "populate via a real Brent/EIA/Pink-Sheet monthly series (see Stooq connector)",
        "trade": "this is now resolved via real UN Comtrade bilateral data (data/trade.csv, "
                "data/eu_trade.csv) -- see load_trade(). This message should only appear if "
                "those files are missing; if so, restore them from UN Comtrade bilateral "
                "merchandise export data (a country-level trade-openness PROXY is also "
                "available in data/total_trade.csv and used in m24_with_tradeopen_proxy, but "
                "it is not a substitute for genuine bilateral trade flows)",
        "RTA (institutional control)": "this is now resolved for both clubs via the WTO RTA-IS "
                                       "signatory-based dyadic construction (data/"
                                       "rta_wto_genuine_dyadic.csv, see build_rta_wto_signatory_based) "
                                       "-- Mario Larch's database is no longer required. This message "
                                       "should only appear if that file is missing; if so, restore it, "
                                       "or as a lower-quality fallback populate data/rta_larch.csv from "
                                       "Larch's database directly (a country-level RTA-activity PROXY "
                                       "is also available in data/rta_activity_by_country.csv and used "
                                       "in m19b_with_rta_proxy, but it is not a substitute for true "
                                       "dyadic RTA membership)",
    }
    for row in offenders:
        var = row["variable"]
        hint = next((v for k, v in remedy.items() if var.startswith(k)), "see this variable's loader docstring")
        lines.append(f"  - {var}: {row['source']}  ->  fix: {hint}")
    raise RuntimeError("\n".join(lines))

YEAR_MIN, YEAR_MAX = 2000, 2023
# TASK ("extend the period from 2007-2021 to 2000-2023"): widened as
# requested. What this ACTUALLY changes, checked directly against the
# embedded real data rather than assumed:
#   - 2000-2006: UPDATE (Chapter 2 DEA/MPI integration): ES itself now has
#     REAL coverage back to 2000 via this dissertation's own Chapter 2
#     bootstrap bias-corrected DEA/MPI results (see load_ch2_dea_mpi,
#     get_common_es_frontier), which cover all 38 canonical union members
#     for the full 2000-2023 window -- checked directly, not assumed. This
#     pipeline's OWN Ch3-WDI-based DEA computation still only covers
#     2007-2023 and remains available as a fallback if the Chapter 2 files
#     are absent. Innovation (R&D/patents) and cooperation still have NO
#     real coverage before 2007 in any source this pipeline uses (GERD,
#     OECD co-invention, CORDIS all start at 2007) -- these years are NOT
#     fabricated to fill that remaining gap, so the estimation sample is
#     still constrained by Innovation/cooperation, not by ES, and will
#     still effectively start around 2008 (2007 + 1 lag) in practice.
#   - 2022-2023: ES inputs (CO2/capita, import dependency, T&D losses,
#     renewable share) are 100% real through 2023. R&D/GDP (GERD) is real
#     for 36/40 countries through 2023. OECD co-invention is real through
#     2022 only (2023 -> NaN for that one component, degrading gracefully
#     since the Innovation composite skips missing components rather than
#     zero-filling them). patents_residents has ZERO real coverage from
#     2022 onward (confirmed directly against the source table) -- the
#     Innovation composite simply drops that component for those years.
#     CORDIS real cooperation data genuinely covers 2007-2023 (H2020
#     project end dates extend that far) -- verified by re-running the
#     aggregation without a year cap, not assumed from the original
#     2007-2021 extract.
YEARS = list(range(YEAR_MIN, YEAR_MAX + 1))

# CRITICAL FIX: this used to be DATA = Path("data") -- a bare relative
# path resolved against the CURRENT WORKING DIRECTORY at run time, not
# against the script's own location, and with no fallback if the
# expected "data/" subfolder didn't exist or didn't contain a file.
# Confirmed directly to be a severe, real bug, not a hypothetical one:
# every real-data file supplied across this project so far has been
# delivered as a FLAT set of files alongside this script (exactly what
# present_files produces), never pre-organised into a "data/" subfolder.
# Under the old DATA = Path("data"); DATA.mkdir(exist_ok=True), that call
# silently created an EMPTY data/ directory next to the flat files, so
# every loader's `if not p.exists(): return None` check failed for every
# single real dataset -- the entire pipeline was reporting simulated or
# absent for variables whose real data was sitting, unused, right next
# to the script the whole time. Reproduced directly (not assumed): copying
# this script and every delivered CSV flat into one folder and running it
# showed price volatility, trade, RTA, UNTS, FDI, GDELT, and the country-
# level RTA/trade proxies ALL reporting SIMULATED/ABSENT despite every
# file being present on disk.
#
# Fixed by resolving against the script's own directory (so it no longer
# matters what directory the user launches Python from), and by checking
# BOTH a "data/" subfolder AND the flat script directory for each
# requested file, preferring an organised data/ subfolder if the person
# has set one up but falling back to the flat layout that matches actual
# delivery. This preserves every existing `DATA / "filename"` call site
# unchanged (21 of them) via a __truediv__ override, and keeps the one
# direct DATA.mkdir() call working via a delegating method.
class _DataPathResolver:
    def __init__(self, script_dir):
        self.script_dir = script_dir
        self.subfolder = script_dir / "data_from_claude"

    def mkdir(self, exist_ok=True):
        self.subfolder.mkdir(exist_ok=exist_ok)

    def __truediv__(self, filename):
        candidate_sub = self.subfolder / filename
        if candidate_sub.exists():
            return candidate_sub
        candidate_flat = self.script_dir / filename
        if candidate_flat.exists():
            return candidate_flat
        # Neither exists: return the data/ subfolder path so template-
        # generation code (--templates) writes new files somewhere
        # organised, and so .exists() checks on the result still
        # correctly report False for genuinely absent data.
        return candidate_sub

    def __fspath__(self):
        return str(self.subfolder)

try:
    _SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    # FIX: __file__ is not defined when this code runs inside a Jupyter/
    # IPython notebook (or any exec'd/interactive context) rather than as
    # a standalone script -- confirmed directly by the exact NameError
    # this fallback now catches. There is no "script file" to anchor to
    # in that case, so the current working directory is the best
    # available anchor: in a notebook, that is normally wherever the
    # person launched Jupyter from, which is also where they would have
    # placed or uploaded their data files sitting flat alongside the
    # notebook -- the same delivery shape this whole DATA resolver exists
    # to handle.
    _SCRIPT_DIR = Path.cwd()
DATA = _DataPathResolver(Path("/content/drive/MyDrive/ppml"))
DATA.mkdir(exist_ok=True)
OUT_BASE = Path("/content/drive/MyDrive/ppml/output_from_claude"); OUT_BASE.mkdir(exist_ok=True)

rng = np.random.default_rng(SEED)


def zscore(s):
    sd = s.std(skipna=True)
    return (s - s.mean(skipna=True)) / sd if sd and sd == sd else s * 0.0


def haversine(a, b):
    (la1, lo1), (la2, lo2) = a, b
    p1, p2 = math.radians(la1), math.radians(la2)
    dphi, dl = math.radians(la2 - la1), math.radians(lo2 - lo1)
    h = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * 6371.0 * math.asin(math.sqrt(h))


def _session():
    import requests
    s = requests.Session()
    # A generic library User-Agent gets blocked or throttled by some hosts
    # (observed on Stooq); use a standard browser-like string everywhere.
    s.headers.update({
        "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                       "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"),
    })
    return s


def _canon(df):
    """Ensure undirected dyads have iso_i < iso_j."""
    swap = df["iso_i"] > df["iso_j"]
    df.loc[swap, ["iso_i", "iso_j"]] = df.loc[swap, ["iso_j", "iso_i"]].values
    return df


_PANEL_ORDER_CACHE = {}

# TASK (reviewer): "a real-only flag is only reliable if its coverage rule
# is reliable. Use an explicit source-specific coverage table; first and
# last positive observations are insufficient." Correct: inferring a
# source's coverage window from the min/max YEAR ACTUALLY OBSERVED in its
# real data conflates "the source has no data before/after this point" with
# "the source happened not to record a positive count in the extreme years
# of its true window" -- these are different claims, and only the first is
# a genuine, verified coverage statement. This table states each source's
# DOCUMENTED coverage explicitly, from its own known construction, rather
# than inferring it from the loaded data itself.
SOURCE_COVERAGE = {
    "cordis": (2007, 2023,
              "FP7 (2007-2013) + Horizon 2020 (2014-2020, with projects running through "
              "2023) by documented programme structure, not inferred from the extract's own "
              "observed year range -- though the two happen to coincide here, confirmed "
              "directly rather than assumed."),
}


def _get_source_coverage(source_name, real_df=None):
    """
    Returns (year_min, year_max) for a named source's DOCUMENTED coverage
    window if known (SOURCE_COVERAGE above); falls back to inferring from
    real_df's own observed min/max ONLY if the source is not in that table,
    printing an explicit warning that this fallback is exactly the
    unreliable inference the reviewer flagged -- so a silent, undocumented
    fallback is never mistaken for a verified coverage statement.
    """
    if source_name in SOURCE_COVERAGE:
        y0, y1, _ = SOURCE_COVERAGE[source_name]
        return y0, y1
    if real_df is not None and len(real_df) > 0:
        print(f"  WARNING: no documented coverage window for source '{source_name}' in "
             f"SOURCE_COVERAGE -- falling back to inferring from observed min/max year, "
             f"which the reviewer correctly notes is NOT a reliable coverage statement "
             f"(a source can have real coverage in years where it happens to record no "
             f"positive observations). Add this source to SOURCE_COVERAGE explicitly.")
        return int(real_df["year"].min()), int(real_df["year"].max())
    return None, None


def _to_panel_order(df, club):
    """
    SINGLE canonical pair-ordering function (reviewer: "establish one
    canonical pair key throughout the program... rather than repairing
    ordering separately in individual loaders"). Reorders every row's
    iso_i/iso_j to match build_panel's own convention -- combinations()
    over this club's members_map in DICTIONARY INSERTION order, which is
    NOT alphabetical -- so that any loader's output merges correctly
    against the panel on [iso_i, iso_j, year].

    Replaces the scattered, one-off fixes previously applied separately
    to individual loaders (build_rta_eu_membership, build_rta_wto_
    signatory_based, load_trade, load_cordis_dyadic_real): those were
    each correct in isolation but left every OTHER loader still calling
    _canon() (alphabetical order), which reintroduces the identical
    mismatch the moment a different loader or code path is used --
    confirmed as a real, not hypothetical, risk by a reviewer directly.
    Every loader that returns a dyadic [iso_i, iso_j, ...] DataFrame
    destined to merge against the panel should call this, not _canon(),
    from here on.

    Cached per club (by id(club)) since members_map is fixed per run.
    """
    key = id(club)
    if key not in _PANEL_ORDER_CACHE:
        members_list = list(club["members_map"].keys())
        _PANEL_ORDER_CACHE[key] = {frozenset(pair): pair for pair in combinations(members_list, 2)}
    canonical_order = _PANEL_ORDER_CACHE[key]

    def _reorder(row):
        key2 = frozenset((row["iso_i"], row["iso_j"]))
        correct = canonical_order.get(key2)
        return correct if correct is not None else (row["iso_i"], row["iso_j"])

    if len(df) == 0:
        return df
    reordered = df.apply(_reorder, axis=1, result_type="expand")
    df = df.copy()
    df["iso_i"], df["iso_j"] = reordered[0], reordered[1]
    return df


# =========================================================================
# SECTION 2 ? DEA / MALMQUIST-LUENBERGER ENGINE (club-agnostic)
# =========================================================================
def _ddf(Xo, Yo, Bo, X, Y, B):
    """Directional distance function by LP. Expand goods, contract the bad
    output (weak disposability = equality). Returns inefficiency beta>=0."""
    K = X.shape[0]
    c = np.zeros(K + 1); c[-1] = -1.0
    A_ub, b_ub = [], []
    for m in range(Y.shape[1]):
        A_ub.append(np.concatenate([-Y[:, m], [Yo[m]]])); b_ub.append(-Yo[m])
    for n in range(X.shape[1]):
        A_ub.append(np.concatenate([X[:, n], [0.0]])); b_ub.append(Xo[n])
    A_eq, b_eq = [], []
    for j in range(B.shape[1]):
        A_eq.append(np.concatenate([B[:, j], [Bo[j]]])); b_eq.append(Bo[j])
    res = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub),
                  A_eq=np.array(A_eq), b_eq=np.array(b_eq),
                  bounds=[(0, None)] * K + [(0, None)], method="highs")
    return float(res.x[-1]) if res.success else np.nan


# =========================================================================
# SECTION 3B ? EMBEDDED REAL DATA (from the Chapter 3 companion pipeline)
# =========================================================================
# The tables below are REAL World Bank WDI data (not simulated), embedded
# directly in this script so it is fully self-contained -- no external CSV
# file is required to get real ES/innovation inputs. Extracted verbatim
# from the user's Chapter 3 "Innovation & Energy Security Efficiency"
# pipeline, whose own comments cite the underlying WDI indicator codes:
#   CO2_PC     -> EN.GHG.CO2.PC.CE.AR5  (t CO2e/capita, excl. LULUCF)
#   IMP_DEP    -> EG.IMP.CONS.ZS        (net energy imports, % of energy
#                                         use; negative = net exporter)
#   TD_LOSS    -> EG.ELC.LOSS.ZS        (T&D losses, % of output)
#   RNW_SHARE  -> renewable electricity share of consumption, %
#   PAT_RESD   -> IP.PAT.RESD           (patent applications, residents only)
#   POP        -> population, 2007-2024
# Covers 40 economies: full overlap with all 19 non-EU G20 members, and
# 22/27 EU-27 members (missing HRV, CYP, EST, MLT, ROU -- simply absent
# below, never fabricated).
#
# An optional override: if data/g20_real_energy_panel.csv or
# data/eu_real_energy_panel.csv exists on disk, load_real_energy_data()
# below prefers it over these embedded tables (e.g. to drop in a more
# complete future extract without editing this file) -- but the script
# runs with real data out of the box even with no data/ directory at all.

CH3_SOURCE_YEARS = list(range(2007, 2025))  # source data year index, independent of the pipeline's own YEAR_MIN/YEAR_MAX window

# =========================================================================
# SECTION 3C ? REAL OECD INTERNATIONAL CO-INVENTION DATA (user-provided)
# =========================================================================
# Country-level indicator, 2007-2022, provided directly by the user (not
# fetched by this script). CONFIRMED BY THE USER: "co-inventions, in % of
# patents" -- a genuine rate (co-invented patents as a share of a
# country's total patents), not a level. Exact OECD dataset/indicator code
# still not confirmed -- verify before the final dissertation citation
# (the numbers themselves are used, z-scored, without depending on that
# precision; the citation does).
#
# EXCLUSION: because this is a RATE, a flat 0.0 across every single year
# (2007-2022) is not read here as "this country never co-invents
# internationally" but as a patent base too small to produce a meaningful
# percentage (an unreliable near-0/0 calculation). Countries with an
# all-zero series are excluded (treated as missing, not a real 0) from
# every use of this variable below -- computed dynamically in
# _oecd_coinvent_excluded_countries() rather than hand-listed, so it stays
# correct if this table is ever corrected or extended.
#
# IMPORTANT: this is COUNTRY-level, not dyadic. It cannot directly replace
# a true dyadic cooperation count (Coop_ij,t) the way CORDIS's real
# project-level data can, because it says how internationally-oriented
# country i's patenting is in year t, not which OTHER country i
# collaborates with. It is used in TWO ways below, kept clearly distinct:
#   (a) as a third real component of the Innovation composite (alongside
#       R&D/GDP and patents/capita) -- a legitimate country-level use;
#   (b) as an optional, explicitly-labelled DYADIC PROXY for cooperation,
#       constructed as sqrt(share_i * share_j) -- a modelling assumption
#       (countries that are each independently active in international
#       co-invention are assumed more likely to co-invent WITH EACH
#       OTHER), not an observed pairwise link. This proxy is deliberately
#       ranked BELOW CORDIS's true dyadic counts in the cooperation
#       priority order (Section 5B) for that reason.
OECD_COINVENT_YEARS = list(range(2007, 2023))  # 2007-2022, 16 years
OECD_COINVENT = {
    "AUT": [0.7,0.0,0.4,0.3,0.1,0.1,0.3,0.2,0.2,0.3,0.5,0.7,0.2,0.2,0.1,0.0],
    "BEL": [0.4,0.3,0.3,0.6,0.4,0.3,0.1,0.6,0.6,0.2,0.1,0.5,0.3,0.3,0.3,0.3],
    "CAN": [0.9,1.4,0.9,1.2,0.6,1.5,1.7,1.8,0.7,1.1,0.8,0.2,0.9,1.7,0.7,1.4],
    "CZE": [0.1,0.1,0.1,0.0,0.1,0.1,0.0,0.0,0.0,0.1,0.1,0.0,0.0,0.0,0.0,0.0],
    "DNK": [0.5,0.3,0.1,0.4,0.6,0.3,0.5,0.6,0.4,0.0,0.3,0.2,0.4,0.3,0.5,0.4],
    "EST": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.0,0.2,0.0,0.0],
    "FIN": [0.3,0.1,0.3,0.1,0.7,0.3,0.3,0.0,0.1,0.3,0.4,0.6,0.1,0.1,0.1,0.0],
    "FRA": [1.1,0.8,1.7,2.4,2.5,1.9,1.2,1.5,1.2,0.6,1.6,0.2,1.7,0.7,1.4,1.4],
    "DEU": [2.8,2.6,4.0,4.2,4.6,2.8,2.6,2.5,2.8,3.2,3.7,1.8,3.2,2.5,2.8,3.2],
    "GRC": [0.1,0.2,0.3,0.7,0.2,0.2,0.0,0.0,0.0,0.0,0.0,0.1,0.0,0.0,0.0,0.0],
    "HUN": [0.2,0.1,0.3,0.8,0.5,0.6,0.2,0.4,0.0,0.1,0.2,0.2,0.0,0.2,0.0,0.4],
    "IRL": [0.4,0.3,0.0,0.1,0.1,0.3,0.2,0.2,0.9,0.6,0.5,0.2,0.8,1.4,0.4,0.3],
    "ITA": [0.5,0.3,0.3,0.4,0.6,0.4,0.2,0.4,0.2,0.2,0.3,0.4,0.7,0.3,0.5,0.4],
    "JPN": [0.7,0.6,0.4,0.4,0.4,0.6,0.8,0.6,0.7,0.5,0.4,0.2,0.8,0.6,0.5,0.7],
    "KOR": [0.1,0.0,0.3,0.1,0.1,0.0,0.1,0.0,0.2,0.2,0.2,0.0,0.3,0.0,0.3,0.0],
    "LUX": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.0,0.0,0.0],
    "MEX": [0.3,0.1,0.0,0.0,0.0,0.1,0.0,0.0,0.0,0.1,0.0,0.0,0.2,0.0,0.0,0.0],
    "NLD": [0.6,1.3,1.4,0.4,0.6,0.8,0.7,0.7,0.6,0.3,0.3,0.3,0.2,0.6,0.1,0.8],
    "NOR": [0.4,0.4,0.3,0.6,0.4,0.2,0.2,0.1,0.2,0.3,0.0,0.2,0.1,0.2,0.2,0.3],
    "POL": [0.0,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.1,0.2,0.3,0.2,0.0,0.2,0.1],
    "PRT": [0.0,0.0,0.0,0.0,0.1,0.1,0.0,0.1,0.0,0.0,0.0,0.2,0.1,0.3,0.0,0.0],
    "SVK": [0.0,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0],
    "SVN": [0.0,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.0,0.0],
    "ESP": [0.1,0.4,0.2,0.3,0.3,0.1,0.9,0.2,0.3,0.3,0.2,0.1,0.3,0.3,0.1,0.4],
    "SWE": [1.0,0.8,1.4,2.5,1.5,0.9,1.4,1.3,1.5,1.7,0.6,0.9,0.5,0.3,0.7,0.1],
    "CHE": [1.3,0.8,0.9,0.6,0.7,1.5,1.0,0.9,1.2,1.8,1.1,0.6,1.1,1.5,1.1,1.4],
    "TUR": [0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.1,0.0,0.0,0.0,0.0,0.0,0.1,0.2,0.0],
    "GBR": [2.7,4.3,3.3,4.6,4.5,4.3,3.9,2.8,2.8,2.0,2.6,2.7,2.2,1.9,2.3,3.2],
    "USA": [11.9,9.8,10.9,11.7,9.5,10.0,11.6,11.6,14.1,10.4,11.0,8.0,8.7,8.8,9.6,10.8],
    "DZA": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],
    "ARG": [0.0,0.0,0.0,0.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1],
    "BRA": [0.3,0.0,0.1,0.1,0.4,0.1,0.1,0.4,0.2,0.0,0.3,0.1,0.1,0.1,0.3,0.0],
    "BGR": [0.0,0.0,0.0,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],
    "CHN": [0.8,1.1,1.0,1.4,1.1,1.6,1.5,1.6,2.1,1.8,1.9,6.7,6.2,3.5,1.4,1.1],
    "HRV": [0.0,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],
    "IND": [0.5,0.4,0.7,0.7,0.7,0.1,0.4,0.6,0.6,0.3,1.0,0.4,0.3,1.0,1.7,1.2],
    "IDN": [0.1,0.1,0.0,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0,0.1,0.0,0.0,0.1,0.0],
    "LVA": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],
    "LTU": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],
    "MLT": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],
    "NGA": [0.0,0.0,0.0,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],
    "ROU": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0],
    "RUS": [0.2,0.3,0.2,0.2,0.2,0.3,0.2,0.0,0.0,0.0,0.0,0.1,0.0,0.0,0.1,0.0],
    "SAU": [0.0,0.0,0.0,0.0,0.2,0.0,0.1,0.0,0.1,0.1,0.2,0.0,0.0,0.3,0.0,0.0],
    "ZAF": [0.1,0.0,0.0,0.3,0.4,0.1,0.0,0.0,0.0,0.2,0.1,0.0,0.1,0.1,0.2,0.1],
}


def _oecd_coinvent_excluded_countries():
    """
    Countries whose OECD co-invention series is EXACTLY 0.0 in every single
    year (2007-2022). Per the user's clarification, this is "co-inventions
    in % of patents" -- a genuine RATE, not a level -- so a flat-zero
    series most plausibly reflects a patent base too small to yield a
    meaningful percentage (a near-0/0 situation), not a reliably-measured
    zero collaboration rate. Excluding them (treating as missing, not 0)
    avoids silently telling the Innovation composite or the dyadic proxy
    "this country never internationally co-invents", which the data does
    not actually support. Computed dynamically from OECD_COINVENT so the
    exclusion list stays correct if the source table is ever corrected or
    extended, rather than being hand-maintained separately from the data.
    """
    return sorted(iso for iso, vals in OECD_COINVENT.items() if max(vals) == 0.0)


_OECD_COINVENT_EXCLUDED = _oecd_coinvent_excluded_countries()


def _oecd_coinvent_long():
    rows = []
    for iso, vals in OECD_COINVENT.items():
        if iso in _OECD_COINVENT_EXCLUDED:
            continue  # all-zero series -- excluded, not treated as a real 0 (see docstring above)
        for y, v in zip(OECD_COINVENT_YEARS, vals):
            rows.append({"iso": iso, "year": y, "coinvent_intensity": float(v)})
    return pd.DataFrame(rows)

CO2_PC = {
    "ARG": [4.325342357,4.433590041,4.03429401,4.294546105,4.429218229,4.478861287,4.536586723,4.416746151,4.51258702,4.422301044,4.321029105,4.250758069,4.075040693,3.720028549,4.097697046,4.295311671,4.169904868,4.010665317],
    "AUS": [19.23892224,19.0281196,18.91516981,18.85095374,18.48082169,18.18344894,17.54324355,16.88735741,16.82195516,16.89118974,16.72094454,16.39399793,16.10070659,15.30437851,14.97872411,14.53050286,14.15967008,14.09866396],
    "AUT": [9.094354557,9.04182373,8.157445181,8.824134288,8.602272523,8.187517638,8.208756244,7.713954345,7.794231871,7.757476878,7.966419348,7.607108224,7.74519365,7.055866278,7.392239909,6.928581327,6.403595101,6.322784246],
    "BEL": [10.62647167,10.66759926,9.937652903,10.54583939,9.466597284,9.274766425,9.351706592,8.723311872,9.0841511,8.944296665,8.770814436,8.791286013,8.731279887,7.966942968,8.287569819,7.677661617,7.277630984,7.306615194],
    "BGR": [7.639034858,7.282263034,6.251119123,6.694440842,7.455328614,6.832365894,6.208375553,6.688982596,7.138544362,6.83625613,7.243835056,6.901340198,6.745904243,6.016933163,7.049174458,7.647696547,5.734902575,5.307602158],
    "BRA": [2.084967186,2.168393919,2.007462507,2.271128131,2.374189679,2.529019887,2.63744451,2.731034576,2.568188589,2.375335498,2.416949554,2.285623504,2.258698818,2.146646183,2.415808589,2.278895772,2.27447827,2.318264661],
    "CAN": [18.0804999,17.30720794,16.14006204,16.52977474,16.76749452,16.33905298,16.39676837,16.3774544,16.08702915,15.89824242,16.19624532,16.52388205,16.19086037,14.42190488,14.68975936,14.78390882,14.16004529,14.00727041],
    "CHE": [5.908569553,6.01237631,5.761747641,5.920145525,5.344460681,5.452164293,5.514252945,4.994169368,4.856287963,4.871235342,4.711198982,4.523891431,4.477451465,4.160616483,4.323039938,3.928466936,3.787464751,3.745687952],
    "CHN": [5.707632457,5.852258664,6.249090185,6.807184768,7.401850658,7.584416515,7.893276312,7.931643316,7.790570493,7.763224839,7.878026522,8.217736249,8.391475871,8.47858302,8.887156603,8.86482511,9.230275535,9.315089338],
    "CZE": [12.849666,12.14498041,11.46240268,11.55144777,11.25360229,10.84876153,10.41753533,10.18865221,10.26844246,10.39289699,10.41874991,10.28650429,9.72456561,8.867158267,9.472430965,9.153646914,7.880059742,7.129665325],
    "DEU": [10.04781273,10.10729655,9.378644243,9.965671247,9.817767261,9.947453071,10.14086012,9.616308462,9.608848872,9.584233839,9.369765189,9.006578094,8.382180431,7.721032648,8.061281446,7.911828603,7.079828391,6.943956634],
    "DNK": [9.863574392,9.21101037,8.764813931,8.753203815,7.806182202,6.902638471,7.183559837,6.413991379,5.994774683,6.228760075,5.818875347,5.788748206,5.17323992,4.765216061,5.047882497,4.827040725,4.407367001,4.278205492],
    "ESP": [8.214323705,7.401310342,6.479366087,6.177996357,6.193930145,6.092657407,5.500723465,5.493692921,5.849337117,5.662041263,5.977615108,5.809692533,5.398121642,4.528234972,4.915133206,4.99422196,4.570462169,4.505406884],
    "FIN": [12.87058116,11.25981693,10.69154883,12.27719158,10.80537508,9.665123807,9.805400726,9.052731185,8.390462614,8.881603071,8.3649437,8.601030727,7.994268334,7.138745462,7.094798662,6.681621985,5.889545876,5.535212212],
    "FRA": [6.090030303,5.942901913,5.717173173,5.801948079,5.566040407,5.559497636,5.45712099,4.930488745,4.991163407,5.017275316,5.061551433,4.892170665,4.792140448,4.246776421,4.668208692,4.512468289,4.15598361,4.003557726],
    "GBR": [8.975790813,8.638357196,7.763306955,7.977973553,7.25917228,7.555834942,7.331374536,6.664561507,6.389673821,5.972950645,5.771508702,5.640353312,5.388321727,4.790176806,5.045161098,4.805057957,4.447751219,4.220773372],
    "GRC": [9.594656203,9.192765991,8.66279398,8.011452935,7.757360062,7.383876757,6.73744445,6.519078922,6.418348669,6.325295419,6.358525438,6.199238937,5.735412927,4.894154833,5.081782343,5.28045887,4.923241274,5.046422276],
    "HUN": [5.881701867,5.761836698,5.158456097,5.23461796,5.158253931,4.803524307,4.537405748,4.559343715,4.852234007,4.945318812,5.257107303,5.280693325,5.312401752,5.134431093,5.293984009,4.952934251,4.480480258,4.457917824],
    "IDN": [1.694885004,1.654142525,1.69126327,1.801349627,2.003993009,1.995489289,1.856402268,1.951024701,1.957214553,1.91626591,2.003942761,2.209539252,2.339794665,2.169765809,2.249932001,2.591895524,2.747471873,2.865039782],
    "IND": [1.1886235,1.258159982,1.322845333,1.391298633,1.463373916,1.576075535,1.599458557,1.70517686,1.715502691,1.720292357,1.795453399,1.859893094,1.836882232,1.680400161,1.837029517,1.96929619,2.096228589,2.173651735],
    "IRL": [10.67186155,10.30870841,9.078301133,8.974453719,8.109894928,8.175221267,7.857968397,7.758934591,8.006878838,8.296369177,7.926180412,7.763378768,7.308212913,6.804567769,7.119067582,6.845812913,6.187078018,6.036057741],
    "ITA": [8.174827776,7.852861438,6.948260356,7.081863918,6.905312575,6.55333989,5.997423747,5.68586837,5.873193092,5.808340759,5.761145432,5.684522333,5.567840563,4.96667407,5.586515711,5.608971901,5.159144609,5.075738341],
    "JPN": [10.09160241,9.532537111,9.039610456,9.503116265,9.933974013,10.25160113,10.38479266,10.00636255,9.694561157,9.668652617,9.552832908,9.172136487,8.874566661,8.397105203,8.594782849,8.395630708,8.050635799,7.842420572],
    "KOR": [11.11262063,11.26026884,11.28368139,12.3690482,12.97026844,12.85285636,12.73173298,12.4261678,12.61981905,12.66777882,12.95294484,13.19172695,12.65318946,11.90944428,12.33872683,11.87374098,11.41626573,11.36225699],
    "LTU": [4.818007894,4.950017682,4.176683794,4.533329545,4.729113657,4.847917745,4.513094133,4.497978213,4.692575922,4.698878298,4.912407098,4.972474461,5.085751465,5.071967418,5.103226771,4.651122548,4.519942819,4.490564966],
    "LUX": [23.62888625,22.94607592,21.31430764,22.11921026,21.3951272,20.46893658,18.9995583,17.65156322,16.34363523,15.53484968,15.40809208,15.69997533,15.63174898,12.67871051,13.09337816,11.12489148,10.41609771,10.55727225],
    "LVA": [4.05135605,3.894095591,3.619560259,4.233452758,3.997360792,3.945988805,3.865258041,3.798760346,3.867456677,3.817534448,3.826545323,4.207144762,4.131941215,3.829147744,3.995988305,3.685677693,3.663196564,3.647774746],
    "MEX": [4.328991685,4.279100763,4.121220981,4.197558093,4.221799781,4.333504243,4.174431171,4.01481961,4.050696779,4.061044691,4.057433296,3.838317059,3.925423779,3.462471416,3.546624899,3.52183829,3.665765249,3.637386804],
    "NLD": [10.91845435,10.94181888,10.57165748,11.20172654,10.42782773,10.24185552,10.0780437,9.59164087,9.997341782,10.02880511,9.816887175,9.464865297,9.10784362,8.150698048,8.321133512,7.520147752,6.875705965,6.60188396],
    "NOR": [9.535854962,9.878923169,9.345094338,9.814364242,9.476694135,9.24360371,9.228578578,9.147046503,9.242210867,9.049752231,8.984022643,8.780334629,8.479334677,8.124937842,8.214713626,7.979858266,7.537537587,7.178301733],
    "POL": [8.664644486,8.513349203,8.161615123,8.658233672,8.58934161,8.399903382,8.283077721,7.959918167,8.050094334,8.341513676,8.695816012,8.680317843,8.219855013,7.913474096,8.761455946,8.363578819,7.64977784,7.551085659],
    "PRT": [5.817917997,5.612872374,5.534531886,5.036224002,4.868331319,4.734135856,4.603370183,4.583262748,4.993350116,4.893054561,5.353591643,4.969728181,4.513582824,3.947176875,3.760146252,3.801719171,3.536659541,3.324764899],
    "RUS": [12.45785918,12.36041933,11.64056203,12.22893459,12.81384023,12.69229817,12.2682756,12.13416942,12.14812778,11.9617467,12.13573224,12.59392471,12.77387392,12.28405303,13.39207574,13.14138384,13.54623391,13.98454112],
    "SAU": [17.34059247,18.05631746,18.33456288,20.30474233,20.29402595,20.61009457,19.80872867,20.69738068,20.50712591,19.69070237,19.47646918,19.20908406,19.28594586,18.29028816,19.17598933,19.2828743,18.85407447,18.48458426],
    "SVK": [7.690773416,7.652057459,6.911250285,7.402714086,7.019563632,6.676999079,6.720424695,6.272818188,6.365646527,6.526591488,6.936420436,6.837574042,6.381822859,5.980020983,6.727857209,6.110128003,5.739781158,5.513116856],
    "SVN": [8.887222874,9.348216706,8.280559248,8.376912236,7.83269836,7.54292692,7.529783447,6.850696903,6.927397747,7.272830286,7.425662557,7.449753941,7.099313584,6.440485935,6.574753603,6.339402559,5.950875777,6.266522516],
    "SWE": [5.776428571,5.608951849,5.046042298,5.63902639,5.12869167,4.803750751,4.579871274,4.411996151,4.3780065,4.281934499,4.159619826,3.896556868,3.842351803,3.674024542,3.886255233,3.59996304,3.480077884,3.57381646],
    "TUR": [4.280793645,4.21593984,4.183513627,4.268019193,4.483322959,4.578038097,4.384562281,4.604460031,4.702763397,4.93533247,5.394164445,5.274019729,5.019136725,5.073959629,5.475006345,5.22956172,5.232154128,5.379264533],
    "USA": [19.49566069,18.70890263,17.1890632,17.84708431,17.0203889,16.1854221,16.49597522,16.42489531,15.95010229,15.50454822,15.18471272,15.57964566,15.04396621,13.47156657,14.31609793,14.41862213,13.71402588,13.62386227],
    "ZAF": [9.102810006,9.545699459,8.833539706,8.8711743,8.377586236,8.57667504,8.688727225,8.75262376,8.149486517,8.037015519,8.147514186,8.077016906,8.014122334,7.444583462,7.293406752,6.882639362,6.759227749,6.876849626],
}

IMP_DEP = {
    "ARG": [-8.758408955,-6.261836322,-7.655871461,-2.402167615,7.23741684,8.114399089,13.98621819,14.06781034,13.54778099,11.99936896,13.21891469,8.608755323,2.49167708,1.228006169,5.966368089,1.958558212,0.41571453,None],
    "AUS": [-127.7642909,-123.5346102,-128.4766172,-146.7083468,-141.942926,-149.1953804,-164.6917741,-185.5557895,-197.3868422,-206.2302513,-213.0392192,-221.1351478,-230.5402481,-237.3285802,-232.1047721,-234.2793459,-214.4715403,None],
    "AUT": [70.17256284,70.47707612,66.71038756,64.31557716,71.89275916,65.52058643,62.88356458,67.4888818,62.14742085,64.11378635,65.94638124,66.52577755,74.4762819,59.76687681,53.15293017,76.81118466,63.96619803,None],
    "BEL": [91.14283498,95.95087727,86.60283944,89.34598401,86.77835315,86.34280285,87.31560366,89.70586102,95.27225513,86.25422818,86.93804423,99.69900066,90.93022608,89.07580989,81.49615621,86.76194408,89.17478019,None],
    "BGR": [51.98949515,53.0049789,46.38260488,40.85265001,37.23748234,37.35401723,39.05392629,35.71519268,37.1318138,39.32935663,40.2667722,37.37150303,39.3231833,38.92174518,36.8953489,37.96266106,40.93104582,None],
    "BRA": [10.75545069,10.97343668,6.602297027,9.460563808,10.70160764,12.13591703,15.77992893,14.2246121,8.613873325,2.67300703,1.14058561,-0.816754027,-4.059636124,-11.16773546,-3.847631075,-7.287210528,-13.68089694,None],
    "CAN": [-54.59004885,-52.79255808,-52.91117037,-55.14223849,-56.29299601,-59.75225499,-65.70481306,-66.69667522,-68.42215597,-69.11730461,-71.73147258,-75.0034002,-75.76917616,-79.70398217,-87.58213629,-89.48809483,-89.62608131,None],
    "CHE": [55.19510378,58.00880524,58.35444899,57.46780383,57.08001048,57.41939428,57.3949323,53.85016575,56.43456601,60.1477702,60.58298053,54.03961198,55.40321991,50.53267476,52.43855307,51.73384764,51.73882573,None],
    "CHN": [7.946098618,8.63245193,12.07848032,13.58229019,14.71873757,16.55190762,17.32603201,17.54117494,16.70642274,19.19307401,20.41898999,21.66874647,22.22097389,22.97457219,21.63866073,20.62455892,23.96175883,None],
    "CZE": [25.13008635,27.93063539,27.00078125,25.55491106,28.88472083,25.42957607,27.7105784,30.28692535,32.11729478,32.87793272,37.28801413,37.08665965,41.01730022,38.78497051,40.02614066,42.02336374,41.97514545,None],
    "DEU": [60.55782924,62.84655305,63.27807318,62.05364863,63.96260267,63.4091868,64.63418612,64.03523602,64.43544967,66.30647282,66.62119661,66.20193066,69.92400065,65.38560057,65.40456187,71.75340939,70.48742654,None],
    "DNK": [-26.75325991,-23.26511462,-20.24214727,-17.63265135,-6.930177351,-3.473025584,13.33464266,13.46039152,14.41914959,14.97374255,12.47132191,25.16462699,43.76761717,48.33434809,34.83075575,47.39571982,43.78719474,None],
    "ESP": [86.04068392,88.43352621,86.4856561,83.53312414,83.47438059,79.8545675,76.65695963,80.05080233,80.0358808,78.88249702,80.8546276,81.00753247,82.91405795,73.4774192,75.4707202,82.86749234,76.98467751,None],
    "FIN": [54.66365717,55.93197326,55.44258313,49.32608353,54.53569366,47.48492547,50.07318591,50.26081124,48.54248941,46.68279579,45.26715371,46.7658274,44.4037103,43.86686235,38.68764767,42.38189164,30.27185341,None],
    "FRA": [51.97025142,52.30051954,52.44617955,50.3709944,50.76691298,49.91921275,49.71873506,47.85957235,47.58401938,49.14706214,50.65012525,48.74056504,49.70652063,45.79096111,45.57152518,54.32017398,47.1279303,None],
    "GBR": [21.82270032,28.16539746,28.86627907,30.84692471,39.47336112,46.00559097,50.60833175,49.74316805,40.27756784,37.95433034,38.03046548,38.16722492,36.5356011,28.49943807,38.92158141,39.1791429,44.04361845,None],
    "GRC": [80.60258642,82.61112292,75.2206017,76.99324161,73.10501571,73.08631845,69.13900833,72.99340147,79.07436522,81.38303797,81.85228149,82.02291231,87.49899898,91.21231613,84.55707294,92.90814048,89.24605623,None],
    "HUN": [60.75015383,63.23134657,58.16966698,57.01349558,50.41619466,50.18486147,50.0908154,59.73173264,53.83685497,55.80333144,62.73205956,58.36338481,69.81266662,56.40687656,53.96490947,64.44471528,62.50981254,None],
    "IDN": [-90.79615295,-90.42674983,-92.42696626,-103.5095619,-131.5766037,-130.2982568,-134.0451583,-122.6894189,-109.2714861,-106.4851795,-100.4014145,-98.01327589,-99.18164887,-86.94797312,-84.93705477,-82.91927561,-89.54881628,None],
    "IND": [27.0601259,28.45557442,29.34264791,30.22192148,31.16095907,32.8228389,34.02400089,36.54230259,37.28278221,37.38741865,37.47131053,37.58434677,39.2996983,37.48125277,35.03557996,36.52941238,36.08718435,None],
    "IRL": [94.21187387,96.51952366,92.73908116,92.12378594,95.52436138,89.72229639,94.59909281,91.27917911,95.23158143,74.2255241,71.40470459,73.32815804,75.41381605,74.19104682,80.64378227,85.98152909,85.76634887,None],
    "ITA": [85.67753037,85.49130575,83.30304459,85.46187865,84.1225458,81.83903237,79.29453949,78.38816137,79.58795511,80.48505065,81.17800645,80.97072172,82.22796403,76.86951866,76.63297818,83.62903406,79.59805807,None],
    "JPN": [84.3107714,84.21477703,80.61633929,81.74253492,90.81959742,96.12221971,95.67297673,95.5684274,94.83962643,93.81328155,93.08867124,91.4437393,90.55903281,89.57960336,88.68172594,89.97075163,87.27988532,None],
    "KOR": [85.36244593,85.59835656,86.26639972,88.08062546,87.23054052,86.54668289,88.08018499,86.86939116,86.73006281,86.93763209,88.47103665,89.4697679,88.15629908,84.51774111,84.9035263,85.4997855,84.6000912,None],
    "LTU": [61.06401135,57.90093837,49.9910096,81.93433871,81.49745031,80.09641004,78.01198418,76.53023163,77.68354975,78.03492726,75.12919733,77.7375707,79.09285752,78.02175799,76.55514273,76.16618602,71.4559727,None],
    "LUX": [106.6437688,107.5697301,107.8477121,106.959854,106.6766104,106.358294,106.2290593,106.8874908,107.7451081,109.5550509,110.1726851,110.1923819,109.5147766,107.1754661,108.4623897,110.2983312,109.5850535,None],
    "LVA": [65.89179714,62.73341734,65.51228508,49.20639125,64.47019217,60.91111147,60.48006339,43.70738847,55.53295909,52.08184884,48.18868314,46.2654243,48.4186366,48.60275318,41.02375312,40.39712717,34.30773846,None],
    "MEX": [-34.27833007,-25.82320089,-23.08671821,-22.45161242,-17.62941629,-11.71105404,-10.43344719,-7.875725611,-0.83822344,4.933837153,11.43404193,16.79131792,18.29943971,10.88954966,15.31175582,21.10507649,21.58435595,None],
    "NLD": [47.04138668,41.97899754,41.84316451,34.04178409,36.30839641,37.26481003,28.84392145,38.13503745,58.81999661,55.48829608,62.3248618,71.68489283,78.25449011,81.76038234,69.94622181,99.35876527,87.01737489,None],
    "NOR": [-679.3467863,-615.6529048,-626.3589805,-557.915249,-620.2578852,-598.4242079,-535.3411462,-596.9992562,-719.2085379,-647.7632884,-626.8607552,-597.2652267,-596.9587884,-648.6590564,-647.6325453,-746.7193656,-703.9593058,None],
    "POL": [25.97177556,31.04492756,32.15465432,31.95158622,34.23673074,31.70237404,26.50121091,29.67857662,30.15320748,31.16093106,38.82238908,44.07005477,45.881671,43.04947437,41.10111162,46.84187551,48.81935506,None],
    "PRT": [85.81629969,88.41039478,85.74631719,79.49915693,82.80985705,85.12133584,78.78243092,77.43895285,84.27732167,80.59114218,87.17364366,85.33897432,84.35654716,71.50919024,73.74412406,81.51279544,77.24489759,None],
    "RUS": [-82.11466438,-77.95494185,-83.14469347,-82.91332676,-80.53987975,-80.16437403,-86.66912112,-83.49141511,-89.86205251,-90.27864762,-91.76702644,-91.98553434,-99.05170322,-90.05135864,-80.50924377,-75.13753584,None,None],
    "SAU": [-260.3431199,-247.6053294,-197.276525,-174.5720512,-194.4791468,-198.4047468,-196.8718136,-172.6351677,-175.630387,-188.3133785,-170.0128445,-186.8699658,-177.7154055,-167.3918023,-159.988473,-178.2004488,None,None],
    "SVK": [69.16252638,65.39786073,67.23418887,61.28021895,62.56626814,58.13886828,57.60048067,59.79532549,57.35796793,57.48943503,61.83030159,60.78920509,69.47289589,55.86089385,51.94048499,68.88728198,57.88699438,None],
    "SVN": [51.24476688,54.16280474,48.29301778,48.69576232,47.77729038,51.51187832,47.30676958,44.8245997,49.08105015,49.17816534,51.10194481,52.20039503,52.84915693,45.79535779,48.71622084,53.54708407,48.90218241,None],
    "SWE": [38.2973061,40.07015771,39.48995651,39.08383767,38.45769125,31.47110241,34.00463859,34.10361063,32.8832024,35.36370603,28.69652314,30.74656876,32.01970348,34.44175139,22.54978448,28.69989149,27.75240498,None],
    "TUR": [75.59779507,73.67415454,71.59938313,71.81136759,72.34552434,76.50451535,76.70820265,78.73347287,80.44431968,77.28781811,79.17251048,75.83164799,72.19736329,71.33300268,72.12471215,72.70412972,71.99286005,None],
    "USA": [30.55072873,27.91841276,25.9120017,24.08502187,20.6799127,17.51845286,14.43308083,11.69570188,11.73894467,12.29977735,8.100348135,3.592072117,-0.898716369,-3.96551326,-4.189931016,-6.495038632,-9.000585752,None],
    "ZAF": [-14.33600403,-7.795952067,-2.217462188,-13.62965161,-11.55505771,-18.06225576,-18.68910274,-12.77922295,-17.66579351,-14.30662692,-15.42684381,-17.08162135,-13.28038292,-19.0992445,-9.883062795,-14.14842491,-12.50729961,None],
}

TD_LOSS = {
    "ARG": [15.32432075,14.89497814,15.53682937,15.81240329,14.63793183,15.18323068,15.98442642,12.85071008,12.28763742,13.87854911,15.78094532,16.1280654,14.95109455,14.42181317,14.5764106,20.51289093,23.85092666,21.83601295],
    "AUS": [6.636864248,6.361292816,5.14139957,5.843798608,5.088898167,5.021513944,4.526852887,3.963250883,3.904167544,3.937581295,4.302229594,4.025189998,3.797353291,3.917745816,4.792622216,4.911409747,4.618652404,4.711341828],
    "AUT": [5.339253887,5.157738852,5.183319583,4.709818918,5.274034005,4.85379392,5.17584411,5.215623949,5.307891392,4.888153657,4.849699961,4.860311585,4.450790743,4.397861022,4.573463735,4.688177639,4.141261857,3.82132369],
    "BEL": [4.508410706,4.764274458,4.317690798,4.385417878,4.373489855,4.95243829,4.739319464,5.282238611,5.586399828,4.587295283,4.496703956,5.100014659,3.877450771,3.909315506,3.592719022,3.461177697,4.366326811,4.297786303],
    "BGR": [10.83677853,10.36703155,10.50181547,9.602812252,8.654054373,8.946166533,8.889649663,8.450200042,7.688869929,7.89990945,7.618609138,6.620833956,6.2471768,6.367473131,5.470063909,4.970395453,5.875046578,None],
    "BRA": [16.14096018,16.64575537,17.1869536,16.62688356,16.45939781,17.08306104,16.64132381,15.38919524,15.4988653,16.8239252,16.39057447,16.1349664,16.73074979,16.82348226,16.52210227,15.35273875,15.10961057,14.97214957],
    "CAN": [3.368313504,3.709438376,4.32322184,4.292127123,4.204701275,4.112326545,3.955692596,4.444663597,5.0523262,4.889627594,5.339109691,5.087755241,5.140825646,5.044840814,4.022456669,4.007422052,3.999829384,4.151916381],
    "CHE": [6.212734634,6.215639914,6.116069472,6.444002064,6.93096885,6.550781082,6.647011907,6.513527452,6.707731395,7.061579122,7.224792446,6.125518897,5.665664576,5.604066569,6.390625472,6.328384465,5.502511548,5.013492745],
    "CHN": [6.282272784,6.166039597,6.078554296,6.103242092,5.726965383,5.799226879,5.765683106,5.328728217,5.103758219,4.940338764,4.759390766,4.692403291,4.437463164,4.163896406,3.786592265,3.421966444,3.428987991,None],
    "CZE": [5.572688723,5.573421641,5.459633753,5.198887117,5.03560936,4.789631426,4.715059887,4.465570878,4.847899681,4.89743005,5.024352141,4.849097539,4.939392199,5.050046612,4.293359995,4.023571008,3.911485118,4.265223275],
    "DEU": [4.570345146,4.696976549,4.191835619,3.786643922,4.044702286,3.909191901,3.831839938,3.848163286,3.94950556,3.960187501,4.133860978,4.162737248,4.526813386,4.68128217,4.484150614,4.545996279,5.035154655,5.115933448],
    "DNK": [4.952182318,5.940026218,6.500288596,6.752097164,6.247693661,7.003257329,5.50918197,6.133482476,6.098825155,6.130067457,5.344594159,7.60973361,6.315218864,8.378585352,6.060514372,5.608860291,5.659314597,6.16417023],
    "ESP": [8.35281044,8.226085072,8.297128505,9.087080096,8.857330516,8.628569892,9.345624249,9.468374774,9.436796708,9.724461461,8.640099229,9.326220979,9.07204573,9.731825206,9.743649567,8.593146273,8.55708689,8.288238849],
    "FIN": [3.740537879,4.300547577,3.840763713,3.422416144,3.663895729,4.131456733,3.651571074,4.062031897,3.549614426,3.7610716,4.099343927,4.303829896,4.460305899,4.54617639,4.425834003,3.901033441,3.551631101,3.669140019],
    "FRA": [5.548398646,5.834898903,6.508000187,6.220763868,6.43683371,6.605419742,6.656769189,6.025763867,6.232076744,6.638890956,6.862276237,6.633151675,6.66626999,6.736099383,7.05383512,7.437847287,6.958490279,6.618484981],
    "GBR": [7.013096321,7.22678098,7.470077758,7.023171974,7.606949801,7.771945294,7.41212238,8.127611315,8.462733149,7.807357241,7.952487783,8.018514675,8.154525018,8.814153137,8.802573897,8.557535469,9.616626229,9.853715295],
    "GRC": [7.65402545,7.926398845,5.254320183,6.590018291,4.744599233,2.642759888,6.815159574,8.220073701,9.436326483,7.268819575,1.992183259,9.841913559,7.59264591,10.80392116,10.25678516,10.87905829,11.98413334,9.831847836],
    "HUN": [9.907407407,9.713928795,10.03676061,10.1709882,10.50556651,10.63663924,12.09150327,12.34908003,12.17061924,11.17798257,10.49977214,10.46870615,9.600186638,8.986544518,8.449612403,7.767939443,6.965817977,7.102429779],
    "IDN": [10.7202904,10.10808566,9.598530369,9.0122864,8.743123548,8.609261939,9.253208294,9.006104123,8.945315035,8.659141179,7.823367052,7.400561929,8.010915892,7.62318307,6.968448021,6.806268229,6.453421308,None],
    "IND": [22.94152336,21.33951078,21.30794798,20.00888658,19.6760811,19.04195797,18.81962416,17.7208362,17.73997185,17.39129828,17.17599411,16.79301295,16.44018727,17.72412936,16.64692877,14.9056435,14.16143541,None],
    "IRL": [7.564633425,7.352989732,7.423072856,7.38758407,7.446027317,7.300973706,7.759184459,7.749331295,7.309541355,6.970894718,7.020034197,7.145597545,7.310176408,7.119751071,7.521543649,7.231605228,7.81426598,8.157627334],
    "ITA": [6.682659683,6.406166766,6.954643776,6.809837683,6.890055886,7.016910755,7.31075271,6.951080489,6.967310145,6.471751442,6.31004293,6.209010452,6.063576006,6.190046733,6.583549256,6.744236004,6.89266988,7.079152937],
    "JPN": [4.267445751,4.430867459,4.432284617,4.02674934,4.216461784,3.935218678,4.3036521,4.027491856,3.833873661,4.259305789,3.760992542,4.218942593,4.245827592,4.549669819,4.695642068,4.096969673,4.934034676,4.933945558],
    "KOR": [3.78007804,3.80554978,3.842769342,3.796024215,3.313971728,3.229801367,3.400589857,3.386017065,3.268802112,3.287861395,3.272931861,3.061286336,3.245239319,3.131599894,3.100473275,3.203660229,3.244389497,3.287401478],
    "LTU": [7.981723424,7.295335298,6.309415288,17.20299182,18.08378266,17.509419,18.31163377,18.53536502,16.0754105,21.26113455,21.82947218,26.60210766,22.31176026,17.23762915,18.2355258,17.66673636,14.03479425,10.17179582],
    "LUX": [2.849287678,3.289288726,2.991232594,2.896972337,4.440258342,4.035639413,5.295950156,5.128205128,5.676066522,7.14610833,6.935123043,7.048658481,8.123689727,6.535362578,6.968325792,6.569664903,4.63654224,3.501094092],
    "LVA": [16.72605324,15.13083049,13.30579996,10.92500377,10.09354998,9.064374899,9.24464487,9.048453006,8.133020061,7.394146949,6.293984863,6.662700773,6.632494564,6.586303284,6.585699624,7.17693837,5.918271489,5.758582503],
    "MEX": [15.26891368,15.37573242,15.85491106,16.06027503,15.06264265,14.33793254,14.30084924,13.70569993,13.07816326,12.50289115,13.57177294,11.94557854,11.76821755,11.15706468,12.00069642,11.27638748,11.1305372,11.11116785],
    "NLD": [5.252698141,5.056158881,4.613381886,4.722897627,4.561129489,5.028382122,5.050674014,4.773699181,4.776206074,4.717865889,4.620754991,4.662569178,4.166117554,3.848212982,3.926495866,4.004145623,4.133508714,3.803202442],
    "NOR": [7.369963263,6.827360097,5.750798722,7.727835252,5.722171257,6.201959868,6.011695209,5.356765514,5.167213205,5.162974195,5.103023816,5.142185089,5.185833099,4.092271014,4.446498905,4.537586042,4.735465961,4.935570758],
    "POL": [9.046865979,8.105984997,8.260611653,7.516951356,6.504512437,6.712758806,6.226151416,6.444149655,6.386409933,5.698082636,5.863960344,5.230564753,5.458326219,6.324268232,5.148888555,5.800899036,6.178224448,6.832393493],
    "PRT": [6.729873868,9.111671815,7.552731691,7.912738029,7.796119096,10.09782469,10.55677046,9.865156623,9.336131248,8.123787896,8.581380088,8.484807834,9.306919517,8.47997287,9.694187803,10.24422226,9.864812511,9.667253866],
    "RUS": [10.32715375,10.5000197,10.76553963,10.10886005,9.955013676,9.962044728,10.10242736,10.01374733,9.981134267,9.830399102,9.61903996,9.247569485,8.834392042,8.770661129,8.43778247,8.557050478,8.388283464,None],
    "SAU": [8.234707534,8.673849167,8.27567463,8.909031078,9.400116762,8.218875072,6.604897332,6.272006455,7.015357457,7.096746218,8.545006035,8.507789491,8.008055761,9.066811956,9.066648403,9.068388797,9.06819721,None],
    "SVK": [5.146849159,3.463158622,2.989868094,3.072725967,1.797180346,4.444599498,2.635960044,2.434217729,5.066349478,4.921667159,4.553320355,4.578992251,5.975240909,5.510090852,4.497601279,4.922125345,3.735411163,3.669162696],
    "SVN": [5.76347803,4.951521434,5.444126074,5.97323601,5.131079146,5.560144881,5.266099485,4.708378735,5.721854305,5.309090909,5.469802769,5.384049982,5.329523573,4.93310064,5.265810028,6.205934195,4.799093085,5.090630885],
    "SWE": [7.156095137,7.322811554,7.248327056,7.109486496,7.045643429,6.580171829,6.530822767,4.773432772,4.310600079,5.492596628,5.780821918,6.448592411,6.287142526,5.069186306,6.191573825,5.606985487,5.771465384,5.478496842],
    "TUR": [13.91066935,13.85005393,14.88152681,14.30911708,14.11071829,14.88834887,15.46257818,14.81606426,13.95354167,12.97739133,11.65882325,10.37857487,10.2946064,9.575061216,9.26019425,8.851662256,8.936487613,8.273025609],
    "USA": [6.139143937,5.634187151,6.224799825,5.96101799,5.966897523,6.263675052,5.928936453,5.882492533,5.914120837,5.432784256,5.275114851,4.738838247,5.558409385,4.647112717,5.580116105,4.560004876,5.482534796,5.311724822],
    "ZAF": [8.356263687,8.697554309,9.729240214,9.425233339,8.378977519,8.615495563,8.391357152,8.290508279,7.968997216,8.466569336,8.399115759,8.86788847,9.065756819,10.47006317,10.1363304,10.04319427,10.24905259,None],
}

PAT_RESD = {
    "ARG": [937,801,640,552,688,735,643,509,546,884,393,425,442,930,406,None,None,None],
    "AUS": [2718,2821,2494,2409,2383,2627,3061,1988,2291,2620,2503,2757,2637,2368,2966,None,None,None],
    "AUT": [2385,2298,2263,2424,2154,2258,2162,2092,2205,2078,2073,2039,2066,2124,1872,None,None,None],
    "BEL": [454,575,669,620,636,755,715,889,949,1054,1001,892,876,862,799,None,None,None],
    "BGR": [211,249,242,243,262,245,282,218,280,230,202,180,186,239,165,None,None,None],
    "BRA": [4194,4280,4271,4228,4695,4798,4959,4659,4641,5200,5480,4980,5464,5280,4666,None,None,None],
    "CAN": [4998,5061,5067,4550,4754,4709,4567,4198,4277,4078,4053,4349,4238,4452,4710,None,None,None],
    "CHE": [1692,1594,1684,1622,1597,1480,1525,1480,1477,1462,1337,1283,1369,1384,1288,None,None,None],
    "CHN": [153060,194579,229096,293066,415829,535313,704936,801135,968252,1204981,1245709,1393815,1243568,1344817,1426644,None,None,None],
    "CZE": [716,712,789,868,783,867,984,910,880,792,794,678,765,673,541,None,None,None],
    "DEU": [47853,49240,47859,47047,46986,46620,47353,48154,47384,48480,47785,46617,46632,42260,39822,None,None,None],
    "DNK": [1660,1634,1518,1626,1574,1406,1341,1377,1462,1552,1490,1262,1351,1261,1090,None,None,None],
    "ESP": [3267,3632,3596,3566,3430,3266,3026,2953,2799,2745,2167,1525,1288,1431,1308,None,None,None],
    "FIN": [1804,1799,1806,1731,1650,1698,1596,1419,1289,1260,1390,1387,1321,1588,1557,None,None,None],
    "FRA": [14722,14658,14100,14748,14655,14540,14690,14500,14306,14206,14415,14303,14103,12771,13386,None,None,None],
    "GBR": [17375,16523,15985,15490,15343,15370,14972,15196,14867,13876,13301,12865,12061,11990,11592,None,None,None],
    "GRC": [575,628,698,728,722,628,698,651,550,606,498,430,356,400,394,None,None,None],
    "HUN": [689,683,757,649,662,692,642,546,569,616,496,407,427,428,433,None,None,None],
    "IDN": [284,386,415,508,533,None,663,702,1058,1101,2271,1407,3093,1309,1397,None,None,None],
    "IND": [6296,6425,7262,8853,8841,9553,10669,12040,12579,13199,14961,16289,19454,23141,26267,None,None,None],
    "IRL": [None,None,None,None,None,None,None,None,None,79,62,76,58,75,75,None,None,None],
    "ITA": [9255,8588,8814,8877,8794,8439,8307,8601,None,8848,8643,8921,9229,10061,10281,None,None,None],
    "JPN": [333498,330110,295315,290081,287580,287013,271731,265959,258839,260244,260292,253630,245372,227348,222452,None,None,None],
    "KOR": [128701,127114,127316,131805,138034,148136,159978,164073,167275,163424,159084,162561,171603,180477,186245,None,None,None],
    "LTU": [62,87,91,108,93,109,117,123,101,95,81,81,90,95,81,None,None,None],
    "LUX": [15,48,60,79,85,109,113,128,128,143,156,152,117,129,112,None,None,None],
    "LVA": [139,206,240,178,173,193,225,103,136,95,90,86,82,93,104,None,None,None],
    "MEX": [629,685,822,951,1065,1294,1210,1246,1364,1310,1334,1555,1305,1132,1117,None,None,None],
    "NLD": [2079,2421,2575,2527,2585,2375,2315,2294,2207,2290,2241,2111,2228,2198,2080,None,None,None],
    "NOR": [1225,1150,1246,1117,1122,1009,1101,1106,1153,1227,1152,1082,957,880,946,None,None,None],
    "POL": [2392,2488,2899,3203,3879,4410,4237,3941,4676,4261,3924,4207,3887,4010,3377,None,None,None],
    "PRT": [250,381,571,499,571,621,647,722,925,724,644,661,703,695,711,None,None,None],
    "RUS": [27505,27712,25598,28722,26495,28701,28765,24072,29269,26795,22777,24926,23337,23759,19569,None,None,None],
    "SAU": [128,None,None,288,347,None,491,652,715,1070,909,1078,1188,1294,1398,None,None,None],
    "SVK": [239,167,176,234,224,168,184,211,228,220,183,217,206,206,146,None,None,None],
    "SVN": [331,301,373,442,470,None,None,None,None,None,None,255,None,232,222,None,None,None],
    "SWE": [2527,2549,2186,2196,2004,2288,2332,1984,2038,2032,1992,1838,1802,1764,1771,None,None,None],
    "TUR": [1810,2221,2555,3180,3885,4434,4392,4766,5352,6230,8175,7156,7871,7920,8234,None,None,None],
    "USA": [241347,231588,224912,241977,247750,268782,287831,285096,288335,295327,293904,285095,285113,269586,262244,None,None,None],
    "ZAF": [915,860,822,821,656,608,638,802,889,704,728,657,567,542,1804,None,None,None],
}

RNW_SHARE = {
    "ARG": [29.22679114,27.4501464,30.49539837,28.63348014,26.3898275,24.28399653,26.3673844,25.67168918,24.42052431,22.73758321,24.5224932,25.03788147,26.50991365,27.26244932,25.53611135,24.24840328,23.99157311,23.73474293],
    "AUS": [8.714945488,8.17116943,7.522908423,8.615496654,10.39285349,10.52669323,13.17011972,14.61686017,13.32533538,14.58357673,15.67847025,17.09709794,19.7037413,22.60029113,26.66086747,23.55353092,24.73621873,25.91890654],
    "AUT": [69.63745314,70.10779196,72.30001303,67.75582049,67.54162551,76.06025922,79.36820469,82.54022128,78.12785801,79.22847983,77.02418541,78.16539633,78.14707546,80.99662339,79.91142801,82.69896817,83.56206889,84.42516961],
    "BEL": [5.425427346,6.82941741,7.532116732,8.441797292,10.54889284,14.23672486,15.63652043,18.46040389,22.32221505,17.9650959,19.55413939,24.25938512,21.74479945,27.40433524,23.54315741,28.42502547,29.94561675,31.46620804],
    "BGR": [7.577568885,7.581965939,10.01070664,13.77015412,9.269976573,12.77394596,17.50779413,16.74007159,18.79399517,17.08032421,14.89947821,20.61748191,17.92808745,19.54522255,22.19180962,22.73855989,23.69514168,24.65172348],
    "BRA": [88.22733838,84.33459089,89.05454592,84.81639788,87.1951775,82.57765826,76.82520085,72.57709273,73.54009627,79.71280896,78.41815459,81.57307523,81.43023575,83.1761424,77.37534922,76.55020065,75.94536019,75.34051973],
    "CAN": [60.24766215,61.32909862,62.96373887,61.23945569,62.46040272,63.63715528,64.0759575,63.19546203,64.12426447,64.97537679,66.56463042,65.94838641,65.87912593,67.05645786,67.01638058,67.76743537,68.2324185,68.69740163],
    "CHE": [55.89400074,56.89667677,56.70255189,57.85593158,55.3378828,60.46907888,60.07947359,59.00784746,63.12446453,63.11583079,64.49711642,60.31068748,61.43811299,63.72397465,67.87623448,65.96920381,66.66260541,67.35600701],
    "CHN": [15.26579793,17.64081957,17.81087901,18.79298682,16.83607333,20.09803736,20.13348927,22.05570112,23.86494888,25.05203571,25.03867255,25.81274095,26.88970237,28.11190797,28.42799793,29.74161417,30.69405102,31.64648787],
    "CZE": [4.359101113,4.881155331,6.336423922,7.559421673,9.086994296,10.06153195,11.75161368,11.86697312,12.75119201,12.70746258,12.40260292,11.90614174,12.91782616,14.33772877,14.02348997,15.91823403,16.60009946,17.28196488],
    "DEU": [15.01250483,15.64343651,17.03173174,17.62383119,21.18302527,23.73606191,24.75696766,26.82086504,30.03290098,30.01926362,34.00951167,35.63722153,40.5775419,44.83962451,39.83056652,44.78389798,46.90892644,49.03395489],
    "DNK": [26.21070302,27.57264584,27.65907154,31.98337708,40.25393852,48.33042345,45.9608543,55.87770321,65.43562543,60.2201192,70.26597254,68.35743686,78.171839,81.61678502,78.95888048,88.78747508,93.16194801,97.53642093],
    "ESP": [20.16108687,20.69210984,26.07799878,33.49059952,30.5731384,30.44108039,40.46947985,40.92230645,35.71155633,39.3497749,32.89207402,38.75869004,37.77249988,44.52772304,47.01491367,46.89180995,48.42949426,49.96717857],
    "FIN": [29.93837159,35.87740211,30.10860425,29.99113717,32.89170215,40.56212808,35.96803121,38.58043293,44.50118806,44.23148189,46.614013,45.74100167,46.44342316,51.86811902,52.93029866,53.51935015,55.1572078,56.79506544],
    "FRA": [12.56009288,13.68749717,13.92544479,14.58104787,12.40904629,15.73566805,17.94905259,17.42406361,16.69125009,18.39548326,17.38139422,20.43545559,20.62870003,24.3565689,22.80387336,23.3109856,24.06682015,24.82265469],
    "GBR": [5.934042119,6.771619526,7.786682944,7.663209593,10.35838816,12.1419134,15.66903129,19.95925534,25.47103111,25.48525409,30.15449176,33.7661244,37.52052469,43.88367585,40.19997602,44.7031595,47.60134427,50.49952904],
    "GRC": [8.472722691,10.33354249,13.74597326,18.39547078,14.14590484,16.96023557,25.17344975,24.38411856,28.72726221,27.39869209,25.11184091,30.34974278,33.20850985,36.58123148,40.56401352,40.2725657,42.36036383,44.44816195],
    "HUN": [4.712937938,5.887145534,8.06190821,8.081185946,7.518698465,7.639208893,9.200699809,10.71666157,10.63580369,10.21255721,10.57268722,11.74416066,13.67705812,15.82880046,19.14451827,16.50348043,17.28613188,18.06878334],
    "IDN": [17.10797673,17.66528937,17.12346435,19.59947465,16.00835925,15.04753015,15.91261281,14.5086391,13.65935046,15.14374571,15.77141276,15.59991066,14.5841229,15.82066223,15.53003982,14.40798354,14.21662655,14.02526957],
    "IND": [16.89583191,17.1072579,15.350479,14.9419431,15.30611413,16.24498719,15.29876355,15.8631842,14.89485275,13.59717257,14.15937861,15.01025978,16.82752321,19.75574749,19.13129245,16.88444974,16.99179939,17.09914904],
    "IRL": [10.96548245,12.96467131,15.76560572,13.75277298,19.91331326,19.94110353,22.97487978,25.49037065,28.59897533,25.49681508,29.31293351,33.38368822,38.70595418,42.38950676,37.15417759,42.20724511,44.34306533,46.47888555],
    "ITA": [17.00669031,19.98055338,25.13541941,26.57085111,28.05822904,31.47954236,39.30872377,43.7427339,38.99396098,37.91326825,35.74225062,40.08991122,40.05219276,42.37352735,40.97357032,47.15354827,48.81922993,50.48491159],
    "JPN": [9.272989366,9.461420188,9.665595575,9.678700454,10.49708387,10.07610449,10.85884659,12.36036823,14.17879571,14.69936867,16.19738317,17.0616438,18.57880562,20.29236919,21.10366784,20.78646597,21.68491469,22.58336341],
    "KOR": [1.470153519,1.623017598,1.722130813,1.889600822,2.046633267,2.099899506,2.393610383,2.524408235,2.598460414,3.51815763,3.975070675,4.537970187,5.271464269,6.23004781,6.588347686,6.107922864,6.467338449,6.826754033],
    "LTU": [7.981723424,8.539423561,9.109845032,28.98051835,35.01182082,33.6506048,43.55312894,50.03411417,47.69916886,62.9395218,74.4411273,77.8467673,77.34575674,60.80659779,65.61441512,86.56139469,91.68528125,96.8091678],
    "LUX": [26.64783804,31.35535564,25.64956163,35.46852538,35.34122713,35.87997904,48.77570093,49.1977058,67.32321041,85.11379153,86.65324385,87.6416553,85.9067086,88.5747538,89.04479638,103.1723046,108.7473922,114.3224799],
    "LVA": [59.28170195,60.9051953,63.85688634,54.8599668,50.49827671,66.63272256,56.91963279,54.53298307,50.18507139,54.18415318,72.52290532,52.03152885,49.5823237,63.75838574,63.59613411,58.19678514,58.19348439,58.19018363],
    "MEX": [14.08873567,17.53743957,12.94707436,16.60202441,14.88821763,13.78543878,13.29741529,17.5321614,15.26460754,15.29570047,15.96099525,15.90793949,16.87713613,20.75288542,23.3762298,19.39955679,19.79021805,20.18087931],
    "NLD": [7.210336138,8.861546043,9.530235463,9.387675023,10.81067539,12.0898638,11.91167962,11.26928733,12.42307169,12.83313361,14.88038441,16.52806896,18.77625033,26.55756907,33.1271998,25.17292871,26.51790333,27.86287795],
    "NOR": [99.14653916,99.41209704,96.59698117,95.75931603,96.58501321,97.98866338,97.69816964,97.67329013,97.70692375,97.81797077,97.84925074,97.79684154,97.78224838,98.43045846,99.10382974,98.14739351,98.18792109,98.22844867],
    "POL": [3.80481713,4.670641641,6.158153177,7.403572312,8.391145107,10.76186482,10.76187265,12.87667469,14.19413862,13.99115432,14.44852022,12.97828733,15.97611411,18.39951405,17.45999855,19.41113047,20.40188454,21.39263861],
    "PRT": [35.07718192,32.93298386,37.87553528,53.15566648,47.06295986,43.78429227,59.23739671,61.3689633,48.66634872,55.52329725,40.90099443,51.37329801,54.2392106,59.62869739,64.90617705,62.23231113,63.79690833,65.36150552],
    "RUS": [17.80797334,16.21177379,17.95282163,16.41679913,16.08316639,15.75052814,17.40447572,16.81253929,16.08600863,17.26767207,17.27743885,17.51289085,17.79043899,20.07658854,19.15447518,18.49233917,18.64050165,18.78866413],
    "SAU": [0.000320151,0.000298727,0.000281,0.001703663,0.002155293,0.008861731,0.014016888,0.013609518,0.012734504,0.012434076,0.01841144,0.056225168,0.068837562,0.067648214,0.065298165,0.064982671,0.070248538,0.075514404],
    "SVK": [18.17080125,16.45373248,19.67715542,22.73652093,18.72972501,20.26932738,23.13748613,23.65607095,23.45835037,25.40644398,24.62686567,22.59834637,24.12252937,24.7555309,23.56076759,26.01925061,26.51032649,27.00140237],
    "SVN": [22.46500033,26.26745533,29.91257697,30.01107056,25.06183449,28.70102307,33.56337328,39.48523255,30.71145695,32.34260606,28.91357344,33.20739924,32.56581154,34.16655032,36.11268582,36.18392873,36.84452274,37.50511675],
    "SWE": [52.05883617,54.35323051,58.45956046,55.32790075,55.98889665,59.09852847,54.07399162,55.86870619,63.30428963,57.20508942,57.88925419,55.80973072,58.74827089,68.48070901,67.39193704,64.17379378,64.91169357,65.64959336],
    "TUR": [19.03410455,17.35063351,19.58219206,26.38180372,25.33197177,27.23822527,28.83539312,20.8888686,31.9582364,32.88814466,29.34585925,32.07699122,43.52259154,41.84769631,35.41894641,41.08363518,42.62157576,44.15951633],
    "USA": [8.913912026,9.530199775,10.78063089,10.61439831,12.6863937,12.41296211,13.02858021,13.37929407,13.63280829,15.25944497,17.28542203,17.16288634,17.93827372,19.91669873,20.27330893,20.58514429,21.3848272,22.18451012],
    "ZAF": [1.733694146,1.640483021,1.754008102,1.941271413,1.999101083,1.807645036,1.73565741,2.142585657,2.671154193,3.101900319,4.181046039,4.876851133,5.241241594,6.011915448,6.447863939,5.966031093,6.317731492,6.66943189],
}

GERD_PCT = {  # R&D expenditure % of GDP (GB.XPD.RSDV.GD.ZS), 2007-2024, real WDI
    "ARG": [0.46007,0.47055,0.58398,0.56104,0.56597,0.63491,0.61849,0.59396,0.62262,0.55815,0.55632,0.4883,0.47813,0.54126,0.5236,0.54951,0.60111,None],
    "AUS": [None,2.39898,None,2.3702,2.23459,None,2.17733,None,1.91969,None,1.88102,None,1.82826,None,1.85783,None,None,None],
    "AUT": [2.4336,2.58632,2.61279,2.74321,2.68567,2.93372,2.97993,3.11263,3.06917,3.13356,3.07376,3.10828,3.14405,3.20758,3.25565,3.17775,3.25517,None],
    "BEL": [1.84999,1.93684,1.99134,2.0613,2.15739,2.27079,2.32047,2.35858,2.43496,2.53291,2.67654,2.86365,3.15154,3.36823,3.40589,3.19592,3.27235,None],
    "BGR": [0.43003,0.44731,0.49338,0.56299,0.5295,0.60039,0.63424,0.79007,0.94947,0.77009,0.74037,0.75505,0.83576,0.84549,0.76933,0.75174,0.79224,None],
    "BRA": [1.08138,1.12904,1.11866,1.15992,1.13966,1.12684,1.19567,1.26971,1.28464,1.19584,1.1175,1.15016,1.15115,1.21907,1.13371,1.18873,1.1938,None],
    "CAN": [1.90358,1.85578,1.91742,1.82504,1.78714,1.77303,1.70624,1.71482,1.6936,1.72903,1.68702,1.7372,1.75578,1.93413,1.87064,1.81291,1.81048,1.7908],
    "CHE": [None,2.65296,None,None,None,2.87456,None,None,3.04577,None,3.04671,None,3.14983,None,3.24862,None,3.22178,None],
    "CHN": [1.35321,1.4233,1.6366,1.68456,1.75244,1.88095,1.96246,1.98475,2.01703,2.0595,2.07771,2.10232,2.20144,2.35712,2.38165,2.4945,2.57729,None],
    "CZE": [1.2967,1.2352,1.28943,1.31352,1.5323,1.757,1.86743,1.94392,1.906,1.65411,1.74513,1.87652,1.89547,1.94537,1.93302,1.89089,1.83369,None],
    "DEU": [2.41918,2.57152,2.68864,2.67632,2.75016,2.82499,2.78067,2.82218,2.87725,2.88393,2.9886,3.05057,3.11256,3.0897,3.0786,3.07096,3.1539,None],
    "DNK": [2.50921,2.76081,3.04209,2.91389,2.94198,2.9845,2.96145,2.91543,3.06422,3.10209,2.93576,2.97895,2.90605,2.96533,2.74323,2.86709,3.04841,None],
    "ESP": [1.23823,1.32156,1.35898,1.35436,1.32726,1.29268,1.26864,1.23401,1.21165,1.1808,1.20198,1.23288,1.24207,1.39637,1.39617,1.40684,1.49362,None],
    "FIN": [3.33727,3.53719,3.73427,3.70524,3.62434,3.4095,3.28462,3.16344,2.88826,2.74716,2.74723,2.77601,2.81534,2.93278,3.01133,2.982,3.0938,None],
    "FRA": [2.02815,2.06363,2.21273,2.17771,2.18761,2.22761,2.23369,2.27173,2.22401,2.22468,2.20422,2.20407,2.19668,2.2742,2.21278,2.21937,2.17832,None],
    "GBR": [1.6183,1.60901,1.66991,1.63887,1.64681,1.57587,1.62096,2.26488,2.27566,2.32051,2.32602,2.70889,2.6707,2.93993,2.89968,2.75224,2.67614,None],
    "GRC": [0.58513,0.67086,0.63414,0.60491,0.6837,0.71953,0.82288,0.84553,0.9716,1.00556,1.1492,1.2066,1.26236,1.48872,1.43353,1.47714,1.4943,None],
    "HUN": [0.95488,0.97817,1.13027,1.13102,1.18116,1.25758,1.38449,1.3437,1.33884,1.17645,1.31498,1.50196,1.46473,1.58067,1.63327,1.39056,1.37748,None],
    "IDN": [None,None,0.08332,None,None,None,0.0847,None,None,0.24535,0.23805,0.22632,0.27129,0.28068,None,None,None,None],
    "IND": [0.80507,0.85876,0.83314,0.78849,0.75502,0.74399,0.70642,0.70158,0.6931,0.66984,0.66603,0.66001,0.65942,0.64558,None,None,None,None],
    "IRL": [1.23409,1.39126,1.61371,1.59425,1.53761,1.5455,1.53562,1.47756,1.14062,1.14954,1.20821,1.08244,1.13733,1.12466,1.0732,1.53215,1.58829,None],
    "ITA": [1.12421,1.15548,1.21261,1.21296,1.19531,1.25559,1.29425,1.33148,1.33213,1.35915,1.36393,1.41934,1.45558,1.49869,1.41065,1.36563,1.37935,None],
    "JPN": [3.29257,3.29224,3.1959,3.10495,3.20537,3.1737,3.27895,3.36788,3.24071,3.10665,3.16636,3.2192,3.21824,3.26556,3.27383,3.40053,3.44125,None],
    "KOR": [2.75895,2.86774,3.02145,3.17913,3.44409,3.6852,3.77487,3.88983,3.78907,3.78638,4.07341,4.27155,4.36379,4.5214,4.59673,4.84753,4.94352,None],
    "LTU": [0.80111,0.78876,0.82848,0.79379,0.90998,0.8976,0.95322,1.03495,1.04077,0.84391,0.8963,0.92781,0.98701,1.12379,1.10044,1.05289,1.04977,None],
    "LUX": [1.57165,1.54663,1.58839,1.42373,1.42453,1.20664,1.23381,1.217,1.25226,1.2669,1.23898,1.17039,1.18209,1.09627,1.03153,1.0653,1.07045,None],
    "LVA": [0.58167,0.60627,0.46114,0.6183,0.73982,0.68884,0.63536,0.71433,0.641,0.45065,0.53004,0.66138,0.6602,0.75663,0.77085,0.8122,0.8236,None],
    "MEX": [0.38039,0.42416,0.45748,0.47353,0.45268,0.40284,0.40806,0.41962,0.41477,0.37601,0.31955,0.29816,0.27629,0.29185,0.27305,0.25727,0.26746,0.25372],
    "NLD": [1.66063,1.61247,1.65161,1.69388,1.8651,1.90094,2.13953,2.15067,2.11792,2.11546,2.14167,2.1027,2.14036,2.26514,2.21542,2.17529,2.26719,None],
    "NOR": [1.55795,1.54556,1.71678,1.6412,1.61714,1.61053,1.64216,1.70369,1.92351,2.03286,2.08168,2.03482,2.13599,2.24436,1.88764,1.55081,1.85142,None],
    "POL": [0.56103,0.59719,0.65932,0.72252,0.74726,0.88476,0.8807,0.9454,0.99807,0.9616,1.03055,1.19426,1.3088,1.37128,1.41558,1.44162,1.55525,None],
    "PRT": [1.12417,1.44335,1.58002,1.53317,1.45558,1.37661,1.32325,1.28892,1.24552,1.2815,1.32224,1.35078,1.39487,1.60979,1.66711,1.69051,1.69491,None],
    "RUS": [1.11611,1.04435,1.25192,1.1302,1.01545,1.02766,1.02732,1.07241,1.10085,1.10238,1.10967,0.99001,1.03531,1.09099,0.96602,0.91494,0.93518,0.93706],
    "SAU": [0.04521,0.04902,0.07338,0.884,0.88541,0.85824,0.79069,None,None,None,None,None,None,0.47618,0.39379,0.41439,0.49487,0.63522],
    "SVK": [0.45,0.46,0.47,0.62,0.66,0.80,0.82,0.88,1.16,0.79,0.89,0.84,0.83,0.92,0.95,0.98,0.98,None],
    "SVN": [1.43261,1.63465,1.83331,2.06914,2.4311,2.58339,2.59426,2.38854,2.21611,2.02922,1.88219,1.96365,2.05725,2.15559,2.13847,2.1,2.13385,None],
    "SWE": [3.24176,3.49083,3.41266,3.18756,3.20686,3.24636,3.27557,3.12457,3.24088,3.25643,3.39374,3.34602,3.39847,3.50181,3.41565,3.47239,3.59978,None],
    "TUR": [0.68616,0.68741,0.80363,0.79369,0.79393,0.82595,0.81206,0.8564,0.96732,1.11979,1.17632,1.27093,1.3196,1.36748,1.40209,1.32342,1.42224,None],
    "USA": [2.61516,2.74481,2.7918,2.71443,2.73803,2.67272,2.69591,2.70882,2.77328,2.83677,2.88315,2.98956,3.14297,3.41788,3.4689,3.48736,3.44716,None],
    "ZAF": [0.79364,0.80567,0.74993,0.66284,0.66753,0.66934,0.6633,0.70987,0.73147,0.74992,0.76257,0.68586,0.61304,0.60296,0.6139,0.61479,None,None],
}

POP = {
    "ARG": [40016763, 40424148, 40854831, 41288694, 41730660, 42161721, 42582455, 43024071, 43477012, 43900313, 44288894, 44654882, 44973465, 45191965, 45312281, 45407904, 45538401, 45696159],
    "AUS": [20827622, 21249199, 21691653, 22031750, 22340024, 22733465, 23128129, 23475686, 23815995, 24190907, 24592588, 24963258, 25334826, 25649248, 25685412, 26018721, 26659922, 27194286],
    "AUT": [8293654, 8319388, 8338752, 8357367, 8384901, 8420328, 8472176, 8538350, 8618709, 8737663, 8790624, 8833192, 8872344, 8912293, 8945279, 9058407, 9125730, 9174908],
    "BEL": [10624835, 10709108, 10796498, 10895591, 10993609, 11067748, 11125033, 11179778, 11238474, 11295003, 11349081, 11403740, 11462023, 11506938, 11552615, 11640788, 11730606, 11794601],
    "BGR": [7545339, 7492568, 7444447, 7395591, 7348329, 7264891, 7160010, 7073576, 6984223, 6894133, 6803473, 6710797, 6616726, 6550692, 6507301, 6465094, 6446597, 6441422],
    "BRA": [189038268, 191010274, 192980905, 194890682, 196603732, 198314934, 200004188, 201717541, 203475683, 205156587, 206804741, 208494900, 210147125, 211755692, 213317639, 214828540, 216284269, 217684462],
    "CAN": [32888886, 33247298, 33630069, 34005902, 34339221, 34713395, 35080992, 35434066, 35704498, 36110803, 36545075, 37072620, 37618495, 38028638, 38239864, 38950132, 40049088, 41262329],
    "CHE": [7551117, 7647676, 7743832, 7824910, 7912396, 7996861, 8089346, 8188646, 8282398, 8373334, 8451834, 8514327, 8575280, 8638169, 8704542, 8777085, 8888818, 9006644],
    "CHN": [1325810000, 1333820000, 1342520000, 1351560000, 1360250000, 1369520000, 1379010000, 1387950000, 1396130000, 1404050000, 1412350000, 1419010000, 1423520000, 1426110000, 1426440000, 1425180000, 1422580000, 1419320000],
    "CZE": [10322689, 10429692, 10491492, 10517247, 10496672, 10509286, 10510719, 10524783, 10542942, 10565284, 10589526, 10626430, 10669324, 10700155, 10500850, 10759525, 10878042, 10886531],
    "DEU": [82266371, 82110098, 81902311, 81776936, 80274981, 80425826, 80645605, 80982495, 81686608, 82348669, 82657000, 82905788, 83092956, 83160877, 83196076, 83797987, 83287281, 83516599],
    "DNK": [5457415, 5489022, 5519441, 5543819, 5566856, 5587085, 5608784, 5639719, 5678348, 5724456, 5760694, 5789957, 5814461, 5825337, 5850189, 5910577, 5944145, 5972420],
    "ESP": [45236004, 45983169, 46367550, 46562483, 46736257, 46749303, 46581124, 46433050, 46384379, 46427100, 46510461, 46715383, 47087778, 47344852, 47346836, 47781354, 48320520, 48821936],
    "FIN": [5288719, 5313398, 5338867, 5363341, 5388272, 5413967, 5438975, 5461507, 5479528, 5495297, 5508209, 5515525, 5521605, 5529545, 5541020, 5556108, 5583901, 5619914],
    "FRA": [63781275, 64133174, 64458715, 64773169, 65087317, 65402998, 65735961, 66276671, 66512558, 66688563, 66883314, 67125071, 67349922, 67569466, 67878650, 68232524, 68521609, 68745188],
    "GBR": [61319075, 61823772, 62260486, 62759456, 63285145, 63710543, 64138221, 64618693, 65086958, 65605841, 65964292, 66286727, 66627507, 66739867, 66977988, 67636134, 68526183, 69281437],
    "GRC": [11048466, 11077839, 11107017, 11121344, 11104900, 11045010, 10965209, 10892415, 10820883, 10775966, 10754679, 10732877, 10721584, 10698597, 10570133, 10437804, 10394869, 10374048],
    "HUN": [10055778, 10038186, 10022647, 10000020, 9958824, 9913588, 9872737, 9833034, 9797756, 9759760, 9726755, 9706961, 9694826, 9670416, 9630930, 9605078, 9592192, 9562069],
    "IDN": [237062337, 240157903, 243220028, 246305322, 249470032, 252698525, 255852467, 258877399, 261799249, 264627418, 267346658, 269951846, 272489381, 269576540, 272679150, 275719910, 278696190, 281603800],
    "IND": [1190680000, 1207930000, 1225520000, 1243480000, 1261220000, 1278670000, 1295830000, 1312280000, 1328020000, 1343940000, 1359660000, 1374660000, 1389030000, 1402620000, 1414200000, 1425420000, 1438070000, 1450940000],
    "IRL": [4375842, 4485070, 4533395, 4554763, 4574888, 4593697, 4614669, 4645440, 4687787, 4739597, 4810895, 4884896, 4958471, 5029875, 5074668, 5183966, 5281612, 5380313],
    "ITA": [58756247, 59211183, 59555456, 59819402, 60026844, 60191243, 60311616, 60320708, 60229599, 60115220, 60002254, 59877216, 59729077, 59438845, 59133173, 59013671, 58984217, 58957352],
    "JPN": [128032743, 128083960, 128031514, 128057352, 127834233, 127592657, 127413888, 127237150, 127094745, 127041812, 126918546, 126748506, 126555078, 126146099, 125502290, 124946789, 124351877, 123801750],
    "KOR": [48683638, 49054708, 49307835, 49554112, 49936638, 50199853, 50428893, 50746659, 51014947, 51217803, 51361911, 51585058, 51764822, 51836239, 51769539, 51672569, 51712619, 51751065],
    "LTU": [3231294, 3198231, 3162916, 3097282, 3028115, 2987773, 2957689, 2932367, 2904910, 2868231, 2828403, 2801543, 2794137, 2794885, 2808380, 2831638, 2871585, 2888277],
    "LUX": [479992, 488647, 497783, 506953, 518351, 530952, 543358, 556322, 569605, 583459, 596337, 607950, 620003, 630413, 640064, 653108, 666428, 677010],
    "LVA": [2200327, 2177327, 2141674, 2097553, 2059710, 2034327, 2012650, 1993780, 1977528, 1959536, 1942250, 1927173, 1913829, 1900445, 1884492, 1879377, 1886915, 1869569],
    "MEX": [109538665, 111275915, 113030218, 114756059, 116446180, 118060514, 119597654, 121048604, 122368490, 123587407, 124777172, 125995825, 127215666, 128209170, 128982939, 129960600, 131135337, 132274416],
    "NLD": [16381696, 16445590, 16530387, 16615390, 16693074, 16754963, 16804430, 16865008, 16939925, 17030314, 17131295, 17231622, 17344876, 17441500, 17533048, 17700981, 17877121, 17993486],
    "NOR": [4709156, 4768215, 4828716, 4889253, 4953089, 5018574, 5080171, 5137427, 5189898, 5236152, 5276965, 5311916, 5347893, 5379472, 5408320, 5457129, 5519596, 5572273],
    "POL": [38115967, 38115909, 38153389, 38516689, 38525670, 38533789, 38502396, 38483957, 38454576, 38426809, 38422346, 38413139, 38386476, 38177724, 37989517, 37827355, 37698294, 37563071],
    "PRT": [10542964, 10558177, 10568247, 10573100, 10574536, 10531420, 10473991, 10419607, 10381838, 10356516, 10340124, 10334633, 10354446, 10384846, 10561281, 10764411, 11067026, 11295785],
    "RUS": [142114903, 141956413, 141909248, 142389969, 143018195, 143378447, 143805638, 146508169, 146963159, 147381167, 147688545, 147818888, 147899994, 147707517, 147217903, 146713735, 146299107, 145855768],
    "SAU": [22368313, 23287877, 24217654, 23978487, 25091867, 26168861, 27624004, 28309273, 29816382, 30954198, 30977355, 30196281, 30063799, 31552510, 30784383, 32175224, 33702731, 35300280],
    "SVK": [5397766,5406972,5418374,5431024,5398384,5407579,5413393,5418649,5423801,5430798,5439232,5446771,5454147,5458827,5441991,5431752,5426740,5422069],
    "SVN": [2019406, 2022629, 2042335, 2049261, 2052496, 2056262, 2059114, 2061623, 2063077, 2064241, 2066161, 2070050, 2089310, 2100126, 2107007, 2108732, 2120937, 2126324],
    "SWE": [9142817, 9215021, 9292359, 9373379, 9446812, 9514406, 9596436, 9694194, 9793172, 9906331, 10053061, 10171524, 10281189, 10352390, 10409248, 10487859, 10545310, 10553341],
    "TUR": [70158112, 71051689, 72039215, 73142162, 74223642, 75175836, 76147634, 77181894, 78218488, 79277971, 80312708, 81407211, 82579448, 83384688, 84147326, 84979919, 85325967, 85518662],
    "USA": [301231207, 304093966, 306771529, 309378227, 311839461, 314339099, 316726282, 319257560, 321815121, 324353340, 326608609, 328529577, 330226227, 331578104, 332100166, 333996304, 336755052, 340003797],
    "ZAF": [49264665, 50068486, 50783099, 51507827, 52285283, 53125563, 53960768, 54837001, 55669436, 56436023, 57210747, 58038876, 58889765, 59694268, 60234725, 60867031, 61578350, 62328654],
}


EU_CH3_CODE_MAP = {  # Ch3 source code -> this pipeline's internal EU code
    "AUT":"AUT","BEL":"BEL","BGR":"BGR","CZE":"CZE","DNK":"DNK","ESP":"ESP",
    "FIN":"FIN","FRA":"FRA_EU","DEU":"DEU_EU","GRC":"GRC","HUN":"HUN","IRL":"IRL",
    "ITA":"ITA_EU","LVA":"LVA","LTU":"LTU","LUX":"LUX","NLD":"NLD","POL":"POL",
    "PRT":"PRT","SVK":"SVK","SVN":"SVN","SWE":"SWE",
}


def _ch3_embedded_to_df(club_key):
    """Builds the real-data DataFrame directly from the embedded Ch3 tables
    above (no file I/O). club_key='g20' keeps source codes as-is (they
    already match MEMBERS_WB); club_key='eu' renames via EU_CH3_CODE_MAP to
    match this pipeline's FRA_EU/DEU_EU/ITA_EU internal EU codes."""
    def to_long(d, value_name):
        rows = []
        for iso, vals in d.items():
            for y, v in zip(CH3_SOURCE_YEARS, vals):
                if v is not None:
                    rows.append({"iso": iso, "year": y, value_name: float(v)})
        return pd.DataFrame(rows)

    merged = to_long(CO2_PC, "co2_pc")
    for d, name in [(IMP_DEP, "import_dep_pct"), (TD_LOSS, "td_loss_pct"),
                    (PAT_RESD, "patents_residents"), (RNW_SHARE, "renew_share_pct"),
                    (POP, "population")]:
        merged = merged.merge(to_long(d, name), on=["iso", "year"], how="outer")

    if club_key == "eu":
        merged = merged[merged["iso"].isin(EU_CH3_CODE_MAP.keys())].copy()
        merged["iso"] = merged["iso"].map(EU_CH3_CODE_MAP)
    else:
        g20_codes = set(MEMBERS_WB.keys()) - {"EU"}
        merged = merged[merged["iso"].isin(g20_codes)].copy()
    return merged


def load_real_energy_data(club_key):
    """
    Real-data source for ES/innovation inputs: CO2 per capita, net energy
    import dependency, T&D losses, renewable electricity share, patents
    (residents only), and population -- all real World Bank WDI values.

    Primary source: the embedded tables in Section 3B above (works with
    zero setup, no external files). If data/g20_real_energy_panel.csv or
    data/eu_real_energy_panel.csv is ALSO present on disk, it overrides
    the embedded data (lets a future, more complete extract be dropped in
    without editing this script).

    Returns a DataFrame filtered to YEAR_MIN..YEAR_MAX, or None if neither
    source yields any rows. Countries not covered (EU: HRV, CYP, EST, MLT,
    ROU) are simply absent -- NOT filled with fabricated values;
    compute_es_dea scores only covered countries and leaves the rest NaN.
    """
    fname = "g20_real_energy_panel.csv" if club_key == "g20" else "eu_real_energy_panel.csv"
    p = DATA / fname
    if p.exists():
        df = pd.read_csv(p)
    else:
        df = _ch3_embedded_to_df(club_key)
    df = df[(df["year"] >= YEAR_MIN) & (df["year"] <= YEAR_MAX)].copy()
    return df if len(df) > 0 else None


def build_real_es_variables(real_df):
    """
    Constructs genuine DEA input/output columns from the real WDI extract.
    No input variable differentiates DMUs in the source data (no GDP or
    primary-energy-supply series was extracted), so a constant unit input
    is used -- a standard, valid DEA practice when the exercise is purely
    about comparing output bundles (Lovell, 1995; common in output-only
    "benefit of the doubt" composite-indicator DEA applications).
      Good outputs (maximise): renew_share_pct (renewable electricity
        share); self_suff = 100 - import_dep_pct (net exporters, which
        have negative import_dep_pct in WDI convention, correctly score
        above 100 -- more self-sufficient than "fully self-sufficient").
      Bad outputs (minimise, weak disposability): co2_pc, td_loss_pct.
    """
    d = real_df.copy()
    d["unit_input"] = 1.0
    d["self_suff"] = 100.0 - d["import_dep_pct"]
    d = d.dropna(subset=["co2_pc", "td_loss_pct", "renew_share_pct", "self_suff"])
    return d[["iso", "year", "unit_input", "self_suff", "renew_share_pct", "co2_pc", "td_loss_pct"]]


def simulate_energy_data(members):
    """Synthetic energy inputs/outputs -- FALLBACK ONLY, used when the real
    Chapter 3 data file (data/g20_real_energy_panel.csv / eu_real_energy_
    panel.csv) is not present. Identical generator for every club so ES
    indices are constructed the same way and remain comparable."""
    recs = []
    for m in members:
        scale, green, eff = rng.uniform(0.6, 1.6), rng.uniform(0.005, 0.03), rng.uniform(0.004, 0.02)
        for k, y in enumerate(YEARS):
            recs.append((m, int(y),
                         scale * (100 + rng.normal(0, 4)) * (1 - eff * k),
                         scale * (60 + rng.normal(0, 3)),
                         scale * (90 + 2.0 * k + rng.normal(0, 3)),
                         scale * (80 + 1.2 * k + rng.normal(0, 2)),
                         scale * (70 + rng.normal(0, 3)) * (1 - green * k)))
    return pd.DataFrame(recs, columns=["iso", "year", "pes", "cap", "gdp_va", "reliab", "co2"])


_ES_FRONTIER_CACHE = {}


def _canonical_union_members():
    """FIXED reference set for the single common ES frontier: 19 G20
    sovereigns union 22 real-data-covered EU members (plain ISO3; DEU/FRA/
    ITA counted once). Every ES computation in this script -- per-club or
    pooled -- draws from this same set, so a country's score is numerically
    identical wherever it's read."""
    return sorted(set(MEMBERS_WB.keys()) | set(EU_CH3_CODE_MAP.keys()))


def load_ch2_dea_mpi():
    """
    TASK: real, bootstrap bias-corrected DEA and Malmquist Productivity
    Index results supplied by the user from this dissertation's own
    Chapter 2 (data/step5_dea_scores.csv, data/mpi_results.csv) --
    methodologically more rigorous than this pipeline's own DEA/ML
    computation (Simar & Wilson, 1998, 2000, bias correction and
    confidence intervals, which this pipeline's own directional-distance-
    function solver does not perform), AND with genuinely wider real
    coverage: 2000-2023 for all 44 countries in the source, versus
    2007-2023 in this pipeline's own Chapter 3 WDI extract. Checked
    directly before use, not assumed: every one of this pipeline's 38
    canonical union members is present in both source files (confirmed by
    set comparison), and the score conventions match this pipeline's own
    (DEA scores bounded at a maximum of 1.0 = frontier-efficient, MPI/EC/TC
    centred on 1.0 = no change), so no re-scaling or sign-flip is needed.

    Column mapping, made explicit rather than assumed: dea_bc (the
    bias-corrected DEA score, preferred over the uncorrected dea_score)
    becomes "es"; mpi_bc/ec_bc/tc_bc (bias-corrected MPI, efficiency
    change, technical change) become ml_index/ml_effch/ml_techch. The MPI
    file's (year_t, year_t1) transition is stored at year_t1 in the
    output, matching this pipeline's OWN convention (get_common_es_
    frontier's rows.append(..., "year": y1, ...)) for the ML columns --
    checked directly against that function's source, not assumed to align.

    The source's 6 extra countries beyond this pipeline's 38-member union
    (EST, HRV, ROU, CYP, MLT, NOR -- the five EU members currently
    excluded from the EU-22 club for lacking real Chapter 3 coverage, plus
    Norway) are NOT used here: restoring them to club membership is a
    separate, larger decision (redefining which countries are "EU-22")
    that this function does not make on its own; only the ES/MPI VALUES
    for the EXISTING 38-member union are replaced.

    Returns a DataFrame [iso, year, es, ml_index, ml_effch, ml_techch] for
    exactly this pipeline's 38 canonical union members, or None if either
    source file is absent.
    """
    print("in load_ch2_dea_mpi")
    p_dea = DATA / "step5_dea_scores.csv"
    p_mpi = DATA / "mpi_results.csv"
    if not p_dea.exists() or not p_mpi.exists():
        print("p_dea or p_mpi does not exist. Return None")
        return None
    union_members = set(_canonical_union_members())

    dea = pd.read_csv(p_dea)
    dea = dea[dea.iso_code.isin(union_members)][["iso_code", "year", "dea_bc"]]
    dea = dea.rename(columns={"iso_code": "iso", "dea_bc": "es"})

    mpi = pd.read_csv(p_mpi)
    mpi = mpi[mpi.iso_code.isin(union_members)][["iso_code", "year_t1", "mpi_bc", "ec_bc", "tc_bc"]]
    mpi = mpi.rename(columns={"iso_code": "iso", "year_t1": "year", "mpi_bc": "ml_index",
                              "ec_bc": "ml_effch", "tc_bc": "ml_techch"})

    out = dea.merge(mpi, on=["iso", "year"], how="left")
    n_covered = out["iso"].nunique()
    missing = union_members - set(out["iso"].unique())
    print(f"  [Chapter 2 DEA/MPI] {n_covered}/{len(union_members)} union members covered "
         f"(2000-2023, bias-corrected)" + (f"; missing: {missing}" if missing else "") + ".")
    return out


def get_common_es_frontier(prov=None):
    """
    CORRECTION (reviewer: "estimate ES under a common frontier"): solves the
    DEA/Malmquist-Luenberger frontier ONCE per process, over the fixed
    canonical union above, and caches the result -- regardless of whether
    the caller is a G20-only run, an EU-only run, or the pooled comparison
    model. This replaces the earlier design, in which compute_es_dea() ran
    a SEPARATE DEA per club (a genuine club-specific-frontier bug: the same
    country -- e.g. Germany, which is in both clubs -- could get two
    different ES scores depending only on which club's table you read).
    Returns a DataFrame keyed by PLAIN ISO3 codes with columns
    [iso, year, es, ml_index, ml_effch, ml_techch].
    """
    if "canonical" in _ES_FRONTIER_CACHE:
        return _ES_FRONTIER_CACHE["canonical"]


    print("\n\n>> In get_common_es_frontier")
    ch2 = load_ch2_dea_mpi()
    print("after load_ch2_dea_mpi: Returns:", ch2)
    if ch2 is not None and ch2["iso"].nunique() == len(_canonical_union_members()):
        print("  [common ES frontier] Using Chapter 2's bootstrap bias-corrected DEA/MPI "
             "results (2000-2023) INSTEAD OF this pipeline's own LP-solved DEA -- more "
             "rigorous (Simar-Wilson bias correction) and wider real coverage (2000-2023 "
             "vs 2007-2023). This pipeline's own directional-distance-function solver "
             "(_ddf, still used elsewhere for the DEA-frontier sensitivity check in "
             "run_block_bootstrap_dea) is bypassed for the primary ES/MLI series specifically.")
        if prov is not None:
            prov.append({"variable": "ES index (common frontier)",
                        "source": "REAL bootstrap bias-corrected DEA/MPI from this dissertation's "
                                 "own Chapter 2 (data/step5_dea_scores.csv, data/mpi_results.csv), "
                                 "covering 2000-2023 for all 38 canonical union members -- "
                                 "preferred over this pipeline's own LP-solved DEA for both wider "
                                 "real coverage and methodological rigor (Simar & Wilson bias "
                                 "correction, not performed by this pipeline's own solver).",
                        "status": "LOADED (real, bias-corrected, Chapter 2 source)",
                        "citation": "This dissertation, Chapter 2 (DEA/Malmquist-Luenberger "
                                   "productivity analysis, bootstrap bias-corrected per "
                                   "Simar & Wilson, 1998, 2000)."})
        _ES_FRONTIER_CACHE["canonical"] = ch2
        return ch2

    print("before _canonical_union_members")
    union_members = _canonical_union_members()
    print("after union_members:,", union_members)
    print("before ch3_embedded_to_df_pooled")
    real_df = _ch3_embedded_to_df_pooled(union_members)
    print("after _ch3_embedded_to_df_pooled:", real_df)
    energy = build_real_es_variables(real_df)
    print("after build_real_es_variables>", energy)
    covered = set(energy["iso"].unique())
    missing = [m for m in union_members if m not in covered]
    print("missing:", missing)
    if missing:
        print(f"  [common ES frontier] no real data for {missing} -- excluded, "
              f"not fabricated (see EU_EXCLUDED_NO_REAL_DATA)")

    Xc, Yc, Bc = ["unit_input"], ["self_suff", "renew_share_pct"], ["co2_pc", "td_loss_pct"]
    by_year = {y: energy[energy.year == y].set_index("iso") for y in YEARS if (energy.year == y).any()}
    beta = {}
    for y, d in by_year.items():
        if len(d) < 3:
            continue
        X, Y, B = d[Xc].to_numpy(), d[Yc].to_numpy(), d[Bc].to_numpy()
        for m in d.index:
            o = d.loc[m]
            beta[(m, y)] = _ddf(o[Xc].to_numpy(), o[Yc].to_numpy(), o[Bc].to_numpy(), X, Y, B)
    es = pd.DataFrame([{"iso": m, "year": y, "es": 1.0 / (1.0 + b) if b == b else np.nan}
                       for (m, y), b in beta.items()])

    rows = []
    for m in union_members:
        for y in YEARS[:-1]:
            y1 = y + 1
            if y not in by_year or y1 not in by_year:
                continue
            if m not in by_year[y].index or m not in by_year[y1].index:
                continue
            d0, d1 = by_year[y], by_year[y1]
            X0, Y0, B0 = d0[Xc].to_numpy(), d0[Yc].to_numpy(), d0[Bc].to_numpy()
            X1, Y1, B1 = d1[Xc].to_numpy(), d1[Yc].to_numpy(), d1[Bc].to_numpy()
            o0, o1 = d0.loc[m], d1.loc[m]
            d_tt, d_t1t1 = beta[(m, y)], beta[(m, y1)]
            d_t_t1 = _ddf(o1[Xc].to_numpy(), o1[Yc].to_numpy(), o1[Bc].to_numpy(), X0, Y0, B0)
            d_t1_t = _ddf(o0[Xc].to_numpy(), o0[Yc].to_numpy(), o0[Bc].to_numpy(), X1, Y1, B1)
            if any(v != v for v in (d_tt, d_t1t1, d_t_t1, d_t1_t)):
                ml = effch = techch = np.nan
            else:
                # FIX (reviewer, critical, confirmed algebraically and
                # numerically): the previous techch formula did not
                # satisfy the required identity ML = EC x TC (a concrete
                # illustrative case gave ML=0.9387 vs EC*TC=1.1093). techch
                # is derived here directly as ML/EC and verified to
                # reproduce ML exactly on test values before use, rather
                # than an independently-guessed formula that happened not
                # to telescope correctly with the other two terms.
                ml = math.sqrt(((1 + d_tt) / (1 + d_t_t1)) * ((1 + d_t1_t) / (1 + d_t1t1)))
                effch = (1 + d_tt) / (1 + d_t1t1)
                techch = math.sqrt(((1 + d_t1t1) / (1 + d_tt)) * ((1 + d_t1_t) / (1 + d_t_t1)))
                assert abs(effch * techch - ml) < 1e-9, \
                    f"ML=EC*TC identity violated: ml={ml}, effch*techch={effch*techch}"
            rows.append({"iso": m, "year": y1, "ml_index": ml, "ml_effch": effch, "ml_techch": techch})
    es = es.merge(pd.DataFrame(rows), on=["iso", "year"], how="left")

    print("prov:", prov)
    if prov is not None:
        prov.append({"variable": "ES index (common frontier)",
                    "source": f"ONE DEA/ML run over {len(union_members)} unique G20+EU countries "
                              f"(same frontier used by every club and the pooled model)",
                    "status": "COMPUTED (real inputs, common frontier)",
                    "citation": "World Bank WDI, via Chapter 3 extract"})
    _ES_FRONTIER_CACHE["canonical"] = es
    return es


def compute_es_dea(members, out_dir, prov, club_key=None):
    """Per-club accessor: filters the SINGLE common ES frontier (above) to
    this club's own member codes. club_key='eu' remaps plain ISO3 -> this
    pipeline's internal EU codes (FRA_EU/DEU_EU/ITA_EU) via EU_CH3_CODE_MAP,
    which is exactly the plain->club-code mapping needed here."""
    print("in compute_es_dea")
    common = get_common_es_frontier(prov)
    print("after get_common_es_frontier", type(common), common)
    if club_key == "eu":
        common = common.copy()
        common["iso"] = common["iso"].map(lambda x: EU_CH3_CODE_MAP.get(x, x))
    club_es = common[common["iso"].isin(members)].copy()
    missing = [m for m in members if m not in set(club_es["iso"].unique())]
    print("missing in compute_es_dea>", missing)
    if missing:
        print(f"  [{club_key}] ES: {len(missing)} member(s) not covered by the common "
              f"frontier and get NO ES score (not fabricated): {missing}")
    (out_dir / "tables").mkdir(parents=True, exist_ok=True)
    print("Saving club_es to csv in: ", out_dir / "tables " / "es_index_dea_csv")
    club_es.to_csv(out_dir / "tables" / "es_index_dea.csv", index=False)
    return club_es


# =========================================================================
# SECTION 3 ? CLUB CONFIGS
# =========================================================================
MEMBERS_WB = {  # G20 = 19 SOVEREIGN states only (EU aggregate removed -- see fix
                # note below). World Bank ISO3 codes.
    "ARG": "ARG", "AUS": "AUS", "BRA": "BRA", "CAN": "CAN", "CHN": "CHN",
    "DEU": "DEU", "FRA": "FRA", "GBR": "GBR", "IDN": "IDN",
    "IND": "IND", "ITA": "ITA", "JPN": "JPN", "KOR": "KOR", "MEX": "MEX",
    "RUS": "RUS", "SAU": "SAU", "TUR": "TUR", "USA": "USA", "ZAF": "ZAF",
}
# FIX (reviewer correction #3, "Resolve the G20 unit"): the previous version
# included "EU" as a 20th G20 "member" alongside its own sovereign states
# (Germany, France, Italy, ...), which double-represents those countries
# when the G20 club is compared against the EU-27 club -- a France-Germany
# dyad would otherwise implicitly appear once as a G20 dyad AND be compared
# against an EU-27 sample that also contains France and Germany individually.
# G20 is now the clean 19-sovereign-state sample: C(19,2) = 171 dyads (was
# 190 with the EU aggregate included). The EU as its own institutional actor
# is out of scope for a country-to-country dyadic model and is not modelled
# as a 20th "country" anywhere below.
G20_LANGS = {"ARG": {"es"}, "AUS": {"en"}, "BRA": {"pt"}, "CAN": {"en", "fr"},
             "CHN": {"zh"}, "DEU": {"de"}, "FRA": {"fr"},
             "GBR": {"en"}, "IDN": {"id"}, "IND": {"hi", "en"}, "ITA": {"it"},
             "JPN": {"ja"}, "KOR": {"ko"}, "MEX": {"es"}, "RUS": {"ru"},
             "SAU": {"ar"}, "TUR": {"tr"}, "USA": {"en"}, "ZAF": {"en"}}
G20_CONTIG = {frozenset(p) for p in [("USA", "CAN"), ("USA", "MEX"), ("ARG", "BRA"),
              ("CHN", "IND"), ("CHN", "RUS"), ("DEU", "FRA"), ("FRA", "ITA")]}
G20_CAPITALS_FALLBACK = {
    "ARG": (-34.6, -58.4), "AUS": (-35.3, 149.1), "BRA": (-15.8, -47.9),
    "CAN": (45.4, -75.7), "CHN": (39.9, 116.4), "DEU": (52.5, 13.4),
    "FRA": (48.9, 2.4), "GBR": (51.5, -0.1),
    "IDN": (-6.2, 106.8), "IND": (28.6, 77.2), "ITA": (41.9, 12.5),
    "JPN": (35.7, 139.7), "KOR": (37.6, 127.0), "MEX": (19.4, -99.1),
    "RUS": (55.8, 37.6), "SAU": (24.7, 46.7), "TUR": (39.9, 32.9),
    "USA": (38.9, -77.0), "ZAF": (-25.7, 28.2)}

MEMBERS_EUROSTAT = {  # EU-22: panel code -> Eurostat geo code (EL, not GR, for Greece)
    "AUT": "AT", "BEL": "BE", "BGR": "BG",
    "CZE": "CZ", "DNK": "DK", "FIN": "FI", "FRA_EU": "FR",
    "DEU_EU": "DE", "GRC": "EL", "HUN": "HU", "IRL": "IE", "ITA_EU": "IT",
    "LVA": "LV", "LTU": "LT", "LUX": "LU", "NLD": "NL",
    "POL": "PL", "PRT": "PT", "SVK": "SK", "SVN": "SI",
    "ESP": "ES", "SWE": "SE",
}
# FIX (reviewer correction: "recover or explicitly exclude the five EU
# countries"): Croatia, Cyprus, Estonia, Malta, and Romania are NOT covered
# by the real WDI extract this pipeline embeds (Section 3B). The previous
# version kept all 27 nominal EU members in MEMBERS_EUROSTAT, so these 5
# silently dropped out only once ES turned out NaN for them deep inside
# dyad-building -- the club looked like "EU-27" (351 dyads) while actually
# running as a 22-country sample. That is now made EXPLICIT: the club is
# defined as EU-22 from the start (231 dyads), not EU-27 with 5 silent
# gaps. RECOVERY remains possible and is the better long-run fix -- these
# are EU member states, so Eurostat (already wired as this club's live
# connector for R&D/GDP/GDP/population) very likely has the missing CO2,
# import-dependency, T&D-loss, renewable-share, and patent series too; add
# them to Section 3B's embedded tables (or point compute_es_dea at a live
# Eurostat pull for just these 5) to restore the full EU-27 sample. That
# data-gathering exercise was not completed in this pass, so exclusion
# rather than a partial/unverified reconstruction was chosen.
EU_LANGS = {
    "AUT": {"de"}, "BEL": {"nl", "fr", "de"}, "BGR": {"bg"},
    "CZE": {"cs"}, "DNK": {"da"},
    "FIN": {"fi", "sv"}, "FRA_EU": {"fr"}, "DEU_EU": {"de"}, "GRC": {"el"},
    "HUN": {"hu"}, "IRL": {"en", "ga"}, "ITA_EU": {"it"}, "LVA": {"lv"},
    "LTU": {"lt"}, "LUX": {"lb", "fr", "de"}, "NLD": {"nl"},
    "POL": {"pl"}, "PRT": {"pt"}, "SVK": {"sk"},
    "SVN": {"sl"}, "ESP": {"es"}, "SWE": {"sv"},
}
EU_CONTIG = {frozenset(p) for p in [
    ("FRA_EU", "DEU_EU"), ("FRA_EU", "BEL"), ("FRA_EU", "LUX"), ("FRA_EU", "ITA_EU"), ("FRA_EU", "ESP"),
    ("DEU_EU", "NLD"), ("DEU_EU", "BEL"), ("DEU_EU", "LUX"), ("DEU_EU", "AUT"), ("DEU_EU", "CZE"),
    ("DEU_EU", "POL"), ("DEU_EU", "DNK"), ("AUT", "ITA_EU"), ("AUT", "CZE"), ("AUT", "SVK"),
    ("AUT", "HUN"), ("AUT", "SVN"), ("POL", "CZE"), ("POL", "SVK"),
    ("CZE", "SVK"), ("HUN", "SVK"), ("HUN", "SVN"),
    ("BGR", "GRC"), ("ESP", "PRT"), ("ITA_EU", "SVN"),
    ("NLD", "BEL"), ("BEL", "LUX"), ("SWE", "FIN"), ("LVA", "LTU"),
]}
EU_CAPITALS = {
    "AUT": (48.2, 16.4), "BEL": (50.8, 4.4), "BGR": (42.7, 23.3),
    "CZE": (50.1, 14.4), "DNK": (55.7, 12.6),
    "FIN": (60.2, 24.9), "FRA_EU": (48.9, 2.4), "DEU_EU": (52.5, 13.4), "GRC": (38.0, 23.7),
    "HUN": (47.5, 19.0), "IRL": (53.3, -6.3), "ITA_EU": (41.9, 12.5), "LVA": (56.9, 24.1),
    "LTU": (54.7, 25.3), "LUX": (49.6, 6.1), "NLD": (52.4, 4.9),
    "POL": (52.2, 21.0), "PRT": (38.7, -9.1), "SVK": (48.1, 17.1),
    "SVN": (46.1, 14.5), "ESP": (40.4, -3.7), "SWE": (59.3, 18.1),
}
EU_EXCLUDED_NO_REAL_DATA = ["HRV", "CYP", "EST", "MLT", "ROU"]  # documented, not silent

CLUBS = {
    "g20": {
        "label": "G20",
        "members_map": MEMBERS_WB,
        "langs": G20_LANGS,
        "contig": G20_CONTIG,
        "capitals_fallback": G20_CAPITALS_FALLBACK,
        "connector": "worldbank",
        "has_live_capitals": True,
        "coop_file": "cooperation.csv",
        "trade_file": "trade.csv",
        "fdi_file": "fdi.csv",
        "out_dir": OUT_BASE / "g20",
        "coop_source_note": ("C-EENRG Bilateral State Energy Agreements Database; "
                              "G20 Research Group (Toronto) coded commitments"),
        "baseline_intercept": -0.2,   # weaker unconditional cooperation floor
    },
    "eu": {
        "label": "EU-22 (real-data-covered; excludes HRV/CYP/EST/MLT/ROU -- see note above MEMBERS_EUROSTAT)",
        "members_map": MEMBERS_EUROSTAT,
        "langs": EU_LANGS,
        "contig": EU_CONTIG,
        "capitals_fallback": EU_CAPITALS,
        "connector": "eurostat",
        "has_live_capitals": False,   # 27 fixed points; no live connector built
        "coop_file": "eu_cooperation.csv",
        "trade_file": "eu_trade.csv",
        "fdi_file": None,
        "out_dir": OUT_BASE / "eu",
        "coop_source_note": ("ENTSO-E Transparency Platform cross-border flows/capacity; "
                              "EC Projects of Common Interest list (Reg. (EU) 2022/869)"),
        "baseline_intercept": 0.05,   # single-market baseline already integrated
    },
}

WB_API = "https://api.worldbank.org/v2"
WB_INDICATORS = {"rd_gdp": "GB.XPD.RSDV.GD.ZS",
                 "gdp_usd": "NY.GDP.MKTP.CD", "pop": "SP.POP.TOTL",
                 "gdp_pc_ppp": "NY.GDP.PCAP.PP.KD", "gdp_growth": "NY.GDP.MKTP.KD.ZG"}
# gdp_pc_ppp: GDP per capita, PPP (constant 2021 international $) -- the
# standard cross-country development-level control (superior to raw
# nominal gdp_usd for comparing economies of very different sizes/price
# levels). gdp_growth: GDP growth (annual %) -- a business-cycle/momentum
# control, distinct from the level variables above.
EUROSTAT_API = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data"
EUROSTAT_DATASETS = {
    "rd_gdp": ("tsc00001", {}),
    "gdp_usd": ("nama_10_gdp", {"na_item": "B1GQ", "unit": "CP_MEUR"}),
    "pop": ("demo_pjan", {"sex": "T", "age": "TOTAL"}),
}
STOOQ_BRENT = "https://stooq.com/q/d/l/?s=cb.f&i=d"
IMF_CDIS_BASE = "http://dataservices.imf.org/REST/SDMX_JSON.svc/CompactData/CDIS"
USE_IMF_CDIS = False  # best-effort, off by default; see fetch_imf_cdis_bilateral


# =========================================================================
# SECTION 4 ? CONNECTORS
# =========================================================================
def parse_wb_payload(payload, value_name):
    if not isinstance(payload, list) or len(payload) < 2 or payload[1] is None:
        return pd.DataFrame(columns=["iso3", "year", value_name])
    out = []
    for r in payload[1]:
        iso3 = r.get("countryiso3code") or (r.get("country") or {}).get("id")
        yr, val = r.get("date"), r.get("value")
        if iso3 and yr is not None:
            out.append((iso3, int(yr), val))
    return pd.DataFrame(out, columns=["iso3", "year", value_name])


def fetch_worldbank(members_map):
    iso3 = ";".join(members_map.values()); back = {v: k for k, v in members_map.items()}
    s = _session(); merged = None
    for name, code in WB_INDICATORS.items():
        url = (f"{WB_API}/country/{iso3}/indicator/{code}"
               f"?format=json&date={YEAR_MIN}:{YEAR_MAX}&per_page=20000")
        r = s.get(url, timeout=60); r.raise_for_status()
        df = parse_wb_payload(r.json(), name)
        df["iso"] = df["iso3"].map(back).fillna(df["iso3"]); df = df[["iso", "year", name]]
        merged = df if merged is None else merged.merge(df, on=["iso", "year"], how="outer")
        time.sleep(0.2)
    return merged[merged.iso.isin(members_map.keys())].sort_values(["iso", "year"])


def fetch_capitals_worldbank(members_map):
    iso3 = ";".join(members_map.values()); back = {v: k for k, v in members_map.items()}
    r = _session().get(f"{WB_API}/country/{iso3}?format=json&per_page=400", timeout=60)
    r.raise_for_status()
    coords = {}
    for c in r.json()[1]:
        iso = back.get(c.get("id")); lat, lon = c.get("latitude"), c.get("longitude")
        if iso and lat and lon:
            coords[iso] = (float(lat), float(lon))
    return coords


def _jsonstat_to_long(js, value_name):
    """Generic JSON-stat 2.0 -> long DataFrame parser (Eurostat's format).
    Reads dimension order from js['id'] and sizes from js['size'] rather
    than assuming a fixed layout, so it tolerates Eurostat re-ordering
    dimensions across datasets/versions."""
    dim_ids = js["id"]; sizes = js["size"]; dims = js["dimension"]
    cat_index = {}
    for d in dim_ids:
        idx_map = dims[d]["category"]["index"]
        if isinstance(idx_map, list):
            idx_map = {code: i for i, code in enumerate(idx_map)}
        cat_index[d] = idx_map

    strides = [1] * len(dim_ids)
    for i in range(len(dim_ids) - 2, -1, -1):
        strides[i] = strides[i + 1] * sizes[i + 1]

    geo_dim = "geo" if "geo" in dim_ids else None
    time_dim = "time" if "time" in dim_ids else None
    if geo_dim is None or time_dim is None:
        raise ValueError(f"Expected geo/time dimensions, got {dim_ids}")

    fixed_pos = {}
    for d in dim_ids:
        if d in (geo_dim, time_dim):
            continue
        fixed_pos[d] = 0 if sizes[dim_ids.index(d)] == 1 else None

    values = js.get("value", {})
    if isinstance(values, list):
        values = {str(i): v for i, v in enumerate(values) if v is not None}

    out = []
    for geo_code, geo_pos in cat_index[geo_dim].items():
        for time_code, time_pos in cat_index[time_dim].items():
            skip = False
            flat = geo_pos * strides[dim_ids.index(geo_dim)] + time_pos * strides[dim_ids.index(time_dim)]
            for d, pos in fixed_pos.items():
                if pos is None:
                    skip = True; break
                flat += pos * strides[dim_ids.index(d)]
            if skip:
                continue
            v = values.get(str(flat))
            if v is not None:
                out.append((geo_code, int(time_code), float(v)))
    return pd.DataFrame(out, columns=["geo", "year", value_name])


def fetch_eurostat_indicator(members_map, dataset_code, filters, value_name):
    """
    IMPORTANT: multi-value filters for Eurostat's Statistics API must be sent
    as REPEATED query keys (geo=AT&geo=BE&geo=BG...), not a single key with
    values joined by '+' (geo=AT+BE+BG...) ? the latter is invalid and
    returns 400. `requests` does this correctly when a list is passed via
    `params=`, so we build a dict/list of tuples instead of a hand-joined
    string. We also use `sinceTimePeriod` instead of enumerating all 19
    years individually, both to shorten the URL and because it is a
    documented Eurostat filter.
    """
    s = _session()
    params = [("format", "JSON"), ("lang", "EN"), ("sinceTimePeriod", str(YEAR_MIN))]
    for g in members_map.values():
        params.append(("geo", g))
    for k, v in filters.items():
        params.append((k, v))

    url = f"{EUROSTAT_API}/{dataset_code}"
    r = s.get(url, params=params, timeout=60)
    if not r.ok:
        # Eurostat returns a JSON body describing exactly what was wrong ?
        # surface it instead of a bare "400 Bad Request".
        detail = r.text[:500]
        try:
            body = r.json()
            detail = (body.get("error", {}).get("label")
                      or body.get("error", {}).get("detail")
                      or str(body)[:500])
        except Exception:
            pass
        raise RuntimeError(
            f"Eurostat API returned {r.status_code} for dataset '{dataset_code}'. "
            f"Request URL: {r.url}\nServer message: {detail}"
        )
    js = r.json()
    df = _jsonstat_to_long(js, value_name)
    df = df[(df["year"] >= YEAR_MIN) & (df["year"] <= YEAR_MAX)]
    back = {v: k for k, v in members_map.items()}
    df["iso"] = df["geo"].map(back)
    return df.dropna(subset=["iso"])[["iso", "year", value_name]]


def fetch_eurostat_all(members_map):
    merged = None
    for name, (code, filters) in EUROSTAT_DATASETS.items():
        df = fetch_eurostat_indicator(members_map, code, filters, name)
        time.sleep(0.2)
        merged = df if merged is None else merged.merge(df, on=["iso", "year"], how="outer")
    return merged


def fetch_brent_volatility():
    s = _session()
    r = s.get(STOOQ_BRENT, timeout=60)
    r.raise_for_status()
    text = r.text
    first_line = text.splitlines()[0] if text else ""
    if "date" not in first_line.lower():
        # Stooq returned something other than the expected CSV (rate-limit
        # page, empty body, changed symbol, etc.) ? surface what we got
        # instead of letting pandas fail later with a bare KeyError.
        raise RuntimeError(
            f"Stooq did not return the expected CSV for {STOOQ_BRENT!r}. "
            f"First line of response was: {first_line[:200]!r}. This usually "
            f"means Stooq rate-limited/blocked the request or the symbol "
            f"changed; try again later or verify the URL in a browser."
        )
    px = pd.read_csv(io.StringIO(text))
    px.columns = [c.lower() for c in px.columns]
    if "date" not in px.columns or "close" not in px.columns:
        raise RuntimeError(f"Stooq CSV parsed but is missing expected columns; "
                           f"got {list(px.columns)}.")
    px["date"] = pd.to_datetime(px["date"]); px = px.sort_values("date")
    px["ret"] = np.log(px["close"]).diff(); px["year"] = px["date"].dt.year
    vol = (px.groupby("year")["ret"].std() * np.sqrt(252)).reset_index()
    vol.columns = ["year", "price_volatility"]
    return vol[(vol.year >= YEAR_MIN) & (vol.year <= YEAR_MAX)]


def parse_imf_compactdata(js, value_name):
    series = (js.get("CompactData", {}).get("DataSet", {}) or {}).get("Series", [])
    if isinstance(series, dict):
        series = [series]
    out = []
    for s in series:
        ref, cp = s.get("@REF_AREA"), s.get("@COUNTERPART_AREA")
        obs = s.get("Obs", [])
        if isinstance(obs, dict):
            obs = [obs]
        for o in obs:
            t, v = o.get("@TIME_PERIOD"), o.get("@OBS_VALUE")
            if t and v not in (None, ""):
                out.append((ref, cp, int(t), float(v)))
    return pd.DataFrame(out, columns=["ref", "cp", "year", value_name])


def fetch_imf_cdis_bilateral(indicator="IIWX_BP6_USD"):
    """Best-effort: IMF is migrating its API; verify dataflow/keys before relying on it."""
    s = _session()
    url = f"{IMF_CDIS_BASE}/A...{indicator}?startPeriod={YEAR_MIN}&endPeriod={YEAR_MAX}"
    r = s.get(url, timeout=90); r.raise_for_status()
    return parse_imf_compactdata(r.json(), "fdi_value")


# =========================================================================
# SECTION 5 ? LOADERS (proprietary/coded data) + simulated fallbacks
# =========================================================================
def load_cooperation(club):
    p = DATA / club["coop_file"]
    if p.exists():
        df0 = pd.read_csv(p)
        if "coop_count" in df0 and df0["coop_count"].notna().any():
            df = _to_panel_order(df0, club)
            if "coop_depth" not in df:
                df["coop_depth"] = df["coop_count"]
            return df, {"variable": "cooperation", "source": f"{club['coop_source_note']} ({p})",
                       "status": "LOADED", "citation": club["coop_source_note"]}
    return None, None


# =========================================================================
# SECTION 5B ? LAYERED REAL-COOPERATION-DATA STRATEGY
# =========================================================================
# Manually-coded "communiqu� cooperation" (the load_cooperation() loader
# above) is one option, but it is a political narrative more than an
# observed cooperation flow. Three genuinely observable, bulk-downloadable
# alternatives are built here instead, in priority order:
#
#   1. OECD (green) patent co-invention  -- PRIMARY cooperation measure.
#      Directly linked to Chapter 3's innovation logic; good G20 coverage.
#      Loader only (data/oecd_coinvention.csv) -- unlike RTA/CORDIS below,
#      no specific live OECD API/bulk-file URL was verified in this
#      session, so this is schema-only until populated.
#   2. CORDIS Horizon project co-participation -- EU-SPECIFIC ROBUSTNESS
#      variable. Real, observed joint R&D participation; EU-centred by
#      construction (see docstring on build_cordis_dyadic_cooperation).
#   3. WTO RTA membership (Mario Larch's Regional Trade Agreements
#      Database, Egger & Larch, 2008) -- INSTITUTIONAL CONTROL, not the
#      cooperation outcome itself; a formal economic-cooperation dummy
#      merged into the gravity controls (see mA_gravity in estimate()).
#
# All three are REAL, citable, downloadable datasets. None could be fully
# fetched and processed inside this sandbox: the RTA file is served as a
# .zip (this environment has no path from a fetched URL to an unzip step),
# and the CORDIS project/organisation files run to tens of thousands of
# rows -- too large to pull through a text-based web-fetch reliably. Every
# function below is therefore a LOADER + PROCESSOR: give it the real, raw,
# unmodified bulk file (exact download instructions in each docstring) and
# it does the real aggregation into a dyadic panel -- it does not require
# you to pre-aggregate anything by hand.

def _find_col(df, candidates, required_desc):
    """Defensive column-name matching: bulk files from external providers
    occasionally rename columns between export vintages. Tries each
    candidate (case-insensitive) and raises a clear, actionable error
    naming exactly which columns WERE found if none match, rather than
    silently mis-parsing."""
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    raise ValueError(f"Could not find a column for {required_desc} -- tried {candidates}, "
                     f"but the file has columns: {list(df.columns)}. Rename the relevant "
                     f"column to one of the candidates above, or extend the candidate list.")


def build_rta_dummy(club):
    """
    WTO RTA institutional control, from Mario Larch's Regional Trade
    Agreements Database (Egger & Larch, 2008), the standard RTA panel used
    throughout the structural-gravity literature (cited in this chapter's
    own reference list via Yotov et al., 2016).

    REAL DATA, not fetched in this session (served as a .zip):
      1. Download rta_20260111.csv (or the current dated version) from
         https://www.ewf.uni-bayreuth.de/en/research/RTA-data/index.html
      2. Unzip it and save as data/rta_larch.csv
    Expected columns (standard Larch/Egger format; auto-detected with
    reasonable name variants via _find_col): two ISO3 country codes, a
    year, and an "rta" dummy (1 = in force that year). Additional dummies
    (fta, cu, eia, psa, ...) are ignored here; "rta" is the broadest,
    most-inclusive category and the natural default control.

    Returns a DataFrame [iso_i, iso_j, year, rta] for this club's members,
    or None if data/rta_larch.csv is not present.
    """
    p = DATA / "rta_larch.csv"
    print(">>> In built_rta_dumy:", p)
    if not p.exists():
        return None, None
    raw = pd.read_csv(p, low_memory=False)
    c1 = _find_col(raw, ["iso3_1", "iso3_o", "importer", "country1", "iso_o"], "first country code")
    c2 = _find_col(raw, ["iso3_2", "iso3_d", "exporter", "country2", "iso_d"], "second country code")
    cy = _find_col(raw, ["year"], "year")
    cr = _find_col(raw, ["rta"], "RTA dummy")
    df = raw[[c1, c2, cy, cr]].rename(columns={c1: "iso_i", c2: "iso_j", cy: "year", cr: "rta"})
    df = df[(df.year >= YEAR_MIN) & (df.year <= YEAR_MAX)]

    members = set(club["members_map"].keys())
    # Larch's data uses plain ISO3; this club's own internal codes may be
    # renamed (EU: FRA_EU/DEU_EU/ITA_EU) -- map via connector type.

    def to_club_code(iso):
        return EU_CH3_CODE_MAP.get(iso, iso) if club.get("connector") == "eurostat" else iso

    df["iso_i"] = df["iso_i"].map(to_club_code)
    df["iso_j"] = df["iso_j"].map(to_club_code)
    df = df[df.iso_i.isin(members) & df.iso_j.isin(members)]
    df = _to_panel_order(df.dropna(subset=["rta"]), club)
    df["rta"] = df["rta"].astype(int)
    prov = {"variable": "RTA (institutional control)",
           "source": f"Mario Larch's RTA Database, Egger & Larch (2008) ({p})",
           "status": "LOADED (real)",
           "citation": "Egger, P.H. & Larch, M. (2008). Journal of International Economics 76(2), 384-399."}
    return df, prov


# TASK: "forget about Larch's database, it cannot be found. have a look if
# this document helps" -- the WTO country-level RTA activity data (already
# integrated as an honest PROXY, see load_rta_country_activity) cannot
# become genuine dyadic RTA membership for a general pair of countries:
# knowing how many agreements each independently signed does not reveal
# whether the SAME agreement covers both. But EU-22 intra-club dyads are a
# genuine special case, not just a proxy: the EU negotiates commercial
# policy collectively (an EU exclusive competence since Lisbon), so EU
# membership itself IS the deepest form of RTA (a customs union plus
# single market) between every pair of members, automatically covering
# them under every EU-negotiated third-country agreement too. This is
# independently verifiable in the supplied WTO data, not merely asserted:
# every "old" EU member (pre-2004) shows an IDENTICAL annual RTA count for
# all 24 years (confirmed directly: 14 members, 24 years, one series) --
# essentially impossible by chance and only explained by a shared,
# EU-level trade policy -- and Bulgaria's series is exactly zero through
# 2006 and jumps to 1 in 2007, precisely matching its known 1 January 2007
# EU accession date. Given that cross-check, EU accession year is used
# here as GENUINE dyadic ground truth (not a proxy) for EU-22 intra-club
# dyads specifically: rta_ij,t = 1 for t >= max(accession_i, accession_j).
# This does NOT extend to G20: G20 members negotiate trade policy
# independently, so no equivalent public, verifiable fact substitutes for
# genuine bilateral RTA membership there, and G20's "rta" remains ABSENT
# (the country-level WTO proxy, m19b_with_rta_proxy, is still the best
# available G20 control, honestly labelled as a proxy, not upgraded).
EU_ACCESSION_YEAR = {
    # Founding members (Treaty of Rome, in force 1958)
    "BEL": 1958, "FRA_EU": 1958, "DEU_EU": 1958, "ITA_EU": 1958, "LUX": 1958, "NLD": 1958,
    "DNK": 1973, "IRL": 1973,
    "GRC": 1981,
    "PRT": 1986, "ESP": 1986,
    "AUT": 1995, "FIN": 1995, "SWE": 1995,
    "CZE": 2004, "HUN": 2004, "LVA": 2004, "LTU": 2004, "POL": 2004, "SVK": 2004, "SVN": 2004,
    "BGR": 2007,
}


def build_rta_eu_membership(club):
    """
    Genuine dyadic RTA membership for EU-22 intra-club dyads, derived from
    EU accession dates (see the module-level comment above for the full
    justification and the empirical cross-check against the supplied WTO
    data). Returns None for any club other than "eu" -- this construction
    is specific to the EU's collective trade-policy competence and does
    not generalise to G20.
    """
    if club.get("connector") != "eurostat":
        return None, None
    members = list(club["members_map"].keys())
    missing = [m for m in members if m not in EU_ACCESSION_YEAR]
    if missing:
        print(f"    [build_rta_eu_membership] WARNING: no accession year for {missing} -- "
             f"dyads involving these members will be excluded from this construction, not "
             f"defaulted to a guess.")
    rows = []
    for i, j in combinations(members, 2):  # SAME order as build_panel's own dyad construction
        if i not in EU_ACCESSION_YEAR or j not in EU_ACCESSION_YEAR:
            continue
        acc = max(EU_ACCESSION_YEAR[i], EU_ACCESSION_YEAR[j])
        for y in YEARS:
            rows.append({"iso_i": i, "iso_j": j, "year": y, "rta": int(y >= acc)})
    df = pd.DataFrame(rows)
    # FIX: _canon() alphabetically sorts iso_i/iso_j, but build_panel's own
    # dyad_id does NOT use alphabetical order -- it uses whichever order
    # combinations(members, 2) happens to produce from the raw dict key
    # order. Confirmed directly: this caused a real merge failure (456 of
    # 5544 dyad-years, exactly 19 dyads x 24 years, silently unmatched)
    # before this fix, because _canon reordered pairs like (DEU_EU, DNK)
    # while the panel's own dyad_id was "DNK_DEU_EU". Not applying _canon
    # here, matching the panel's own convention exactly instead, resolved
    # it -- verified below, not just assumed.
    prov = {"variable": "RTA (institutional control)",
           "source": "GENUINE dyadic RTA membership derived from EU accession dates (not Larch's "
                    "database, which could not be located) -- justified by the EU's exclusive "
                    "competence over commercial policy since the Lisbon Treaty, and empirically "
                    "cross-checked against the supplied WTO country-level RTA data (identical "
                    "series across all pre-2004 members; Bulgaria's series transitions from 0 to "
                    "1 exactly at its 2007 accession year).",
           "status": "LOADED (real, EU-specific construction)",
           "citation": "Treaty of Lisbon (2007), Art. 3(1)(e) TFEU (exclusive EU competence over "
                      "the common commercial policy); accession dates per the EU's own accession "
                      "treaties."}
    return df, prov


def build_rta_wto_signatory_based(club):
    """
    TASK: real, GENUINELY DYADIC RTA membership supplied by the user,
    built directly from the WTO's own RTA-IS database export (agreement-
    level records with actual signatory lists), replacing this pipeline's
    earlier two RTA constructions:
      - load_rta_country_activity()'s country-level proxy (kept, still
        used in m19b_with_rta_proxy for comparison, but no longer the
        preferred dyadic source);
      - build_rta_eu_membership()'s EU-accession-based construction
        (kept as a final fallback below, but superseded here by data that
        does the same thing more precisely AND extends genuine dyadic
        coverage to G20 as well, which accession dates alone cannot do).

    Construction: for each of 237 WTO-notified agreements (2000-2023)
    with 2+ panel-economy signatories, every pairwise combination of
    signatories is marked as RTA-covered from that agreement's
    notification year onward (no termination dates in the source, so
    treated as ongoing, matching its own "In Force" / "in force for at
    least one Party" status categories). The source itself already
    handles time-varying EU composition correctly (2004/2007/2013
    accessions, UK included through 2019 and excluded from 2020 for
    Brexit) -- confirmed directly in the source's own Notes sheet, not
    re-derived here.

    KNOWN LIMITATION, PARTIALLY CORRECTED (reviewer: "too important to
    leave only as a caveat"): the source only carries a NOTIFICATION
    year, not each agreement's entry-into-force date, so a pair whose
    ONLY qualifying RTA predates 2000 and was never renegotiated since
    can show a FALSE ZERO until a later notification eventually covers
    them. Checked directly across every currently-covered dyad in both
    clubs (not assumed): two genuinely uncontroversial cases were found
    and are now hand-corrected below to their real historical start date
    -- Argentina-Brazil (Mercosur, in force 1991, originally showing 0
    through 2009), and the NAFTA trio USA-Canada, USA-Mexico, Canada-
    Mexico (in force 1 January 1994, originally showing 0 until their
    2018-2020 USMCA renegotiation). Preferring each agreement's entry-
    into-force date generally, as would be preferable, is not possible
    from this file alone since it does not carry that field; this
    correction is deliberately narrow -- two specific, extremely
    well-documented treaty relationships, not a general attempt to
    hand-encode this pipeline's author's own historical knowledge of
    every G20 pair's treaty history. Other left-censoring cases may
    remain; see m19c_with_rta_excl_left_censored for a sensitivity check
    that excludes dyads whose first RTA activation looks implausibly late.

    Coverage, checked directly: 231/231 EU-22 dyads show at least some
    RTA coverage (expected, given the EU's own internal integration); only
    58/171 G20 dyads do, which is very plausibly a real reflection of the
    world (most G20 pairs genuinely have no bilateral/regional agreement)
    rather than a data gap, and -- unlike the EU-accession construction,
    which is CONSTANT across the entire 2008-2023 estimation window --
    this gives G20 genuine, usable within-sample variation for the first
    time.

    Returns a DataFrame [iso_i, iso_j, year, rta] for this club's members
    (dyads never covered by any agreement are correctly included as
    genuine 0, not omitted as unknown), or (None, None) if absent.
    """
    p = DATA / "rta_wto_genuine_dyadic.csv"
    if not p.exists():
        return None, None
    raw = pd.read_csv(p)
    raw = raw[(raw.year >= YEAR_MIN) & (raw.year <= YEAR_MAX)]
    members = list(club["members_map"].keys())

    def to_club_code(iso):
        return EU_CH3_CODE_MAP.get(iso, iso) if club.get("connector") == "eurostat" else iso

    raw["iso_i"] = raw["iso_i"].map(to_club_code)
    raw["iso_j"] = raw["iso_j"].map(to_club_code)
    covered = raw[raw.iso_i.isin(members) & raw.iso_j.isin(members)].copy()

    # FIX: the SAME bug already found and fixed once for
    # build_rta_eu_membership -- _canon() alphabetically sorts iso_i/
    # iso_j, but build_panel's own dyad_id does NOT use alphabetical
    # order (it uses whichever order combinations(members, 2) happens to
    # produce from the raw dict key order). Reintroduced here by mistake
    # (this is a NEW function, not the one fixed before) and caught the
    # same way: confirmed directly, 456/5544 EU dyad-years (the identical
    # 19 dyads as before, e.g. DNK_DEU_EU) went unmatched before this fix.
    # Rather than apply _canon()'s alphabetical convention, this maps
    # each unordered pair to whichever (i, j) order the panel's OWN
    # combinations(members, 2) actually produced, via all_pairs below.
    all_pairs = pd.DataFrame(list(combinations(members, 2)), columns=["iso_i", "iso_j"])
    canonical_order = {frozenset((r.iso_i, r.iso_j)): (r.iso_i, r.iso_j) for r in all_pairs.itertuples()}

    def _reorder(row):
        key = frozenset((row["iso_i"], row["iso_j"]))
        correct = canonical_order.get(key)
        return pd.Series(correct) if correct is not None else pd.Series([row["iso_i"], row["iso_j"]])

    if len(covered) > 0:
        covered[["iso_i", "iso_j"]] = covered.apply(_reorder, axis=1)

    # Dyads NEVER appearing in the source (no agreement ever covers them)
    # are genuine, informative zeros -- built explicitly here rather than
    # left as an accidental gap from an inner join.
    skeleton = all_pairs.merge(pd.DataFrame({"year": YEARS}), how="cross")
    df = skeleton.merge(covered, on=["iso_i", "iso_j", "year"], how="left")
    df["rta"] = df["rta"].fillna(0).astype(int)

    # TASK (reviewer): "the code itself identifies Argentina-Brazil as a
    # false-zero example. That is too important to leave only as a
    # caveat... complete pre-2000 agreements relevant to the sample." The
    # underlying WTO RTA-IS export used here only carries a NOTIFICATION
    # year, not each agreement's entry-into-force date, so preferring
    # entry-into-force generally (as the reviewer's first-preference
    # suggests) is not possible from this file alone. What IS possible,
    # and done here: checked every one of this club's currently-covered
    # dyads for implausibly late first-activation years (see the
    # investigation this correction is based on), found that all but two
    # relationships look like genuinely NEW agreements (Korea's, CETA's
    # and USMCA's 2010s-2020s FTAs are real, new instruments, not
    # left-censoring artefacts) -- but NAFTA (in force 1 January 1994)
    # was completely missed until its 2020 USMCA renegotiation, exactly
    # the same failure mode as Argentina-Brazil's Mercosur (in force
    # 1991, missed until a 2010 notification). Both are hand-supplemented
    # here from well-established, uncontroversial public treaty dates --
    # NOT from the WTO notification file, and explicitly labelled as such
    # -- rather than left as a caveat only. This is deliberately narrow:
    # only two treaty relationships with essentially no factual dispute,
    # not a general attempt to hand-encode this pipeline's author's own
    # historical knowledge of every G20 pair's treaty history.
    PRE_2000_RTA_SUPPLEMENT = {
        frozenset(("ARG", "BRA")): 1991,  # Mercosur, Treaty of Asuncion
        frozenset(("USA", "CAN")): 1994,  # NAFTA (superseded by USMCA, 2020)
        frozenset(("USA", "MEX")): 1994,  # NAFTA (superseded by USMCA, 2020)
        frozenset(("CAN", "MEX")): 1994,  # NAFTA (superseded by USMCA, 2020)
    }
    df["left_censoring_corrected"] = False
    for pair_key, true_start_year in PRE_2000_RTA_SUPPLEMENT.items():
        plain_pair = tuple(pair_key)
        club_pair = tuple(to_club_code(iso) for iso in plain_pair)
        mask_a = (df.iso_i == club_pair[0]) & (df.iso_j == club_pair[1])
        mask_b = (df.iso_i == club_pair[1]) & (df.iso_j == club_pair[0])
        mask = (mask_a | mask_b) & (df.year >= true_start_year) & (df.rta == 0)
        if mask.any():
            df.loc[mask, "rta"] = 1
            df.loc[mask, "left_censoring_corrected"] = True
    n_corrected_dyads = df.loc[df["left_censoring_corrected"], ["iso_i", "iso_j"]].drop_duplicates().shape[0]
    if n_corrected_dyads > 0:
        print(f"    [RTA left-censoring fix] {n_corrected_dyads} dyad(s) hand-corrected to their "
             f"real pre-2000 treaty start date (Mercosur 1991 and/or NAFTA 1994), not left as a "
             f"caveat-only false zero.")

    n_dyads_any_coverage = covered.groupby(["iso_i", "iso_j"]).ngroups if len(covered) > 0 else 0
    n_dyads_total = all_pairs.shape[0]
    prov = {"variable": "RTA (institutional control)",
           "source": f"GENUINE dyadic RTA membership from the WTO's own RTA-IS database export "
                    f"(237 agreements, signatory-level, 2000-2023 notifications); "
                    f"{n_dyads_any_coverage}/{n_dyads_total} of this club's dyads show coverage "
                    f"at some point. Preferred over both this pipeline's earlier country-level "
                    f"RTA proxy and its EU-accession-based construction. KNOWN LIMITATION, "
                    f"partially addressed: agreements notified before 2000 (Mercosur 1991, NAFTA "
                    f"1994) were originally missed as false zeros (verified directly on "
                    f"Argentina-Brazil and the NAFTA trio) -- these two specific, uncontroversial "
                    f"treaty relationships are now hand-corrected to their real start dates "
                    f"({n_corrected_dyads} dyad(s) in this club); other left-censoring cases may "
                    f"remain uncorrected (see left_censoring_corrected column and "
                    f"m19c_with_rta_excl_left_censored for a sensitivity check).",
           "status": "LOADED (real, genuinely dyadic, WTO RTA-IS signatory data, partially "
                    "corrected for known pre-2000 left-censoring)",
           "citation": "World Trade Organization, Regional Trade Agreements Information System "
                      "(RTA-IS), agreement-level signatory records; pre-2000 correction from the "
                      "Treaty of Asuncion (1991) and NAFTA (1994) directly, not from WTO RTA-IS."}
    return df, prov


def load_rta_country_activity(club):
    """
    TASK: real WTO RTA notification data was supplied (data/
    rta_activity_by_country.csv) -- country-year counts of NEW regional
    trade agreements entering into force each year, for all 19 G20
    sovereigns and all 22 EU-22 members (full coverage, verified directly
    by mapping every country name to ISO3 before trusting the file).
    Sanity-checked before use, not just accepted at face value: EU member
    states show IDENTICAL annual values because the EU negotiates trade
    agreements as a bloc (a real, expected feature of WTO RTA
    notifications, not a data error), and Croatia's series is correctly
    all-zero before its 2013 EU accession and jumps immediately after --
    both independently verifiable facts, both confirmed in the data.

    IMPORTANT, exactly analogous to the OECD co-invention variable's own
    documented limitation above: this is COUNTRY-level (how many RTAs did
    country i enter into that year), not DYADIC (do countries i and j
    share a specific common agreement with each other). It CANNOT
    directly populate the "rta" dummy that build_rta_dummy() above expects
    from Mario Larch's true dyadic database, because knowing how many
    agreements each country has does not reveal whether any of THOSE
    agreements are the same one. It is used here, honestly labelled, as a
    dyadic PROXY only: rta_stock_min_l1_z = the lagged, z-scored minimum
    of each pair's cumulative RTA stock (countries that are each
    independently heavily embedded in the RTA system are more likely,
    though not certain, to share one) -- ranked explicitly BELOW true
    dyadic Larch data in every place this pipeline reports RTA provenance,
    never presented as equivalent to it.

    Returns (DataFrame [iso, year, rta_stock] for this club's members,
    provenance dict) or (None, None) if the file is absent.
    """
    print ("\n\n>>> In load_rta_country_activity")
    p = DATA / "rta_activity_by_country.csv"
    if not p.exists():
        return None, None
    raw = pd.read_csv(p)
    raw = raw[(raw.year >= YEAR_MIN) & (raw.year <= YEAR_MAX)]
    members = set(club["members_map"].keys())
    plain_to_club = {v: k for k, v in EU_CH3_CODE_MAP.items()}
    club_to_plain = {v: k for k, v in plain_to_club.items()}

    def to_club_code(iso):
        return EU_CH3_CODE_MAP.get(iso, iso) if club.get("connector") == "eurostat" else iso

    raw["iso"] = raw["iso"].map(to_club_code)
    df = raw[raw.iso.isin(members)][["iso", "year", "rta_stock"]].copy()
    n_matched = df["iso"].nunique()
    prov = {"variable": "RTA activity proxy (country-level)",
           "source": f"REAL WTO RTA notification data, country-level annual new-RTA counts "
                    f"({p}); {n_matched}/{len(members)} club members matched. NOT a substitute "
                    f"for true dyadic Larch RTA membership (still absent) -- used only as an "
                    f"explicitly labelled dyadic proxy (pair minimum of cumulative RTA stock), "
                    f"ranked below true dyadic RTA data.",
           "status": "LOADED (real, country-level proxy only)",
           "citation": "World Trade Organization RTA notifications (WTO RTA Database)."}

    print(">>> df in load_rta_country_activity\n",df)

    return df, prov


def load_fdi_country_level(club):
    """
    TASK: real World Bank data was supplied (data/fdi_outflows_pct_gdp.csv,
    indicator BM.KLT.DINV.WD.GD.ZS, "Foreign direct investment, net
    outflows (% of GDP)") -- verified before use, not just accepted at
    face value: coverage is 22/22 for EU-22 but only 18/19 for G20
    (Turkiye is absent from this series and is reported as missing, not
    silently dropped or zero-filled); 7 country-year cells are missing
    (World Bank ".." markers, correctly parsed to NaN, not to 0); and the
    extreme values for Cyprus, Malta and Luxembourg (into the hundreds of
    percent of GDP, occasionally negative) are a well-documented real
    phenomenon for small offshore-financial-conduit economies -- large
    multinational flows routed through special-purpose entities relative
    to a tiny GDP base (Damgaard, Elkjaer & Johannesen, 2019, "The Rise of
    Phantom Investments") -- not treated as a data error or winsorised.

    IMPORTANT, exactly the same distinction already documented for the
    OECD co-invention variable and the RTA activity proxy above: this is
    COUNTRY-level (how much does country i invest abroad in total,
    relative to its own economy), not BILATERAL (how much does i invest
    specifically in j). It CANNOT populate the genuine dyadic "fdi_count"/
    "fdi_value" slot load_fdi() above expects from true bilateral data
    (fDi Markets, IMF CDIS) -- knowing a country's total outward
    investment intensity does not reveal how much of it goes to any one
    partner. Used here, honestly labelled, as a dyadic CONTROL proxy only:
    the pair mean and pair minimum of each country's FDI-outflow
    intensity, representing how internationally financially engaged the
    pair is on average / at the weaker-linked partner -- not a stand-in
    for the missing genuine bilateral "FDI as an outcome" check, which
    still requires load_fdi()'s real bilateral data to be populated.

    Returns (DataFrame [iso, year, fdi_outflow_pct_gdp] for this club's
    members, provenance dict) or (None, None) if the file is absent.
    """
    p = DATA / "fdi_outflows_pct_gdp.csv"
    if not p.exists():
        return None, None
    raw = pd.read_csv(p)
    raw = raw[(raw.year >= YEAR_MIN) & (raw.year <= YEAR_MAX)]
    members = set(club["members_map"].keys())

    def to_club_code(iso):
        return EU_CH3_CODE_MAP.get(iso, iso) if club.get("connector") == "eurostat" else iso

    raw["iso"] = raw["iso"].map(to_club_code)
    df = raw[raw.iso.isin(members)][["iso", "year", "fdi_outflow_pct_gdp"]].copy()
    n_matched = df["iso"].nunique()
    n_expected = len(members)
    prov = {"variable": "FDI outflow-intensity proxy (country-level)",
           "source": f"REAL World Bank data (BM.KLT.DINV.WD.GD.ZS, net FDI outflows %% GDP) "
                    f"({p}); {n_matched}/{n_expected} club members matched"
                    + (f" -- {n_expected - n_matched} member(s) missing from this series, left as "
                       f"NaN, not zero-filled" if n_matched < n_expected else "")
                    + ". NOT a substitute for true bilateral FDI flow data (still absent unless "
                    "load_fdi() finds a real bilateral file) -- used only as an explicitly "
                    "labelled dyadic CONTROL proxy (pair mean / pair minimum of country-level "
                    "outflow intensity), never as the bilateral FDI outcome itself.",
           "status": "LOADED (real, country-level proxy only)",
           "citation": "World Bank World Development Indicators, BM.KLT.DINV.WD.GD.ZS."}
    return df, prov


def load_gdelt_agreement_activity(club):
    """
    TASK: real GDELT event data was supplied (data/gdelt_agreement_events.csv),
    built by the user from GDELT 1.0 (Google BigQuery gdelt-bq.full.events),
    counting CAMEO EventCode 057 ("Sign formal agreement") events where a
    panel country appears as Actor1 or Actor2 -- genuinely full G20
    coverage (19/19) and near-full EU-22 coverage (21/22), spanning the
    complete 2000-2023 window (the only variable in this entire pipeline
    that does), with the source's OWN documented data-quality diagnosis
    carried forward here rather than re-discovered or ignored:
      - Romania is flagged UNRELIABLE by the source itself (implausibly
        low counts, zero after 2016, inconsistent with comparable
        neighbours) -- kept in the data (not silently dropped) but a
        romania_unreliable_flag column travels with it, and every run
        that uses this variable prints an explicit warning when Romania
        observations are included, rather than presenting Romania's
        numbers as equally trustworthy.
      - Slovenia is absent, not fabricated: the source found no reliable
        series (SVN/SLO returned nothing; SLV was actually El Salvador's
        series, confirmed via a 2019 event-profile mismatch, and
        correctly excluded rather than misattributed).
    Per the source's own explicit methodological instruction, the
    PER-100K-normalised column is used (raw counts are not comparable
    across years because GDELT's total event volume grew sharply over
    the period from expanding media coverage; per-100k divides out that
    trend), never the raw count.

    IMPORTANT, the same distinction already documented for OECD
    co-invention, the RTA activity proxy, and the FDI outflow-intensity
    proxy above: this is COUNTRY-level (how many formal-agreement events
    did country i appear in), not DYADIC (which specific country did i
    sign with). It is used here, honestly labelled, as a dyadic CONTROL
    proxy only (pair mean / pair minimum of per-100k activity) -- not a
    substitute for genuine bilateral cooperation-event data, and
    critically NOT a like-for-like measure of ENERGY cooperation
    specifically: CAMEO 057 counts formal agreements of any kind (trade,
    diplomatic, security, etc.), so a high value reflects a country's
    general diplomatic/agreement-signing activity rather than energy
    cooperation in particular.

    Returns (DataFrame [iso, year, per_100k, romania_unreliable_flag] for
    this club's members, provenance dict) or (None, None) if absent.
    """
    p = DATA / "gdelt_agreement_events.csv"
    if not p.exists():
        return None, None
    raw = pd.read_csv(p)
    raw = raw[(raw.year >= YEAR_MIN) & (raw.year <= YEAR_MAX)]
    members = set(club["members_map"].keys())

    def to_club_code(iso):
        return EU_CH3_CODE_MAP.get(iso, iso) if club.get("connector") == "eurostat" else iso

    raw["iso"] = raw["iso"].map(to_club_code)
    df = raw[raw.iso.isin(members)][["iso", "year", "per_100k", "romania_unreliable_flag"]].copy()
    n_matched = df["iso"].nunique()
    n_expected = len(members)
    n_romania_rows = int(df["romania_unreliable_flag"].sum())
    if n_romania_rows > 0:
        print(f"  [GDELT] WARNING: {n_romania_rows} Romania dyad-year rows included in this "
             f"variable's construction -- the SOURCE ITSELF flags Romania's GDELT agreement-event "
             f"count as unreliable (undercoded, near-zero after 2016). Not excluded automatically "
             f"(a modelling choice, not a data-cleaning one), but never silently trusted either.")
    prov = {"variable": "GDELT agreement-activity proxy (country-level)",
           "source": f"REAL GDELT 1.0 event data (CAMEO 057 'sign formal agreement', per-100k "
                    f"normalised) ({p}); {n_matched}/{n_expected} club members matched"
                    + (f" -- {n_expected - n_matched} member(s) missing (Slovenia: no reliable "
                       f"GDELT series found by the source, see load_gdelt_agreement_activity() "
                       f"docstring)" if n_matched < n_expected else "")
                    + ". NOT energy-specific (counts formal agreements of any kind) and NOT "
                    "dyadic (country-level activity, not who-signed-with-whom) -- used only as an "
                    "explicitly labelled dyadic CONTROL proxy (pair mean / pair minimum).",
           "status": "LOADED (real, country-level proxy only)",
           "citation": "Leetaru, K. & Schrodt, P.A. (2013). GDELT: Global Data on Events, "
                      "Location and Tone. ISA Annual Convention."}
    return df, prov


def load_unts_bilateral_treaties(club):
    """
     TASK: real, GENUINELY DYADIC bilateral treaty data, built by the user
    from the UN Treaty Series (UNTS) Cumulative Indexes Nos. 38-54
    (covering treaties registered with the UN Secretariat under Article
    102 of the UN Charter, concluded 2000-2012, with 2012 only partially
    covered and Index No. 50 -- roughly volumes 2651-2700,mainly 2008-
    2010 conclusions -- missing from the source). UNLIKE the RTA/FDI/
    GDELT proxies elsewhere in this file, this is NOT a country-level
    construct requiring a pair-mean/pair-minimum approximation: each
    treaty's entry names its specific parties, so a treaty between two
    named panel countries is genuine dyadic evidence of cooperation
    between THOSE two countries specifically.

    Construction, done here rather than trusting a pre-aggregated file
    blindly: treaties are kept only if they have 2-5 identified panel-
    country parties after removing "EU*" (the EU-as-party marker, not
    attributable to a specific member per the source's own methodology
    note); every pairwise combination of parties on a kept treaty
    contributes one count to that pair's (iso_i, iso_j, year) total.
    Treaties with 6+ panel parties are EXCLUDED from this pairwise count,
    not included -- checked directly, not assumed: the four such cases in
    the source (6-27 parties) are the 2003 EU Accession Treaty, the 2009
    Lisbon Treaty, and two narrower Nordic/Baltic conventions, all broad
    multilateral instruments rather than bilateral cooperation; naively
    pairing all C(27,2)=351 combinations from one Lisbon Treaty
    registration would swamp genuine bilateral signings entirely.

    IMPORTANT LIMITATION, stated directly: coverage is 2000-2012 only,
    which is a real, meaningful window (overlapping this pipeline's real-
    ES-covered years from 2007) but does NOT extend through 2023 the way
    CORDIS does for the primary events outcome. Registration practice
    also varies by country (some register nearly all bilateral agreements
    with the UN, others far fewer), so cross-country level differences
    partly reflect registration diligence, not only treaty-making
    activity -- a caveat from the source's own documentation, carried
    forward rather than smoothed over.

    Returns (DataFrame [iso_i, iso_j, year, unts_treaty_count] for this
    club's members, provenance dict) or (None, None) if absent.
    """
    p = DATA / "unts_bilateral_treaties.csv"
    if not p.exists():
        return None, None
    raw = pd.read_csv(p)
    members = set(club["members_map"].keys())

    def to_club_code(iso):
        return EU_CH3_CODE_MAP.get(iso, iso) if club.get("connector") == "eurostat" else iso

    raw["iso_i"] = raw["iso_i"].map(to_club_code)
    raw["iso_j"] = raw["iso_j"].map(to_club_code)
    df = raw[raw.iso_i.isin(members) & raw.iso_j.isin(members)].copy()
    df = _to_panel_order(df, club)
    n_dyads_matched = df.groupby(["iso_i", "iso_j"]).ngroups
    prov = {"variable": "UNTS bilateral treaty count (genuine dyadic)",
           "source": f"REAL UN Treaty Series data, genuinely dyadic (named treaty parties, not a "
                    f"country-level aggregate) ({p}); {n_dyads_matched} matched dyads with at "
                    f"least one registered treaty, 2000-2012 only (2012 partial; Index No. 50 -- "
                    f"~2008-2010 -- missing from the source's own coverage). Registration "
                    f"diligence varies by country (source's own caveat).",
           "status": "LOADED (real, genuinely dyadic, 2000-2012 only)",
           "citation": "United Nations Treaty Series, Cumulative Indexes Nos. 38-54 "
                      "(UNTS volumes 2051-2900), registered under UN Charter Article 102."}
    return df, prov


def load_trade_openness_country_level(club):
    """
    TASK: real country-level total trade data supplied by the user
    (total_trade_USD.txt, total_trade_pct_GDP.txt), full coverage for
    both clubs (19/19 G20, 22/22 EU-22). Like RTA/FDI/GDELT above, this is
    COUNTRY-level (a total for country i), not bilateral trade FLOWS
    between specific pairs, so it cannot become the "trade" variable
    load_trade() expects from genuine bilateral data (still absent unless
    a real UN Comtrade file is supplied) -- used here only as an
    explicitly labelled dyadic CONTROL proxy (pair mean of trade
    openness), never as a replacement.

    An open question is flagged here rather than resolved silently: the
    magnitude of the %-of-GDP series does not match standard "trade
    (%% of GDP)" figures as commonly reported (e.g. World Bank
    NE.TRD.GNFS.ZS). Checked directly, not assumed -- cross-validating the
    two supplied files against each other (implied GDP = USD value /
    (pct/100)) reproduces Germany's actual GDP almost exactly in 2000,
    2010 and 2020, confirming the two files ARE mutually consistent and
    genuinely real. But the resulting trade-openness LEVEL is roughly
    10-15x smaller than standard total-goods-and-services-trade/GDP
    figures would suggest (e.g. this series gives Germany ~4-5%% in
    2010-2020 where the standard WDI series shows ~80%%). This is
    consistent with the data representing a NARROWER trade category
    (plausibly energy-specific, given this dissertation's focus, though
    not confirmed) rather than total merchandise-and-services trade.
    Labelled here as "trade openness (category unconfirmed)" rather than
    asserting either interpretation; the person supplying this data
    should confirm which WDI/UN Comtrade series it is before it is
    described as "total trade" in any written results.

    Returns (DataFrame [iso, year, trade_usd, trade_pct_gdp] for this
    club's members, provenance dict) or (None, None) if absent.
    """
    p = DATA / "total_trade.csv"
    if not p.exists():
        return None, None
    raw = pd.read_csv(p)
    raw = raw[(raw.year >= YEAR_MIN) & (raw.year <= YEAR_MAX)]
    members = set(club["members_map"].keys())

    def to_club_code(iso):
        return EU_CH3_CODE_MAP.get(iso, iso) if club.get("connector") == "eurostat" else iso

    raw["iso"] = raw["iso"].map(to_club_code)
    df = raw[raw.iso.isin(members)][["iso", "year", "trade_usd", "trade_pct_gdp"]].copy()
    n_matched_usd = df.loc[df["trade_usd"].notna(), "iso"].nunique()
    n_matched_pct = df.loc[df["trade_pct_gdp"].notna(), "iso"].nunique()
    n_expected = len(members)
    prov = {"variable": "Trade openness proxy (country-level, category unconfirmed)",
           "source": f"REAL country-level trade data supplied by the user (total_trade_USD.txt, "
                    f"total_trade_pct_GDP.txt); {n_matched_usd}/{n_expected} members with USD "
                    f"coverage, {n_matched_pct}/{n_expected} with %%-of-GDP coverage (Argentina "
                    f"absent from the %%-of-GDP file specifically). UNIT/CATEGORY FLAG: magnitudes "
                    f"are ~10-15x smaller than standard total-trade/GDP series, suggesting a "
                    f"narrower category (possibly energy-specific) rather than total "
                    f"merchandise-and-services trade -- not confirmed, stated explicitly rather "
                    f"than asserted either way. NOT bilateral (country-level only) -- used as an "
                    f"explicitly labelled dyadic CONTROL proxy (pair mean), not a substitute for "
                    f"genuine bilateral trade flows.",
           "status": "LOADED (real, country-level proxy only, category unconfirmed)",
           "citation": "User-supplied; likely WDI or UN Comtrade derived (exact series unconfirmed)."}
    return df, prov


def load_cordis_dyadic_real(club):
    """
    Loads data/cordis_dyadic_real.csv -- a REAL, already-aggregated dyadic
    cooperation panel built from the actual CORDIS FP7 (2007-2013) and
    H2020 (2014-2023, by project end date) bulk project/organisation
    files, via KTH-Library/cordis-data's GitHub-hosted Parquet mirror
    (github.com/KTH-Library/cordis-data). This is genuine data, not a
    placeholder: 25,667 FP7 + 35,371 H2020 real projects were processed,
    yielding 9,122 (country-pair, year) rows across 692 unique dyads for
    2007-2023 (re-verified directly against the current file, not assumed
    -- the period was widened from an earlier 2007-2021 extract when this
    pipeline's YEAR_MAX was extended to 2023), counted as the number of
    Framework Programme projects with at least one participating
    organisation from each of two countries.

    Checked BEFORE build_cordis_dyadic_cooperation's raw-file path below:
    if this precomputed file is present, no raw CORDIS bulk files are
    needed at all. Returns (DataFrame [iso_i, iso_j, year, cordis_projects]
    in this club's own codes, provenance dict) or (None, None).

    IMPORTANT ESTIMAND CAVEAT (reviewer point 2): CORDIS is genuinely
    dyadic and extends past 2012, which is a real advance over a country-
    level proxy, but it is NOT an institutionally neutral basis for
    comparing the EU's and G20's overall cooperation intensity. It records
    EU-funded Framework Programme participation specifically, so EU dyads
    are structurally far more likely to generate a real observation than
    most G20 pairs, independent of any true difference in cooperation
    levels. It is defensible as an EU cooperation outcome on its own, and
    as a robustness sample for internationally-participating G20
    economies, but the pooled G20-EU comparison it feeds should be
    described as cooperation WITHIN THE CORDIS/EU RESEARCH-PROGRAMME
    ENVIRONMENT, not general bilateral energy cooperation -- see the
    caveat printed in run_pooled_comparison() where this matters most.
    """
    p = DATA / "cordis_dyadic_real.csv"
    if not p.exists():
        return None, None
    df = pd.read_csv(p)

    def to_club_code(iso):
        return EU_CH3_CODE_MAP.get(iso, iso) if club.get("connector") == "eurostat" else iso
    df = df.copy()
    df["iso_i"] = df["iso_i"].map(to_club_code)
    df["iso_j"] = df["iso_j"].map(to_club_code)
    members_list = list(club["members_map"].keys())
    members = set(members_list)
    df = df[df.iso_i.isin(members) & df.iso_j.isin(members)]
    # FIX (reviewer, critical, confirmed by direct count: 19/231 EU pairs
    # conflict, 456 affected pair-years across the configured period):
    # _canon() alphabetically reorders iso_i/iso_j, but the panel's own
    # dyad_id uses combinations(members, 2) -- dictionary insertion order,
    # NOT alphabetical (the exact same mismatch already found and fixed
    # for build_rta_eu_membership and build_rta_wto_signatory_based
    # earlier in this file; missed here until now). A pair like
    # (DNK, DEU_EU) sorts as (DEU_EU, DNK) under _canon() but the panel's
    # own combinations() order keeps it as (DNK, DEU_EU) -- real CORDIS
    # observations for every such pair silently failed to match and were
    # then zero-filled, misreported as "no cooperation" rather than
    # merged. Fixed by reordering to the panel's own canonical order
    # directly instead of alphabetically.
    canonical_order = {frozenset(pair): pair for pair in combinations(members_list, 2)}

    def _reorder(row):
        key = frozenset((row["iso_i"], row["iso_j"]))
        correct = canonical_order.get(key)
        return correct if correct is not None else (row["iso_i"], row["iso_j"])

    reordered = df.apply(_reorder, axis=1, result_type="expand")
    if len(reordered) > 0:
        df["iso_i"], df["iso_j"] = reordered[0], reordered[1]
    df = df[(df.year >= YEAR_MIN) & (df.year <= YEAR_MAX)]
    if df.empty:
        return None, None
    prov = {"variable": "cooperation (CORDIS FP7+H2020 co-participation)",
           "source": f"REAL, precomputed from actual CORDIS bulk data ({p}); "
                     f"25,667 FP7 + 35,371 H2020 projects processed via "
                     f"github.com/KTH-Library/cordis-data",
           "status": "LOADED (real, EU-centred)",
           "citation": "European Commission, CORDIS (cordis.europa.eu); KTH-Library/cordis-data GitHub mirror"}
    return df, prov


def build_cordis_dyadic_cooperation(club):
    """
    CORDIS Horizon project co-participation -- EU-specific robustness
    cooperation measure. Counts, for each country pair and year, the
    number of EU Framework Programme (Horizon Europe / H2020) projects in
    which BOTH countries have at least one participating organisation.

    REAL DATA, not fetched in this session (files run to 10,000+ rows,
    too large for a reliable text-based web-fetch here):
      1. Download the Horizon Europe or H2020 bulk CSVs from
         https://cordis.europa.eu/projects (or directly:
         http://cordis.europa.eu/data/cordis-h2020projects.csv and the
         matching organizations file, e.g. cordis-h2020organizations.csv)
      2. Save the projects file as data/cordis_projects.csv and the
         organisations file as data/cordis_organizations.csv

    Expected columns (auto-detected with reasonable name variants):
      projects file:  a project id column, a start-date/year column
      organisations file: a project id column, a country column (ISO2
        or ISO3 -- both handled)

    LIMITATION (stated plainly, matching the source review): this measure
    is EU-centred by construction. It captures a non-EU G20 country's
    cooperation with the EU only when that country's organisations
    participate in an EU Framework Programme project -- it says nothing
    about, e.g., purely bilateral G20 cooperation outside the EU's own
    programmes. Use it as the EU club's robustness variable, not as the
    G20 club's primary cooperation measure.

    Returns a DataFrame [iso_i, iso_j, year, cordis_projects] for this
    club's members, or None if the required files aren't present.
    """
    p_proj, p_org = DATA / "cordis_projects.csv", DATA / "cordis_organizations.csv"
    if not (p_proj.exists() and p_org.exists()):
        return None, None

    proj = pd.read_csv(p_proj, sep=None, engine="python")
    org = pd.read_csv(p_org, sep=None, engine="python")

    proj_id_p = _find_col(proj, ["id", "projectid", "rcn", "project_id"], "project id (projects file)")
    date_col = _find_col(proj, ["startdate", "start_date", "startDate"], "project start date")
    proj["_year"] = pd.to_datetime(proj[date_col], errors="coerce").dt.year

    org_id_o = _find_col(org, ["projectid", "id", "project_id", "rcn"], "project id (organisations file)")
    country_col = _find_col(org, ["country", "countrycode", "country_code"], "organisation country")

    merged = org[[org_id_o, country_col]].rename(columns={org_id_o: "_pid", country_col: "_country"})
    merged = merged.merge(proj[[proj_id_p, "_year"]].rename(columns={proj_id_p: "_pid"}), on="_pid", how="inner")
    merged = merged.dropna(subset=["_country", "_year"])
    merged["_year"] = merged["_year"].astype(int)

    # ISO2 -> ISO3 for the handful of country-code schemes CORDIS has used
    # historically; extend this map if a country code doesn't resolve.
    iso2_to_iso3 = {"AT":"AUT","BE":"BEL","BG":"BGR","CZ":"CZE","DK":"DNK","EE":"EST","FI":"FIN",
                    "FR":"FRA","DE":"DEU","EL":"GRC","GR":"GRC","HU":"HUN","IE":"IRL","IT":"ITA",
                    "LV":"LVA","LT":"LTU","LU":"LUX","MT":"MLT","NL":"NLD","PL":"POL","PT":"PRT",
                    "RO":"ROU","SK":"SVK","SI":"SVN","ES":"ESP","SE":"SWE","HR":"HRV","CY":"CYP",
                    "US":"USA","CN":"CHN","JP":"JPN","KR":"KOR","GB":"GBR","UK":"GBR","IN":"IND",
                    "BR":"BRA","CA":"CAN","AU":"AUS","ZA":"ZAF","TR":"TUR","MX":"MEX","AR":"ARG",
                    "RU":"RUS","SA":"SAU","ID":"IDN"}
    merged["_iso3"] = merged["_country"].astype(str).str.upper().map(
        lambda c: c if len(c) == 3 else iso2_to_iso3.get(c, None))
    merged = merged.dropna(subset=["_iso3"])

    def to_club_code(iso):
        return EU_CH3_CODE_MAP.get(iso, iso) if club.get("connector") == "eurostat" else iso
    merged["_iso3"] = merged["_iso3"].map(to_club_code)

    members = set(club["members_map"].keys())
    rows = []
    for (pid, yr), g in merged.groupby(["_pid", "_year"]):
        countries = sorted(set(g["_iso3"]) & members)
        for i, j in combinations(countries, 2):
            rows.append({"iso_i": i, "iso_j": j, "year": yr})
    if not rows:
        return None, None
    dyadic = pd.DataFrame(rows).groupby(["iso_i", "iso_j", "year"]).size().reset_index(name="cordis_projects")
    dyadic = _to_panel_order(dyadic, club)
    dyadic = dyadic[(dyadic.year >= YEAR_MIN) & (dyadic.year <= YEAR_MAX)]
    prov = {"variable": "cooperation (CORDIS Horizon co-participation)",
           "source": f"CORDIS bulk data, EU Open Data Portal ({p_proj.name}, {p_org.name})",
           "status": "LOADED (real, EU-centred)",
           "citation": "European Commission, CORDIS (cordis.europa.eu)"}
    return dyadic, prov


def load_oecd_coinvention(club):
    """
    OECD (green) patent co-invention -- PRIMARY cooperation measure in the
    layered strategy. No specific OECD bulk-file or API URL was verified
    in this session (OECD's patent/co-invention statistics are split
    across REGPAT, PATSTAT-derived tables, and the OECD Data Explorer,
    and none of these were confirmed reachable/parseable here) -- this is
    therefore schema-only pending that follow-up.

    Populate data/oecd_coinvention.csv with columns:
      iso_i, iso_j, year, coinvent_count[, green_coinvent_count]
    (a dyadic count of patents with co-inventors resident in both
    countries; green_coinvent_count restricts this to environment-related
    technology classes, per OECD's environmental-technology patent
    statistics -- the single most theoretically direct cooperation
    measure for an energy-security thesis, per the source review this
    section implements).
    """
    p = DATA / "oecd_coinvention.csv"
    if not p.exists():
        return None, None
    df = pd.read_csv(p)
    if "coinvent_count" not in df.columns or df["coinvent_count"].isna().all():
        return None, None
    df = df.copy()

    # FIX: the file is expected in plain ISO3 (matching OECD's own country
    # coding); this club's internal codes may differ (EU: FRA_EU/DEU_EU/
    # ITA_EU). Without this translation, EU's per-club merge would never
    # match a plain "FRA"/"DEU" row at all -- and without filtering to this
    # club's own members, an unfiltered file leaks non-member dyads into
    # both clubs and duplicates real observations when later pooled
    # (Section 8B), since the same unfiltered row would match G20 AND EU.
    def to_club_code(iso):
        return EU_CH3_CODE_MAP.get(iso, iso) if club.get("connector") == "eurostat" else iso
    df["iso_i"] = df["iso_i"].map(to_club_code)
    df["iso_j"] = df["iso_j"].map(to_club_code)
    members = set(club["members_map"].keys())
    df = df[df.iso_i.isin(members) & df.iso_j.isin(members)]
    if df.empty:
        return None, None

    df = _to_panel_order(df, club)
    df = df[(df.year >= YEAR_MIN) & (df.year <= YEAR_MAX)]
    prov = {"variable": "cooperation (OECD patent co-invention)",
           "source": f"OECD patent co-invention statistics ({p})",
           "status": "LOADED (real)",
           "citation": "OECD patent statistics (REGPAT / co-invention indicators)"}
    return df, prov


def load_cooperation_layered(club):
    """
    """
    oecd_df, oecd_prov = load_oecd_coinvention(club)
    if oecd_df is not None:
        val_col = "green_coinvent_count" if "green_coinvent_count" in oecd_df.columns else "coinvent_count"
        oecd_df = oecd_df.rename(columns={val_col: "coop_count"})
        return oecd_df[["iso_i", "iso_j", "year", "coop_count"]], oecd_prov

    cordis_real_df, cordis_real_prov = load_cordis_dyadic_real(club)
    if cordis_real_df is not None:
        cordis_real_df = cordis_real_df.rename(columns={"cordis_projects": "coop_count"})
        return cordis_real_df[["iso_i", "iso_j", "year", "coop_count"]], cordis_real_prov

    cordis_df, cordis_prov = build_cordis_dyadic_cooperation(club)
    if cordis_df is not None:
        cordis_df = cordis_df.rename(columns={"cordis_projects": "coop_count"})
        return cordis_df[["iso_i", "iso_j", "year", "coop_count"]], cordis_prov

    return load_cooperation(club)




def load_trade(club):
    """
    Loads data/trade.csv (G20) or data/eu_trade.csv (EU), now populated
    with REAL UN Comtrade bilateral merchandise export data (built by the
    user from two UN Comtrade download sets, 2000-2023, all 44 panel
    economies -- see the source's own Notes sheet for the exact
    construction and the 115 remaining USA-destination gaps, zero-filled
    per the source's own explicit instruction that Comtrade non-reporting
    represents a genuine absent/negligible flow).

    FIX: this function used to apply _canon() (alphabetical iso_i/iso_j
    reordering) to whatever file it found -- harmless while no real trade
    file existed to expose it, but WRONG now: build_panel's own dyad_id
    does not use alphabetical order (confirmed and fixed for this exact
    reason twice already in this project, for build_rta_eu_membership and
    build_rta_wto_signatory_based). The supplied file is already written
    in the panel's own combinations(members, 2) order; re-sorting it here
    reintroduced the identical 19-dyad-year mismatch (e.g. DNK_DEU_EU)
    both those earlier fixes had already resolved elsewhere. Not
    reordering here at all is the correct fix, matching those precedents.

    Returns (DataFrame [iso_i, iso_j, year, trade], provenance dict) or
    (None, None) if the file is absent or empty.
    """
    print("\n\n\n>>> In load_trade club[trade_file]:", club["trade_file"])

    p = DATA / club["trade_file"]
    if p.exists():
        df0 = pd.read_csv(p)
        if "trade" in df0 and df0["trade"].notna().any():
            n_dyads = df0.groupby(["iso_i", "iso_j"]).ngroups
            n_zero = (df0["trade"] == 0).sum()
            return df0, {"variable": "trade",
                        "source": f"REAL UN Comtrade bilateral merchandise export data "
                                 f"(total of both reported directions) ({p}); {n_dyads} dyads, "
                                 f"{len(df0)} dyad-years, {n_zero} zero-filled (Comtrade "
                                 f"non-reporting, per the source's own instruction that this "
                                 f"represents a genuine absent/negligible flow, not missing data).",
                        "status": "LOADED (real, genuinely bilateral)",
                        "citation": "UN Comtrade, annual total merchandise exports (flow X, "
                                   "commodity TOTAL), current US$."}
    return None, None


def load_fdi(club):
    print("\n\n\n>>> In load_fdi club[fdi_file]:", club["fdi_file"])
    if club["fdi_file"] is None:
        return None, None
    p = DATA / club["fdi_file"]
    if p.exists():
        _df = pd.read_csv(p)
        _val = "fdi_count" if "fdi_count" in _df else ("fdi_value" if "fdi_value" in _df else None)
        if _val and _df[_val].notna().any():
            # FIX (reviewer): "Net bilateral FDI flows can be negative,
            # whereas PPML requires a non-negative dependent variable." A
            # divestment/repatriation year can genuinely make net FDI
            # negative -- PPML (and the log1p transform used for the
            # FE-OLS benchmark) are undefined/invalid on negative values.
            # This is a real data-validity issue, not silently patched:
            # flagged explicitly, and greenfield PROJECT COUNTS (fdi_count)
            # are structurally non-negative by construction and unaffected
            # -- only a net FDI VALUE series (fdi_value) can trigger this.
            n_neg = int((_df[_val] < 0).sum())
            if n_neg > 0:
                print(f"  [FDI] WARNING: {n_neg}/{len(_df)} rows of '{_val}' are NEGATIVE "
                     f"(net divestment/repatriation years). PPML requires a non-negative "
                     f"outcome -- greenfield PROJECT COUNTS do not have this problem, but a "
                     f"net FDI VALUE series does. These rows are dropped here (NOT clipped to "
                     f"zero, which would fabricate a false floor); use a project-count series "
                     f"instead if this drops too much of the sample.")
                _df = _df[_df[_val] >= 0].copy()
            return _to_panel_order(_df, club), {"variable": "FDI", "source": f"fDi Markets / IMF CDIS ({p})",
                                 "status": "LOADED", "citation": "fDi Intelligence (FT); IMF CDIS"}
    if USE_IMF_CDIS and TRY_ONLINE:
        try:
            raw = fetch_imf_cdis_bilateral()
            back = {v: k for k, v in club["members_map"].items()}
            raw["iso_i"] = raw["ref"].map(back); raw["iso_j"] = raw["cp"].map(back)
            raw = _to_panel_order(raw.dropna(subset=["iso_i", "iso_j"]), club)
            df = raw.groupby(["iso_i", "iso_j", "year"])["fdi_value"].sum().reset_index()
            n_neg = int((df["fdi_value"] < 0).sum())
            if n_neg > 0:
                print(f"  [FDI] WARNING: {n_neg}/{len(df)} IMF CDIS rows are negative net "
                     f"positions -- dropped (PPML requires non-negative outcomes), not clipped.")
                df = df[df["fdi_value"] >= 0].copy()
            print("[online] IMF CDIS bilateral FDI fetched:", len(df), "rows")
            print("  [FDI] NOTE (reviewer): OECD/IMF harmonised bilateral FDI (BMD4) is mainly "
                 "reliable from 2014 onward; present any FDI robustness result as a shorter-"
                 "sample check with an explicit coverage table, not a full-period replication.")
            return df, {"variable": "FDI", "source": "IMF CDIS API (bilateral positions)",
                       "status": "LIVE", "citation": "IMF CDIS"}
        except Exception as e:
            print(f"[FALLBACK] IMF CDIS failed ({e}); FDI will be simulated")
    return None, None


def _sim_country_year(members):
    recs = []
    for m in members:
        base, g0 = rng.uniform(0.4, 3.2), rng.uniform(3e11, 2e13)
        for k, y in enumerate(YEARS):
            recs.append((m, int(y), max(0.1, base + 0.02 * k + rng.normal(0, 0.08)),
                         g0 * (1.03 ** k) * rng.uniform(0.95, 1.05), rng.uniform(3e7, 1.4e9)))
    return pd.DataFrame(recs, columns=["iso", "year", "rd_gdp", "gdp_usd", "pop"])


def _linear_fill(vals, floor=0.01):
    """Linear interpolation for interior gaps; linear extrapolation at the
    edges using the slope of the nearest two known points. Same convention
    used throughout the Chapter 3 source this was extracted from."""
    v = np.array([np.nan if x is None else float(x) for x in vals], dtype=float)
    known = np.where(~np.isnan(v))[0]
    if len(known) == 0:
        return np.full_like(v, np.nan)
    if len(known) == 1:
        return np.full_like(v, v[known[0]])
    v_interp = np.interp(np.arange(len(v)), known, v[known])
    k0, k1 = known[0], known[1]
    slope_lo = (v[k1] - v[k0]) / (k1 - k0)
    for j in range(k0):
        v_interp[j] = v[k0] - slope_lo * (k0 - j)
    km1, km0 = known[-2], known[-1]
    slope_hi = (v[km0] - v[km1]) / (km0 - km1)
    for j in range(km0 + 1, len(v)):
        v_interp[j] = v[km0] + slope_hi * (j - km0)
    return v_interp if floor is None else np.clip(v_interp, floor, None)


def _real_gerd_fallback(members):
    """
    REAL offline fallback for R&D/GDP, replacing the previously-simulated
    _sim_country_year() output. Built from GERD_PCT (embedded, real WDI
    GB.XPD.RSDV.GD.ZS, 2007-2024) -- the same source cross-checked earlier
    in this project against an independent direct WDI/OWID fetch (USA 2007
    = 2.61516%, matching exactly). Missing years within a country's own
    series are linearly interpolated/extrapolated (_linear_fill); GDP and
    population are NOT available in this fallback (GERD_PCT covers R&D
    intensity only) and remain the simulated placeholder for those two
    columns specifically -- only rd_gdp is real here.
    """
    recs = []
    club_to_plain = {v: k for k, v in EU_CH3_CODE_MAP.items()}
    covered = []
    for m in members:
        plain = club_to_plain.get(m, m)  # translate FRA_EU->FRA etc.; no-op for G20's plain codes
        if plain in GERD_PCT:
            covered.append(m)
            filled = _linear_fill(GERD_PCT[plain])
            for y, v in zip(CH3_SOURCE_YEARS, filled):
                if YEAR_MIN <= y <= YEAR_MAX:
                    recs.append((m, int(y), float(v)))
        else:
            base = rng.uniform(0.4, 3.2)
            for k, y in enumerate(YEARS):
                recs.append((m, int(y), max(0.1, base + 0.02 * k + rng.normal(0, 0.08))))
    df = pd.DataFrame(recs, columns=["iso", "year", "rd_gdp"])
    # GDP/population still simulated (not in GERD_PCT); merged in separately
    # so callers see a consistent 3-column frame as before.
    sim_gdp_pop = _sim_country_year(members)[["iso", "year", "gdp_usd", "pop"]]
    df = df.merge(sim_gdp_pop, on=["iso", "year"], how="left")
    return df, covered


def load_real_brent_annual():
    """
    Real EIA Brent spot price data supplied by the user (data/
    brent_annual.csv, Europe Brent Spot Price FOB / RBRTE, from
    https://www.eia.gov/dnav/pet/pet_pri_spt_s1_d.htm). ANNUAL frequency,
    confirmed intentional by the user directly (not an upload mistake) --
    checked before that confirmation, not assumed: the file's own embedded
    metadata (Contents sheet: "Data 1 | Crude Oil | ... | Annual") and its
    actual date spacing (mean 365.26 days between consecutive
    observations) both show annual observations, one per year.

    This shapes what can honestly be computed. "Volatility" normally means
    dispersion of RETURNS WITHIN a period (e.g. the standard deviation of
    monthly log returns within a year) -- that requires sub-annual
    observations and cannot be computed from one price point per year.
    What CAN be built from annual-only data, and what is built here, is a
    lower-frequency substitute: the rolling 3-year trailing standard
    deviation of YEAR-OVER-YEAR log returns, a genuine measure of
    multi-year price turbulence. This is not the same statistic as
    intra-year volatility, and this function's provenance says so
    explicitly -- but it is real, and it behaves sensibly (a clear spike
    around 2020-2022, matching the COVID and Ukraine-war oil shocks).

    Returns a DataFrame [year, price_volatility] or None if the file is
    absent.
    """
    p = DATA / "brent_annual.csv"
    if not p.exists():
        return None
    raw = pd.read_csv(p)
    raw = raw[(raw.year >= YEAR_MIN) & (raw.year <= YEAR_MAX)][["year", "price_volatility"]]
    n_covered = raw["price_volatility"].notna().sum()
    print(f"  [Brent, real annual EIA data] {n_covered}/{len(YEARS)} years covered by a "
         f"rolling 3-year trailing std of annual log returns (a multi-year price-turbulence "
         f"measure, distinct from intra-year/monthly volatility -- see "
         f"load_real_brent_annual() docstring).")
    return raw



def _sim_volatility():
    v = np.abs(0.25 + 0.1 * rng.standard_normal(len(YEARS)).cumsum() / 5) + 0.15
    v[rng.choice(len(YEARS), 3, replace=False)] *= rng.uniform(1.8, 2.6, 3)
    return pd.DataFrame({"year": YEARS, "price_volatility": v})


# =========================================================================
# SECTION 6 ? PANEL ASSEMBLY (club-agnostic; driven by CLUBS[club])
# =========================================================================
def merge_real_patents(club_key, cy, prov):
    """
    Adds patents_pc (patent applications by RESIDENTS, per million population)
    from the real Chapter 3 WDI extract as a second, genuine innovation
    signal alongside R&D/GDP -- matching the chapter's own stated
    methodology ("innovation captured via R&D intensity and patent stocks").
    Falls back silently (no column added) if the real data file isn't
    present; downstream code checks for the column's existence.
    """
    real_df = load_real_energy_data(club_key)  # same source file has patents+population
    if real_df is None or "patents_residents" not in real_df.columns:
        return cy
    pat = real_df.dropna(subset=["patents_residents", "population"]).copy()
    pat["patents_pc"] = 1e6 * pat["patents_residents"] / pat["population"]
    cy = cy.merge(pat[["iso", "year", "patents_pc"]], on=["iso", "year"], how="left")
    prov.append({"variable": "patents (residents, per capita)",
                "source": "REAL WDI data (IP.PAT.RESD) extracted from the Chapter 3 companion pipeline",
                "status": "LOADED (real)", "citation": "World Bank WDI IP.PAT.RESD, via Chapter 3 extract"})
    return cy


def merge_oecd_coinvent(club_key, cy, prov):
    """
    Adds coinvent_intensity -- the real, user-provided OECD international
    co-invention country-level indicator (Section 3C) -- as a THIRD real
    Innovation signal alongside R&D/GDP and patents/capita. Country-level
    only (see the Section 3C docstring for why this cannot serve as a
    dyadic cooperation count on its own); legitimate here because
    Innovation is a country-level construct in this chapter's design.
    """
    df = _oecd_coinvent_long()
    if club_key == "eu":
        df = df.copy()
        df["iso"] = df["iso"].map(lambda x: EU_CH3_CODE_MAP.get(x, x))
    cy = cy.merge(df, on=["iso", "year"], how="left")
    prov.append({"variable": "OECD international co-invention intensity",
                "source": f"User-provided OECD data (Section 3C), confirmed as co-inventions "
                          f"in % of patents; {_OECD_COINVENT_EXCLUDED} excluded as all-zero/"
                          f"unreliable, not treated as a real 0",
                "status": "LOADED (real)",
                "citation": "OECD, co-inventions as % of patents (dataset code pending confirmation)"})
    return cy


def build_oecd_coinvention_proxy(club):
    """
    NOT USED as a cooperation outcome anywhere in this pipeline's automatic
    loading chain (load_cooperation_layered) -- retained only for
    reference/illustration. Constructed as sqrt(intensity_i * intensity_j)
    from the real OECD co-invention COUNTRY-LEVEL data (Section 3C).

    WHY THIS WAS REMOVED (reviewer, correctly): this is not "a weaker proxy
    than CORDIS" -- it carries ZERO pair-specific information under any
    interpretation. Algebraically, ln(sqrt(share_i*share_j)) =
    0.5*ln(share_i) + 0.5*ln(share_j) is an ADDITIVE function of two
    country marginals; two countries could each co-invent heavily with
    entirely unrelated third parties and still be assigned a large
    constructed "i-j cooperation" value here. A PPML coefficient estimated
    on this outcome reflects correlation with national co-invention SCALE,
    not bilateral behaviour between i and j specifically -- and since the
    same OECD series also enters the Innovation regressor
    (merge_oecd_coinvent), any "Innovation predicts cooperation" result
    risked partly reading as the OECD data predicting a function of itself.

    Kept in the file, callable directly, only so its output can be
    inspected/demonstrated if useful -- e.g. to show a reader exactly why
    this construction fails, per the docstring above.
    """
    df = _oecd_coinvent_long()
    if club.get("connector") == "eurostat":
        df = df.copy()
        df["iso"] = df["iso"].map(lambda x: EU_CH3_CODE_MAP.get(x, x))
    members = set(club["members_map"].keys())
    df = df[df.iso.isin(members)]
    if df.empty:
        return None, None

    wide = df.pivot(index="year", columns="iso", values="coinvent_intensity")
    rows = []
    for y in wide.index:
        for i, j in combinations(sorted(wide.columns), 2):
            vi, vj = wide.loc[y, i], wide.loc[y, j]
            if pd.isna(vi) or pd.isna(vj):
                continue
            rows.append({"iso_i": i, "iso_j": j, "year": int(y),
                        "coop_count": math.sqrt(max(vi, 0) * max(vj, 0))})
    if not rows:
        return None, None
    out = _to_panel_order(pd.DataFrame(rows), club)
    out = out[(out.year >= YEAR_MIN) & (out.year <= YEAR_MAX)]
    if out.empty or out["coop_count"].sum() == 0:
        return None, None
    prov = {"variable": "cooperation (OECD co-invention PROXY, sqrt(share_i*share_j)) -- NOT USED IN PIPELINE",
           "source": f"Constructed from real OECD co-inventions-%-of-patents data (Section 3C) -- "
                     f"carries ZERO pair-specific information (see docstring); diagnostic/reference "
                     f"only; {_OECD_COINVENT_EXCLUDED} excluded as all-zero/unreliable",
           "status": "NOT A VALID COOPERATION MEASURE (reference only, not called by the pipeline)",
           "citation": "OECD, co-inventions as % of patents (dataset code pending confirmation)"}
    return out, prov


PATENT_STOCK_DELTA = 0.15  # baseline depreciation rate; 0.10/0.20 also computed for robustness


def compute_patent_stock(cy, deltas=(0.10, 0.15, 0.20)):
    """
    TASK (reviewer point 7): perpetual-inventory patent STOCK,
    K_it = (1-delta)*K_i,t-1 + P_it, from the real patents/capita FLOW
    already in cy (patents_pc, from the Chapter 3 WDI extract). This is
    still GENERAL resident patent applications, not the energy-specific
    (OECD/IEA climate-mitigation-technology) patent flow the reviewer
    recommends as the ideal P_it -- that series was not obtained (see the
    chapter's data-provenance notes) -- but the perpetual-inventory
    MACHINERY itself is implemented correctly and is ready to accept an
    energy-specific flow the moment one is supplied, by construction (it
    only needs a per-country-year flow column).
    Initialised at each country's first observed year as K = P / delta
    (standard steady-state perpetual-inventory initialisation), then
    recursed forward year by year. Adds patent_stock_pc_d{10,15,20}.
    """
    if "patents_pc" not in cy.columns:
        return cy
    cy = cy.copy()
    wide = cy.pivot_table(index="iso", columns="year", values="patents_pc")
    years_sorted = sorted(wide.columns)
    for delta in deltas:
        stock = pd.DataFrame(index=wide.index, columns=years_sorted, dtype=float)
        for iso in wide.index:
            k_prev = None
            for y in years_sorted:
                p = wide.loc[iso, y]
                if pd.isna(p):
                    stock.loc[iso, y] = k_prev
                    continue
                if k_prev is None:
                    k_prev = p / delta  # steady-state initialisation at first observed flow
                else:
                    k_prev = (1 - delta) * k_prev + p
                stock.loc[iso, y] = k_prev
        long = stock.reset_index().melt(id_vars="iso", var_name="year", value_name=f"patent_stock_pc_d{int(delta*100)}")
        long["year"] = long["year"].astype(int)
        cy = cy.merge(long, on=["iso", "year"], how="left")
    return cy

###
### IMPORTANT
def get_country_year(club, es, prov, club_key=None):
    members_map = club["members_map"]
    wb = None
    print("TRY_ONLINE", TRY_ONLINE)
    if TRY_ONLINE:
        try:
            if club["connector"] == "worldbank":
                wb = fetch_worldbank(members_map)
                src, cite = "World Bank Indicators API", "WDI GB.XPD.RSDV.GD.ZS; NY.GDP.MKTP.CD; SP.POP.TOTL; NY.GDP.PCAP.PP.KD; NY.GDP.MKTP.KD.ZG"
            elif club["connector"] == "eurostat":
                wb = fetch_eurostat_all(members_map)
                src, cite = "Eurostat API", "tsc00001; nama_10_gdp (B1GQ, CP_MEUR); demo_pjan"
            print(f"[online] {club['label']} country-year data fetched:", len(wb), "rows")
            prov.append({"variable": "R&D/GDP, GDP, pop", "source": src, "status": "LIVE", "citation": cite})
        except Exception as e:
            print ("### Could not download worldbank or eurostat!")
            print(f"[FALLBACK] {club['label']} online fetch failed ({e}); simulating"); wb = None
    if wb is None or wb["rd_gdp"].notna().mean() < 0.2:
        wb, covered = _real_gerd_fallback(list(members_map.keys()))
        if covered:
            prov.append({"variable": "R&D/GDP", "source": f"REAL embedded WDI GERD data ({len(covered)}/"
                        f"{len(members_map)} members covered: {covered}); GDP/population still simulated "
                        f"(not in this source)", "status": "LOADED (real, R&D/GDP only)", "citation":
                        "World Bank WDI GB.XPD.RSDV.GD.ZS, embedded from Chapter 3 extract"})
        else:
            prov.append({"variable": "R&D/GDP, GDP, pop", "source": "simulated (no real GERD coverage for any member)",
                        "status": "SIMULATED", "citation": ""})
    cy = es.merge(wb, on=["iso", "year"], how="left")
    if club_key:
        cy = merge_real_patents(club_key, cy, prov)
        cy = merge_oecd_coinvent(club_key, cy, prov)
        if "patents_pc" in cy.columns:
            cy = compute_patent_stock(cy)
            prov.append({"variable": "patent stock (perpetual inventory)",
                        "source": f"K_it=(1-delta)K_i,t-1+P_it from real patents/capita flow, "
                                  f"delta={PATENT_STOCK_DELTA} baseline (0.10/0.20 also computed); "
                                  f"STILL general resident patents, not yet energy-specific",
                        "status": "COMPUTED (real flow, general not energy-specific)", "citation": ""})
    return cy


def get_distance(club, prov):
    coords = {}
    print (">>> In getdistnce")
    print ("club[has_live_capitals] and TRY_ONLINE", club["has_live_capitals"], TRY_ONLINE)
    if club["has_live_capitals"] and TRY_ONLINE:
        try:
            coords = fetch_capitals_worldbank(club["members_map"])
            print(f"[online] {club['label']} capital coords:", len(coords))
            prov.append({"variable": "distance", "source": "World Bank capital coords -> haversine",
                        "status": "LIVE", "citation": "WB country API"})
            print(">>> coords downloaded...")
        except Exception as e:
            print(f"[FALLBACK] {club['label']} capitals failed ({e}); using built-in capitals")

    if not coords:
        # FIX: this branch's fallback is G20_CAPITALS_FALLBACK / EU_CAPITALS
        # -- real, hardcoded national capital coordinates, not randomly
        # simulated data. Mislabeling it "SIMULATED" whenever the live
        # World Bank fetch isn't available (e.g. offline mode) was a real
        # bug, caught by --strict mode incorrectly halting on a variable
        # that was actually fine. Always FIXED (real data); "has_live_capitals"
        # only controls whether a LIVE fetch is attempted first, not whether
        # the fallback itself is trustworthy.
        status = "FIXED"
        prov.append({"variable": "distance", "source": "built-in capital coords -> haversine",
                    "status": status, "citation": ""})
    for m in club["members_map"]:
        coords.setdefault(m, club["capitals_fallback"][m])
    prov.append({"variable": "language, contiguity", "source": "fixed lookups", "status": "FIXED", "citation": ""})

    rows = []
    members = list(club["members_map"].keys())
    langs, contig = club["langs"], club["contig"]
    for i, j in combinations(members, 2):
        rows.append((i, j, haversine(coords[i], coords[j]),
                     int(len(langs[i] & langs[j]) > 0),
                     int(frozenset((i, j)) in contig)))
    return pd.DataFrame(rows, columns=["iso_i", "iso_j", "distance", "comlang", "contig"])


def get_volatility(prov):
    if TRY_ONLINE:
        try:
            vol = fetch_brent_volatility()
            print("[online] Brent volatility:", len(vol), "yrs")
            prov.append({"variable": "price volatility", "source": "Stooq Brent -> realised vol",
                        "status": "LIVE", "citation": "stooq.com cb.f"})
            if vol["price_volatility"].notna().sum() >= len(YEARS) * 0.5:
                return vol
        except Exception as e:
            print(f"[FALLBACK] Brent failed ({e}); trying real annual EIA data")
    real_annual = load_real_brent_annual()
    if real_annual is not None and real_annual["price_volatility"].notna().sum() >= len(YEARS) * 0.3:
        prov.append({"variable": "price volatility",
                    "source": "REAL EIA Brent spot price (RBRTE), annual frequency (confirmed "
                             "intentional). Rolling 3-year trailing std of annual log returns "
                             "used as the volatility measure -- a genuine multi-year price-"
                             "turbulence statistic, distinct from intra-year/monthly volatility, "
                             "which annual-only data cannot support computing.",
                    "status": "LOADED (real, annual frequency)",
                    "citation": "U.S. Energy Information Administration, Europe Brent Spot Price "
                               "FOB (RBRTE), https://www.eia.gov/dnav/pet/pet_pri_spt_s1_d.htm"})
        return real_annual
    prov.append({"variable": "price volatility", "source": "simulated", "status": "SIMULATED", "citation": ""})
    return _sim_volatility()


def _simulate_negbin(mean, dispersion, rng):
    """
    NB2-parametrised negative binomial: Var = mean + dispersion*mean^2.
    Used for energy_events_sim -- real event-count data (e.g. GDELT-style
    cooperation events) is characteristically overdispersed relative to a
    plain Poisson, and PPML remains consistent under this overdispersion
    (Santos Silva & Tenreyro, 2006) -- so simulating from NB rather than
    Poisson makes the validation DGP more realistic without requiring any
    change to the estimator itself.
    """
    mean = np.clip(mean, 1e-6, None)
    r = 1.0 / max(dispersion, 1e-6)          # NB "size" parameter
    p = r / (r + mean)                        # NB "success probability"
    return rng.negative_binomial(r, p)


def build_panel(club_key):
    print("\n\n\n\n\n\n>>>>> In build_panel")
    club = CLUBS[club_key]
    prov = []
    members = list(club["members_map"].keys())

    print(">>> before compute_es_dea 2nd time, with ",members, club['out_dir'], prov, club_key)
    es = compute_es_dea(members, club["out_dir"], prov, club_key=club_key)
    print("<<<< after compute_es_dea es:", es)
    print("<<<< after compute_es_dea prov: ",prov)
    print(">>>> before get_countr_year", club, es, prov, club_key)
    cy = get_country_year(club, es, prov, club_key=club_key)
    print("<<< after get_country_year cy:", cy)
    print("<<<< after get_country_year cy.columns:", cy.columns)

    print("after get_country_year prov:", prov)
    dyad_static = get_distance(club, prov)
    print("after get_distance dyadstatic:", dyad_static)
    print("after get_distance prov:", prov)
    vol = get_volatility(prov)
    print("after get_volatility", vol)
    coop, coop_prov = load_cooperation_layered(club)
    print("after load_cooperation_layered:", coop, coop_prov)

    rta, rta_prov = build_rta_dummy(club)
    print("after build_Rta_dummy", rta, rta_prov)
    if rta is None:
        rta, rta_prov = build_rta_wto_signatory_based(club)  # genuine dyadic, both G20 and EU

    if rta is None:
        print ("after first rta is None>", rta, rta_prov)
        rta, rta_prov = build_rta_eu_membership(club)  # final fallback: EU-accession-based, EU only

        print ("after second rta is None:", rta, rta_prov)

    rta_activity, rta_activity_prov = load_rta_country_activity(club)
    print(">>> After load_rta_country_activity:", rta_activity, rta_activity_prov)
    fdi_activity, fdi_activity_prov = load_fdi_country_level(club)
    print(">>> After load_fdi_country_level:", fdi_activity, fdi_activity_prov)

    gdelt_activity, gdelt_activity_prov = load_gdelt_agreement_activity(club)
    print("\n\n\n>>> after load_gdelt_agreement...:", gdelt_activity, gdelt_activity_prov)
    unts_treaties, unts_treaties_prov = load_unts_bilateral_treaties(club)
    print("\n\n\n>>>after load_unts_bilateral_treaties:", unts_treaties, unts_treaties_prov)

    trade_openness, trade_openness_prov = load_trade_openness_country_level(club)
    print("\n\n\n>>>after load_trade_openness_country_level",trade_openness, trade_openness_prov )

    trade, trade_prov = load_trade(club)
    print("\n\n\n>>>After load_trade", trade, trade_prov )

    fdi, fdi_prov = load_fdi(club)
    print ("\n\n\n>>>After load_fdi>", fdi, fdi_prov)

    #print("\n\n\n>>> Stopping")
    #sys.exit()


    dyads = pd.DataFrame(list(combinations(members, 2)), columns=["iso_i", "iso_j"])
    f = dyads.merge(pd.DataFrame({"year": YEARS}), how="cross")
    f["dyad_id"] = f.iso_i + "_" + f.iso_j

    li_cols = {"iso": "iso_i", "es": "es_i", "rd_gdp": "rd_i", "gdp_usd": "gdp_i"}
    lj_cols = {"iso": "iso_j", "es": "es_j", "rd_gdp": "rd_j", "gdp_usd": "gdp_j"}
    has_patents = "patents_pc" in cy.columns
    has_coinvent = "coinvent_intensity" in cy.columns
    has_mli = "ml_index" in cy.columns
    has_ectc = "ml_effch" in cy.columns and "ml_techch" in cy.columns
    has_patstock = f"patent_stock_pc_d{int(PATENT_STOCK_DELTA*100)}" in cy.columns
    patstock_col = f"patent_stock_pc_d{int(PATENT_STOCK_DELTA*100)}"
    has_patstock_all = all(f"patent_stock_pc_d{d}" in cy.columns for d in (10, 15, 20))
    print("has_patents:", has_patents)
    print("has_coinvent:", has_coinvent)
    print("has_mli:", has_mli)
    print("has_ectc:", has_ectc)
    print("has_patstock:", has_patstock)


    if has_patents:
        li_cols["patents_pc"] = "pat_i"; lj_cols["patents_pc"] = "pat_j"
    if has_patstock:
        li_cols[patstock_col] = "patstock_i"; lj_cols[patstock_col] = "patstock_j"

    print("has_patstock_all ", has_patstock_all)
    if has_patstock_all:
        for d in (10, 20):  # d15 already covered by has_patstock above
            col = f"patent_stock_pc_d{d}"
            li_cols[col] = f"patstock{d}_i"; lj_cols[col] = f"patstock{d}_j"

    print("has coinvent: ", has_coinvent)
    if has_coinvent:
        li_cols["coinvent_intensity"] = "coinv_i"; lj_cols["coinvent_intensity"] = "coinv_j"

    print("has_mli: ", has_mli)
    if has_mli:
        li_cols["ml_index"] = "mli_i"; lj_cols["ml_index"] = "mli_j"

    print("has_ectc: ", has_ectc)
    if has_ectc:
        li_cols["ml_effch"] = "ec_i"; lj_cols["ml_effch"] = "ec_j"
        li_cols["ml_techch"] = "tc_i"; lj_cols["ml_techch"] = "tc_j"
    li = cy.rename(columns=li_cols)
    lj = cy.rename(columns=lj_cols)
    li_keep = (["iso_i", "year", "es_i", "rd_i", "gdp_i"] + (["pat_i"] if has_patents else [])
              + (["coinv_i"] if has_coinvent else []) + (["mli_i"] if has_mli else [])
              + (["patstock_i"] if has_patstock else [])
              + ([f"patstock{d}_i" for d in (10, 20)] if has_patstock_all else [])
              + (["ec_i", "tc_i"] if has_ectc else []))
    lj_keep = (["iso_j", "year", "es_j", "rd_j", "gdp_j"] + (["pat_j"] if has_patents else [])
              + (["coinv_j"] if has_coinvent else []) + (["mli_j"] if has_mli else [])
              + (["patstock_j"] if has_patstock else [])
              + ([f"patstock{d}_j" for d in (10, 20)] if has_patstock_all else [])
              + (["ec_j", "tc_j"] if has_ectc else []))
    f = f.merge(li[li_keep], on=["iso_i", "year"], how="left")
    f = f.merge(lj[lj_keep], on=["iso_j", "year"], how="left")

    if has_patents:
        # Composite innovation = average of standardised R&D/GDP, patents-
        # per-capita, and (where available) OECD co-invention intensity --
        # all real, when available -- matching the chapter's own stated
        # methodology: "innovation captured via R&D intensity AND patent
        # stocks", extended with a third real signal once it was supplied
        # (Section 3C). Standardise each series before averaging so no
        # component (each on a very different natural scale) mechanically
        # dominates the composite.
        rd_z = zscore(pd.concat([f.rd_i, f.rd_j]))
        n = len(f)
        rd_i_z, rd_j_z = rd_z.iloc[:n].values, rd_z.iloc[n:].values
        pat_z = zscore(pd.concat([f.pat_i, f.pat_j]))
        pat_i_z, pat_j_z = pat_z.iloc[:n].values, pat_z.iloc[n:].values
        comps_i, comps_j = [rd_i_z, pat_i_z], [rd_j_z, pat_j_z]
        if has_coinvent:
            coinv_z = zscore(pd.concat([f.coinv_i, f.coinv_j]))
            comps_i.append(coinv_z.iloc[:n].values)
            comps_j.append(coinv_z.iloc[n:].values)
        f["innov_i"] = np.nanmean(np.column_stack(comps_i), axis=1)
        f["innov_j"] = np.nanmean(np.column_stack(comps_j), axis=1)

        # TASK (reviewer point 6): "the innovation composite averages
        # whatever components happen to be available... this creates a
        # measurement break that could be mistaken for temporal change."
        # Confirmed directly (not assumed): R&D/GDP covers 2007-2023 in
        # full, patents stop after 2021, co-invention after 2022 -- so the
        # SAME variable name silently goes from a 3-component to a
        # 2-component to a 1-component average across the panel with no
        # flag. Four things are added here in response, in the order the
        # reviewer listed them:
        #
        # (a) Effective composition reported explicitly, by year --
        # printed once per club run rather than left implicit.
        n_comps_per_row = 1 + np.asarray(pd.notna(pd.Series(pat_i_z))).astype(int)
        if has_coinvent:
            n_comps_per_row = n_comps_per_row + np.asarray(pd.notna(pd.Series(coinv_z.iloc[:n].values))).astype(int)
        comp_by_year = pd.DataFrame({"year": f["year"].values, "n_components": n_comps_per_row}).groupby("year")["n_components"].agg(["min", "max", "mean"])
        print(f"    [Innovation composite composition by year] (1=R&D only, 2=+patents, "
             f"3=+co-invention; showing min/max/mean components actually averaged per row):")
        for yr, row in comp_by_year.iterrows():
            if row["min"] != row["max"]:
                print(f"      {int(yr)}: MIXED within year, {row['min']:.0f}-{row['max']:.0f} "
                     f"components (mean {row['mean']:.2f})")
        yr_summary = comp_by_year.groupby(comp_by_year["mean"].round(1))
        print(f"      Full range: {int(comp_by_year.index.min())}-{int(comp_by_year.index.max())}; "
             f"typical composition drops from 3 components (through ~2021) to 2 (2022) to "
             f"1 (2023) as patents then co-invention stop -- see table above for any within-year mixing.")

        # (b) A STABLE one-component index: R&D/GDP alone is the only
        # signal available for the pipeline's ENTIRE 2007-2023 real-data
        # window, so it never changes composition across years by
        # construction. This is what "innov_stable1" means throughout --
        # not a claim that R&D alone is a BETTER measure, only a stable
        # one.
        f["innov_stable1_i"], f["innov_stable1_j"] = rd_i_z, rd_j_z

        # (c) A STABLE two-component index: R&D + patents, both available
        # 2007-2021 (their common window) -- NaN outside it rather than
        # silently falling back to one component, so its own composition
        # never changes within its valid range either.
        both_pat_valid = pd.notna(f.pat_i) & pd.notna(f.pat_j)
        f["innov_stable2_i"] = np.where(both_pat_valid, (rd_i_z + pat_i_z) / 2, np.nan)
        f["innov_stable2_j"] = np.where(both_pat_valid, (rd_j_z + pat_j_z) / 2, np.nan)

        # (d) Component-specific series, for component-specific models
        # below (rd_z alone is identical to innov_stable1 and is not
        # duplicated; patents-alone and co-invention-alone are new).
        f["innov_patonly_i"], f["innov_patonly_j"] = pat_i_z, pat_j_z
        if has_coinvent:
            f["innov_coinvonly_i"] = coinv_z.iloc[:n].values
            f["innov_coinvonly_j"] = coinv_z.iloc[n:].values
        else:
            f["innov_coinvonly_i"] = np.nan
            f["innov_coinvonly_j"] = np.nan
    else:
        f["innov_i"], f["innov_j"] = f.rd_i, f.rd_j
        f["innov_stable1_i"], f["innov_stable1_j"] = f.rd_i, f.rd_j
        f["innov_stable2_i"], f["innov_stable2_j"] = np.nan, np.nan
        f["innov_patonly_i"], f["innov_patonly_j"] = np.nan, np.nan
        f["innov_coinvonly_i"], f["innov_coinvonly_j"] = np.nan, np.nan

    f["es_mean"] = (f.es_i + f.es_j) / 2
    f["es_min"] = f[["es_i", "es_j"]].min(axis=1)
    f["es_diff"] = (f.es_i - f.es_j).abs()
    f["innov_mean"] = (f.innov_i + f.innov_j) / 2
    f["innov_min"] = f[["innov_i", "innov_j"]].min(axis=1)
    f["innov_diff"] = (f.innov_i - f.innov_j).abs()
    # TASK (reviewer point 6): pair-mean versions of the stable and
    # component-specific innovation series built above, for the
    # missing-component sensitivity analysis and component-specific
    # models below.
    f["innov_stable1_mean"] = (f.innov_stable1_i + f.innov_stable1_j) / 2
    f["innov_stable2_mean"] = (f.innov_stable2_i + f.innov_stable2_j) / 2
    f["innov_patonly_mean"] = (f.innov_patonly_i + f.innov_patonly_j) / 2
    f["innov_coinvonly_mean"] = (f.innov_coinvonly_i + f.innov_coinvonly_j) / 2

    # TASK: real WTO RTA country-level activity data supplied by the user
    # (data/rta_activity_by_country.csv, full 19/19 G20 and 22/22 EU-22
    # coverage, verified) -- merged here as an explicitly-labelled DYADIC
    # PROXY (pair minimum of cumulative RTA stock), never as a substitute
    # for the true dyadic Larch "rta" dummy above, which remains absent
    # unless data/rta_larch.csv is separately supplied. See
    # load_rta_country_activity()'s docstring for the full reasoning and
    # the sanity checks (EU-bloc-identical values, Croatia's 2013 jump)
    # that were run on this file before trusting it.
    print("rta_activity:", rta_activity)
    if rta_activity is not None:
        ri = rta_activity.rename(columns={"iso": "iso_i", "rta_stock": "rta_stock_i"})[["iso_i", "year", "rta_stock_i"]]
        rj = rta_activity.rename(columns={"iso": "iso_j", "rta_stock": "rta_stock_j"})[["iso_j", "year", "rta_stock_j"]]
        f = f.merge(ri, on=["iso_i", "year"], how="left").merge(rj, on=["iso_j", "year"], how="left")
        f["rta_stock_min"] = f[["rta_stock_i", "rta_stock_j"]].min(axis=1)
        has_rta_activity = True
    else:
        f["rta_stock_min"] = np.nan
        has_rta_activity = False

    # TASK: real World Bank FDI outflow-intensity data (BM.KLT.DINV.WD.GD.ZS)
    # supplied by the user -- merged as an explicitly-labelled dyadic
    # CONTROL proxy (pair mean AND pair minimum of country-level outflow
    # intensity), exactly analogous to the RTA activity proxy above and
    # for the same reason: country-level totals cannot reveal bilateral
    # flows. See load_fdi_country_level()'s docstring for the coverage
    # gap (Turkiye absent from this series, 18/19 not 19/19 for G20) and
    # the genuine extreme-value phenomenon (Cyprus/Malta/Luxembourg
    # conduit FDI) already checked there.
    print("fdi_activity: ", fdi_activity)
    if fdi_activity is not None:
        fi = fdi_activity.rename(columns={"iso": "iso_i", "fdi_outflow_pct_gdp": "fdi_out_i"})[["iso_i", "year", "fdi_out_i"]]
        fj = fdi_activity.rename(columns={"iso": "iso_j", "fdi_outflow_pct_gdp": "fdi_out_j"})[["iso_j", "year", "fdi_out_j"]]
        f = f.merge(fi, on=["iso_i", "year"], how="left").merge(fj, on=["iso_j", "year"], how="left")
        f["fdi_out_pair_mean"] = (f.fdi_out_i + f.fdi_out_j) / 2
        f["fdi_out_pair_min"] = f[["fdi_out_i", "fdi_out_j"]].min(axis=1)
        has_fdi_activity = True
    else:
        f["fdi_out_pair_mean"] = np.nan
        f["fdi_out_pair_min"] = np.nan
        has_fdi_activity = False

    # TASK: real GDELT CAMEO-057 agreement-event activity (per-100k,
    # country-level) supplied by the user -- merged as an explicitly-
    # labelled dyadic CONTROL proxy (pair mean and pair minimum),
    # following exactly the same pattern as RTA and FDI above and for the
    # same reason (country-level totals cannot reveal who-signed-with-
    # whom). Also carries the Romania unreliability flag through to the
    # pair level: a dyad is flagged if EITHER member is Romania, so any
    # model using this variable can report or exclude affected pairs
    # explicitly rather than silently blending unreliable rows in.
    print("gdelt_activity: ", gdelt_activity)
    if gdelt_activity is not None:
        gi = gdelt_activity.rename(columns={"iso": "iso_i", "per_100k": "gdelt_i",
                                            "romania_unreliable_flag": "rou_flag_i"})[["iso_i", "year", "gdelt_i", "rou_flag_i"]]
        gj = gdelt_activity.rename(columns={"iso": "iso_j", "per_100k": "gdelt_j",
                                            "romania_unreliable_flag": "rou_flag_j"})[["iso_j", "year", "gdelt_j", "rou_flag_j"]]
        f = f.merge(gi, on=["iso_i", "year"], how="left").merge(gj, on=["iso_j", "year"], how="left")
        f["gdelt_pair_mean"] = (f.gdelt_i + f.gdelt_j) / 2
        f["gdelt_pair_min"] = f[["gdelt_i", "gdelt_j"]].min(axis=1)
        f["gdelt_romania_dyad"] = ((f["rou_flag_i"] == 1) | (f["rou_flag_j"] == 1)).astype(float)
        has_gdelt_activity = True
    else:
        f["gdelt_pair_mean"] = np.nan
        f["gdelt_pair_min"] = np.nan
        f["gdelt_romania_dyad"] = np.nan
        has_gdelt_activity = False

    # TASK: real, genuinely DYADIC UNTS bilateral treaty data (see
    # load_unts_bilateral_treaties docstring). Merged directly on
    # [iso_i, iso_j, year] (unlike the country-level proxies above, no
    # pair-mean/pair-minimum approximation is needed -- this data already
    # IS dyadic). Coverage window is 2000-2012 only: a dyad-year in that
    # window not appearing in the source is a genuine, informative ZERO
    # (no treaty registered that year, not "unknown"), so it is filled
    # with 0 -- but a dyad-year OUTSIDE 2000-2012 is left as NaN, since
    # this source provides no information about it at all. Conflating
    # these two would either fabricate zeros for 2013-2023 (wrong) or
    # discard genuine zeros within 2000-2012 (equally wrong).
    print("unts_treaties: ", unts_treaties)
    if unts_treaties is not None:
        f = f.merge(unts_treaties[["iso_i", "iso_j", "year", "unts_treaty_count"]],
                   on=["iso_i", "iso_j", "year"], how="left")
        in_coverage_window = (f["year"] >= 2000) & (f["year"] <= 2012)
        f.loc[in_coverage_window & f["unts_treaty_count"].isna(), "unts_treaty_count"] = 0.0
        # rows outside the window keep whatever the left-merge gave them,
        # which is NaN for every such row since unts_treaties has no
        # entries there at all -- explicit for clarity, not relied on implicitly
        f.loc[~in_coverage_window, "unts_treaty_count"] = np.nan
        has_unts_treaties = True
    else:
        f["unts_treaty_count"] = np.nan
        has_unts_treaties = False

    # TASK: real country-level trade openness data supplied by the user --
    # a dyadic CONTROL proxy (pair mean), exactly analogous to RTA/FDI/
    # GDELT above and for the same country-level-not-bilateral reason. See
    # load_trade_openness_country_level()'s docstring for the flagged,
    # unresolved unit/category question (magnitudes suggest a narrower
    # category than total merchandise trade, possibly energy-specific).
    print("trade_openess: ", trade_openness)
    if trade_openness is not None:
        toi = trade_openness.rename(columns={"iso": "iso_i", "trade_pct_gdp": "tradeopen_i"})[["iso_i", "year", "tradeopen_i"]]
        toj = trade_openness.rename(columns={"iso": "iso_j", "trade_pct_gdp": "tradeopen_j"})[["iso_j", "year", "tradeopen_j"]]
        f = f.merge(toi, on=["iso_i", "year"], how="left").merge(toj, on=["iso_j", "year"], how="left")
        f["tradeopen_pair_mean"] = (f.tradeopen_i + f.tradeopen_j) / 2
        has_trade_openness = True
    else:
        f["tradeopen_pair_mean"] = np.nan
        has_trade_openness = False

    # TASK (reviewer point 6): "Use ES improvement -- not only the ES
    # level." LevelES = the DEA score itself (es_mean above, a STOCK);
    # ImprovementES = ln(MLI), the year-over-year Malmquist-Luenberger
    # productivity CHANGE -- a genuinely distinct quantity, not a
    # transformation of the level. Lag alignment is automatic and correct
    # by construction: ml_index at country-year (i, t-1) is ALREADY the
    # t-2-to-t-1 transition (see get_common_es_frontier), so applying the
    # SAME .shift(1) used for es_mean/innov_mean below yields exactly
    # ImprovementES_{i,t-1} = ln(MLI_{i,t-2->t-1}) for the row predicting
    # cooperation at year t, as specified.
    print("has_mli: ", has_mli)
    if has_mli:
        f["es_impr_mean"] = np.log(np.clip((f.mli_i + f.mli_j) / 2, 1e-6, None))
    else:
        f["es_impr_mean"] = np.nan

    # TASK (reviewer point 7): "Keep R&D and patent stock separate in the
    # baseline." rd_pair_mean/patstock_pair_mean are SEPARATE pair-level
    # variables (not folded into innov_mean's composite above), used
    # together in their own model (m12) so cooperation's response to
    # research INPUTS (R&D/GDP) and technological OUTPUTS (patent stock)
    # can be distinguished, not hidden inside one averaged index.
    f["rd_pair_mean"] = (f.rd_i + f.rd_j) / 2
    f["patstock_pair_mean"] = (f.patstock_i + f.patstock_j) / 2 if has_patstock else np.nan
    for d in (10, 20):
        col_i, col_j = f"patstock{d}_i", f"patstock{d}_j"
        f[f"patstock{d}_pair_mean"] = (f[col_i] + f[col_j]) / 2 if (has_patstock_all and col_i in f.columns) else np.nan

    # TASK (reviewer): "MLI decomposition into efficiency change and
    # technological change" -- EC and TC as their own pair-level, lagged
    # regressors, mirroring es_impr_mean's construction (ln, since both are
    # ratio-type indices where >1 means improvement, exactly like MLI).
    if has_ectc:
        f["ec_pair_mean"] = np.log(np.clip((f.ec_i + f.ec_j) / 2, 1e-6, None))
        f["tc_pair_mean"] = np.log(np.clip((f.tc_i + f.tc_j) / 2, 1e-6, None))
    else:
        f["ec_pair_mean"] = np.nan
        f["tc_pair_mean"] = np.nan

    f = f.merge(dyad_static, on=["iso_i", "iso_j"], how="left")
    f = f.merge(vol, on="year", how="left")
    f["log_dist"] = np.log(f["distance"])

    print("trade: ", trade)
    if trade is not None:
        f = f.merge(trade, on=["iso_i", "iso_j", "year"], how="left")
        f["log_trade"] = np.log1p(f["trade"])
        prov.append(trade_prov)
    else:
        f["log_trade"] = (np.log(f.gdp_i * f.gdp_j + 1) / 2 - 0.6 * f.log_dist
                          + rng.normal(0, 0.4, len(f)))
        # FIX (reviewer): this used to unconditionally claim "gravity proxy
        # from REAL GDP & distance" -- true only when the R&D/GDP/pop fetch
        # was LIVE (World Bank/Eurostat). The real-GERD-only fallback
        # (_real_gerd_fallback) gives real R&D/GDP but SIMULATED gdp_usd,
        # and this message was contradicting that adjacent, correctly-
        # labeled provenance entry. Now checked against what was actually
        # recorded for R&D/GDP, not assumed.
        gdp_is_real = any(row.get("status") == "LIVE" and "GDP" in row.get("variable", "")
                          for row in prov)
        gdp_note = "real GDP (live fetch)" if gdp_is_real else "SIMULATED GDP (no live fetch; R&D/GDP may still be real separately)"
        prov.append({"variable": "trade", "source": f"gravity proxy from {gdp_note} & real distance",
                    "status": "SIMULATED", "citation": ""})

    print("rta: ", rta)
    if rta is not None:
        f = f.merge(rta, on=["iso_i", "iso_j", "year"], how="left")
        if "left_censoring_corrected" not in f.columns:
            f["left_censoring_corrected"] = False
        else:
            f["left_censoring_corrected"] = f["left_censoring_corrected"].fillna(False)
        # FIX (reviewer: "load genuine RTA data without treating missing
        # observations as zero"): a dyad-year absent from Larch's panel
        # after country-code matching is UNKNOWN, not "no RTA in force".
        # The previous version's fillna(0) conflated the two, which
        # silently manufactures a large block of assumed-zero observations
        # for any country/year combination our matching didn't reach (e.g.
        # a country outside Larch's coverage, or a year past the panel's
        # last update). f["rta"] is left as NaN here; estimate() decides,
        # and reports, how it handles the missing share explicitly.
        n_obs_rta = f["rta"].notna().sum()
        coverage = n_obs_rta / len(f) if len(f) else 0.0
        rta_source_label = rta_prov.get("source", "")[:60] if rta_prov else "unknown source"
        print(f"  [{club_key if club_key else 'pooled'}] RTA: {n_obs_rta}/{len(f)} dyad-years "
             f"({coverage:.1%}) matched ({rta_source_label}...); the rest are NaN (unknown), not 0.")
        is_real_dyadic = rta_prov is not None and "SIMULATED" not in rta_prov.get("status", "").upper()
        if is_real_dyadic:
            n_unique_in_sample = f.loc[(f["year"] >= 2008), "rta"].dropna().nunique()
            if n_unique_in_sample <= 1:
                print(f"    NOTE: RTA is now genuine dyadic data for this club (not simulated, not "
                     f"absent) -- strict mode no longer flags it. Checked directly: this specific "
                     f"variable has NO within-2008-2023-estimation-sample variation ({n_unique_in_sample} "
                     f"unique value(s)), so it cannot identify anything once dyad fixed effects are "
                     f"applied; watch for an 'insufficient variation' message from mA_gravity or "
                     f"m19_with_rta below -- that reflects a real identification limit for THIS "
                     f"specific model, not a data-availability problem.")
        prov.append(rta_prov)
        prov.append({"variable": "RTA coverage", "source": f"{n_obs_rta}/{len(f)} dyad-years matched "
                    f"({coverage:.1%}); remainder NaN, not assumed 0", "status": "INFO", "citation": ""})
    else:
        f["rta"] = np.nan
        f["left_censoring_corrected"] = False
        prov.append({"variable": "RTA (institutional control)",
                    "source": "not populated (data/rta_larch.csv absent) -- left as NaN, not defaulted to 0",
                    "status": "ABSENT", "citation": ""})

    print("rta_activity: ", rta_activity)
    if rta_activity is not None:
        n_obs_proxy = f["rta_stock_min"].notna().sum()
        print(f"  [{club_key if club_key else 'pooled'}] RTA activity proxy (country-level, NOT true "
             f"dyadic RTA): {n_obs_proxy}/{len(f)} dyad-years covered.")
        prov.append(rta_activity_prov)
    else:
        prov.append({"variable": "RTA activity proxy (country-level)",
                    "source": "not populated (data/rta_activity_by_country.csv absent)",
                    "status": "ABSENT", "citation": ""})

    print("fdi_activity: ", fdi_activity)
    if fdi_activity is not None:
        n_obs_fdi_proxy = f["fdi_out_pair_mean"].notna().sum()
        print(f"  [{club_key if club_key else 'pooled'}] FDI outflow-intensity proxy (country-level, "
             f"NOT true bilateral FDI): {n_obs_fdi_proxy}/{len(f)} dyad-years covered.")
        prov.append(fdi_activity_prov)
    else:
        prov.append({"variable": "FDI outflow-intensity proxy (country-level)",
                    "source": "not populated (data/fdi_outflows_pct_gdp.csv absent)",
                    "status": "ABSENT", "citation": ""})

    print("gdelt_activity: ", gdelt_activity)
    if gdelt_activity is not None:
        n_obs_gdelt_proxy = f["gdelt_pair_mean"].notna().sum()
        n_romania_dyads = int(f["gdelt_romania_dyad"].sum()) if "gdelt_romania_dyad" in f.columns else 0
        print(f"  [{club_key if club_key else 'pooled'}] GDELT agreement-activity proxy "
             f"(country-level, NOT energy-specific, NOT dyadic): {n_obs_gdelt_proxy}/{len(f)} "
             f"dyad-years covered ({n_romania_dyads} involve the flagged-unreliable Romania series).")
        prov.append(gdelt_activity_prov)
    else:
        prov.append({"variable": "GDELT agreement-activity proxy (country-level)",
                    "source": "not populated (data/gdelt_agreement_events.csv absent)",
                    "status": "ABSENT", "citation": ""})

    print("unts_treaties: ", unts_treaties)
    if unts_treaties is not None:
        n_obs_unts = f["unts_treaty_count"].notna().sum()
        n_nonzero_unts = (f["unts_treaty_count"].fillna(0) > 0).sum()
        print(f"  [{club_key if club_key else 'pooled'}] UNTS bilateral treaty count (GENUINELY "
             f"dyadic, 2000-2012 only): {n_obs_unts}/{len(f)} dyad-years within coverage "
             f"({n_nonzero_unts} with at least one registered treaty).")
        prov.append(unts_treaties_prov)
    else:
        prov.append({"variable": "UNTS bilateral treaty count (genuine dyadic)",
                    "source": "not populated (data/unts_bilateral_treaties.csv absent)",
                    "status": "ABSENT", "citation": ""})

    print("trade_openess: ", trade_openness)
    if trade_openness is not None:
        n_obs_tradeopen = f["tradeopen_pair_mean"].notna().sum()
        print(f"  [{club_key if club_key else 'pooled'}] Trade openness proxy (country-level, "
             f"category unconfirmed -- see docstring): {n_obs_tradeopen}/{len(f)} dyad-years covered.")
        prov.append(trade_openness_prov)
    else:
        prov.append({"variable": "Trade openness proxy (country-level, category unconfirmed)",
                    "source": "not populated (data/total_trade.csv absent)",
                    "status": "ABSENT", "citation": ""})

    f["vol_z"] = zscore(f["price_volatility"])
    f = f.sort_values(["dyad_id", "year"])
    g = f.groupby("dyad_id", sort=False)
    for base in ["es_mean", "innov_mean", "es_impr_mean"]:
        lagged = g[base].shift(1)
        print(f"  [{club_key}] standardisation for {base}_l1_z: "
             f"mean={lagged.mean():.4f}  sd={lagged.std():.4f}  n={lagged.notna().sum()} "
             f"(1 SD in original units = {lagged.std():.4f})")
        f[f"{base}_l1_z"] = zscore(lagged)
        f[f"{base}_l2_z"] = zscore(g[base].shift(2))
        f[f"{base}_f1_z"] = zscore(g[base].shift(-1))
    for base in ["es_min", "es_diff", "innov_min", "innov_diff", "log_trade", "rd_pair_mean", "patstock_pair_mean",
                "ec_pair_mean", "tc_pair_mean", "patstock10_pair_mean", "patstock20_pair_mean", "rta_stock_min",
                "fdi_out_pair_mean", "fdi_out_pair_min", "gdelt_pair_mean", "gdelt_pair_min",
                "tradeopen_pair_mean", "innov_stable1_mean", "innov_stable2_mean",
                "innov_patonly_mean", "innov_coinvonly_mean"]:
        f[f"{base}_l1_z"] = zscore(g[base].shift(1))

    es_l1 = f["es_mean_l1_z"].fillna(0).to_numpy()
    in_l1 = f["innov_mean_l1_z"].fillna(0).to_numpy()
    volz = f["vol_z"].fillna(0).to_numpy()
    dfe = f["dyad_id"].map({d: rng.normal(0, 0.5) for d in f.dyad_id.unique()}).to_numpy()
    yfe = f["year"].map({y: rng.normal(0, 0.3) for y in YEARS}).to_numpy()

    print("coop: ", coop)
    if coop is not None:
        f = f.merge(coop, on=["iso_i", "iso_j", "year"], how="left")
        # FIX (reviewer): "the separate-group models also still fill
        # unmatched cooperation observations with zero across the
        # configured panel, including years outside source coverage."
        # Confirmed: the previous unconditional fillna(0) treated a
        # dyad-year in, say, 2003 -- years before CORDIS/FP7 existed -- as
        # a verified zero, identical to a genuine non-match WITHIN 2007-
        # 2023. Restricted here to the source's own documented coverage
        # window (SOURCE_COVERAGE) so only in-window non-matches become
        # verified zeros; out-of-window years stay NaN (genuinely unknown)
        # and are excluded by estimate()'s own dropna, not silently
        # counted as "no cooperation". Hardcoded to the "cordis" key since
        # that is what load_cooperation_layered's top-priority branch
        # currently returns; if a different source becomes the active one,
        # this key -- and ideally the source name itself -- should be
        # threaded through from coop_prov rather than assumed.
        cc_year_min, cc_year_max = _get_source_coverage("cordis", coop)
        if cc_year_min is not None:
            out_of_window = (f["year"] < cc_year_min) | (f["year"] > cc_year_max)
            f.loc[~out_of_window, "coop_count"] = f.loc[~out_of_window, "coop_count"].fillna(0)
            # out_of_window rows keep whatever the left-merge produced,
            # which is NaN for every such row since coop has no entries
            # there -- left explicit for clarity rather than implicit.
        else:
            f["coop_count"] = f["coop_count"].fillna(0)
        if "coop_depth" not in f or f["coop_depth"].isna().all():
            f["coop_depth"] = f["coop_count"]
        prov.append(coop_prov)
        # TASK (reviewer, critical): "one loaded outcome is labelled as
        # both events and formal agreements... these cannot constitute
        # independent robustness outcomes." Confirmed exactly: when a
        # single real source (CORDIS) drives coop_count, both aliases
        # below take the IDENTICAL value, so any "formal agreements
        # robustness check" run against formal_agreements_sim in this
        # branch is mathematically the same regression as the primary
        # outcome under a different name, not an independent check. Not
        # removed (other code depends on both columns existing), but
        # flagged explicitly via this column so every place that reports
        # the formal-agreements results can and does warn accordingly.
        f["_formal_agreements_is_alias"] = True
        f["energy_events_sim"] = f["coop_count"]      # aliases kept so downstream
        f["formal_agreements_sim"] = f["coop_count"]   # code always finds both columns
        n_dyads_real = f["dyad_id"].nunique()
        always_zero_real = f.groupby("dyad_id")["coop_count"].max().eq(0).sum()
        print(f"  [{club_key}] PRIMARY OUTCOME: REAL COOPERATION DATA ACTIVE")
        print(f"    coop_count (from {coop_prov.get('source', 'real source')[:60]}...): "
             f"mean={f['coop_count'].mean():.3f}  var={f['coop_count'].var():.3f}  "
             f"zero_share={(f['coop_count']==0).mean():.1%}  max={f['coop_count'].max():.2f}  "
             f"all_zero_dyads={always_zero_real}/{n_dyads_real}")
        print(f"  [{club_key}] NOTE: formal_agreements_sim is an ALIAS of the above (identical "
             f"values, not an independently measured series) -- estimate_formal_agreements() "
             f"skips fitting on it accordingly; see that function's own message.")
    else:
        f["_formal_agreements_is_alias"] = False
        b0 = club["baseline_intercept"]
        idx = (b0 + 0.35 * es_l1 + 0.20 * in_l1 + 0.15 * es_l1 * in_l1
               + 0.10 * es_l1 * volz + dfe + yfe)

        # TASK (reviewer, repeatedly): TWO genuinely distinct simulated
        # cooperation outcomes, not one generic count relabelled.
        #   1. energy_events_sim -- PRIMARY. Dense, higher-frequency,
        #      negative-binomial (overdispersed, like real event data);
        #      same ES/Innovation signal (idx) as before.
        #   2. formal_agreements_sim -- ROBUSTNESS. Sparse: real formal
        #      energy agreements are rare (most dyad-years sign nothing),
        #      so the baseline rate is pushed far lower (same relative
        #      ES/Innovation signal, much smaller intercept) to produce a
        #      realistically high zero share, and drawn as a simple
        #      Poisson (a sparse count doesn't need overdispersion to look
        #      realistic the way a dense event count does).
        f["energy_events_sim"] = _simulate_negbin(np.exp(idx), dispersion=0.5, rng=rng)
        agreements_idx = idx - 3.2  # calibrated so zero share lands near 85-95% (see printed check)
        f["formal_agreements_sim"] = rng.poisson(np.exp(np.clip(agreements_idx, -10, 10)))

        # TASK: "three formal-institutional outcomes: number of new
        # agreements [already formal_agreements_sim]; any new agreement,
        # binary; active stock or depth of existing agreements." The
        # binary indicator is immediate. The active stock is a genuine
        # DYAD-LEVEL perpetual inventory of the flow above
        # (Stock_t = (1-delta)*Stock_{t-1} + NewAgreements_t, delta=0.15,
        # same convention as the patent stock elsewhere in this pipeline,
        # implying an average ~6-7 year active lifespan per agreement) --
        # this is exactly why it "retains more variation than new
        # agreements alone": a dyad with zero NEW agreements this year but
        # an agreement signed 3 years ago still shows a positive stock,
        # rather than being indistinguishable from a dyad with no
        # cooperation history at all.
        f["formal_agreements_any"] = (f["formal_agreements_sim"] > 0).astype(float)
        f = f.sort_values(["dyad_id", "year"])
        stock_delta = 0.15
        stock_vals = np.zeros(len(f))
        for dyad_id, idxs in f.groupby("dyad_id", sort=False).groups.items():
            idxs = f.index.get_indexer(idxs)
            flows = f["formal_agreements_sim"].to_numpy()[idxs]
            s = 0.0
            stock_series = np.empty(len(flows))
            for k, flow in enumerate(flows):
                s = (1 - stock_delta) * s + flow
                stock_series[k] = s
            stock_vals[idxs] = stock_series
        f["formal_agreements_stock"] = stock_vals

        f["coop_count"] = f["energy_events_sim"]       # primary outcome used by estimate()
        f["coop_depth"] = f["coop_count"] * rng.uniform(0.6, 1.4, len(f))

        def _outcome_stats(col):
            v = f[col]
            always_zero = f.groupby("dyad_id")[col].max().eq(0).sum()
            n_dyads = f["dyad_id"].nunique()
            return (f"mean={v.mean():.3f}  var={v.var():.3f}  zero_share={(v==0).mean():.1%}  "
                   f"max={v.max():.2f}  all_zero_dyads={always_zero}/{n_dyads}")

        print(f"  [{club_key}] PRIMARY OUTCOME: EVENT-BASED ENERGY COOPERATION")
        print(f"    energy_events_sim (negative binomial, dispersion=0.5): {_outcome_stats('energy_events_sim')}")
        print(f"  [{club_key}] ROBUSTNESS OUTCOMES: FORMAL ENERGY AGREEMENTS (three definitions)")
        print(f"    formal_agreements_sim (count, sparse Poisson):     {_outcome_stats('formal_agreements_sim')}")
        print(f"    formal_agreements_any (binary, any new agreement): {_outcome_stats('formal_agreements_any')}")
        print(f"    formal_agreements_stock (active stock, delta={stock_delta}):    {_outcome_stats('formal_agreements_stock')}")

        prov.append({"variable": "cooperation", "source": "SIMULATED (four outcomes: energy_events_sim "
                    "[primary, negative-binomial]; formal_agreements_sim/any/stock [robustness -- count, "
                    "binary, and perpetual-inventory active stock], all from real ES/Innovation regressors, "
                    "not observed cooperation)",
                    "status": "SIMULATED", "citation": ""})

    print("fdi: ", fdi)
    if fdi is not None:
        val_col = "fdi_count" if "fdi_count" in fdi.columns else "fdi_value"
        f = f.merge(fdi[["iso_i", "iso_j", "year", val_col]], on=["iso_i", "iso_j", "year"], how="left")
        f["fdi_count"] = f[val_col].fillna(0)
        prov.append(fdi_prov)
    elif club["fdi_file"] is not None:
        f["fdi_count"] = rng.poisson(np.exp(-0.4 + 0.40 * es_l1 + 0.20 * in_l1 + dfe + yfe))
        prov.append({"variable": "FDI", "source": "simulated from real + DEA regressors",
                    "status": "SIMULATED", "citation": ""})
    # else: club has no FDI outcome at all (e.g. EU) ? no column, no model

    f["log1p_coop"] = np.log1p(f["coop_count"])
    # TASK ("the program should stop with an explicit missing-data report
    # if any required non-cooperation variable is absent, rather than
    # silently simulating it"): --strict mode. Placed HERE, after every
    # prov.append() in this function has run (an earlier version of this
    # check ran too early, before trade/RTA/cooperation provenance was even
    # recorded -- a real bug, caught by testing rather than assumed fixed).
    # Cooperation and FDI are the two OUTCOME variables the reviewer
    # explicitly allows to remain simulated pending real data; every other
    # variable (ES, Innovation, distance, trade, RTA, price volatility)
    # must be real in strict mode, or the run stops here with a full report
    # of exactly what's missing.
    print("STRICT_MODE: ", STRICT_MODE)
    if STRICT_MODE:
        _enforce_strict_mode(club_key, prov, allowed_simulated={"cooperation", "FDI"})

    return f, prov


# =========================================================================
# SECTION 7 ? ESTIMATION + REPORTING (club-agnostic)
# =========================================================================
def _diagnose_sparsity(est, outcome_col):
    """Pre-flight check so a too-sparse real outcome gives an actionable
    message instead of a raw pyfixest/linear-algebra traceback."""
    nz_share = (est[outcome_col] > 0).mean()
    dyads_with_variation = est.groupby("dyad_id")[outcome_col].std().gt(0).sum()
    n_dyads = est["dyad_id"].nunique()
    if nz_share < 0.005 or dyads_with_variation < 5:
        raise ValueError(
            f"'{outcome_col}' is too sparse to identify dyad+year fixed effects: "
            f"only {nz_share:.2%} of observations are non-zero, and only "
            f"{dyads_with_variation}/{n_dyads} dyads show any within-dyad variation. "
            f"PPML/FE-OLS need enough non-zero, varying observations per fixed-effect "
            f"cell or the design matrix becomes collinear. Check the loaded CSV for "
            f"missing/incorrectly-parsed rows (e.g. `pd.read_csv(...)['{outcome_col}']"
            f".value_counts()`), or widen the year range / dyad set."
        )


def _fit_or_raise(fit_fn, formula, est, vcov, label):
    import pyfixest as pf
    from pyfixest.demeaners import LsmrDemeaner
    # FIX: a user reported a reproducible Rust-level crash --
    # "thread panicked at src/demean.rs:71:48, called Option::unwrap() on
    # a None value" -- that terminates the whole Python process (not a
    # catchable exception) and restarts the Jupyter/Colab kernel entirely,
    # confirmed to occur specifically near the end of a run in their
    # environment. Could not be reproduced directly in this sandbox
    # (pyfixest 0.60.0) despite trying, but traced the panic to pyfixest's
    # default demeaner (MapDemeaner), which calls into a compiled Rust
    # extension (pyfixest/core/_core_impl...so implementing "demean.rs").
    # pyfixest also ships an alternative, LsmrDemeaner (a conjugate-
    # gradient/Schwarz-based solver, a genuinely different algorithm and
    # code path from MapDemeaner's method of alternating projections, not
    # just a different wrapper around the same Rust code) -- switched to
    # here, universally, via this single shared fitting helper. Verified
    # directly, not assumed: refitting this pipeline's own baseline model
    # with each demeaner in this sandbox gives numerically identical
    # coefficients and standard errors to 9+ decimal places, so this is
    # not expected to change any reported result, only the internal
    # numerical routine used to obtain it.
    #
    # ALTERNATIVE CONSIDERED: a reviewer suggested pyfixest's own
    # documented alternative for this exact situation, MapDemeaner(
    # backend="numba") -- same algorithm as the default, different
    # (non-Rust) backend, which is arguably a more targeted fix since it
    # changes nothing but the compute engine. Not adopted as the default
    # here because numba is NOT a base pyfixest dependency (confirmed:
    # pyfixest's own declared requirements list formulaic, joblib,
    # maketables, narwhals, numpy, pandas, scipy, seaborn, tabulate, tqdm
    # -- no numba), so requiring it risks trading one environment-
    # dependent failure for another (an ImportError where numba happens
    # not to be installed, e.g. outside Colab-like environments where
    # it's commonly bundled). LsmrDemeaner's "within" backend uses only
    # scipy, already a base dependency, at the cost of using a genuinely
    # different algorithm rather than the same one on a different engine.
    # If numba is confirmed present in the target environment, swapping
    # to MapDemeaner(backend="numba") throughout this file is a
    # reasonable alternative worth trying.
    n_input = len(est)
    print(f"[FIT START] {label}: rows={n_input}, formula={formula}", flush=True)
    try:
        m = fit_fn(formula, est, vcov=vcov, demeaner=LsmrDemeaner(backend="within"))
    except ValueError as e:
        if "collinear" in str(e).lower():
            raise ValueError(
                f"Model '{label}' failed with a multicollinearity error from pyfixest "
                f"(formula: {formula!r}). This almost always means the outcome or a "
                f"regressor has too little independent variation once dyad and year "
                f"fixed effects are partialled out ? check for a too-sparse loaded CSV, "
                f"a constant regressor, or a near-duplicate control. Original error: {e}"
            ) from e
        raise
    print(f"[FIT OK] {label}", flush=True)
    # TASK (reviewer: "report the number of separated/dropped observations
    # in every PPML model"): pyfixest silently drops rows internally (e.g.
    # singleton fixed-effect groups, perfect-separation cases in PPML) --
    # a real, confirmed behaviour (caught earlier building the wild
    # bootstrap: 2394 input rows -> 2324 used by one model). Reported here
    # for every single model fit, not just the ones where it happened to
    # matter for another feature.
    n_used = int(m._N) if hasattr(m, "_N") else n_input
    if n_used < n_input:
        print(f"    [{label}] {n_input - n_used}/{n_input} observations dropped by pyfixest "
             f"internally (singleton FE groups / separation) -- {n_used} used.")
    return m



def estimate_formal_agreements(df):
    """
    TASK: fit the SAME core models on THREE formal-institutional outcome
    definitions -- formal_agreements_sim (count), formal_agreements_any
    (binary, any new agreement), and formal_agreements_stock (active
    stock/depth, perpetual inventory) -- so their properties (especially
    power) can be compared directly. Returns None if these columns aren't
    present (real cooperation data active instead). Each outcome gets its
    own sparsity check; a failure on one does not block the others.
    """
    if "formal_agreements_sim" not in df.columns:
        return None
    # FIX (reviewer): "estimate_formal_agreements() does not use that flag
    # to stop estimation. It continues to fit the duplicated outcome...
    # research-project counts must not appear in a formal-energy-agreement
    # results table." Correct: a warning printed AFTER fitting does not
    # stop the fitted numbers from reaching the results tables. This now
    # checks the flag FIRST and returns before any model is fit when
    # formal_agreements_sim is only an alias of the primary real outcome
    # (both are the identical CORDIS-derived series) -- not an
    # independently measured formal-agreement construct.
    if "_formal_agreements_is_alias" in df.columns and df["_formal_agreements_is_alias"].iloc[0]:
        print(f"    [estimate_formal_agreements] SKIPPED: formal_agreements_sim is an alias of "
             f"the primary real outcome in this run (identical CORDIS-derived values), not an "
             f"independently measured formal-agreement series. Fitting it would put research-"
             f"project counts into a results table labelled as formal energy agreements.")
        return None
    import pyfixest as pf
    fe, cl = "| dyad_id + year", {"CRV1": "dyad_id"}
    m = {}
    specs = [("formal_agreements_sim", "fa1_ppml", "fa3_innov", pf.fepois),
            ("formal_agreements_any", "fa2_extensive", "fa2b_extensive_innov", pf.feols),
            ("formal_agreements_stock", "fa4_stock", "fa5_stock_innov", pf.fepois)]
    est_by_outcome = {}
    for outcome_col, key1, key2, fit_fn in specs:
        if outcome_col not in df.columns:
            continue
        est = df.dropna(subset=["es_mean_l1_z", "innov_mean_l1_z", outcome_col]).copy()
        try:
            _diagnose_sparsity(est, outcome_col)
        except ValueError as e:
            print(f"    [{outcome_col}] {e}")
            continue
        m[key1] = _fit_or_raise(fit_fn, f"{outcome_col} ~ es_mean_l1_z + innov_mean_l1_z {fe}", est, cl, key1)
        m[key2] = _fit_or_raise(fit_fn, f"{outcome_col} ~ es_mean_l1_z + innov_mean_l1_z + es_mean_l1_z:innov_mean_l1_z {fe}", est, cl, key2)
        est_by_outcome[outcome_col] = est
    if not m:
        return None
    return m, est_by_outcome.get("formal_agreements_sim", next(iter(est_by_outcome.values())))


# =========================================================================
# SECTION 7B ? DYADIC-ROBUST INFERENCE: MULTIWAY CLUSTERING + WILD BOOTSTRAP
# =========================================================================
# TASK: "dyadic/network-robust standard errors" and "wild-cluster bootstrap".
# Motivation (Cameron & Miller, 2014, already cited in this chapter):
# dyad-clustered SEs (the default throughout this pipeline) only allow
# correlation WITHIN a given (i,j) pair over time. They do NOT allow for
# correlation across DIFFERENT dyads that share a common country -- e.g.
# the USA-Germany and USA-France dyads could plausibly have correlated
# shocks (both involve the USA) that dyad-only clustering misses entirely.
# The standard fix is two-way ("multiway") clustering by country i AND
# country j separately (Cameron, Gelbach & Miller, 2011), combined as:
#     V_multiway = V_cluster(i) + V_cluster(j) - V_cluster(dyad)
# implemented here by fitting the SAME model three times with pyfixest's
# existing, already-used CRV1 clustering (by iso_i, by iso_j, by dyad_id)
# and combining the resulting covariance matrices with this formula --
# not a from-scratch sandwich-estimator reimplementation, which would risk
# silently diverging from what the rest of this pipeline already relies on.
#
# The wild-cluster bootstrap (Cameron, Gelbach & Miller, 2008) addresses a
# DIFFERENT problem: with a SMALL number of clusters (G20's country-level
# clustering has only 19), asymptotic cluster-robust SEs are unreliable
# regardless of which clustering scheme is used, and wild-cluster bootstrap
# p-values are the standard small-G correction.

def fit_multiway_cluster(fit_fn, formula, data, label, col_i="iso_i", col_j="iso_j", dyad_col="dyad_id"):
    """
    GENUINE dyadic-robust inference for UNDIRECTED dyads (Fafchamps &
    Gubert, 2007; Cameron & Miller, 2014; Aronow, Samii & Assenova, 2015),
    replacing an earlier, confirmed-incorrect implementation.

    THE BUG THIS REPLACES: the previous version clustered separately by
    the iso_i COLUMN and the iso_j COLUMN and combined via the standard
    Cameron-Gelbach-Miller (2011) two-way formula V_i + V_j - V_dyad. That
    formula is valid for DIRECTED dyadic data where i/j are meaningful,
    distinct roles (e.g. exporter/importer, each observed in both roles
    across the panel). It is NOT valid here: these dyads are undirected,
    and iso_i/iso_j are assigned by alphabetical order (an arbitrary
    labelling), not a real role. Two concrete, confirmed symptoms: (1) the
    alphabetically LAST country never appears in iso_i and the FIRST never
    appears in iso_j, so clustering by column undercounts by exactly one
    country in each direction (18 clusters were reported for 19 G20
    members, 21 for 22 EU members -- confirmed directly, not assumed);
    (2) more fundamentally, a shock to a country produces a SPLIT signal
    across the two clustering variables depending on which column that
    country happens to sit in for a given dyad, so column-based two-way
    clustering never captures a country's correlation across dyads where
    it appears in *different* columns (e.g. Germany's shock correlating
    "France_Germany" [Germany in j] with "Germany_Japan" [Germany in i]
    was NOT captured by either V_i or V_j alone).

    CORRECT APPROACH implemented here: for every unique COUNTRY c
    (regardless of column), let S_c = sum of that country's dyad-year
    score contributions over EVERY dyad containing c, whichever column it
    sits in. The dyadic-robust "meat" matrix is
        M = sum_c (S_c S_c') - sum_d (S_d S_d'),   S_d = sum_t s_{dt}
    -- i.e. the second term requires TIME-AGGREGATING each dyad's scores
    across every year it appears before taking the outer product, not
    summing individual observations' own outer products (these differ
    whenever a dyad has more than one year in the panel; a reviewer
    found this distinction had been implemented incorrectly here, and it
    is fixed below). The formula was originally derived and verified by
    hand on a toy 3-country, single-year example (dyads AB, AC, BC) --
    which could not have caught this specific bug, since with exactly one
    observation per dyad, "sum of individual outer products" and "outer
    product of the time-aggregated sum" are numerically identical; the
    error only manifests with repeated (multi-year) dyad observations,
    which every real panel here has. Re-confirmed on the reviewer's own
    toy case (one dyad, two years, scalar scores 1 and 2): the correct
    dyad-aggregated term is (1+2)^2=9, not the previous 1^2+2^2=5.
    Built on pyfixest's own validated internal score/bread matrices (a
    manual reproduction of pyfixest's own dyad-clustered SE from these
    matrices matched to within its small-sample correction convention --
    checked directly, not assumed) rather than re-deriving the GLM
    sandwich estimator from scratch.

    Returns (model, V_dyadic, se_dyadic, n_countries) -- se_dyadic should
    be used in place of model.se() for inference. n_countries is reported
    directly so it can be checked against the true member count.
    """
    m = _fit_or_raise(fit_fn, formula, data, {"CRV1": dyad_col}, f"{label}_base")
    scores = m._scores                    # (N_used, K) -- validated against pyfixest's own dyad SE
    bread = m._bread                      # (K, K) = (X'WX)^-1
    used = m._data
    iso_i_arr = used[col_i].to_numpy()
    iso_j_arr = used[col_j].to_numpy()
    all_countries = np.unique(np.concatenate([iso_i_arr, iso_j_arr]))
    K = scores.shape[1]

    meat_any_member = np.zeros((K, K))
    for c in all_countries:
        mask = (iso_i_arr == c) | (iso_j_arr == c)
        s_c = scores[mask].sum(axis=0)
        meat_any_member += np.outer(s_c, s_c)
    # FIX (reviewer, confirmed by direct numerical reproduction): the
    # previous "own_diag = scores.T @ scores" summed outer products of
    # INDIVIDUAL observation-level scores (sum_t s_t s_t'), not the outer
    # product of each dyad's TIME-AGGREGATED score (S_d S_d', S_d = sum_t
    # s_dt) that the derivation in this function's own docstring actually
    # requires. These are algebraically different whenever a dyad has more
    # than one observation (i.e. every dyad with more than one year in the
    # panel) -- reproduced directly on the reviewer's own toy case (one
    # dyad, two years, scalar scores 1 and 2): the old code gives 1^2+2^2=5,
    # the correct dyad-aggregated term gives (1+2)^2=9, matching the
    # reviewer's independently-derived value exactly. Fixed by summing
    # scores within each dyad (across all its observations) first, then
    # taking the outer product of that per-dyad sum.
    dyad_arr = used[dyad_col].to_numpy()
    own_diag = np.zeros((K, K))
    for d in np.unique(dyad_arr):
        s_d = scores[dyad_arr == d].sum(axis=0)
        own_diag += np.outer(s_d, s_d)
    meat_dyadic = meat_any_member - own_diag

    G, N, k = len(all_countries), int(m._N), int(m._k)
    ssc = (G / (G - 1)) * ((N - 1) / (N - k)) if G > 1 and N > k else 1.0
    V_dyadic = bread @ meat_dyadic @ bread * ssc

    diag = np.diag(V_dyadic)
    if (diag < 0).any():
        # Same practical PSD safeguard as the old implementation's own
        # comment described -- floor at the dyad-clustered variance rather
        # than propagate a negative variance silently.
        dyad_diag = np.diag(m._vcov)
        diag = np.where(diag < 0, dyad_diag, diag)
        print(f"    [{label}] dyadic-robust V had negative diagonal entries -- "
             f"floored at the dyad-clustered variance (practical PSD safeguard).")
    se_dyadic = np.sqrt(diag)
    print(f"    [{label}] dyadic-robust clustering: {G} countries "
         f"(should equal the true member count -- verify against the club's own member list)")
    return m, V_dyadic, se_dyadic, G


def wild_cluster_bootstrap(fit_fn, formula, data, coef_name, cluster_col=None, B=999, is_count_model=True):
    """
    SUSPENDED (reviewer, mathematically proven, confirmed independently by
    simulation here): the country-membership weighting below (w_ij = w_i *
    w_j, independent Rademacher w_i/w_j per country) does NOT reproduce
    correlated perturbations for two dyads sharing a country, contrary to
    this function's own earlier claim. For dyads {i,j} and {i,k} sharing
    country i: E[(w_i w_j)(w_i w_k)] = E[w_i^2] E[w_j] E[w_k] = 1*0*0 = 0
    -- exactly zero in expectation, confirmed empirically over 2,000,000
    simulated draws (0.0003, statistical noise around the proven zero).
    The shared-country correlation this procedure was designed to induce
    simply is not there. More bootstrap repetitions cannot fix this -- it
    is a property of the weighting scheme itself, not sampling noise.

    The p-value this function returns is NOT a validated wild-cluster
    bootstrap p-value for dyadic data and must not be reported or relied
    upon as one. It is retained here, computed but clearly labelled at
    every call site, only so a reader can see the previously-reported
    (invalid) number rather than have it silently vanish; it should be
    replaced with a properly-derived dyadic wild bootstrap (e.g. a
    dyad-cluster-level, not actor-level, resampling scheme) or dropped
    from the reported inference battery entirely.

    Kept below for reference only -- procedure as originally written,
    now known invalid:

    CORRECT PROCEDURE: draw ONE Rademacher weight PER COUNTRY (not per
    column value). For a dyad {i,j}, the perturbation weight applied to
    that observation's residual is w_i * w_j (the product of its two
    member countries' independently-drawn weights) -- if either member's
    weight flips, the dyad's perturbation flips, so dyads sharing a common
    country get correlated (though not identical) perturbations through
    the shared factor, exactly mirroring the actor-effects structure used
    for the dyadic-robust covariance estimator above. cluster_col is
    accepted for backward-compatible call signatures but ignored -- the
    country set is always read from iso_i/iso_j directly.

    Procedure (restricted/null wild bootstrap, the standard variant):
      1. Fit the RESTRICTED model (coef_name dropped) for residuals under
         H0: beta_coef_name = 0.
      2. For B reps: one Rademacher weight per COUNTRY; each dyad's
         perturbation = product of its two members' weights; pseudo-y =
         fitted_restricted + weighted residual (clipped >=0, rounded for a
         count outcome); refit the UNRESTRICTED model; record the t-stat.
      3. Bootstrap p = share of |t*_b| >= |t_observed|.
    """
    lhs, rhs = formula.split("~", 1)
    rhs_main, rhs_fe = rhs.split("|", 1) if "|" in rhs else (rhs, "")
    terms = [t.strip() for t in rhs_main.split("+")]
    terms_restricted = [t for t in terms if t != coef_name]
    if len(terms_restricted) == len(terms):
        return None
    formula_restricted = f"{lhs}~{' + '.join(terms_restricted)}" + (f"|{rhs_fe}" if rhs_fe else "")

    m_full = _fit_or_raise(fit_fn, formula, data, {"CRV1": "dyad_id"}, "wcb_full")
    if coef_name not in m_full.coef().index:
        return None
    t_obs = float(m_full.coef()[coef_name] / m_full.se()[coef_name])

    m_restr = _fit_or_raise(fit_fn, formula_restricted, data, {"CRV1": "dyad_id"}, "wcb_restr")
    used = m_restr._data.copy()               # pyfixest's OWN retained sample -- correctly aligned
    used["_resid"] = np.asarray(m_restr.resid())
    used["_fitted"] = used[lhs.strip()] - used["_resid"]

    iso_i_arr = used["iso_i"].to_numpy()
    iso_j_arr = used["iso_j"].to_numpy()
    all_countries = np.unique(np.concatenate([iso_i_arr, iso_j_arr]))
    y_col = lhs.strip()
    t_boot = np.full(B, np.nan)
    n_failed = 0
    data_b = used.copy()
    for b in range(B):
        country_weights = dict(zip(all_countries, rng.choice([-1.0, 1.0], size=len(all_countries))))
        w_obs = np.array([country_weights[i] * country_weights[j] for i, j in zip(iso_i_arr, iso_j_arr)])
        pseudo_y = used["_fitted"].to_numpy() + w_obs * used["_resid"].to_numpy()
        if is_count_model:
            pseudo_y = np.clip(np.round(pseudo_y), 0, None)
        data_b[y_col] = pseudo_y
        try:
            m_b = fit_fn(formula, data_b, vcov={"CRV1": "dyad_id"})
            if coef_name in m_b.coef().index and m_b.se()[coef_name] > 0:
                t_boot[b] = float(m_b.coef()[coef_name] / m_b.se()[coef_name])
            else:
                n_failed += 1
        except Exception:
            n_failed += 1
    valid = t_boot[~np.isnan(t_boot)]
    # TASK (reviewer): "with B=399, never print a bootstrap p-value as
    # 0.0000... use p=(r+1)/(B+1)". Fixed: r = count of replications at
    # least as extreme as the observed statistic, B = valid replications
    # actually used (not the requested B, since some can fail to fit) --
    # this is the standard finite-bootstrap correction (Davison & Hinkley,
    # 1997), guaranteeing p >= 1/(B+1) rather than a naive r/B that can
    # report an impossible exact zero.
    if len(valid) > 0:
        r = int(np.sum(np.abs(valid) >= abs(t_obs)))
        p_boot = (r + 1) / (len(valid) + 1)
    else:
        p_boot = np.nan
    return {"coef_name": coef_name, "t_observed": t_obs, "p_wild_bootstrap": p_boot,
           "n_clusters": len(all_countries), "B": B, "n_valid": len(valid), "n_failed": n_failed}


def estimate(df):
    import pyfixest as pf
    est = df.dropna(subset=["es_mean_l1_z", "innov_mean_l1_z", "coop_count"]).copy()
    if est["coop_count"].sum() == 0:
        raise ValueError("cooperation outcome is all zeros ? fill the club's cooperation CSV "
                         "or delete it to use the simulated outcome.")
    _diagnose_sparsity(est, "coop_count")

    fe, cl, cl2 = "| dyad_id + year", {"CRV1": "dyad_id"}, {"CRV1": "iso_i + iso_j"}
    m = {}
    m["m1_ppml"] = _fit_or_raise(pf.fepois, f"coop_count ~ es_mean_l1_z + innov_mean_l1_z {fe}", est, cl, "m1_ppml")
    m["m2_ols"] = _fit_or_raise(pf.feols, f"log1p_coop ~ es_mean_l1_z + innov_mean_l1_z {fe}", est, cl, "m2_ols")
    m["m3_innov"] = _fit_or_raise(pf.fepois, f"coop_count ~ es_mean_l1_z + innov_mean_l1_z + es_mean_l1_z:innov_mean_l1_z {fe}", est, cl, "m3_innov")
    m["m4_vol"] = _fit_or_raise(pf.fepois, f"coop_count ~ es_mean_l1_z + innov_mean_l1_z + es_mean_l1_z:vol_z {fe}", est, cl, "m4_vol")

    # TASK (reviewer, "the most important methodological issue"): the
    # simulated events outcome is generated from ES, Innovation, ES x
    # Innovation, AND ES x Volatility together (see build_panel's DGP:
    # idx = b0 + 0.35*es + 0.20*innov + 0.15*es*innov + 0.10*es*vol + FE).
    # m1_ppml reports only ES and Innovation -- confirmed via the events-
    # vs-agreements Monte Carlo (Section 8C-equivalent per-club battery)
    # that this is not a noisy version of the true beta_ES=0.35, it is a
    # pseudo-true COMPOSITE that partly absorbs the two omitted
    # interactions (mean estimate ~0.55 there, matching this single run's
    # ~0.52-0.59 closely). m_full_ppml is the PREFERRED, correctly
    # specified model: it matches the DGP's functional form exactly, so
    # beta_1 (es_mean_l1_z) is interpretable as the ES effect AT AVERAGE
    # innovation and average volatility (both z-scored, so "average" =
    # z=0), beta_3 as the change in the ES slope per 1-SD of innovation,
    # and beta_4 as the change in the ES slope per 1-SD of volatility --
    # exactly the interpretation the DGP supports. m3_innov and m4_vol
    # remain available as simplified robustness models (each omits one of
    # the two true interactions), not replaced, but m_full_ppml -- not
    # m1_ppml -- is the model whose ES coefficient should be read as "the"
    # structural estimate in any headline IRR statement.
    m["m_full_ppml"] = _fit_or_raise(
        pf.fepois,
        f"coop_count ~ es_mean_l1_z + innov_mean_l1_z + es_mean_l1_z:innov_mean_l1_z + es_mean_l1_z:vol_z {fe}",
        est, cl, "m_full_ppml")
    if "fdi_count" in est.columns:
        _diagnose_sparsity(est, "fdi_count")
        m["m5_fdi"] = _fit_or_raise(pf.fepois, f"fdi_count ~ es_mean_l1_z + innov_mean_l1_z {fe}", est, cl, "m5_fdi")

    # TASK: real, genuinely DYADIC UNTS bilateral treaty counts now
    # available (see load_unts_bilateral_treaties) -- an outcome-
    # substitution check in the same spirit as m5_fdi, but using GENUINE
    # bilateral data rather than a simulated or country-level-proxy
    # outcome: does ES/Innovation explain REAL bilateral treaty-signing
    # the way they explain the (mostly real, partly simulated) primary
    # events outcome? The sample is restricted to this source's own
    # coverage window (2000-2012, non-missing unts_treaty_count) -- a
    # real but smaller sample than the main analysis, stated as such
    # rather than padded with fabricated coverage.
    if "unts_treaty_count" in est.columns and est["unts_treaty_count"].notna().sum() >= 30:
        est_unts = est.dropna(subset=["unts_treaty_count"])
        try:
            _diagnose_sparsity(est_unts, "unts_treaty_count")
            print(f"    [m23_unts_real_treaties] fit on {len(est_unts)} dyad-years within the "
                 f"source's real 2000-2012 coverage window (of {len(est)} in the main sample).")
            m["m23_unts_real_treaties"] = _fit_or_raise(
                pf.fepois, f"unts_treaty_count ~ es_mean_l1_z + innov_mean_l1_z {fe}",
                est_unts, cl, "m23_unts_real_treaties")
        except ValueError as e:
            print(f"    [m23_unts_real_treaties] {e}")
    else:
        print(f"    [m23_unts_real_treaties] insufficient real UNTS coverage in this sample -- skipped.")

    # TASK: real country-level trade openness data now available (see
    # load_trade_openness_country_level) -- a dyadic CONTROL proxy,
    # labelled "category unconfirmed" throughout given the flagged unit
    # ambiguity (magnitudes suggest a narrower category than total
    # merchandise trade, possibly energy-specific, not confirmed).
    if est["tradeopen_pair_mean_l1_z"].notna().sum() >= 30:
        m["m24_with_tradeopen_proxy"] = _fit_or_raise(
            pf.fepois, f"coop_count ~ es_mean_l1_z + innov_mean_l1_z + tradeopen_pair_mean_l1_z {fe}",
            est.dropna(subset=["tradeopen_pair_mean_l1_z"]), cl, "m24_with_tradeopen_proxy")

    # TASK (reviewer point 6): "the innovation composite averages whatever
    # components happen to be available... a measurement break that could
    # be mistaken for temporal change." Four responses, matching the
    # reviewer's own list: (a) composition-by-year reporting is printed
    # in build_panel above; (b)-(c) below refit the headline specification
    # with two ALTERNATIVE, composition-STABLE innovation measures instead
    # of the changing composite, so their own meaning never shifts across
    # the panel; (d) genuinely component-specific models (patents alone,
    # co-invention alone) isolate which signal is doing the work, rather
    # than assuming the composite's pooled coefficient reflects all three
    # components equally.
    if est["innov_stable1_mean_l1_z"].notna().sum() >= 30:
        m["m25a_innov_stable1"] = _fit_or_raise(
            pf.fepois, f"coop_count ~ es_mean_l1_z + innov_stable1_mean_l1_z {fe}",
            est.dropna(subset=["innov_stable1_mean_l1_z"]), cl, "m25a_innov_stable1")
    if est["innov_stable2_mean_l1_z"].notna().sum() >= 30:
        m["m25b_innov_stable2"] = _fit_or_raise(
            pf.fepois, f"coop_count ~ es_mean_l1_z + innov_stable2_mean_l1_z {fe}",
            est.dropna(subset=["innov_stable2_mean_l1_z"]), cl, "m25b_innov_stable2")
    if est["innov_patonly_mean_l1_z"].notna().sum() >= 30:
        m["m25c_innov_patonly"] = _fit_or_raise(
            pf.fepois, f"coop_count ~ es_mean_l1_z + innov_patonly_mean_l1_z {fe}",
            est.dropna(subset=["innov_patonly_mean_l1_z"]), cl, "m25c_innov_patonly")
    if est["innov_coinvonly_mean_l1_z"].notna().sum() >= 30:
        m["m25d_innov_coinvonly"] = _fit_or_raise(
            pf.fepois, f"coop_count ~ es_mean_l1_z + innov_coinvonly_mean_l1_z {fe}",
            est.dropna(subset=["innov_coinvonly_mean_l1_z"]), cl, "m25d_innov_coinvonly")
    # Missing-component sensitivity: does the ES coefficient itself move
    # materially depending on which innovation measure accompanies it?
    if "m25a_innov_stable1" in m and "m1_ppml" in m:
        es_main = float(m["m1_ppml"].coef().get("es_mean_l1_z", np.nan))
        es_stable1 = float(m["m25a_innov_stable1"].coef().get("es_mean_l1_z", np.nan))
        print(f"    [Missing-component sensitivity] ES coefficient with the full (changing-"
             f"composition) composite: {es_main:+.4f}; with the stable one-component (R&D-only) "
             f"index instead: {es_stable1:+.4f}; difference: {es_stable1 - es_main:+.4f}.")

    m["m6_lag2"] = _fit_or_raise(pf.fepois, f"coop_count ~ es_mean_l2_z + innov_mean_l2_z {fe}", est, cl, "m6_lag2")
    m["m6_lead1"] = _fit_or_raise(pf.fepois, f"coop_count ~ es_mean_f1_z + innov_mean_f1_z {fe}", est, cl, "m6_lead1")
    m["m7_min"] = _fit_or_raise(pf.fepois, f"coop_count ~ es_min_l1_z + innov_min_l1_z {fe}", est, cl, "m7_min")
    m["m7_diff"] = _fit_or_raise(pf.fepois, f"coop_count ~ es_diff_l1_z + innov_diff_l1_z {fe}", est, cl, "m7_diff")

    # TASK: "implement joint one- and two-year lags" -- t-1 and t-2 in the
    # SAME regression, so each coefficient is estimated conditional on the
    # other rather than as two separate univariate specifications that
    # ignore their mutual correlation.
    m["m8_joint_lags"] = _fit_or_raise(
        pf.fepois, f"coop_count ~ es_mean_l1_z + es_mean_l2_z + innov_mean_l1_z + innov_mean_l2_z {fe}",
        est, cl, "m8_joint_lags")

    # TASK: "implement the conditional lead/placebo test" -- t-1 (real
    # effect expected) and t+1 (placebo, no effect expected) together in
    # ONE regression, so the placebo test is conditional on the genuine
    # lag also being controlled for, rather than run as an independent
    # univariate regression (m6_lead1 above) that cannot separate "no lead
    # effect" from "lead and lag are correlated and only one was fit".
    m["m9_conditional_placebo"] = _fit_or_raise(
        pf.fepois, f"coop_count ~ es_mean_l1_z + es_mean_f1_z + innov_mean_l1_z {fe}",
        est, cl, "m9_conditional_placebo")

    # TASK (reviewer point 6): "Use average ES and the absolute ES gap
    # jointly. Do not include mean, minimum and gap simultaneously because
    # they are mechanically related." -- mean+gap together, as its own
    # specification, distinct from the univariate m1 (mean alone), m7_min
    # (min alone) and m7_diff (gap alone) already above.
    m["m10_mean_gap"] = _fit_or_raise(
        pf.fepois, f"coop_count ~ es_mean_l1_z + es_diff_l1_z + innov_mean_l1_z {fe}",
        est, cl, "m10_mean_gap")

    # TASK (reviewer point 6): ES IMPROVEMENT (ln MLI) as the preferred
    # regressor, distinct from the ES LEVEL used in m1/m3/m4/m7/m8/m9/m10
    # above. Guarded: only fit if es_impr_mean_l1_z actually has enough
    # non-missing variation (it will be all-NaN before the first
    # DEA/ML transition exists for a country, or if get_common_es_frontier
    # ever runs without an ml_index column for some reason).
    if est["es_impr_mean_l1_z"].notna().sum() >= 30 and est["es_impr_mean_l1_z"].nunique() > 5:
        m["m11_es_improvement"] = _fit_or_raise(
            pf.fepois, f"coop_count ~ es_impr_mean_l1_z + innov_mean_l1_z {fe}",
            est.dropna(subset=["es_impr_mean_l1_z"]), cl, "m11_es_improvement")
    else:
        print(f"    [m11_es_improvement] insufficient non-missing ES-improvement (MLI) data "
             f"({est['es_impr_mean_l1_z'].notna().sum()} obs) -- model skipped, not fit on a degenerate regressor.")

    # TASK (reviewer point 7): R&D and patent stock as SEPARATE regressors
    # in one model, instead of the pre-combined composite used in m1/m3/etc.
    if est["patstock_pair_mean_l1_z"].notna().sum() >= 30 and est["patstock_pair_mean_l1_z"].nunique() > 5:
        m["m12_rd_patent_separate"] = _fit_or_raise(
            pf.fepois, f"coop_count ~ es_mean_l1_z + rd_pair_mean_l1_z + patstock_pair_mean_l1_z {fe}",
            est.dropna(subset=["patstock_pair_mean_l1_z"]), cl, "m12_rd_patent_separate")
    else:
        print(f"    [m12_rd_patent_separate] insufficient non-missing patent-stock data "
             f"({est['patstock_pair_mean_l1_z'].notna().sum()} obs) -- model skipped.")

    # TASK (reviewer): "Binary extensive margin: any cooperation versus
    # none." A hurdle-style companion to the count models above: does
    # cooperation START, versus how INTENSE it is conditional on starting.
    est["any_coop"] = (est["coop_count"] > 0).astype(float)
    m["m13_extensive"] = _fit_or_raise(pf.feols, f"any_coop ~ es_mean_l1_z + innov_mean_l1_z {fe}", est, cl, "m13_extensive")

    # TASK (reviewer): "MLI decomposition into efficiency change and
    # technological change." Catch-up (EC) vs frontier-shifting (TC)
    # improvements may relate to cooperation differently.
    if est["ec_pair_mean_l1_z"].notna().sum() >= 30 and est["tc_pair_mean_l1_z"].notna().sum() >= 30:
        m["m14_ec_tc"] = _fit_or_raise(
            pf.fepois, f"coop_count ~ ec_pair_mean_l1_z + tc_pair_mean_l1_z + innov_mean_l1_z {fe}",
            est.dropna(subset=["ec_pair_mean_l1_z", "tc_pair_mean_l1_z"]), cl, "m14_ec_tc")
    else:
        print(f"    [m14_ec_tc] insufficient EC/TC data -- model skipped.")

    # TASK (reviewer): "Patent-stock depreciation at 10%, 15% and 20%."
    # PATENT_STOCK_DELTA (0.15) is already m12's patstock_pair_mean_l1_z;
    # the two additional rates are fit here for direct comparison.
    for d in (10, 20):
        col = f"patstock{d}_pair_mean_l1_z"
        if col in est.columns and est[col].notna().sum() >= 30:
            m[f"m15_patstock_d{d}"] = _fit_or_raise(
                pf.fepois, f"coop_count ~ es_mean_l1_z + {col} {fe}",
                est.dropna(subset=[col]), cl, f"m15_patstock_d{d}")

    # TASK (reviewer): "Placebo leads for both ES and innovation included
    # simultaneously" -- both leads together (m9 above only conditions on
    # the ES lead; this one adds the innovation lead too).
    m["m16_joint_leads"] = _fit_or_raise(
        pf.fepois, f"coop_count ~ es_mean_l1_z + innov_mean_l1_z + es_mean_f1_z + innov_mean_f1_z {fe}",
        est, cl, "m16_joint_leads")

    # TASK (reviewer): "Results with and without simulated trade,
    # volatility and RTA controls" -- the baseline (m1_ppml) already omits
    # these; this adds them back one at a time so their marginal effect on
    # the ES coefficient is visible directly.
    m["m17_with_trade"] = _fit_or_raise(
        pf.fepois, f"coop_count ~ es_mean_l1_z + innov_mean_l1_z + log_trade_l1_z {fe}", est, cl, "m17_with_trade")
    # TASK (reviewer, confirmed empirically): m18_with_vol previously added
    # vol_z as a MAIN EFFECT alongside year fixed effects. Verified
    # directly: its ES/Innovation coefficients were numerically identical
    # to m1_ppml's to 12 decimal places, and pyfixest silently DROPPED
    # vol_z from the coefficient table entirely -- because price
    # volatility is a single annual value (identical across every dyad in
    # a given year), its main effect is perfectly collinear with, and
    # fully absorbed by, the year fixed effects already in {fe}. A model
    # that appeared to "control for volatility" was actually running
    # m1_ppml with an invisible, silently-dropped extra term. Only
    # volatility's INTERACTION with a dyad-varying regressor (which m4_vol
    # already estimates, es_mean_l1_z:vol_z) is identified -- removed
    # rather than kept in a form that cannot be fixed to do what its name
    # implied.
    if est["rta"].notna().sum() > 0 and est.loc[est["rta"].notna(), "rta"].nunique() > 1:
        m["m19_with_rta"] = _fit_or_raise(
            pf.fepois, f"coop_count ~ es_mean_l1_z + innov_mean_l1_z + rta {fe}",
            est.dropna(subset=["rta"]), cl, "m19_with_rta")
    else:
        n_rta_obs = int(est["rta"].notna().sum())
        n_unique = int(est.loc[est["rta"].notna(), "rta"].nunique()) if n_rta_obs > 0 else 0
        print(f"    [m19_with_rta] SKIPPED explicitly: {n_rta_obs} real RTA observations but "
             f"{n_unique} unique value(s) once dyad fixed effects are applied -- insufficient "
             f"within-sample variation to identify anything, not a data-availability problem "
             f"(this is expected for EU, where RTA membership predates the 2008+ estimation "
             f"window for every dyad; see the RTA coverage note printed earlier in this run).")

        # TASK (reviewer): "at minimum, run sensitivity tests excluding
        # dyads known to have left-censored RTA histories." The dyads
        # KNOWN to have had a left-censoring correction applied (Mercosur,
        # NAFTA -- see build_rta_wto_signatory_based) are flagged via
        # left_censoring_corrected; this refits the same model excluding
        # them entirely, showing whether the main RTA result depends on
        # that correction rather than holding regardless.
        if "left_censoring_corrected" in est.columns and est["left_censoring_corrected"].any():
            est_excl = est[~est["left_censoring_corrected"]].dropna(subset=["rta"])
            n_excluded_dyads = est.loc[est["left_censoring_corrected"], ["iso_i", "iso_j"]].drop_duplicates().shape[0]
            print(f"    [m19c_with_rta_excl_left_censored] excluding {n_excluded_dyads} dyad(s) "
                 f"with a known pre-2000 left-censoring correction (Mercosur/NAFTA): "
                 f"{len(est_excl)}/{len(est.dropna(subset=['rta']))} obs remain.")
            if len(est_excl) >= 30:
                m["m19c_with_rta_excl_left_censored"] = _fit_or_raise(
                    pf.fepois, f"coop_count ~ es_mean_l1_z + innov_mean_l1_z + rta {fe}",
                    est_excl, cl, "m19c_with_rta_excl_left_censored")
            else:
                print(f"    [m19c_with_rta_excl_left_censored] SKIPPED explicitly: only "
                     f"{len(est_excl)} observations remain after excluding left-censoring-"
                     f"corrected dyads (need >= 30) -- too few to fit reliably.")

    # TASK: real WTO RTA country-level activity data now available (see
    # load_rta_country_activity()) -- fit alongside, never instead of,
    # m19_with_rta above. Labelled "proxy" throughout since rta_stock_min
    # is a country-level construct (pair minimum of cumulative RTA
    # count), not the true dyadic "do i and j share an agreement"
    # indicator m19_with_rta uses when the genuine Larch file is present.
    if est["rta_stock_min_l1_z"].notna().sum() >= 30:
        m["m19b_with_rta_proxy"] = _fit_or_raise(
            pf.fepois, f"coop_count ~ es_mean_l1_z + innov_mean_l1_z + rta_stock_min_l1_z {fe}",
            est.dropna(subset=["rta_stock_min_l1_z"]), cl, "m19b_with_rta_proxy")
    else:
        print(f"    [m19b_with_rta_proxy] SKIPPED explicitly: only "
             f"{int(est['rta_stock_min_l1_z'].notna().sum())} non-missing observations "
             f"(need >= 30) -- too few to fit reliably.")

    # TASK: real World Bank FDI outflow-intensity data now available (see
    # load_fdi_country_level()) -- a dyadic CONTROL proxy, not a
    # substitute for m5_fdi's genuine bilateral FDI OUTCOME check (which
    # tests a completely different question: can ES/Innovation explain
    # bilateral FDI the way they explain cooperation -- still requires
    # real bilateral data, still absent). This model instead asks: is the
    # ES-cooperation relationship robust to controlling for how
    # internationally financially engaged the pair is on average.
    if est["fdi_out_pair_mean_l1_z"].notna().sum() >= 30:
        m["m21_with_fdi_proxy"] = _fit_or_raise(
            pf.fepois, f"coop_count ~ es_mean_l1_z + innov_mean_l1_z + fdi_out_pair_mean_l1_z {fe}",
            est.dropna(subset=["fdi_out_pair_mean_l1_z"]), cl, "m21_with_fdi_proxy")

    # TASK: real GDELT CAMEO-057 agreement-event activity now available
    # (see load_gdelt_agreement_activity()) -- a dyadic CONTROL proxy for
    # general diplomatic/agreement-signing engagement, NOT energy-specific
    # and NOT dyadic (see that function's docstring). Fit twice: once on
    # the full sample, once excluding any dyad involving Romania (whose
    # GDELT series the SOURCE ITSELF flags as unreliable) -- reported
    # side by side so a reader can see directly whether Romania's
    # questionable data is driving the result, rather than only being
    # warned about it in text.
    if est["gdelt_pair_mean_l1_z"].notna().sum() >= 30:
        est_gdelt = est.dropna(subset=["gdelt_pair_mean_l1_z"])
        m["m22_with_gdelt_proxy"] = _fit_or_raise(
            pf.fepois, f"coop_count ~ es_mean_l1_z + innov_mean_l1_z + gdelt_pair_mean_l1_z {fe}",
            est_gdelt, cl, "m22_with_gdelt_proxy")
        est_gdelt_no_rou = est_gdelt[est_gdelt.get("gdelt_romania_dyad", 0) != 1]
        if len(est_gdelt_no_rou) >= 30 and len(est_gdelt_no_rou) < len(est_gdelt):
            m["m22b_with_gdelt_proxy_no_romania"] = _fit_or_raise(
                pf.fepois, f"coop_count ~ es_mean_l1_z + innov_mean_l1_z + gdelt_pair_mean_l1_z {fe}",
                est_gdelt_no_rou, cl, "m22b_with_gdelt_proxy_no_romania")
        elif len(est_gdelt_no_rou) < 30:
            print(f"    [m22b_with_gdelt_proxy_no_romania] SKIPPED explicitly: only "
                 f"{len(est_gdelt_no_rou)} observations remain after excluding Romania (need "
                 f">= 30) -- too few to fit reliably.")
        # else: no Romania dyads present to exclude in the first place -- this club's own
        # m22_with_gdelt_proxy above already covers the identical sample, so no separate
        # "no_romania" variant is needed or run.

    # TASK (reviewer): "Alternative event definitions: verbal, material and
    # high-intensity cooperation." Since the active outcome (real or
    # simulated) is a single undifferentiated count, these are constructed
    # as stylised SPLITS of that count -- not independently observed
    # sub-types -- and reported as such, not as if genuinely distinct
    # verbal/material data existed. "Verbal" = thinned low-intensity share;
    # "material" = the complementary higher-intensity share; "high-
    # intensity" = binary indicator for count >= 2.
    if est["coop_count"].max() >= 1:
        est["event_verbal"] = rng.binomial(est["coop_count"].astype(int).clip(0, 50), 0.5)
        est["event_material"] = est["coop_count"] - est["event_verbal"]
        est["event_high_intensity"] = (est["coop_count"] >= 2).astype(float)
        for outcome, model_key in [("event_verbal", "m20_verbal"), ("event_material", "m20_material")]:
            try:
                _diagnose_sparsity(est.assign(**{outcome: est[outcome]}), outcome)
                m[model_key] = _fit_or_raise(pf.fepois, f"{outcome} ~ es_mean_l1_z + innov_mean_l1_z {fe}", est, cl, model_key)
            except ValueError as e:
                print(f"    [{model_key}] {e}")
        m["m20_high_intensity"] = _fit_or_raise(pf.feols, f"event_high_intensity ~ es_mean_l1_z + innov_mean_l1_z {fe}", est, cl, "m20_high_intensity")

    # FIX (reviewer: "the gravity model reports coefficients for distance,
    # common language and contiguity... a model with genuine dyad fixed
    # effects cannot estimate their coefficients: they must be absorbed").
    # This was a real specification bug: mA_gravity previously used
    # "| iso_i + iso_j + year" (separate exporter/importer/year FE, a
    # three-way structural-gravity design) instead of the SAME
    # "| dyad_id + year" genuine dyad FE used by every other model in this
    # suite -- which is exactly why distance/comlang/contig did not get
    # absorbed and could be (mis)reported. Fixed to use the same `fe` as
    # every other model; distance/comlang/contig are dropped entirely
    # (they are time-invariant within a dyad and are, correctly, perfectly
    # collinear with dyad fixed effects -- not estimated, not needed).
    # Time-varying controls only: RTA and trade.
    #
    # FIX (reviewer, this turn): "verify this model is actually PPML if
    # presented as a PPML gravity specification." It was not -- confirmed
    # directly: mA_gravity uses pf.feols on log1p_coop, a genuine FE-OLS
    # specification (log1p_coop only makes sense as an OLS dependent
    # variable; PPML's whole point is to model the count directly without
    # a log transform). It was never mislabeled as PPML in a table
    # (it doesn't appear in main_results/robustness.md at all), but its
    # figure/robustness-plot label ("Gravity (dyad FE, no distance/lang)")
    # did not say which estimator it was, unlike m1's explicit "Baseline
    # PPML" -- an omission that invites exactly this reasonable
    # misreading. Renamed to mA_gravity_feols throughout, and a genuine
    # PPML gravity specification (mA_gravity_ppml, same RTA+trade controls,
    # fit on coop_count directly) is added alongside it, so a real "PPML
    # gravity specification" actually exists rather than only implying one.
    rta_valid = est["rta"].notna()
    rta_n, rta_total = int(rta_valid.sum()), len(est)
    if rta_n > 0 and est.loc[rta_valid, "rta"].nunique() > 1:
        est_rta = est.loc[rta_valid].copy()
        print(f"    [mA_gravity] RTA: fitting on {rta_n}/{rta_total} obs with a genuine "
             f"(non-missing) RTA observation ({rta_n/rta_total:.1%}); the rest excluded, not zero-filled.")
        m["mA_gravity_feols"] = _fit_or_raise(
            pf.feols, f"log1p_coop ~ es_mean_l1_z + innov_mean_l1_z + log_trade_l1_z + rta {fe}",
            est_rta, cl, "mA_gravity_feols")
        m["mA_gravity_ppml"] = _fit_or_raise(
            pf.fepois, f"coop_count ~ es_mean_l1_z + innov_mean_l1_z + log_trade_l1_z + rta {fe}",
            est_rta, cl, "mA_gravity_ppml")
    else:
        print(f"    [mA_gravity] RTA: {rta_n}/{rta_total} genuine observations, insufficient "
             f"variation -- 'rta' omitted from the gravity model (not fit as a constant, not zero-filled).")
        m["mA_gravity_feols"] = _fit_or_raise(
            pf.feols, f"log1p_coop ~ es_mean_l1_z + innov_mean_l1_z + log_trade_l1_z {fe}",
            est, cl, "mA_gravity_feols")
        m["mA_gravity_ppml"] = _fit_or_raise(
            pf.fepois, f"coop_count ~ es_mean_l1_z + innov_mean_l1_z + log_trade_l1_z {fe}",
            est, cl, "mA_gravity_ppml")
    m["m1_cl2"] = _fit_or_raise(pf.fepois, f"coop_count ~ es_mean_l1_z + innov_mean_l1_z {fe}", est, cl2, "m1_cl2")
    return m, est


LABELS = {"es_mean_l1_z": "ES (pair mean, t-1)", "innov_mean_l1_z": "Innovation (t-1)",
          "es_mean_l1_z:innov_mean_l1_z": "ES x Innovation", "es_mean_l1_z:vol_z": "ES x Brent turbulence",
          "es_mean_l2_z": "ES (t-2)", "innov_mean_l2_z": "Innovation (t-2)",
          "es_mean_f1_z": "ES (t+1, placebo)", "innov_mean_f1_z": "Innovation (t+1)",
          "es_min_l1_z": "ES (pair min)", "innov_min_l1_z": "Innovation (pair min)",
          "es_diff_l1_z": "ES (pair diff)", "innov_diff_l1_z": "Innovation (pair diff)",
          "log_dist": "log distance", "comlang": "common language", "contig": "contiguity",
          "log_trade_l1_z": "log trade (t-1)"}


def write_table(out_dir, models, heads, stub, title):
    import pyfixest as pf
    (out_dir / "tables" / f"{stub}.tex").write_text(
        str(pf.etable(models, type="tex", labels=LABELS, model_heads=heads, notes=title)))
    dft = pf.etable(models, type="df", labels=LABELS, model_heads=heads)
    (out_dir / "tables" / f"{stub}.md").write_text(f"**{title}**\n\n" + dft.to_markdown())


def run_events_agreements_mc_validation(club_key, B=200):
    """
    TASK (reviewer): "Separate Monte Carlo validation for events and
    agreements." Builds the club's real ES/Innovation regressors and fixed
    effects ONCE (same pattern as the pooled Monte Carlo in Section 8C),
    then for B replications draws BOTH energy_events_sim (dense, NB) and
    formal_agreements_sim (sparse) from the SAME underlying signal and
    fixed effects, fitting the baseline model on each -- so their bias,
    coverage, and power can be compared directly under conditions that
    differ ONLY in outcome sparsity, not in the underlying regressors.
    Also reports, per replication, how many dyads are always-zero and
    therefore effectively dropped by the dyad fixed effects -- expected to
    be far higher for the sparse agreements outcome.
    """
    print("=" * 70)
    print(f"SEPARATE MONTE CARLO VALIDATION: EVENTS vs AGREEMENTS ({club_key.upper()}, B={B})")
    print("=" * 70)
    club = CLUBS[club_key]
    members = list(club["members_map"].keys())
    prov = []
    es = compute_es_dea(members, club["out_dir"], prov, club_key=club_key)
    cy = get_country_year(club, es, prov, club_key=club_key)
    dyad_static = get_distance(club, prov)
    vol = get_volatility(prov)

    dyads = pd.DataFrame(list(combinations(members, 2)), columns=["iso_i", "iso_j"])
    f = dyads.merge(pd.DataFrame({"year": YEARS}), how="cross")
    f["dyad_id"] = f.iso_i + "_" + f.iso_j
    li = cy.rename(columns={"iso": "iso_i", "es": "es_i", "rd_gdp": "rd_i"})
    lj = cy.rename(columns={"iso": "iso_j", "es": "es_j", "rd_gdp": "rd_j"})
    f = f.merge(li[["iso_i", "year", "es_i", "rd_i"]], on=["iso_i", "year"], how="left")
    f = f.merge(lj[["iso_j", "year", "es_j", "rd_j"]], on=["iso_j", "year"], how="left")
    f["es_mean"] = (f.es_i + f.es_j) / 2
    f["innov_mean"] = (f.rd_i + f.rd_j) / 2
    f = f.merge(vol, on="year", how="left")
    f = f.sort_values(["dyad_id", "year"])
    g = f.groupby("dyad_id", sort=False)
    f["es_mean_l1_z"] = zscore(g["es_mean"].shift(1))
    f["innov_mean_l1_z"] = zscore(g["innov_mean"].shift(1))
    f["vol_z"] = zscore(f["price_volatility"])
    f = f.dropna(subset=["es_mean_l1_z", "innov_mean_l1_z"]).copy()

    b0 = club["baseline_intercept"]
    dfe = f["dyad_id"].map({d: rng.normal(0, 0.5) for d in f.dyad_id.unique()}).to_numpy()
    yfe = f["year"].map({y: rng.normal(0, 0.3) for y in YEARS}).to_numpy()
    es_l1 = f["es_mean_l1_z"].to_numpy(); in_l1 = f["innov_mean_l1_z"].to_numpy(); volz = f["vol_z"].fillna(0).to_numpy()
    TRUE_ES_PERCLUB = 0.35  # matches build_panel's simulated DGP exactly
    idx = b0 + TRUE_ES_PERCLUB * es_l1 + 0.20 * in_l1 + 0.15 * es_l1 * in_l1 + 0.10 * es_l1 * volz + dfe + yfe

    import pyfixest as pf
    # TASK: two DISTINCT experiments, not one mislabeled comparison.
    #   1. CORRECT-SPECIFICATION VALIDATION (primary): estimate the SAME
    #      complete specification used to generate the data (ES, Innovation,
    #      ES x Innovation, ES x Volatility). Each coefficient compared to
    #      its OWN true value -- this is the genuine estimator-recovery test.
    #   2. MISSPECIFICATION EXPERIMENT: estimate only ES + Innovation
    #      (omitting the true interactions) on the SAME data. The resulting
    #      gap is NOT reported as "bias" -- under omitted, correlated
    #      interaction terms, PPML converges to a pseudo-true composite
    #      effect, not beta_ES itself (a real, distinct statistical fact,
    #      not a labeling nicety). Reported as "omitted-moderation
    #      distortion" explicitly, to demonstrate why the interactions
    #      matter, not to imply the estimator is broken.
    formula_full = ("y ~ es_mean_l1_z + innov_mean_l1_z + es_mean_l1_z:innov_mean_l1_z "
                    "+ es_mean_l1_z:vol_z | dyad_id + year")
    formula_misspec = "y ~ es_mean_l1_z + innov_mean_l1_z | dyad_id + year"
    TRUE_FULL = {"es_mean_l1_z": TRUE_ES_PERCLUB, "innov_mean_l1_z": 0.20,
                "es_mean_l1_z:innov_mean_l1_z": 0.15, "es_mean_l1_z:vol_z": 0.10}

    results = {}
    for label in ("events", "agreements", "agreements_stock"):
        results[label] = {"full": {p: {"est": [], "se": []} for p in TRUE_FULL},
                          "misspec": {"est": [], "se": []}, "always_zero": []}
    n_dyads = f["dyad_id"].nunique()
    stock_delta = 0.15
    f = f.sort_values(["dyad_id", "year"]).reset_index(drop=True)
    dyad_groups = {d: f.index.get_indexer(idxs) for d, idxs in f.groupby("dyad_id", sort=False).groups.items()}

    def _fit_both_specs(label, y_vals, verbose=False):
        f["y"] = y_vals
        results[label]["always_zero"].append(int((f.groupby("dyad_id")["y"].max() == 0).sum()))
        try:
            if verbose:
                print(f"[FIT START] m_full (misspec-experiment correct-spec, {label}): rows={len(f)}", flush=True)
            m_full = pf.fepois(formula_full, f, vcov={"CRV1": "dyad_id"}, demeaner=LsmrDemeaner(backend="within"))
            if verbose:
                print("[FIT OK] m_full", flush=True)
            for p in TRUE_FULL:
                if p in m_full.coef().index:
                    results[label]["full"][p]["est"].append(float(m_full.coef()[p]))
                    results[label]["full"][p]["se"].append(float(m_full.se()[p]))
        except Exception:
            pass
        try:
            if verbose:
                print(f"[FIT START] m_mis (misspec-experiment omitted-interaction, {label}): rows={len(f)}", flush=True)
            m_mis = pf.fepois(formula_misspec, f, vcov={"CRV1": "dyad_id"}, demeaner=LsmrDemeaner(backend="within"))
            if verbose:
                print("[FIT OK] m_mis", flush=True)
            if "es_mean_l1_z" in m_mis.coef().index:
                results[label]["misspec"]["est"].append(float(m_mis.coef()["es_mean_l1_z"]))
                results[label]["misspec"]["se"].append(float(m_mis.se()["es_mean_l1_z"]))
        except Exception:
            pass

    for b in range(B):
        v = (b % 25 == 0)
        _fit_both_specs("events", _simulate_negbin(np.exp(idx), dispersion=0.5, rng=rng), verbose=v)
        agree_flow = rng.poisson(np.exp(np.clip(idx - 3.2, -10, 10)))
        _fit_both_specs("agreements", agree_flow, verbose=v)
        stock_vals = np.zeros(len(f))
        for d, ix in dyad_groups.items():
            s = 0.0
            flows = agree_flow[ix]
            stock_series = np.empty(len(flows))
            for k, flow in enumerate(flows):
                s = (1 - stock_delta) * s + flow
                stock_series[k] = s
            stock_vals[ix] = stock_series
        _fit_both_specs("agreements_stock", stock_vals, verbose=v)

    print("\n" + "=" * 70)
    print("EXPERIMENT 1 (PRIMARY): CORRECT-SPECIFICATION VALIDATION")
    print("=" * 70)
    print("  Estimated model matches the DGP exactly (ES, Innovation, ES x Innovation,")
    print("  ES x Volatility all included). Each coefficient compared to ITS OWN true")
    print("  value -- this is the genuine estimator-recovery test.")
    full_rows = []
    for label in ("events", "agreements", "agreements_stock"):
        print(f"\n  --- {label} ---")
        print(f"  {'Parameter':30s} {'True':>8s} {'Mean est':>10s} {'Bias':>9s} {'RMSE':>8s} {'Coverage':>9s} {'Power':>7s} {'n':>5s}")
        for p, true_v in TRUE_FULL.items():
            vals = np.array(results[label]["full"][p]["est"]); ses_ = np.array(results[label]["full"][p]["se"])
            if len(vals) == 0:
                print(f"  {p:30s}  -- all replications failed to fit --")
                continue
            bias = vals.mean() - true_v
            rmse = math.sqrt(np.mean((vals - true_v) ** 2))
            covered = np.mean((vals - 1.96 * ses_ <= true_v) & (true_v <= vals + 1.96 * ses_))
            power = np.mean(np.abs(vals / ses_) > 1.96)
            print(f"  {p:30s} {true_v:+8.4f} {vals.mean():+10.4f} {bias:+9.4f} {rmse:8.4f} {covered:9.1%} {power:6.1%} {len(vals):5d}")
            full_rows.append({"outcome": label, "parameter": p, "true_value": true_v, "mean_estimate": vals.mean(),
                              "bias": bias, "rmse": rmse, "coverage": covered, "power": power, "n_fit": len(vals)})
        mean_az = np.mean(results[label]["always_zero"])
        print(f"  (mean all-zero dyads: {mean_az:.1f}/{n_dyads})")

    print("\n" + "=" * 70)
    print("EXPERIMENT 2: DELIBERATE MISSPECIFICATION (omitted-moderation distortion)")
    print("=" * 70)
    print("  TASK: generated WITH interactions, estimated WITHOUT them (ES + Innovation")
    print("  only). The gap below is NOT estimator bias -- under an omitted, correlated")
    print("  interaction term, PPML converges to a pseudo-true COMPOSITE effect, not the")
    print("  original beta_ES. Labeled 'omitted-moderation distortion' throughout, never")
    print("  'bias', per the reviewer's precise distinction. This experiment exists to")
    print("  show WHY the interaction terms matter, not to evaluate the estimator itself")
    print("  -- Experiment 1 above is the actual estimator-recovery test.")
    misspec_rows = []
    print(f"\n  {'Outcome':18s} {'True beta_ES':>13s} {'Mean est':>10s} {'Distortion':>12s} {'n':>5s}")
    for label in ("events", "agreements", "agreements_stock"):
        vals = np.array(results[label]["misspec"]["est"])
        if len(vals) == 0:
            print(f"  {label:18s}  -- all replications failed to fit --")
            continue
        distortion = vals.mean() - TRUE_ES_PERCLUB
        print(f"  {label:18s} {TRUE_ES_PERCLUB:13.4f} {vals.mean():+10.4f} {distortion:+12.4f} {len(vals):5d}")
        misspec_rows.append({"outcome": label, "true_beta_es": TRUE_ES_PERCLUB, "mean_estimate": vals.mean(),
                             "omitted_moderation_distortion": distortion, "n_fit": len(vals)})
    print(f"\n  This distortion reflects the OMITTED ES x Innovation and ES x Volatility terms")
    print(f"  being correlated with es_mean_l1_z in this DGP -- it demonstrates the")
    print(f"  practical importance of m3_innov/m4_vol (which include the interactions)")
    print(f"  alongside the main-effects-only m1_ppml, not a defect in PPML itself.")

    if any(r["outcome"] == "agreements_stock" for r in full_rows) and any(r["outcome"] == "agreements" for r in full_rows):
        stock_power = next(r for r in full_rows if r["outcome"] == "agreements_stock" and r["parameter"] == "es_mean_l1_z")["power"]
        count_power = next(r for r in full_rows if r["outcome"] == "agreements" and r["parameter"] == "es_mean_l1_z")["power"]
        print(f"\n  IMPORTANT, TESTED RATHER THAN ASSUMED: the active-stock construction reduces "
             f"the zero share substantially (see the per-club run's outcome distributions), but "
             f"here it does NOT translate into better power for the ES coefficient under the "
             f"CORRECT specification either -- stock power={stock_power:.1%} vs count "
             f"power={count_power:.1%}. The likely reason: Stock_t=(1-delta)*Stock_{{t-1}}+Flow_t "
             f"accumulates agreements driven by THEIR OWN historical ES values, not the CURRENT "
             f"es_mean_l1_z regressor -- 'more variation' from reduced sparsity is largely "
             f"persistence/memory of past draws, not contemporaneous signal.")

    out_dir = club["out_dir"] / "tables"; out_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(full_rows).to_csv(out_dir / "mc_correct_specification.csv", index=False)
    pd.DataFrame(misspec_rows).to_csv(out_dir / "mc_misspecification_distortion.csv", index=False)
    return pd.DataFrame(full_rows), pd.DataFrame(misspec_rows)


def run_block_bootstrap_dea(club_key, B=50, df_main=None):
    """
    TASK (reviewer): "Block bootstrap that recalculates DEA/MLI before
    rebuilding the dyadic panel." Standard approach for bootstrapping a
    DEA-based regressor (Simar & Wilson, 1998, 2000): the ORIGINAL
    countries are NOT resampled themselves (that would break the dyadic
    panel's country identities); instead, for each year, the REFERENCE SET
    used to define the efficient frontier is resampled WITH REPLACEMENT,
    and every original country is re-scored (via the same _ddf distance
    function) against this new bootstrap frontier. The dyadic panel is
    then rebuilt from these bootstrap ES scores (same dyad structure,
    same cooperation outcome -- only the DEA-derived regressor changes),
    and the baseline model is refit. This isolates how much of the final
    ES-coefficient's sampling uncertainty comes from DEA-frontier
    estimation error specifically, on top of (not instead of) the
    dyad-clustered SE already reported elsewhere.

    df_main: pass the panel ALREADY built by run_club() (df) when calling
    from there, so the fixed cooperation outcome is bit-for-bit identical
    to what the reported main coefficient was estimated on -- not just the
    same data-generating process redrawn with different random noise.
    Only calls build_panel() itself (a fresh, independently-drawn
    simulation if cooperation is simulated) when used standalone.
    """
    print("=" * 70)
    print(f"SENSITIVITY OF PPML ESTIMATES TO DEA-FRONTIER RE-ESTIMATION ({club_key.upper()}, B={B})")
    print("=" * 70)
    if load_ch2_dea_mpi() is not None:
        print("  IMPORTANT MISMATCH, flagged directly rather than silently run: this function's")
        print("  reference-set resampling re-solves THIS PIPELINE'S OWN LP-based DEA (Chapter 3")
        print("  WDI extract), but the PRIMARY ES source used everywhere else in this run is now")
        print("  Chapter 2's bootstrap bias-corrected DEA/MPI data (see get_common_es_frontier).")
        print("  The 'Original' point estimate below uses Chapter 2 ES; the resampled estimates")
        print("  do NOT -- they test sensitivity of a now-secondary computation, not the ES score")
        print("  actually used in the reported results. Chapter 2's own dea_lo/dea_hi bounds")
        print("  (already a proper Simar-Wilson bootstrap CI) are a more direct, better-justified")
        print("  sensitivity check for the ACTUALLY-used ES source -- see the comparison below.")
    print("  TASK (reviewer): relabeled from \"block bootstrap\" / \"sampling distribution\".")
    print("  This resamples the DEA REFERENCE SET only, while the cooperation outcome is")
    print("  held FIXED at its original (non-resampled) values. That is a genuine,")
    print("  informative sensitivity check on how much the ES-derived regressor's own")
    print("  estimation uncertainty could shift the PPML coefficient -- but holding the")
    print("  outcome fixed while its generating regressor changes is NOT equivalent to")
    print("  drawing a new sample from the joint ES-cooperation process, so the resulting")
    print("  percentile range is NOT a valid sampling-distribution CI for the coefficient")
    print("  and must not be reported as the main result's confidence interval. Simar &")
    print("  Wilson (1998, 2000) require a justified frontier DGP and matching estimator")
    print("  for a valid nonparametric-frontier bootstrap; naive reference-set resampling")
    print("  does not automatically satisfy that, and is not claimed to here.")
    club = CLUBS[club_key]
    members = list(club["members_map"].keys())
    club_to_plain = {v: k for k, v in EU_CH3_CODE_MAP.items()}

    # FIX: this used to build the reference set from ONLY this club's own
    # members (e.g. 19 for G20), a DIFFERENT frontier from the common
    # 38-country union frontier get_common_es_frontier() uses everywhere
    # else in this pipeline (the same club-specific-frontier bug an
    # earlier reviewer flagged and this file already fixed once for the
    # main analysis -- reintroduced here in a second, separate function).
    # Confirmed by direct test: an IDENTITY resample (no perturbation)
    # against the club-only reference set did NOT reproduce the main
    # analysis's ES coefficient (0.526 vs 0.591), proving this was a real
    # frontier-definition mismatch, not just bootstrap noise. Fixed to use
    # the canonical union, exactly like every other ES computation here.
    plain_members = _canonical_union_members()

    real_df = _ch3_embedded_to_df_pooled(plain_members)
    energy = build_real_es_variables(real_df)
    Xc, Yc, Bc = ["unit_input"], ["self_suff", "renew_share_pct"], ["co2_pc", "td_loss_pct"]
    by_year = {y: energy[energy.year == y].set_index("iso") for y in YEARS if (energy.year == y).any()}
    by_year = {y: d for y, d in by_year.items() if len(d) >= 3}
    if not by_year:
        print("  Insufficient real ES data to bootstrap -- skipped.")
        return None

    # FIX (reviewer, confirmed correct): the previous version fell back to
    # "skeleton['coop_count'] = rng.poisson(1.0, len(skeleton))" whenever no
    # real cooperation file was present -- pure noise with ZERO relationship
    # to ES, Innovation, or anything else, completely disconnected from
    # whatever run_club()'s own main model actually fit. That explained the
    # reported mismatch exactly (e.g. G20 main ES=+0.659 vs bootstrap
    # mean=-0.009: a coefficient estimated on random noise centres near
    # zero regardless of the true relationship, which is not a "different
    # but valid" estimate, it is a different, uninformative outcome).
    #
    # FIX: reuse build_panel()'s output DIRECTLY -- passed in as df_main
    # when called from run_club() (bit-for-bit the same coop_count as the
    # reported main coefficient), or built fresh here only for standalone
    # use. Either way this is the SAME function run_club() uses, never a
    # second, parallel, potentially-diverging construction.
    if df_main is None:
        df_main, prov_main = build_panel(club_key)
    else:
        prov_main = []
    skeleton = df_main[["iso_i", "iso_j", "year", "dyad_id", "innov_mean", "coop_count"]].drop_duplicates(
        subset=["iso_i", "iso_j", "year"]).copy()
    coop_source = next((r["source"] for r in prov_main if r.get("variable", "").startswith("cooperation")),
                       "same as run_club()'s main analysis (identical coop_count values, not recomputed)")
    print(f"  Cooperation outcome held fixed across replications: {coop_source}")

    import pyfixest as pf
    formula = "coop_count ~ es_mean_l1_z + innov_mean_l1_z | dyad_id + year"
    boot_estimates = []
    n_failed = 0
    for b in range(B):
        beta_boot = {}
        for y, d in by_year.items():
            n = len(d)
            boot_idx = rng.choice(n, size=n, replace=True)
            Xb, Yb, Bb = d[Xc].to_numpy()[boot_idx], d[Yc].to_numpy()[boot_idx], d[Bc].to_numpy()[boot_idx]
            for m in d.index:  # score EVERY ORIGINAL country against the bootstrap reference set
                o = d.loc[m]
                beta_boot[(m, y)] = _ddf(o[Xc].to_numpy(), o[Yc].to_numpy(), o[Bc].to_numpy(), Xb, Yb, Bb)
        es_boot = pd.DataFrame([{"iso": mm, "year": y, "es": 1.0 / (1.0 + bb) if bb == bb else np.nan}
                               for (mm, y), bb in beta_boot.items()])
        plain_to_club = {v: k for k, v in club_to_plain.items()}
        es_boot["iso_club"] = es_boot["iso"].map(lambda x: plain_to_club.get(x, x))

        f = skeleton.copy()
        li_es = es_boot.rename(columns={"iso_club": "iso_i", "es": "es_i"})[["iso_i", "year", "es_i"]]
        lj_es = es_boot.rename(columns={"iso_club": "iso_j", "es": "es_j"})[["iso_j", "year", "es_j"]]
        f = f.merge(li_es, on=["iso_i", "year"], how="left").merge(lj_es, on=["iso_j", "year"], how="left")
        f["es_mean"] = (f.es_i + f.es_j) / 2
        f = f.sort_values(["dyad_id", "year"])
        g = f.groupby("dyad_id", sort=False)
        f["es_mean_l1_z"] = zscore(g["es_mean"].shift(1))
        f["innov_mean_l1_z"] = zscore(g["innov_mean"].shift(1))
        est_b = f.dropna(subset=["es_mean_l1_z", "innov_mean_l1_z", "coop_count"])
        try:
            print(f"[FIT START] DEA-bootstrap replication refit: rows={len(est_b)}", flush=True)
            m = pf.fepois(formula, est_b, vcov={"CRV1": "dyad_id"}, demeaner=LsmrDemeaner(backend="within"))
            print("[FIT OK] DEA-bootstrap replication refit", flush=True)
            if "es_mean_l1_z" in m.coef().index:
                boot_estimates.append(float(m.coef()["es_mean_l1_z"]))
            else:
                n_failed += 1
        except Exception:
            n_failed += 1

    if not boot_estimates:
        print("  All bootstrap replications failed to fit -- reporting nothing.")
        return None
    boot_arr = np.array(boot_estimates)
    orig_est = df_main.dropna(subset=["es_mean_l1_z", "innov_mean_l1_z", "coop_count"]).pipe(
        lambda d: pf.fepois("coop_count ~ es_mean_l1_z + innov_mean_l1_z | dyad_id + year", d,
                           vcov={"CRV1": "dyad_id"}).coef()["es_mean_l1_z"])
    print(f"\n  {len(boot_estimates)}/{B} replications converged ({n_failed} failed)")
    print(f"  Original (non-bootstrapped, common-frontier) ES coefficient: {orig_est:+.4f}")
    print(f"  Distribution of the ES coefficient under DEA reference-set resampling:")
    print(f"    mean={boot_arr.mean():+.4f}  sd={boot_arr.std():.4f}  "
         f"2.5-97.5 pct CI=[{np.percentile(boot_arr, 2.5):+.4f}, {np.percentile(boot_arr, 97.5):+.4f}]")
    if B < 200:
        print(f"  WARNING (reviewer, correct): B={B} is FAR too small for the percentile range "
             f"above to be credible -- a 2.5th percentile from {B} draws is effectively set by "
             f"the single lowest observation, not a genuine tail estimate. This run is a quick, "
             f"directional check only (kept small here so it runs as part of the default per-club "
             f"pipeline in reasonable time -- roughly 3s/replication makes B=999 take about an "
             f"hour). Call run_block_bootstrap_dea(club_key, B=999) directly (or B=2000 for a more")
        print(f"  precise range) for a result whose percentile range is actually usable.")
    if abs(boot_arr.mean() - orig_est) > boot_arr.std():
        print(f"  NOTE: the mean above sits notably below the original estimate. VERIFIED (not "
             f"assumed) that the ES-recomputation machinery itself is correct: an IDENTITY "
             f"resample (the unperturbed reference set, zero bootstrap noise) reproduces the "
             f"original estimate EXACTLY. But per the reviewer's correction, the SIZE and even "
             f"the DIRECTION of the gap should NOT be presented as a known, expected property of "
             f"DEA bootstrapping in general -- Simar & Wilson require a justified frontier "
             f"data-generating process and matching estimator for that claim, which this simple "
             f"reference-set resampling has not been shown to satisfy. Two further caveats this "
             f"procedure does NOT resolve: (1) the cooperation outcome is held FIXED at its "
             f"original values while the ES regressor that generated it is resampled -- this is a "
             f"measurement-error sensitivity experiment on the existing sample, not equivalent to "
             f"drawing a new sample from the joint ES-cooperation process; (2) this conflates two "
             f"distinct questions -- how uncertain the DEA frontier is, and how that uncertainty "
             f"propagates into the PPML coefficient -- that a rigorous treatment would keep "
             f"separate. The ORIGINAL point estimate above remains the one used in every other "
             f"results table in this pipeline; this section is a sensitivity check, not an "
             f"alternative estimate or confidence interval.")
    print(f"  Compare this bootstrap SD to the dyad-clustered SE reported elsewhere for the same "
         f"coefficient: if the bootstrap SD is much larger, DEA-frontier estimation error is a "
         f"material, previously-unquantified source of uncertainty this pipeline's other "
         f"inference methods do not capture at all.")

    # TASK: when Chapter 2 data is the active ES source, its OWN dea_lo/
    # dea_hi bounds (a genuine Simar-Wilson bootstrap CI, already computed
    # by that chapter's own methodology) give a MORE DIRECT, better-
    # justified sensitivity check than the reference-set resampling above,
    # which -- as flagged at the top of this function -- no longer tests
    # the ES source actually in use. Refits the SAME model with es
    # replaced by dea_lo, then by dea_hi, holding everything else (the
    # cooperation outcome, Innovation) fixed at df_main's own values.
    ch2 = load_ch2_dea_mpi()
    if ch2 is not None:
        p_dea = DATA / "step5_dea_scores.csv"
        raw_bounds = pd.read_csv(p_dea)
        union_members = set(_canonical_union_members())
        raw_bounds = raw_bounds[raw_bounds.iso_code.isin(union_members)][["iso_code", "year", "dea_lo", "dea_hi"]]
        raw_bounds = raw_bounds.rename(columns={"iso_code": "iso"})
        plain_to_club_bounds = {v: k for k, v in club_to_plain.items()}
        raw_bounds["iso"] = raw_bounds["iso"].map(lambda x: plain_to_club_bounds.get(x, x))

        print(f"\n  Chapter 2's OWN bootstrap CI bounds (dea_lo/dea_hi, Simar-Wilson), a more "
             f"direct check for the ES source actually in use:")
        for bound_col, label in [("dea_lo", "lower"), ("dea_hi", "upper")]:
            bi = raw_bounds.rename(columns={"iso": "iso_i", bound_col: "es_bound_i"})[["iso_i", "year", "es_bound_i"]]
            bj = raw_bounds.rename(columns={"iso": "iso_j", bound_col: "es_bound_j"})[["iso_j", "year", "es_bound_j"]]
            fb = df_main.copy()
            fb = fb.merge(bi, on=["iso_i", "year"], how="left").merge(bj, on=["iso_j", "year"], how="left")
            fb["es_mean_bound"] = (fb["es_bound_i"] + fb["es_bound_j"]) / 2
            fb = fb.sort_values(["dyad_id", "year"])
            gb = fb.groupby("dyad_id", sort=False)
            fb["es_mean_bound_l1_z"] = zscore(gb["es_mean_bound"].shift(1))
            est_b = fb.dropna(subset=["es_mean_bound_l1_z", "innov_mean_l1_z", "coop_count"])
            try:
                print(f"[FIT START] es-bound sensitivity refit: rows={len(est_b)}", flush=True)
                mb = pf.fepois("coop_count ~ es_mean_bound_l1_z + innov_mean_l1_z | dyad_id + year",
                              est_b, vcov={"CRV1": "dyad_id"}, demeaner=LsmrDemeaner(backend="within"))
                print("[FIT OK] es-bound sensitivity refit", flush=True)
                if "es_mean_bound_l1_z" in mb.coef().index:
                    print(f"    Using {label} bound (dea_{'lo' if label=='lower' else 'hi'}): "
                         f"ES coefficient = {mb.coef()['es_mean_bound_l1_z']:+.4f}")
            except Exception as e:
                print(f"    Using {label} bound: fit failed ({e})")
        print(f"  Compare this range to the 'Original' point estimate above: this reflects "
             f"Chapter 2's own documented DEA-frontier uncertainty for the ES source THIS run "
             f"actually used, unlike the reference-set resampling earlier in this function.")

    out_dir = club["out_dir"] / "tables"; out_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({"boot_estimate": boot_arr}).to_csv(out_dir / "block_bootstrap_dea.csv", index=False)
    return boot_arr


def run_panel_granger_test(club_key, df, min_T=6, B=999):
    """
    TASK: the chapter's own abstract/methodology promises "panel Granger
    non-causality tests" (in the spirit of Dumitrescu & Hurlin, 2012) --
    confirmed by direct search that no such test existed anywhere in this
    pipeline before this function. Implemented here as a genuine,
    validated procedure, with one explicit, honest adaptation from DH's
    own published method:

    DH (2012)'s test standardises the average per-unit Wald statistic
    using an asymptotic formula with small-T correction CONSTANTS that
    depend on T and the lag order K in a way this implementation cannot
    independently verify against a reference implementation in this
    environment (no R plm/dynpanel package available to cross-check).
    Rather than risk publishing an authoritative-looking p-value built on
    a possibly-misremembered constant, the exact same underlying test
    statistic (the average per-dyad Wald statistic for the lagged-ES
    coefficient, testing whether ES has predictive content for
    cooperation beyond cooperation's own persistence) is used, but its
    null distribution is obtained by PERMUTATION -- reassigning which
    dyad's ES history feeds which dyad's outcome history, refitting, and
    repeating B times. This is validated below on toy cases (a strong
    true signal and a no-signal null) before being reported.

    Per-dyad regression (K=1 lag, linear -- DH's own theory is built for
    linear panel VAR, so a linear specification is used for this test
    specifically, distinct from the PPML models used everywhere else):
        coop_count_t = alpha_i + rho_i * coop_count_{t-1} + gamma_i * ES_mean_{t-1} + e_{i,t}
    H0 (non-causality): gamma_i = 0 for every dyad i.
    """
    print("=" * 70)
    print(f"PANEL GRANGER NON-CAUSALITY TEST ({club_key.upper()}, permutation-based, B={B})")
    print("=" * 70)
    print("  Adapted from Dumitrescu & Hurlin (2012): same average-Wald-statistic idea,")
    print("  but the null distribution is obtained by permutation rather than DH's own")
    print("  asymptotic small-T-correction formula (not independently verifiable in this")
    print("  environment) -- validated on toy signal/no-signal cases before being used here.")

    f = df.sort_values(["dyad_id", "year"]).copy()
    f["coop_l1"] = f.groupby("dyad_id")["coop_count"].shift(1)
    valid = f.dropna(subset=["coop_count", "coop_l1", "es_mean_l1_z"])
    dyad_groups = {d: g for d, g in valid.groupby("dyad_id") if len(g) >= min_T}
    if len(dyad_groups) < 5:
        print(f"  Only {len(dyad_groups)} dyads have >= {min_T} time points -- too few for a "
             f"credible panel test. Skipped.")
        return None

    def _wald_stat(y, y_l1, x_l1):
        n = len(y)
        X = np.column_stack([np.ones(n), y_l1, x_l1])
        # FIX: near-singular designs (common under permutation, when the
        # reassigned ES series happens to be highly collinear with the
        # dyad's own coop_l1 for that particular random pairing) produced
        # astronomically large, numerically meaningless Wald statistics
        # (observed directly: a permutation-null mean of ~7.9e24) that
        # would silently distort the permutation null. Guarded here via a
        # condition-number check -- a degenerate fit is excluded from the
        # average entirely (treated as "no information from this dyad/
        # permutation", not as a valid, very-large statistic.
        if np.linalg.cond(X) > 1e8:
            return np.nan
        try:
            beta, res, rank, sv = np.linalg.lstsq(X, y, rcond=None)
        except Exception:
            return np.nan
        if rank < 3:
            return np.nan
        resid = y - X @ beta
        dof = n - 3
        if dof <= 0:
            return np.nan
        sigma2 = np.sum(resid ** 2) / dof
        try:
            XtX_inv = np.linalg.inv(X.T @ X)
        except Exception:
            return np.nan
        se_gamma = math.sqrt(sigma2 * XtX_inv[2, 2])
        if se_gamma <= 0 or not np.isfinite(se_gamma):
            return np.nan
        w = (beta[2] / se_gamma) ** 2
        return w if np.isfinite(w) and w < 1e6 else np.nan  # second safeguard: cap absurd outliers

    def _average_wald(dyad_data, x_key):
        stats = []
        for d, g in dyad_data.items():
            w = _wald_stat(g["coop_count"].to_numpy(), g["coop_l1"].to_numpy(), g[x_key].to_numpy())
            if np.isfinite(w):
                stats.append(w)
        return np.mean(stats) if stats else np.nan, len(stats)

    # --- Validation on toy cases before trusting this on real data --------
    rng_local = np.random.default_rng(12345)
    toy_signal, toy_null = {}, {}
    for i in range(30):
        T = 12
        x = rng_local.normal(0, 1, T)
        y_l1_seed = rng_local.normal(0, 1, T)
        y_signal = 2.0 * x + 0.3 * y_l1_seed + rng_local.normal(0, 0.5, T)
        y_null = 0.3 * y_l1_seed + rng_local.normal(0, 0.5, T)
        toy_signal[i] = pd.DataFrame({"coop_count": y_signal, "coop_l1": y_l1_seed, "x": x})
        toy_null[i] = pd.DataFrame({"coop_count": y_null, "coop_l1": y_l1_seed, "x": x})
    w_signal, _ = _average_wald(toy_signal, "x")
    w_null, _ = _average_wald(toy_null, "x")
    print(f"  Self-check: toy strong-signal avg Wald={w_signal:.1f} (should be large), "
         f"toy no-signal avg Wald={w_null:.1f} (should be small, near 1-3) -- "
         f"{'PASSED' if w_signal > 10 * max(w_null, 0.5) else 'FAILED, see below'}")
    if not (w_signal > 10 * max(w_null, 0.5)):
        print("  Toy validation did not show a clear separation -- results below are not reported.")
        return None

    # --- Real test on this club's dyad panel --------------------------------
    w_obs, n_dyads_used = _average_wald(dyad_groups, "es_mean_l1_z")
    print(f"  Observed average Wald statistic across {n_dyads_used} dyads (>= {min_T} time points "
         f"each): {w_obs:.4f}")

    all_es_series = {d: g["es_mean_l1_z"].to_numpy() for d, g in dyad_groups.items()}
    dyad_ids = list(dyad_groups.keys())
    w_perm = np.full(B, np.nan)
    for b in range(B):
        perm_order = rng.permutation(dyad_ids)
        perm_dyads = {}
        for orig_id, perm_id in zip(dyad_ids, perm_order):
            g = dyad_groups[orig_id].copy()
            x_perm = all_es_series[perm_id]
            if len(x_perm) != len(g):
                continue  # skip mismatched lengths rather than truncate silently
            g = g.assign(es_perm=x_perm)
            perm_dyads[orig_id] = g
        w_b, _ = _average_wald(perm_dyads, "es_perm")
        w_perm[b] = w_b
    valid_perm = w_perm[~np.isnan(w_perm)]
    r = int(np.sum(valid_perm >= w_obs))
    p_perm = (r + 1) / (len(valid_perm) + 1) if len(valid_perm) > 0 else np.nan
    print(f"  Permutation null: {len(valid_perm)}/{B} valid permutations, mean W*={valid_perm.mean():.4f}")
    print(f"  H0 (ES does not Granger-cause cooperation, jointly across dyads): p={p_perm:.4f}")
    print(f"  {'REJECT H0 -- ES has lagged predictive content beyond cooperation persistence' if p_perm < 0.05 else 'fail to reject H0'}")

    out_dir = CLUBS[club_key]["out_dir"] / "tables"; out_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([{"club": club_key, "n_dyads_used": n_dyads_used, "w_observed": w_obs,
                  "p_permutation": p_perm, "B": B}]).to_csv(out_dir / "panel_granger_test.csv", index=False)
    return {"w_observed": w_obs, "p_permutation": p_perm, "n_dyads_used": n_dyads_used}


def run_dyadic_robust_inference(club_key, df, formula="coop_count ~ es_mean_l1_z + innov_mean_l1_z | dyad_id + year",
                                coef_of_interest="es_mean_l1_z", B_wild=399):
    """
    TASK ("dyadic/network-robust standard errors", "wild-cluster
    bootstrap"): reports THREE inference approaches side by side for the
    same key coefficient, so the sensitivity of significance to the
    clustering/small-sample assumption is visible directly, rather than
    presenting only the dyad-clustered baseline as if it were the only
    reasonable choice.
    """
    import pyfixest as pf
    est = df.dropna(subset=["es_mean_l1_z", "innov_mean_l1_z", "coop_count"]).copy()
    try:
        _diagnose_sparsity(est, "coop_count")
    except ValueError as e:
        print(f"  [{club_key}] dyadic-robust inference skipped: {e}")
        return None

    print(f"\n  [{club_key}] DYADIC-ROBUST INFERENCE for {coef_of_interest} (three approaches):")

    m_dyad = _fit_or_raise(pf.fepois, formula, est, {"CRV1": "dyad_id"}, "dyad_baseline")
    b = float(m_dyad.coef()[coef_of_interest]); se_dyad = float(m_dyad.se()[coef_of_interest])
    p_dyad = float(m_dyad.pvalue()[coef_of_interest])
    print(f"    1. Dyad-clustered (baseline, used throughout this pipeline): "
         f"se={se_dyad:.4f}  p={p_dyad:.4f}")

    m_multi, V_multi, se_multi_arr, n_countries_used = fit_multiway_cluster(pf.fepois, formula, est, "dyadic_robust")
    n_countries_true = len(set(est["iso_i"]) | set(est["iso_j"]))
    if n_countries_used != n_countries_true:
        print(f"    WARNING: dyadic-robust clustering used {n_countries_used} countries but "
             f"{n_countries_true} appear in the estimation sample -- investigate before trusting this SE.")
    idx = list(m_multi.coef().index).index(coef_of_interest)
    se_multi = se_multi_arr[idx]
    z_multi = b / se_multi if se_multi > 0 else np.nan
    p_multi = 2 * (1 - norm.cdf(abs(z_multi))) if np.isfinite(z_multi) else np.nan
    print(f"    2. Dyadic-robust (actor-membership, any-shared-country; "
         f"Fafchamps-Gubert/Aronow-Samii-Assenova; {n_countries_used} countries): "
         f"se={se_multi:.4f}  p={p_multi:.4f}  "
         f"({'WIDER' if se_multi > se_dyad else 'narrower'} than dyad-only)")

    # FIX (reviewer): "the invalid wild bootstrap is labelled suspended but
    # still runs and is exported... the invalid number remains available
    # to downstream tables, figures and manuscript text, despite the
    # console warning." Correct: a warning after computing an invalid
    # statistic still lets that statistic reach every downstream consumer.
    # The call is now NOT made at all -- wild_cluster_bootstrap() itself is
    # preserved, unchanged, for documentation/reference (its docstring
    # already explains the proven flaw in detail), but this analysis path
    # no longer invokes it, and the exported record carries an explicit
    # disabled status rather than a silently-omitted or NaN-only field
    # that could be mistaken for "not computed" rather than "invalid method".
    print(f"    3. Wild-cluster bootstrap -- DISABLED, not computed (reviewer, proven "
         f"mathematically and confirmed by simulation: the country-membership weighting "
         f"does not reproduce shared-country correlation -- E[(w_i*w_j)(w_i*w_k)]=0 exactly, "
         f"not the claimed positive covariance; see wild_cluster_bootstrap()'s docstring, "
         f"preserved for reference but not called here). NOTE on a possible replacement: "
         f"ordinary dyad-level (not country-level) resampling would fix the specific flaw "
         f"above, but on its own still would not capture dependence BETWEEN different dyads "
         f"that share a country -- a full fix needs both properties simultaneously, not "
         f"either alone.")
    wb_country = None

    result = {"club": club_key, "coefficient": coef_of_interest, "point_estimate": b,
             "se_dyad": se_dyad, "p_dyad": p_dyad, "se_multiway": se_multi, "p_multiway": p_multi,
             "p_wild_bootstrap": np.nan, "wild_bootstrap_status": "disabled_invalid_method",
             "n_clusters_wild": np.nan}
    out_dir = CLUBS[club_key]["out_dir"]; (out_dir / "tables").mkdir(parents=True, exist_ok=True)
    pd.DataFrame([result]).to_csv(out_dir / "tables" / "dyadic_robust_inference.csv", index=False)
    return result


def run_club(club_key):
    club = CLUBS[club_key]
    print("=" * 70, "in run_club")
    print(f"{club['label']} ? DEA + connectors + PPML")
    print("=" * 70)
    n_dyads_theoretical = len(list(combinations(club["members_map"].keys(), 2)))
    # TASK (reviewer): "the sentence claiming 2000-2006 has no real ES or
    # innovation information appears inconsistent with 456 G20 country-year
    # rows = 19x24". Verified directly (not re-asserted): in this delivered
    # version, BOTH the real ES inputs (get_common_es_frontier, via the
    # embedded Ch3 WDI extract) and the real R&D/GDP fallback
    # (_real_gerd_fallback) are confirmed to return rows ONLY for
    # 2007-2023 for every G20/EU member checked -- never a false
    # 2000-2023 range. The 19x24=456 figure matches _sim_country_year()
    # (the FULLY simulated fallback with no real-data constraint at all),
    # which is not on the path used for R&D/GDP in this version. If a
    # future change reintroduces that fallback path, this printed
    # cross-check is designed to catch it: it directly counts and prints
    # the real ES-covered country-years below, not just asserts a range.
    prov_check = []
    print("Before compute_es_des", club["members_map"].keys(), club["out_dir"], prov_check, club_key)
    es_check = compute_es_dea(list(club["members_map"].keys()), club["out_dir"], prov_check, club_key=club_key)
    print("After compute_es_dea: es_check:", type(es_check), es_check)
    n_real_es_years = es_check["es"].notna().sum() if len(es_check) else 0
    es_min_year = int(es_check.year.min()) if len(es_check) else None
    es_max_year = int(es_check.year.max()) if len(es_check) else None
    print(f"[period] configured YEAR_MIN={YEAR_MIN}, YEAR_MAX={YEAR_MAX} "
         f"({YEAR_MAX - YEAR_MIN + 1} years, {n_dyads_theoretical} x "
         f"{YEAR_MAX - YEAR_MIN + 1} = {n_dyads_theoretical * (YEAR_MAX - YEAR_MIN + 1)} dyad-years "
         f"before lags). VERIFIED (not asserted): {n_real_es_years} real ES country-year "
         f"observations exist, spanning {es_min_year}-{es_max_year}. Note this reflects ES "
         f"specifically -- since Chapter 2's bias-corrected DEA/MPI data was integrated, ES "
         f"itself is real back to {es_min_year}, but Innovation (R&D/patents) and cooperation "
         f"still have no real coverage before 2007 in any source this pipeline uses (checked "
         f"separately, not fabricated to fill that gap), so the estimation sample below is still "
         f"constrained by THOSE variables, not by ES.")

    print("before build_panel")
    df, prov = build_panel(club_key)
    print("After build_panel> df",type(df))
    print("df",df)
    print ("prov",prov)
    print(">> df columns")
    for col in df.columns:
        print(col)

    print(df.head(20)) # first 20 rows
    print(df.tail(20)) # last 20 rows
    print(">> prov type", type(prov))
    #for col in prov.columns:
    #    print(col)

    estn = df.dropna(subset=["es_mean_l1_z", "innov_mean_l1_z", "coop_count"])
    print(f"[panel] rows={len(df)}, estimation obs={len(estn)}, "
          f"dyads={estn.dyad_id.nunique()}, zero-coop={(estn.coop_count==0).mean():.1%}, "
          f"actual year range in estimation sample = {int(estn.year.min())}-{int(estn.year.max())}")

    print("estn: ", estn)

    print("\n\n>> Before run_dyadic_robust_inference...")
    run_dyadic_robust_inference(club_key, df)

    print("\n\n>> Before run_panel_granger_test...")
    run_panel_granger_test(club_key, df, B=30)

    print("\n\n>> Before run_events_agreements_mc_validation...")
    run_events_agreements_mc_validation(club_key, B=60)

    print("\n\n>> Before run_block_bootstrap_dea...")
    run_block_bootstrap_dea(club_key, B=20, df_main=df)

    models, est = estimate(df)

    print("\n\n>> Before generate_diagrams...")
    generate_diagrams(club_key, df, models, club["out_dir"], prov)

    # TASK: run the same core models on formal_agreements_sim and report
    # side by side with the primary energy_events_sim results. Note:
    # estimate_formal_agreements() now returns None immediately whenever
    # the outcome is an alias of the primary real series (see its own
    # skip message there), so fa_result being non-None here already
    # guarantees this is a genuine, independently-measured construct --
    # no separate alias check is needed or reachable at this point.
    print("\n\n>> Before estimate_formal_agreements...")
    fa_result = estimate_formal_agreements(df)
    if fa_result is not None:
        fa_models, fa_est = fa_result
        print(f"\n  [{club_key}] FORMAL-AGREEMENTS ROBUSTNESS (three outcome definitions: "
             f"count, binary any-agreement, active stock):")
        for k, mm in fa_models.items():
            tidy_fa = mm.tidy().reset_index()
            term_col_fa = tidy_fa.columns[0]
            for _, r in tidy_fa.iterrows():
                print(f"        {k}.{r[term_col_fa]:24s} b={r['Estimate']:+.4f}  se={r['Std. Error']:.4f}  p={r['Pr(>|t|)']:.4f}")
        pd.DataFrame([{"model": k, "term": r[tidy.columns[0]], "coef": r["Estimate"], "se": r["Std. Error"], "p_value": r["Pr(>|t|)"]}
                     for k, mm in fa_models.items() for tidy in [mm.tidy().reset_index()] for _, r in tidy.iterrows()]
                    ).to_csv(club["out_dir"] / "tables" / "formal_agreements_coefficients.csv", index=False)

    coefs = {}
    full_coef_rows = []
    print("[estimating]")
    for k, mm in models.items():
        b = mm.coef(); key = "es_mean_l1_z" if "es_mean_l1_z" in b.index else b.index[0]
        coefs[k] = float(b[key])  # headline term, kept for the naive compare_clubs() summary

        # FIX (reviewer: "report the complete innovation interaction"): the
        # previous version printed only ONE coefficient per model, even for
        # m3_innov, which has three terms (ES, Innovation, ES x Innovation).
        # Every term of every model is now printed/recorded, with its own
        # SE, p-value, and 95% CI -- not just the headline ES coefficient.
        tidy = mm.tidy().reset_index()
        term_col = tidy.columns[0]
        print(f"    --- {k} ---")
        for _, r in tidy.iterrows():
            term, coef_v, se_v, p_v = r[term_col], r["Estimate"], r["Std. Error"], r["Pr(>|t|)"]
            lo, hi = coef_v - 1.96 * se_v, coef_v + 1.96 * se_v
            print(f"        {term:28s} b={coef_v:+.4f}  se={se_v:.4f}  p={p_v:.4f}  "
                  f"95%CI=[{lo:+.4f},{hi:+.4f}]")
            full_coef_rows.append({"model": k, "term": term, "coef": coef_v, "se": se_v,
                                   "p_value": p_v, "ci_lo": lo, "ci_hi": hi})
    pd.DataFrame(full_coef_rows).to_csv(club["out_dir"] / "tables" / "full_coefficients.csv", index=False)

    heads1 = ["(1) PPML", "(2) FE-OLS", "(3) +Innov", "(4) +Vol", "(5) Full (preferred)"]
    models1 = [models["m1_ppml"], models["m2_ols"], models["m3_innov"], models["m4_vol"], models["m_full_ppml"]]
    # FIX: coop_prov is local to build_panel(), not in scope here -- an
    # earlier version of this line referenced it directly and would have
    # raised NameError the first time this code path executed. The
    # cooperation entry is pulled from `prov` (returned by build_panel and
    # already in scope) instead, so the label can't drift from what
    # build_panel actually did AND doesn't depend on cross-function scope
    # that doesn't exist.
    coop_prov_entries = [r for r in prov if r.get("variable", "").startswith("cooperation")]
    outcome_label = coop_prov_entries[0]["source"] if coop_prov_entries else "cooperation outcome"
    write_table(club["out_dir"], models1, heads1, "main_results",
                f"Main results: ES, innovation and {club['label']} cooperation "
                f"[outcome: {outcome_label}]")

    heads2, models2 = [], []
    if "m5_fdi" in models:
        heads2.append("(5) FDI"); models2.append(models["m5_fdi"])
    heads2 += ["(6) t-2", "(6) t+1", "(7) min", "(7) diff"]
    models2 += [models["m6_lag2"], models["m6_lead1"], models["m7_min"], models["m7_diff"]]
    write_table(club["out_dir"], models2, heads2, "robustness",
                f"Robustness: {club['label']} outcome swap/timing/aggregation")

    prov_df = pd.DataFrame(prov)
    prov_df.to_csv(club["out_dir"] / "tables" / "data_provenance.csv", index=False)
    print(f"\n--- {club['label']} DATA PROVENANCE ---")
    print(prov_df.to_string(index=False))
    print(f"Outputs in {club['out_dir']}/\n")

    return {"club": club_key, "label": club["label"], "n_members": len(club["members_map"]),
            "n_dyads": estn.dyad_id.nunique(), "n_obs": len(estn),
            "zero_share": (estn.coop_count == 0).mean(), "coefs": coefs, "provenance": prov_df, "df": df}


# =========================================================================
# SECTION 8B ? POOLED CROSS-CLUB INTERACTION MODEL
# =========================================================================
# Implements all four corrections from the methodological review:
#   (1) COMMON DEA FRONTIER: ES is computed ONCE, in a single DEA run, over
#       the union of unique G20-sovereign + EU-27-covered countries (38
#       countries; Germany/France/Italy counted once, not once per club).
#       A country's ES score is therefore identical whether it is used in
#       a G20 dyad or an EU dyad -- club-specific frontiers are no longer
#       used for this comparison (they remain available as the per-club
#       Section 7 models above, which can be read as a robustness check
#       against club-specific frontiers).
#   (2) COMMON STANDARDISATION: ES and Innovation are z-scored ONCE using
#       the pooled mean/SD across both clubs' dyad-year observations
#       combined, not separately within each club.
#   (3) CORRECTED G20 UNIT: uses the 19-sovereign-state G20 (already fixed
#       in Section 3 above -- MEMBERS_WB no longer includes "EU").
#   (4) HARMONISED SOURCE: both clubs' R&D/GDP now come from a single World
#       Bank fetch over the pooled country list (Eurostat is not used here),
#       and patents/CO2/import-dependency/T&D-losses/renewable-share are
#       already the same WDI-sourced table for every country regardless of
#       club. The one remaining, explicitly flagged asymmetry is that, if
#       the World Bank fetch fails, the offline fallback is simulated
#       identically for every country (same generator, no club asymmetry).
#
# The core econometric fix: rather than comparing two SEPARATELY estimated
# coefficients (which cannot support a claim that the two clubs differ),
# this stacks the G20 and EU-27 dyad panels into ONE regression with an EU
# club dummy and full interactions, and tests the EU-G20 difference
# directly (delta_1, delta_3) via the model's own standard errors --
# exactly the "pooled PPML with EU x ES, EU x Innov, EU x ES x Innov"
# specification requested in the review.
#
# UPDATED CAVEAT (was "STANDING CAVEAT... cooperation is SIMULATED,
# period" -- no longer accurate since real cooperation data overlay was
# added below, and left uncorrected here was itself a source of the
# reviewer's "contradictory provenance statements" finding). The
# cooperation outcome below starts from a simulated baseline with known
# EU-G20 differences (TRUE_DELTA1, TRUE_DELTA3) built in -- useful for the
# Monte Carlo validation in Section 8C -- but is then OVERWRITTEN with real
# OBSERVED BILATERAL data (the same loader as the per-club models; see
# _load_real_cooperation_plain) wherever that real data actually covers a
# given dyad-year. The exact real/simulated split is computed and printed
# for every run (search this function's output for "Cooperation outcome:"),
# not asserted here as a fixed percentage, because it depends entirely on
# which real-data files are present in data/ when the script runs.
# ES and (when online) R&D/GDP are real; patents are real; cooperation is
# a real/simulated MIX whose exact composition is reported at runtime, not
# uniformly one or the other.

TRUE_BETA_ES, TRUE_BETA_INNOV, TRUE_BETA_INT = 0.30, 0.15, 0.10
TRUE_DELTA1, TRUE_DELTA2, TRUE_DELTA3 = 0.08, -0.05, -0.06  # EU-G20 differences


def _ch3_embedded_to_df_pooled(members):
    """Same embedded Ch3 tables as Section 3B, but keyed on PLAIN ISO3
    codes for an arbitrary member list (no G20/EU-specific renaming) --
    used only by the pooled model so the same country's data is read
    exactly once, from exactly one place, regardless of which club(s)
    it belongs to."""
    def to_long(d, value_name):
        rows = []
        for iso, vals in d.items():
            if iso not in members:
                continue
            for y, v in zip(CH3_SOURCE_YEARS, vals):
                if v is not None:
                    rows.append({"iso": iso, "year": y, value_name: float(v)})
        return pd.DataFrame(rows)

    merged = to_long(CO2_PC, "co2_pc")
    for d, name in [(IMP_DEP, "import_dep_pct"), (TD_LOSS, "td_loss_pct"),
                    (PAT_RESD, "patents_residents"), (RNW_SHARE, "renew_share_pct"),
                    (POP, "population")]:
        merged = merged.merge(to_long(d, name), on=["iso", "year"], how="outer")
    merged = merged[(merged["year"] >= YEAR_MIN) & (merged["year"] <= YEAR_MAX)]
    return merged


def compute_pooled_es_dea(union_members, prov):
    """Delegates to get_common_es_frontier() -- the SAME cached computation
    used by every per-club run too, so pooled and per-club ES numbers are
    guaranteed identical (single source of truth, not two DEA implementations
    that happen to use the same specification)."""
    es = get_common_es_frontier(prov)
    missing = [m for m in union_members if m not in set(es["iso"].unique())]
    if missing:
        print(f"  [pooled] ES: no real data for {missing} -- excluded from the "
              f"common frontier (not fabricated)")
    return es[es["iso"].isin(union_members)].copy()


def compute_pooled_innovation(union_members, prov):
    """Innovation (R&D/GDP + patents/capita) for the pooled union, using a
    SINGLE World Bank fetch (not World Bank for G20 + Eurostat for EU) --
    the harmonised-source fix. Falls back to one simulated generator for
    everyone if offline, so there's no club asymmetry either way."""
    members_map = {m: m for m in union_members}  # WB ISO3 == plain ISO3 for all of these
    wb = None
    if TRY_ONLINE:
        try:
            wb = fetch_worldbank(members_map)
            prov.append({"variable": "R&D/GDP, GDP, pop (pooled)", "source": "World Bank Indicators API",
                        "status": "LIVE", "citation": "WDI GB.XPD.RSDV.GD.ZS; NY.GDP.MKTP.CD; SP.POP.TOTL; NY.GDP.PCAP.PP.KD; NY.GDP.MKTP.KD.ZG"})
        except Exception as e:
            print(f"  [pooled] World Bank fetch failed ({e}); simulating"); wb = None
    if wb is None or wb["rd_gdp"].notna().mean() < 0.2:
        wb, covered = _real_gerd_fallback(union_members)
        if covered:
            prov.append({"variable": "R&D/GDP (pooled)", "source": f"REAL embedded WDI GERD data "
                        f"({len(covered)}/{len(union_members)} members covered); GDP/population still "
                        f"simulated identically for every country (no club asymmetry)",
                        "status": "LOADED (real, R&D/GDP only)",
                        "citation": "World Bank WDI GB.XPD.RSDV.GD.ZS, embedded from Chapter 3 extract"})
        else:
            prov.append({"variable": "R&D/GDP, GDP, pop (pooled)", "source": "simulated (identical generator, no club asymmetry)",
                        "status": "SIMULATED", "citation": ""})
    real_df = _ch3_embedded_to_df_pooled(union_members)
    pat = real_df.dropna(subset=["patents_residents", "population"]).copy()
    pat["patents_pc"] = 1e6 * pat["patents_residents"] / pat["population"]
    prov.append({"variable": "patents/capita (pooled)", "source": "REAL WDI IP.PAT.RESD, Chapter 3 extract",
                "status": "LOADED (real)", "citation": "World Bank WDI IP.PAT.RESD"})
    out = wb.merge(pat[["iso", "year", "patents_pc"]], on=["iso", "year"], how="left")

    coinv = _oecd_coinvent_long()
    coinv = coinv[coinv["iso"].isin(union_members)]
    prov.append({"variable": "OECD co-invention intensity (pooled)",
                "source": f"User-provided OECD data (Section 3C), co-inventions in % of patents; "
                          f"{_OECD_COINVENT_EXCLUDED} excluded as all-zero/unreliable",
                "status": "LOADED (real)", "citation": "OECD, co-inventions as % of patents (dataset code pending confirmation)"})
    out = out.merge(coinv, on=["iso", "year"], how="left")
    return out


def _linear_combo_var(vcov, names, weight_dict):
    """Var(sum_i w_i * beta_i) = w' V w, for an arbitrary linear combination
    of model coefficients -- used for every marginal-effect SE below."""
    w = np.zeros(len(names))
    for nm, wt in weight_dict.items():
        if nm in names:
            w[names.index(nm)] = wt
    return float(w @ vcov @ w)


def build_pooled_club_dyads(club_label, members, es_pooled, innov_pooled, true_delta=(0.0, 0.0, 0.0)):
    """Builds one club's dyad-year rows using the POOLED (common-frontier,
    common-source) ES/innovation series -- raw (unstandardised) at this
    stage, since standardisation must happen AFTER stacking both clubs
    (correction #2)."""
    d1, d2, d3 = true_delta
    dyads = list(combinations(sorted(members), 2))
    rows = []
    es_i = es_pooled.set_index(["iso", "year"])["es"]
    in_i = innov_pooled.set_index(["iso", "year"])
    for (i, j) in dyads:
        for t_idx in range(1, len(YEARS)):
            t, t_lag = YEARS[t_idx], YEARS[t_idx - 1]
            try:
                es_ii, es_jj = es_i.loc[(i, t_lag)], es_i.loc[(j, t_lag)]
                rd_ii, rd_jj = in_i.loc[(i, t_lag), "rd_gdp"], in_i.loc[(j, t_lag), "rd_gdp"]
            except KeyError:
                continue
            pat_ii = in_i.loc[(i, t_lag), "patents_pc"] if (i, t_lag) in in_i.index else np.nan
            pat_jj = in_i.loc[(j, t_lag), "patents_pc"] if (j, t_lag) in in_i.index else np.nan
            coinv_ii = in_i.loc[(i, t_lag), "coinvent_intensity"] if (i, t_lag) in in_i.index else np.nan
            coinv_jj = in_i.loc[(j, t_lag), "coinvent_intensity"] if (j, t_lag) in in_i.index else np.nan
            rows.append({"club": club_label, "iso_i": i, "iso_j": j, "year": t,
                        "clubdyad_id": f"{club_label}_{i}_{j}", "club_year": f"{club_label}_{t}",
                        "es_mean_raw": (es_ii + es_jj) / 2,
                        "rd_mean_raw": (rd_ii + rd_jj) / 2,
                        "pat_mean_raw": np.nanmean([pat_ii, pat_jj]) if not (np.isnan(pat_ii) and np.isnan(pat_jj)) else np.nan,
                        "coinv_mean_raw": np.nanmean([coinv_ii, coinv_jj]) if not (np.isnan(coinv_ii) and np.isnan(coinv_jj)) else np.nan})
    df = pd.DataFrame(rows)
    return df, (d1, d2, d3)


def _load_real_cooperation_plain(club_key):
    """
    Loads real cooperation data via the EXACT SAME call run_club() uses
    (load_cooperation_layered) -- not a parallel implementation -- so that
    wherever real data exists, the pooled model and the per-club model see
    literally the same numbers. Translates EU's internal FRA_EU/DEU_EU/
    ITA_EU codes back to plain ISO3, since the pooled panel (Section 8B)
    uses plain codes throughout for both clubs. Returns (DataFrame with
    [iso_i, iso_j, year, coop_count] or None, provenance dict or None).
    """
    club = CLUBS[club_key]
    df, prov = load_cooperation_layered(club)
    if df is None:
        return None, None
    df = df.copy()
    if club_key == "eu":
        club_to_plain = {v: k for k, v in EU_CH3_CODE_MAP.items()}
        df["iso_i"] = df["iso_i"].map(lambda x: club_to_plain.get(x, x))
        df["iso_j"] = df["iso_j"].map(lambda x: club_to_plain.get(x, x))
        # NOTE, not a bug: _canon() (alphabetical order) is CORRECT here,
        # unlike the per-club loaders elsewhere in this file. Discovered
        # while auditing every remaining _canon() call for the reviewer's
        # "one canonical pair key throughout the program" point: the
        # POOLED panel's own dyad construction (build_pooled_club_dyads,
        # "dyads = list(combinations(sorted(members), 2))") uses
        # ALPHABETICAL order, genuinely different from build_panel's own
        # insertion-order combinations() used for the per-club models.
        # This function feeds the pooled panel specifically (translating
        # EU's internal codes back to plain ISO3 first), so alphabetical
        # _canon() is the matching convention here, not a mismatch.
        # Two different canonical orders genuinely coexist in this file by
        # historical accident, not by a considered design choice -- a
        # complete fix would standardise on one throughout both the
        # per-club and pooled panel constructions, which was judged too
        # large and too load-bearing a change to make in this pass without
        # separately re-validating every existing result against it.
        df = _canon(df)
    return df[["iso_i", "iso_j", "year", "coop_count"]], prov


def build_pooled_regressors(prov=None):
    """
    Builds the pooled G20+EU dyad panel's REGRESSORS (es_z, innov_z,
    es_z:innov_z, club/clubdyad_id/club_year identifiers) -- everything
    except the cooperation outcome itself. Factored out so both
    run_pooled_comparison() and the Monte Carlo validator below build the
    ES/Innovation data exactly once and share it, rather than re-running
    the DEA and World Bank fetch on every one of B replications.
    """
    if prov is None:
        prov = []
    g20_members = list(MEMBERS_WB.keys())
    eu_members = list(EU_CH3_CODE_MAP.keys())
    union_members = sorted(set(g20_members) | set(eu_members))

    es_pooled = compute_pooled_es_dea(union_members, prov)
    innov_pooled = compute_pooled_innovation(union_members, prov)

    g20_df, _ = build_pooled_club_dyads("G20", g20_members, es_pooled, innov_pooled)
    eu_df, _ = build_pooled_club_dyads("EU", eu_members, es_pooled, innov_pooled)
    g20_df["EU"], eu_df["EU"] = 0, 1
    pooled = pd.concat([g20_df, eu_df], ignore_index=True)
    pooled = pooled.dropna(subset=["es_mean_raw", "rd_mean_raw"]).copy()

    rd_z_pool = zscore(pooled["rd_mean_raw"])
    comps = [rd_z_pool]
    if pooled["pat_mean_raw"].notna().mean() > 0.2:
        comps.append(zscore(pooled["pat_mean_raw"]))
    if pooled["coinv_mean_raw"].notna().mean() > 0.2:
        comps.append(zscore(pooled["coinv_mean_raw"]))
    pooled["innov_mean_raw"] = np.nanmean(np.column_stack(comps), axis=1)

    print(f"  [pooled] standardisation for es_z: mean={pooled['es_mean_raw'].mean():.4f}  "
         f"sd={pooled['es_mean_raw'].std():.4f}  n={pooled['es_mean_raw'].notna().sum()} "
         f"(POOLED across both clubs, per correction #2 -- not computed separately per club)")
    print(f"  [pooled] standardisation for innov_z: mean={pooled['innov_mean_raw'].mean():.4f}  "
         f"sd={pooled['innov_mean_raw'].std():.4f}  n={pooled['innov_mean_raw'].notna().sum()}")
    pooled["es_z"] = zscore(pooled["es_mean_raw"])
    pooled["innov_z"] = zscore(pooled["innov_mean_raw"])
    pooled["es_z:innov_z"] = pooled["es_z"] * pooled["innov_z"]
    return pooled, g20_members, eu_members, union_members


def _drop_degenerate_fe_groups(df, fe_cols, min_size=2):
    """
    DEFENSIVE MITIGATION, added after a reported Rust-level panic
    ("thread panicked at src/demean.rs:71:48, called Option::unwrap() on
    a None value") in one user's environment, immediately after
    restarting their kernel following a filtered-subsample refit. Could
    NOT be reproduced directly here (the full pooled run, including the
    Monte Carlo battery, completed cleanly with pyfixest 0.60.0) -- most
    likely a pyfixest/Rust-backend version difference rather than a bug
    in this pipeline's own Python logic, since a Rust panic bypasses
    Python's exception handling entirely and this pipeline cannot patch
    a compiled third-party dependency. This is a best-effort mitigation,
    not a confirmed fix: filtering a panel (e.g. to verified-real
    observations only, or excluding overlap countries) can leave a
    fixed-effect factor level -- a specific dyad, or a specific year --
    with zero or one remaining observations, a classic degenerate input
    for a demeaning routine. Dropping such groups BEFORE they reach
    pyfixest removes one plausible trigger regardless of which pyfixest
    version is installed; it does not address the possibility that the
    installed pyfixest/Rust build itself has a bug independent of input
    degeneracy.

    FIX (reviewer, confirmed by direct reproduction): the original
    version filtered each fe_col ONCE, sequentially. Removing rows for
    the SECOND column can shrink a group in the FIRST column below
    min_size again -- confirmed directly on a toy 6-row, 2-column case
    here: filtering dyad_id first correctly keeps a 2-observation dyad,
    but the subsequent club_year pass then removes one of that dyad's two
    remaining rows, leaving a singleton dyad despite it having passed its
    own filter. Fixed by iterating both filters to a fixed point (re-
    checking every column after every pass) rather than a single sweep.
    """
    d = df.dropna(subset=fe_cols).copy()
    while not d.empty:
        n_before = len(d)
        for col in fe_cols:
            sizes = d.groupby(col, observed=True)[col].transform("size")
            d = d.loc[sizes >= min_size].copy()
        if len(d) == n_before:
            break
    for col in fe_cols:
        if isinstance(d[col].dtype, pd.CategoricalDtype):
            d[col] = d[col].cat.remove_unused_categories()
    return d.reset_index(drop=True)


def run_pooled_comparison():
    print("=" * 70)
    print("POOLED CROSS-CLUB INTERACTION MODEL (common frontier, common")
    print("standardisation, corrected G20 unit, harmonised source)")
    print("=" * 70)

    prov = []
    pooled, g20_members, eu_members, union_members = build_pooled_regressors(prov)
    overlap = sorted(set(g20_members) & set(eu_members))
    print(f"  G20 sovereigns: {len(g20_members)}  |  EU-22 covered: {len(eu_members)}  |  "
          f"union (unique): {len(union_members)}  |  overlap (in both clubs): {overlap}")

    # TASK ("the pooled comparison needs an overlap rule"): Germany, France
    # and Italy are members of BOTH clubs. Of the three options a reviewer
    # laid out -- (a) remove these countries' EU observations from the G20
    # comparison; (b) index dyads as club x country pair; (c) treat this as
    # "EU institutional setting vs G20 setting", allowing membership
    # overlap -- this pipeline implements (b) AS THE MECHANISM and (c) AS
    # THE JUSTIFICATION, together, not left implicit:
    #   - No country or dyad is removed from either club. Germany-France
    #     contributes a G20 observation (from the 19-sovereign roster) AND
    #     a SEPARATE EU observation (from the 22-country roster) to the
    #     pooled sample.
    #   - clubdyad_id = f"{club}_{iso_i}_{iso_j}" (e.g. "G20_DEU_FRA" vs
    #     "EU_DEU_FRA") makes these TWO DISTINCT dyad fixed-effect units,
    #     never pooled into one "DEU_FRA" identity -- so a common ES/
    #     Innovation shock to the Germany-France pair is allowed to have a
    #     DIFFERENT estimated cooperation relationship depending on which
    #     institutional setting (G20 forum vs EU internal market) it is
    #     observed under. Dyad-clustered SEs are computed on this SAME
    #     clubdyad_id, so "G20_DEU_FRA" and "EU_DEU_FRA" are treated as
    #     independent clusters, not artificially correlated with each
    #     other or merged.
    #   - This is therefore explicitly a comparison of "does the ES-
    #     cooperation relationship differ by institutional setting",
    #     NOT a comparison of two disjoint, non-overlapping country
    #     samples -- calling G20 and EU "independent samples" would
    #     misdescribe this design, since 3 countries and their behaviour
    #     inform both sides of the comparison simultaneously.
    print(f"  OVERLAP RULE: {overlap} are members of BOTH clubs. No country or dyad is removed --")
    print(f"    each contributes a SEPARATE G20 observation and EU observation (indexed by")
    print(f"    clubdyad_id = club+pair, e.g. 'G20_DEU_FRA' vs 'EU_DEU_FRA', distinct dyad FE")
    print(f"    and distinct dyad clusters). This is a comparison of INSTITUTIONAL SETTING")
    print(f"    (G20 forum vs EU internal market), not of two disjoint country samples --")
    print(f"    describing G20 and EU as 'independent samples' would be inaccurate.")

    # --- Simulated baseline (fixed effects + known true deltas) -- used as
    # the fallback wherever real cooperation data is not available for a
    # given dyad-year, and as the sole source if NEITHER club has any real
    # data at all (pure code validation, as in previous versions).
    dfe = pooled["clubdyad_id"].map({d: rng.normal(0, 0.4) for d in pooled["clubdyad_id"].unique()}).to_numpy()
    cyfe = pooled["club_year"].map({c: rng.normal(0, 0.2) for c in pooled["club_year"].unique()}).to_numpy()
    EU = pooled["EU"].to_numpy()
    lin = (np.log(0.5)
          + TRUE_BETA_ES * pooled["es_z"] + TRUE_BETA_INNOV * pooled["innov_z"]
          + TRUE_BETA_INT * pooled["es_z:innov_z"]
          + TRUE_DELTA1 * EU * pooled["es_z"] + TRUE_DELTA2 * EU * pooled["innov_z"]
          + TRUE_DELTA3 * EU * pooled["es_z:innov_z"]
          + dfe + cyfe)
    pooled["coop_count"] = rng.poisson(np.exp(np.clip(lin, -10, 10)))
    pooled["coop_is_real"] = False

    # TASK ("use the same real cooperation outcome in separate and pooled
    # models"): reuse load_cooperation_layered() -- the IDENTICAL loader
    # run_club() uses for the per-club Section 7 models, not a parallel
    # implementation -- so real data, where available, is literally the
    # same numbers in both places. Real counts overlay the simulated
    # baseline row-by-row; rows without a real match keep the simulated
    # value, and every row is flagged so the split is auditable, not
    # blended silently.
    real_g20, real_g20_prov = _load_real_cooperation_plain("g20")
    real_eu, real_eu_prov = _load_real_cooperation_plain("eu")
    # FIX: merge PER CLUB, not on one combined table keyed only by
    # [iso_i, iso_j, year]. A country pair like Germany-France is a valid
    # dyad in BOTH clubs, so a combined real-data table has two rows with
    # an identical (iso_i, iso_j, year) key (one meant for the G20 sub-
    # panel, one for the EU sub-panel) -- merging that combined table
    # against the full pooled panel in one shot fans out into a Cartesian
    # duplicate (every matching pooled row hits BOTH real rows), silently
    # inflating the "real" count. Restricting each merge to its own club's
    # rows of `pooled` first avoids the ambiguity entirely.
    for club_lbl, real_df, real_prov in [("G20", real_g20, real_g20_prov), ("EU", real_eu, real_eu_prov)]:
        if real_df is None:
            continue
        sub_mask = pooled["club"] == club_lbl
        matched = pooled.loc[sub_mask, ["iso_i", "iso_j", "year"]].merge(
            real_df.rename(columns={"coop_count": "coop_count_real"}),
            on=["iso_i", "iso_j", "year"], how="left")
        real_vals = matched["coop_count_real"].to_numpy()
        has_real = ~pd.isna(real_vals)
        idx = pooled.index[sub_mask]
        pooled.loc[idx[has_real], "coop_count"] = real_vals[has_real]
        pooled.loc[idx[has_real], "coop_is_real"] = True

        # CRITICAL FIX (reviewer: "simply retaining positive project
        # matches would also distort the sample" -- exactly the bug found
        # here): the block above only marked POSITIVE matches as real.
        # CORDIS's own file (confirmed directly: 9,122 rows, ALL with
        # cordis_projects > 0, zero rows with a count of exactly 0) only
        # lists dyad-years with at least one joint project -- a dyad-year
        # within CORDIS's actual coverage window (2007-2023, confirmed
        # directly from the file's own year range) that does NOT appear
        # is a GENUINE, VERIFIED ZERO (CORDIS is an exhaustive project
        # registry; absence of a matching project record is real
        # evidence of zero co-participation that year, not an unknown).
        # This is exactly what the PER-CLUB build_panel already does
        # correctly (f["coop_count"] = f["coop_count"].fillna(0) after
        # its own real-data merge) -- the pooled construction had
        # silently diverged from that and was mis-marking every verified
        # zero as "simulated" purely because it had no positive count to
        # match against. Fixed by marking every (dyad, year) row inside
        # CORDIS's own coverage window as real, not only the rows that
        # happened to match a positive count.
        if real_df is not None and len(real_df) > 0:
            cordis_year_min, cordis_year_max = _get_source_coverage("cordis", real_df)
            in_coverage = sub_mask & (pooled["year"] >= cordis_year_min) & (pooled["year"] <= cordis_year_max)
            newly_verified_zero = in_coverage & (~pooled["coop_is_real"])
            pooled.loc[newly_verified_zero, "coop_count"] = 0.0
            pooled.loc[newly_verified_zero, "coop_is_real"] = True
        if real_prov is not None:
            prov.append(real_prov)

    n_real = int(pooled["coop_is_real"].sum())
    print(f"  Cooperation outcome: {n_real}/{len(pooled)} dyad-years ({n_real/max(len(pooled),1):.1%}) "
         f"use REAL data from the same loader as the per-club models; the remainder is simulated "
         f"(known-true-parameter baseline, for the Monte Carlo validation in Section 8C).")
    if n_real == len(pooled):
        print(f"  NOTE: this is 100%, not a bug -- the estimation window (constrained to 2008-2023 by "
             f"when ES/Innovation lags become available) falls entirely within CORDIS's documented "
             f"coverage window ({SOURCE_COVERAGE['cordis'][0]}-{SOURCE_COVERAGE['cordis'][1]}), so "
             f"every dyad-year here is either a genuine positive match or a genuine verified zero -- "
             f"none are simulated for THIS configuration. Were the estimation window ever extended "
             f"earlier than CORDIS's coverage (e.g. if ES/Innovation gained real data before 2007), "
             f"a genuine simulated remainder would reappear. The real-only check below is therefore "
             f"IDENTICAL to the mixed-sample model above under the current configuration -- expected, "
             f"not a sign the filter failed to run.")

    n_g20, n_eu = (pooled.EU == 0).sum(), (pooled.EU == 1).sum()
    print(f"  Pooled panel: {len(pooled)} dyad-year obs ({n_g20} G20 + {n_eu} EU), "
          f"zero-coop share={ (pooled.coop_count==0).mean():.1%}")

    import pyfixest as pf
    formula = ("coop_count ~ es_z + innov_z + es_z:innov_z "
              "+ EU:es_z + EU:innov_z + EU:es_z:innov_z | clubdyad_id + club_year")
    print(f"[FIT START] pooled main interaction model: rows={len(pooled)}", flush=True)
    model = pf.fepois(formula, pooled, vcov={"CRV1": "clubdyad_id"}, demeaner=LsmrDemeaner(backend="within"))
    print("[FIT OK] pooled main interaction model", flush=True)

    tidy = model.tidy().reset_index()
    term_col = tidy.columns[0]
    names = list(model.coef().index)
    vcov = model._vcov

    print("\n" + "-" * 70)
    print("Table: Pooled interaction model (coef, SE, p, 95% CI, IRR)")
    print("-" * 70)
    rows_out = []
    for _, r in tidy.iterrows():
        term, b, se, p = r[term_col], r["Estimate"], r["Std. Error"], r["Pr(>|t|)"]
        lo, hi = b - 1.96 * se, b + 1.96 * se
        irr, irr_lo, irr_hi = np.exp(b), np.exp(lo), np.exp(hi)
        rows_out.append({"term": term, "coef": b, "se": se, "p_value": p,
                         "ci_lo": lo, "ci_hi": hi, "IRR": irr, "IRR_ci_lo": irr_lo, "IRR_ci_hi": irr_hi})
        print(f"  {term:20s} b={b:+.4f}  se={se:.4f}  p={p:.4f}  "
              f"95%CI=[{lo:+.4f},{hi:+.4f}]  IRR={irr:.3f} [{irr_lo:.3f},{irr_hi:.3f}]")

    # --- Marginal ES effect (IRR) at Innovation = -1, 0, +1 SD, by club ----
    print("\n" + "-" * 70)
    print("Marginal ES effect (IRR) across Innovation, G20 vs EU")
    print("-" * 70)
    marg_rows = []
    for club_lbl, eu_flag in [("G20", 0), ("EU", 1)]:
        for z in (-1, 0, 1):
            w = {"es_z": 1.0, "es_z:innov_z": z}
            if eu_flag:
                w["EU:es_z"] = 1.0; w["EU:es_z:innov_z"] = z
            point = sum(w.get(nm, 0.0) * model.coef()[nm] for nm in w if nm in names)
            var = _linear_combo_var(vcov, names, w)
            se = math.sqrt(max(var, 0))
            lo, hi = point - 1.96 * se, point + 1.96 * se
            marg_rows.append({"club": club_lbl, "innov_z": z, "beta": point, "se": se,
                              "IRR": np.exp(point), "IRR_ci_lo": np.exp(lo), "IRR_ci_hi": np.exp(hi)})
            print(f"  {club_lbl:4s} innov_z={z:+d}  beta_ES={point:+.4f} (se={se:.4f})  "
                  f"IRR={np.exp(point):.3f} [{np.exp(lo):.3f},{np.exp(hi):.3f}]")

    # --- EU-G20 differences + Wald tests -----------------------------------
    print("\n" + "-" * 70)
    print("EU-G20 differences and Wald tests")
    print("-" * 70)
    d1_row = tidy[tidy[term_col] == "EU:es_z"].iloc[0]
    d3_row = tidy[tidy[term_col] == "EU:es_z:innov_z"].iloc[0]
    print(f"  delta_1 (EU - G20, ES main effect):        b={d1_row['Estimate']:+.4f}  "
          f"se={d1_row['Std. Error']:.4f}  p={d1_row['Pr(>|t|)']:.4f}  "
          f"{'REJECT H0 (differ)' if d1_row['Pr(>|t|)'] < 0.05 else 'fail to reject H0 (no evidence of difference)'}")
    print(f"  delta_3 (EU - G20, ES x Innov moderation): b={d3_row['Estimate']:+.4f}  "
          f"se={d3_row['Std. Error']:.4f}  p={d3_row['Pr(>|t|)']:.4f}  "
          f"{'REJECT H0 (differ)' if d3_row['Pr(>|t|)'] < 0.05 else 'fail to reject H0 (no evidence of difference)'}")

    idx1, idx3 = names.index("EU:es_z"), names.index("EU:es_z:innov_z")
    sub_v = vcov[np.ix_([idx1, idx3], [idx1, idx3])]
    sub_b = np.array([model.coef()["EU:es_z"], model.coef()["EU:es_z:innov_z"]])
    try:
        wald_stat = float(sub_b @ np.linalg.solve(sub_v, sub_b))
        wald_p = 1 - chi2.cdf(wald_stat, df=2)
        print(f"  Joint Wald test H0: delta_1=0 AND delta_3=0:  chi2={wald_stat:.3f}  df=2  p={wald_p:.4f}  "
              f"{'REJECT H0' if wald_p < 0.05 else 'fail to reject H0'}")
    except np.linalg.LinAlgError:
        wald_stat, wald_p = np.nan, np.nan
        print("  Joint Wald test: singular covariance submatrix, skipped")

    # TASK (reviewer): "the immediate question is whether the reported
    # EU-G20 difference persists when the simulated outcome observations
    # are excluded." Answered directly here, not merely discussed: refits
    # the IDENTICAL specification on ONLY the rows verified real
    # (coop_is_real, corrected above to include genuine verified zeros,
    # not only positive project matches -- simply keeping positive
    # matches would itself distort the sample, per the same review).
    # This is now the PRIMARY analysis; the full mixed-sample model above
    # is retained as a secondary sensitivity/validation exercise, not the
    # headline result.
    print("\n" + "=" * 70)
    print("PRIMARY ANALYSIS: EU-G20 test restricted to VERIFIED REAL observations only")
    print("(simulated fallback rows excluded entirely, not imputed -- ordinary")
    print(" estimation on a completed dataset does not supply valid imputation-")
    print(" uncertainty-adjusted inference; see Van Buuren, Flexible Imputation")
    print(" of Missing Data, on why simulated fill-ins cannot be treated as a")
    print(" defensible multiple-imputation analysis without accounting for that")
    print(" uncertainty in the standard errors)")
    print("=" * 70)
    pooled_real = pooled[pooled["coop_is_real"]].copy()
    pooled_real = _drop_degenerate_fe_groups(pooled_real, ["clubdyad_id", "club_year"])
    n_real_g20 = int((pooled_real["club"] == "G20").sum())
    n_real_eu = int((pooled_real["club"] == "EU").sum())
    print(f"  Real-only sample: {len(pooled_real)} dyad-years ({n_real_g20} G20 + {n_real_eu} EU), "
         f"zero-coop share={(pooled_real.coop_count==0).mean():.1%}")
    print(f"  Outcome definition, stated precisely: country-pair co-participation in EU Framework")
    print(f"  Programme (FP7 2007-2013 / Horizon 2020 2014-2023) funded research projects of any")
    print(f"  kind -- NOT filtered to energy-specific projects (no such filter exists in this")
    print(f"  pipeline's CORDIS extraction; confirmed directly, not assumed).")
    try:
        print(f"[FIT START] pooled real-only refit: rows={len(pooled_real)}", flush=True)
        model_real = pf.fepois(formula, pooled_real, vcov={"CRV1": "clubdyad_id"}, demeaner=LsmrDemeaner(backend="within"))
        print("[FIT OK] pooled real-only refit", flush=True)
        terms_real = list(model_real.coef().index)
        if "EU:es_z" in terms_real and "EU:es_z:innov_z" in terms_real:
            tidy_real = model_real.tidy().reset_index()
            d1_real = tidy_real[tidy_real[tidy_real.columns[0]] == "EU:es_z"].iloc[0]
            d3_real = tidy_real[tidy_real[tidy_real.columns[0]] == "EU:es_z:innov_z"].iloc[0]
            print(f"  delta_1 (EU - G20, ES main effect):        b={d1_real['Estimate']:+.4f}  "
                 f"se={d1_real['Std. Error']:.4f}  p={d1_real['Pr(>|t|)']:.4f}")
            print(f"  delta_3 (EU - G20, ES x Innov moderation): b={d3_real['Estimate']:+.4f}  "
                 f"se={d3_real['Std. Error']:.4f}  p={d3_real['Pr(>|t|)']:.4f}")
            idx1r, idx3r = terms_real.index("EU:es_z"), terms_real.index("EU:es_z:innov_z")
            sub_v_r = model_real._vcov[np.ix_([idx1r, idx3r], [idx1r, idx3r])]
            sub_b_r = np.array([model_real.coef()["EU:es_z"], model_real.coef()["EU:es_z:innov_z"]])
            wald_real = float(sub_b_r @ np.linalg.solve(sub_v_r, sub_b_r))
            wald_real_p = 1 - chi2.cdf(wald_real, df=2)
            real_reject = wald_real_p < 0.05
            mixed_reject = wald_p < 0.05 if not np.isnan(wald_p) else None
            print(f"  Joint Wald test H0: delta_1=0 AND delta_3=0:  chi2={wald_real:.3f}  df=2  "
                 f"p={wald_real_p:.4f}  {'REJECT H0' if real_reject else 'fail to reject H0'}")
            print(f"\n  DIRECT ANSWER: the rejection on the mixed (real+simulated) sample "
                 f"{'DOES' if (mixed_reject == real_reject) else 'DOES NOT'} persist on verified-real "
                 f"data alone (mixed: {'reject' if mixed_reject else 'fail to reject'}, p={wald_p:.4f}; "
                 f"real-only: {'reject' if real_reject else 'fail to reject'}, p={wald_real_p:.4f}).")
            print(f"  Even where it persists, this establishes a difference in CORDIS-based EU")
            print(f"  research-programme co-participation specifically -- NOT a claim about general")
            print(f"  bilateral energy cooperation, which remains a separate, unestablished question.")
        else:
            print("  EU/G20 interaction terms not identified on the real-only sample (likely too few")
            print("  observations or insufficient within-dyad variation once restricted) -- cannot")
            print("  answer the persistence question from this sample as currently sized.")
    except Exception as e:
        print(f"  Real-only refit failed: {e}")

    # TASK (reviewer point 7): "clustering only by clubdyad_id treats the
    # G20 Germany-France dyad and EU Germany-France dyad as independent.
    # They share the same countries, covariates and potentially the same
    # outcome observations." Two of the reviewer's four suggested checks
    # are implemented directly here (the other two -- full multiway
    # country x dyad clustering, which pyfixest's CRV1 does not support
    # directly, and a fully separate refit pipeline -- are noted as not
    # attempted rather than approximated poorly): (1) reclustering by the
    # UNDERLYING unordered country pair, stripping the club prefix, so
    # G20_DEU_FRA and EU_DEU_FRA share one cluster; (2) refitting on a
    # pooled sample that excludes DEU, FRA and ITA (the three overlap
    # countries) entirely, removing the shared-country dependence at its
    # source rather than merely reweighting for it.
    print("\n" + "-" * 70)
    print("Sensitivity to the EU-G20 overlap countries (reviewer point 7)")
    print("-" * 70)
    pooled["country_pair_id"] = pooled.apply(lambda r: "_".join(sorted([r["iso_i"], r["iso_j"]])), axis=1)
    try:
        print(f"[FIT START] pooled country-pair-recluster refit: rows={len(pooled)}", flush=True)
        model_altclust = pf.fepois(formula, pooled, vcov={"CRV1": "country_pair_id"}, demeaner=LsmrDemeaner(backend="within"))
        print("[FIT OK] pooled country-pair-recluster refit", flush=True)
        d1_alt = model_altclust.tidy().reset_index()
        d1_alt_row = d1_alt[d1_alt[d1_alt.columns[0]] == "EU:es_z"].iloc[0]
        d3_alt_row = d1_alt[d1_alt.columns[0]].eq("EU:es_z:innov_z")
        d3_alt_row = d1_alt[d3_alt_row].iloc[0]
        idx1a, idx3a = list(model_altclust.coef().index).index("EU:es_z"), list(model_altclust.coef().index).index("EU:es_z:innov_z")
        sub_v_alt = model_altclust._vcov[np.ix_([idx1a, idx3a], [idx1a, idx3a])]
        sub_b_alt = np.array([model_altclust.coef()["EU:es_z"], model_altclust.coef()["EU:es_z:innov_z"]])
        wald_alt = float(sub_b_alt @ np.linalg.solve(sub_v_alt, sub_b_alt))
        wald_alt_p = 1 - chi2.cdf(wald_alt, df=2)
        print(f"  (1) Reclustered by UNDERLYING country pair (not club+pair): delta_1 SE "
             f"{d1_row['Std. Error']:.4f} -> {d1_alt_row['Std. Error']:.4f}; delta_3 SE "
             f"{d3_row['Std. Error']:.4f} -> {d3_alt_row['Std. Error']:.4f}; joint Wald "
             f"chi2={wald_alt:.3f} p={wald_alt_p:.4f} "
             f"({'REJECT H0' if wald_alt_p < 0.05 else 'fail to reject H0'}, vs. "
             f"{'REJECT H0' if wald_p < 0.05 else 'fail to reject H0'} under clubdyad_id "
             f"clustering) -- {'CONCLUSION CHANGES' if (wald_alt_p < 0.05) != (wald_p < 0.05) else 'conclusion unchanged'}.")
    except Exception as e:
        print(f"  (1) Reclustering by underlying country pair failed: {e}")

    overlap_countries = {"DEU", "FRA", "ITA"}
    pooled_no_overlap = pooled[~(pooled.iso_i.isin(overlap_countries) | pooled.iso_j.isin(overlap_countries))]
    pooled_no_overlap = _drop_degenerate_fe_groups(pooled_no_overlap, ["clubdyad_id", "club_year"])
    n_dropped = len(pooled) - len(pooled_no_overlap)
    try:
        print(f"[FIT START] pooled overlap-excluded refit: rows={len(pooled_no_overlap)}", flush=True)
        model_nooverlap = pf.fepois(formula, pooled_no_overlap, vcov={"CRV1": "clubdyad_id"}, demeaner=LsmrDemeaner(backend="within"))
        print("[FIT OK] pooled overlap-excluded refit", flush=True)
        d_terms = list(model_nooverlap.coef().index)
        if "EU:es_z" in d_terms and "EU:es_z:innov_z" in d_terms:
            tidy_no = model_nooverlap.tidy().reset_index()
            d1_no_row = tidy_no[tidy_no[tidy_no.columns[0]] == "EU:es_z"].iloc[0]
            d3_no_row = tidy_no[tidy_no[tidy_no.columns[0]] == "EU:es_z:innov_z"].iloc[0]
            idx1n, idx3n = d_terms.index("EU:es_z"), d_terms.index("EU:es_z:innov_z")
            sub_v_no = model_nooverlap._vcov[np.ix_([idx1n, idx3n], [idx1n, idx3n])]
            sub_b_no = np.array([model_nooverlap.coef()["EU:es_z"], model_nooverlap.coef()["EU:es_z:innov_z"]])
            wald_no = float(sub_b_no @ np.linalg.solve(sub_v_no, sub_b_no))
            wald_no_p = 1 - chi2.cdf(wald_no, df=2)
            print(f"  (2) Excluding DEU/FRA/ITA entirely ({n_dropped} obs dropped, "
                 f"{len(pooled_no_overlap)} remain): delta_1={d1_no_row['Estimate']:+.4f} "
                 f"(was {d1_row['Estimate']:+.4f}), delta_3={d3_no_row['Estimate']:+.4f} "
                 f"(was {d3_row['Estimate']:+.4f}); joint Wald chi2={wald_no:.3f} p={wald_no_p:.4f} "
                 f"({'REJECT H0' if wald_no_p < 0.05 else 'fail to reject H0'}) -- "
                 f"{'CONCLUSION CHANGES' if (wald_no_p < 0.05) != (wald_p < 0.05) else 'conclusion unchanged'}.")
    except Exception as e:
        print(f"  (2) Overlap-excluding refit failed: {e}")
    print("  NOT attempted: full multiway (country x dyad) clustering -- pyfixest's CRV1 "
         "does not support a two-way cluster variable directly, and approximating it poorly "
         "would be worse than stating the gap. (1) and (2) above are the two most directly "
         "actionable of the reviewer's four suggestions.")

    # TASK (reviewer point 8): "the previous pooled test found chi2=0.775,
    # p=0.679... nothing in the attached code changes that empirical
    # conclusion yet. Only a new run using the revised real variables
    # could do so." This IS that new run, using the real Chapter 2 ES,
    # real WTO RTA, real Brent turbulence, and (after the verified-zero
    # fix above) 96.4% real CORDIS cooperation data integrated since that
    # comment was written. Flagged explicitly, in either direction, rather
    # than letting a changed
    # number pass without comment.
    PREVIOUSLY_REPORTED_WALD_P = 0.679
    real_share_now = n_real / max(len(pooled), 1)
    if not np.isnan(wald_p):
        prev_reject, now_reject = PREVIOUSLY_REPORTED_WALD_P < 0.05, wald_p < 0.05
        if prev_reject != now_reject:
            print(f"\n  *** CONCLUSION HAS CHANGED from the previously-reported run (chi2=0.775, "
                 f"p={PREVIOUSLY_REPORTED_WALD_P}, fail to reject) to THIS run (chi2={wald_stat:.3f}, "
                 f"p={wald_p:.4f}, {'REJECT' if now_reject else 'fail to reject'}). Expected and "
                 f"appropriate -- point 8 anticipated exactly this. BUT per the point-2 caveat "
                 f"above, the cooperation outcome driving this is now {real_share_now:.1%} real via "
                 f"CORDIS specifically, which is NOT club-neutral -- this rejection may partly "
                 f"reflect CORDIS's EU-programme institutional structure rather than a purely "
                 f"substantive EU-G20 difference in GENERAL cooperation intensity. Read this "
                 f"result alongside, not instead of, that caveat.")
        else:
            print(f"\n  Conclusion unchanged from the previously-reported run: still "
                 f"{'REJECT' if now_reject else 'fail to reject'} H0 (was p={PREVIOUSLY_REPORTED_WALD_P}, "
                 f"now p={wald_p:.4f}).")

    real_share = n_real / max(len(pooled), 1)
    if real_share < 0.005:
        # FIX (reviewer, quoting the exact bug): the old message asserted
        # "a genuine empirical result on that real share blended with..."
        # UNCONDITIONALLY, which is self-contradictory when real_share is
        # actually ~0% (no real cooperation data file present) -- there is
        # no real share to speak of. Wording below matches what the
        # reviewer proposed almost verbatim.
        print(f"\n  (VALIDATION CHECK CAVEAT: the pooled cooperation outcome is FULLY SIMULATED "
             f"({real_share:.1%} real). These estimates constitute a single-draw estimator-"
             f"validation exercise and must not be interpreted as empirical evidence. The known "
             f"coefficients (true delta_1={TRUE_DELTA1}, true delta_3={TRUE_DELTA3}) provide a "
             f"benchmark for evaluating estimator recovery; the repeated Monte Carlo experiment "
             f"in Section 8C evaluates bias, coverage, type-I error, and power properly -- this "
             f"single draw does not.\n"
             f"  Estimated: {d1_row['Estimate']:+.4f} / {d3_row['Estimate']:+.4f})")
    else:
        print(f"\n  (VALIDATION CHECK CAVEAT: true delta_1={TRUE_DELTA1}, true delta_3={TRUE_DELTA3} "
             f"are properties of the SIMULATED fraction of the outcome only ({1-real_share:.1%} of "
             f"observations). With {real_share:.1%} of observations now real (see below), the "
             f"delta_1/delta_3 ESTIMATES above are a genuine empirical result on that real share "
             f"blended with a known-answer simulated remainder -- comparing the estimate to the "
             f"'true' value is only a partial code-check here, not a full validation; see Section 8C "
             f"for a clean, fully-simulated Monte Carlo validation of the estimator itself.\n"
             f"  Estimated: {d1_row['Estimate']:+.4f} / {d3_row['Estimate']:+.4f})")
        # TASK (reviewer point 2): "CORDIS does not provide a neutral
        # G20-EU comparison... It records participation in EU-funded
        # research programmes. It will naturally provide richer and
        # denser coverage for EU dyads than for many G20 dyads." This is
        # a real, structural feature of the ONLY real bilateral source
        # driving delta_1/delta_3 above whenever real_share > 0: CORDIS is
        # an EU-programme-participation measure, and EU dyads are
        # near-automatically eligible for it in a way most G20-only pairs
        # are not (a G20 pair with no EU-programme participant simply
        # cannot generate a CORDIS-real observation at all). The estimand
        # this real share actually identifies is therefore NOT "general
        # bilateral energy cooperation intensity" in a club-neutral sense
        # -- it is cooperation WITHIN THE CORDIS/EU RESEARCH-PROGRAMME
        # ENVIRONMENT specifically, which is a defensible EU outcome and a
        # legitimate ROBUSTNESS check for internationally-participating
        # G20 economies, but not an institutionally neutral basis for
        # comparing the EU's and G20's overall cooperation intensity.
        print(f"\n  IMPORTANT ESTIMAND CAVEAT (institutional neutrality): the real share driving "
             f"the estimates above comes from CORDIS, an EU-funded-research-programme "
             f"participation record. It is genuinely dyadic and extends past 2012, which makes it "
             f"far better than a country-level proxy, but it is NOT club-neutral: EU dyads are "
             f"structurally far more likely to generate a CORDIS-real observation than most G20 "
             f"pairs, simply by virtue of EU programme eligibility, independent of any true "
             f"difference in cooperation intensity. The delta_1/delta_3 comparison above should be "
             f"read as cooperation WITHIN THE CORDIS/EU RESEARCH-PROGRAMME ENVIRONMENT, not as a "
             f"neutral estimate of general bilateral energy cooperation across the two clubs. "
             f"CORDIS remains defensible as an EU cooperation outcome on its own, and as a "
             f"robustness sample for internationally-participating G20 economies specifically.")

    # --- Real vs simulated / imputed observation share, per variable -------
    print("\n" + "-" * 70)
    print("Real vs. simulated observation share (pooled sample)")
    print("-" * 70)
    prov_df = pd.DataFrame(prov)
    print(prov_df.to_string(index=False))
    # FIX (reviewer point 7): this used to unconditionally print "0% real --
    # fully simulated" regardless of the actual real-data share computed
    # above (n_real/len(pooled)) -- a genuine, confirmed contradiction with
    # the "Cooperation outcome: X/Y ... use REAL data" line printed earlier
    # in this same function. Now derived from the same n_real value, not
    # hardcoded, so the two statements cannot disagree.
    print(f"  cooperation (outcome): {n_real}/{len(pooled)} ({real_share:.1%}) real "
         f"(observed bilateral data via the same loader as the per-club models); "
         f"remainder simulated. See the 'Cooperation outcome:' line above for the "
         f"identical figure -- restated here so it appears in the same block as "
         f"every other variable's real/simulated share.")

    out_dir = OUT_BASE / "pooled"; (out_dir / "tables").mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows_out).to_csv(out_dir / "tables" / "pooled_coefficients.csv", index=False)
    pd.DataFrame(marg_rows).to_csv(out_dir / "tables" / "marginal_effects.csv", index=False)
    prov_df.to_csv(out_dir / "tables" / "data_provenance.csv", index=False)
    pd.DataFrame([{
        "n_g20_dyads": len(list(combinations(g20_members, 2))), "n_eu_dyads": len(list(combinations(eu_members, 2))),
        "n_obs_g20": int(n_g20), "n_obs_eu": int(n_eu),
        "zero_share_pooled": float((pooled.coop_count == 0).mean()),
        "wald_delta1_delta3_chi2": wald_stat, "wald_delta1_delta3_p": wald_p,
    }]).to_csv(out_dir / "tables" / "sample_summary.csv", index=False)
    print(f"\nOutputs in {out_dir}/tables/  "
          f"(pooled_coefficients.csv, marginal_effects.csv, data_provenance.csv, sample_summary.csv)")


# =========================================================================
# SECTION 8 ? CROSS-CLUB COMPARISON (only when both clubs are run)
# =========================================================================
# =========================================================================
# SECTION 8C ? MONTE CARLO VALIDATION OF THE POOLED ESTIMATOR
# =========================================================================
# TASK ("validate the estimator with repeated Monte Carlo replications"):
# replaces the earlier single-draw "does the point estimate land in a
# plausible range" check with an actual Monte Carlo study. The regressors
# (es_z, innov_z, their interaction, EU indicator, dyad/club-year identity)
# and the "true" unobserved dyad and club-year heterogeneity are fixed
# ONCE -- they represent one population -- and only the Poisson sampling
# noise is redrawn on each of B replications, exactly as a Monte Carlo
# study of a fixed-design estimator should be set up. For every
# replication the pooled PPML interaction model (Section 8B's formula) is
# refit from scratch and every coefficient's point estimate and 95% CI are
# recorded. Reported per parameter: mean estimate, bias (mean - true),
# RMSE, and EMPIRICAL COVERAGE -- the share of replications whose reported
# 95% CI actually contained the true value, which should be close to 95%
# for a well-calibrated estimator and is the single most informative
# Monte Carlo diagnostic (a biased or overconfident estimator shows up
# here even when the average point estimate looks fine).
B_MC = 1000  # TASK (reviewer: "increase to at least 1,000... preferably 2,000 if
            # computation time remains reasonable"): raised from 100. Timed at
            # ~0.15s/replication for the combined main+null-DGP battery, so
            # B=1000 takes ~2.5 minutes -- confirmed feasible, not assumed.
            # MC standard error on a coverage/type-I-error rate near 0.95/0.05
            # drops from ~2.2 pts at B=100 to ~0.7 pts at B=1000. Pass a
            # smaller B explicitly (--montecarlo runs with this default; call
            # run_monte_carlo_validation(B=...) directly for a custom value)
            # for fast iteration during development.

MC_TRUE = {"es_z": TRUE_BETA_ES, "innov_z": TRUE_BETA_INNOV, "es_z:innov_z": TRUE_BETA_INT,
          "EU:es_z": TRUE_DELTA1, "EU:innov_z": TRUE_DELTA2, "EU:es_z:innov_z": TRUE_DELTA3}


def run_monte_carlo_validation(B=B_MC):
    print("=" * 70)
    print(f"MONTE CARLO VALIDATION OF THE POOLED ESTIMATOR (B={B} replications)")
    print("=" * 70)
    print("  Regressors and the 'true' dyad/club-year heterogeneity are fixed once")
    print("  (one population); only the Poisson outcome noise is redrawn each")
    print("  replication and the full pooled interaction model is refit from")
    print("  scratch every time -- this is a genuine repeated-sampling check, not")
    print("  a single simulated draw.")

    pooled, *_ = build_pooled_regressors()
    dfe = pooled["clubdyad_id"].map({d: rng.normal(0, 0.4) for d in pooled["clubdyad_id"].unique()}).to_numpy()
    cyfe = pooled["club_year"].map({c: rng.normal(0, 0.2) for c in pooled["club_year"].unique()}).to_numpy()
    EU = pooled["EU"].to_numpy()
    lin_true = (np.log(0.5)
               + TRUE_BETA_ES * pooled["es_z"] + TRUE_BETA_INNOV * pooled["innov_z"]
               + TRUE_BETA_INT * pooled["es_z:innov_z"]
               + TRUE_DELTA1 * EU * pooled["es_z"] + TRUE_DELTA2 * EU * pooled["innov_z"]
               + TRUE_DELTA3 * EU * pooled["es_z:innov_z"]
               + dfe + cyfe)
    lam = np.exp(np.clip(lin_true, -10, 10))

    import pyfixest as pf
    formula = ("coop_count ~ es_z + innov_z + es_z:innov_z "
              "+ EU:es_z + EU:innov_z + EU:es_z:innov_z | clubdyad_id + club_year")

    params = list(MC_TRUE.keys())
    estimates = {p: [] for p in params}
    ses = {p: [] for p in params}
    covered = {p: [] for p in params}
    pvals = {p: [] for p in params}       # TASK (item I): power = P(reject H0 | true effect)
    n_always_zero_dyads = []              # TASK (item I): dyads dropped because always zero
    n_failed = 0
    t0 = time.time()
    for b in range(B):
        if b % 25 == 0:
            print(f"[MC ITER] main MC validation loop: b={b}/{B}", flush=True)
        pooled["coop_count"] = rng.poisson(lam)
        always_zero = pooled.groupby("clubdyad_id")["coop_count"].max().eq(0).sum()
        n_always_zero_dyads.append(int(always_zero))
        try:
            model = pf.fepois(formula, pooled, vcov={"CRV1": "clubdyad_id"}, demeaner=LsmrDemeaner(backend="within"))
            coefs, ses_b = model.coef(), model.se()
            for p in params:
                if p not in coefs.index:
                    continue
                est_b, se_b = float(coefs[p]), float(ses_b[p])
                estimates[p].append(est_b)
                ses[p].append(se_b)
                lo, hi = est_b - 1.96 * se_b, est_b + 1.96 * se_b
                covered[p].append(lo <= MC_TRUE[p] <= hi)
                z = est_b / se_b if se_b > 0 else 0.0
                pvals[p].append(2 * (1 - norm.cdf(abs(z))))
        except Exception:
            n_failed += 1
        if (b + 1) % max(1, B // 5) == 0:
            print(f"    {b+1}/{B} replications ({time.time()-t0:.0f}s elapsed)", end="\r")
    print()
    if n_failed:
        print(f"  {n_failed}/{B} replications failed to fit (e.g. degenerate draw) and were skipped.")
    print(f"  Dyads with an all-zero outcome (no within-dyad variation, effectively dropped "
         f"by dyad FE): mean {np.mean(n_always_zero_dyads):.1f} of {pooled['clubdyad_id'].nunique()} "
         f"across replications (range {min(n_always_zero_dyads)}-{max(n_always_zero_dyads)}).")

    print("\n" + "-" * 70)
    print("Table: Monte Carlo properties of the pooled estimator")
    print("-" * 70)
    print(f"  {'Parameter':20s} {'True':>8s} {'Mean est':>10s} {'Bias':>9s} {'RMSE':>8s} {'Coverage':>9s} {'Power':>7s} {'n':>5s}")
    rows_out = []
    for p in params:
        vals = np.array(estimates[p])
        if len(vals) == 0:
            continue
        true_v = MC_TRUE[p]
        mean_est = vals.mean()
        bias = mean_est - true_v
        rmse = math.sqrt(np.mean((vals - true_v) ** 2))
        cov_rate = np.mean(covered[p])
        # TASK (item I, "power"): share of replications correctly rejecting
        # H0:beta=0 at 5%, for parameters with a nonzero true effect. For
        # a true-zero parameter this same statistic IS the type-I error
        # rate -- none of the six pooled parameters here are exactly zero
        # by construction, so a dedicated null-DGP run (below) is used for
        # a genuine type-I-error / false-difference-detection check.
        power = np.mean(np.array(pvals[p]) < 0.05)
        print(f"  {p:20s} {true_v:+8.4f} {mean_est:+10.4f} {bias:+9.4f} {rmse:8.4f} {cov_rate:9.1%} {power:6.1%} {len(vals):5d}")
        rows_out.append({"parameter": p, "true_value": true_v, "mean_estimate": mean_est,
                         "bias": bias, "rmse": rmse, "coverage_95pct_CI": cov_rate,
                         "power_5pct": power, "n_replications": len(vals)})
    print("-" * 70)
    print("  Coverage should be near 95% for a well-calibrated estimator (Monte Carlo")
    print(f"  standard error on the reported coverage rates here is roughly "
         f"{100*math.sqrt(0.95*0.05/max(B,1)):.1f} pts at B={B}).")

    # TASK (item I: "type-I error" / "false detection of EU-G20 differences"):
    # none of the six parameters above has a true value of exactly zero, so
    # their rejection rates measure POWER, not type-I error. A genuine
    # type-I-error check needs a scenario where the null is actually true --
    # run here as a SEPARATE null-DGP battery with delta_1=delta_3=0 (no real
    # EU-G20 difference), checking how often the pooled model's own joint
    # Wald test (Section 8B) falsely rejects H0 anyway.
    print("\n" + "-" * 70)
    print(f"Null-DGP check: TYPE-I ERROR for the EU-G20 joint difference test (B={B})")
    print("-" * 70)
    print("  Same population, but delta_1=delta_3=0 imposed (no true EU-G20")
    print("  difference) -- the joint Wald test should reject H0 at close to the")
    print("  nominal 5% rate; a much higher rate would mean the test over-rejects.")
    lin_null = (np.log(0.5)
               + TRUE_BETA_ES * pooled["es_z"] + TRUE_BETA_INNOV * pooled["innov_z"]
               + TRUE_BETA_INT * pooled["es_z:innov_z"]
               + dfe + cyfe)  # no EU:... terms at all -> true delta_1=delta_2=delta_3=0
    lam_null = np.exp(np.clip(lin_null, -10, 10))
    n_false_reject = 0
    n_null_failed = 0
    for b in range(B):
        if b % 25 == 0:
            print(f"[MC ITER] type-1-error null loop: b={b}/{B}", flush=True)
        pooled["coop_count"] = rng.poisson(lam_null)
        try:
            model = pf.fepois(formula, pooled, vcov={"CRV1": "clubdyad_id"}, demeaner=LsmrDemeaner(backend="within"))
            names_b = list(model.coef().index)
            if "EU:es_z" not in names_b or "EU:es_z:innov_z" not in names_b:
                n_null_failed += 1; continue
            idx1, idx3 = names_b.index("EU:es_z"), names_b.index("EU:es_z:innov_z")
            sub_v = model._vcov[np.ix_([idx1, idx3], [idx1, idx3])]
            sub_b = np.array([model.coef()["EU:es_z"], model.coef()["EU:es_z:innov_z"]])
            wald_stat = float(sub_b @ np.linalg.solve(sub_v, sub_b))
            wald_p = 1 - chi2.cdf(wald_stat, df=2)
            if wald_p < 0.05:
                n_false_reject += 1
        except Exception:
            n_null_failed += 1
    type1_rate = n_false_reject / max(B - n_null_failed, 1)
    print(f"  False-positive rate (type-I error): {n_false_reject}/{B - n_null_failed} = {type1_rate:.1%} "
         f"(nominal target: 5%; {n_null_failed} replications failed and were excluded)")

    # TASK (reviewer): "Monte Carlo coefficient distributions against the
    # true values" -- histogram of the actual per-replication estimates,
    # not just their summary statistics.
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, p in zip(axes.flat, params):
        vals = np.array(estimates[p])
        if len(vals) == 0:
            continue
        ax.hist(vals, bins=30, color="#2E5A88", edgecolor="white", alpha=0.85)
        ax.axvline(MC_TRUE[p], color="#C00000", lw=2, label=f"true={MC_TRUE[p]:+.3f}")
        ax.axvline(vals.mean(), color="#1A7A4A", lw=2, ls="--", label=f"mean est={vals.mean():+.3f}")
        ax.set_title(p, fontsize=9); ax.legend(fontsize=7)
    fig.suptitle(f"Monte Carlo coefficient distributions vs true values (B={B})", y=1.01)
    plt.tight_layout()
    _save_fig(fig, OUT_BASE / "pooled", "fig_mc_coefficient_distributions")

    # TASK (reviewer): "Minimum detectable effect and power curves."
    # MDE at 80% power, two-sided 5% test: MDE = SE * (z_.025 + z_.20).
    # Power curve: re-run a SMALLER battery (fewer reps -- this is run once
    # per effect-size multiplier, so full B would be very expensive) at
    # several assumed true effect sizes for es_z specifically, holding
    # everything else fixed, to show how power scales with effect size.
    z_mde = norm.ppf(0.975) + norm.ppf(0.80)
    mde_rows = []
    for p in params:
        se_mean = np.mean(ses[p]) if len(ses[p]) > 0 else np.nan
        mde_rows.append({"parameter": p, "mean_se": se_mean, "mde_80pct_power": z_mde * se_mean})
    print("\n" + "-" * 70)
    print("Minimum detectable effect (80% power, two-sided 5% test)")
    print("-" * 70)
    for r in mde_rows:
        print(f"  {r['parameter']:20s} mean SE={r['mean_se']:.4f}  MDE={r['mde_80pct_power']:.4f}")
    (OUT_BASE / "pooled" / "tables").mkdir(parents=True, exist_ok=True)
    pd.DataFrame(mde_rows).to_csv(OUT_BASE / "pooled" / "tables" / "minimum_detectable_effect.csv", index=False)

    B_curve = min(200, B)
    print(f"\n  Power curve for es_z across assumed effect sizes (B={B_curve}/point, faster "
         f"than the main battery since this repeats it 5x):")
    multipliers = [0.25, 0.5, 1.0, 1.5, 2.0]
    power_curve = []
    for mult in multipliers:
        true_es_mult = TRUE_BETA_ES * mult
        lin_mult = (np.log(0.5) + true_es_mult * pooled["es_z"] + TRUE_BETA_INNOV * pooled["innov_z"]
                   + TRUE_BETA_INT * pooled["es_z:innov_z"] + TRUE_DELTA1 * EU * pooled["es_z"]
                   + TRUE_DELTA2 * EU * pooled["innov_z"] + TRUE_DELTA3 * EU * pooled["es_z:innov_z"]
                   + dfe + cyfe)
        lam_mult = np.exp(np.clip(lin_mult, -10, 10))
        n_reject = 0; n_valid = 0
        for b in range(B_curve):
            if b % 25 == 0:
                print(f"[MC ITER] power-curve loop: b={b}/{B_curve}", flush=True)
            pooled["coop_count"] = rng.poisson(lam_mult)
            try:
                mm = pf.fepois(formula, pooled, vcov={"CRV1": "clubdyad_id"}, demeaner=LsmrDemeaner(backend="within"))
                if "es_z" in mm.coef().index and mm.se()["es_z"] > 0:
                    t = float(mm.coef()["es_z"] / mm.se()["es_z"])
                    p_val = 2 * (1 - norm.cdf(abs(t)))
                    n_reject += int(p_val < 0.05); n_valid += 1
            except Exception:
                pass
        power_curve.append({"effect_multiplier": mult, "true_effect": true_es_mult,
                            "power": n_reject / max(n_valid, 1), "n_valid": n_valid})
        print(f"    effect={true_es_mult:+.4f} ({mult}x true): power={n_reject/max(n_valid,1):.1%}")
    pooled["coop_count"] = rng.poisson(lam)  # restore original DGP before returning
    pd.DataFrame(power_curve).to_csv(OUT_BASE / "pooled" / "tables" / "power_curve.csv", index=False)

    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    pc_df = pd.DataFrame(power_curve)
    ax.plot(pc_df.true_effect, pc_df.power * 100, "-o", color="#1F3864")
    ax.axhline(80, color="#C00000", ls="--", lw=1, label="80% power target")
    ax.set_xlabel("True effect size (es_z coefficient)"); ax.set_ylabel("Power (%)")
    ax.set_title(f"Power curve for ES effect (B={B_curve} per point)")
    ax.legend()
    plt.tight_layout()
    _save_fig(fig, OUT_BASE / "pooled", "fig_mc_power_curve")

    out_dir = OUT_BASE / "pooled"; (out_dir / "tables").mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows_out).to_csv(out_dir / "tables" / "monte_carlo_validation.csv", index=False)
    pd.DataFrame([{"type1_error_rate": type1_rate, "n_false_reject": n_false_reject,
                  "n_valid": B - n_null_failed, "mean_always_zero_dyads": float(np.mean(n_always_zero_dyads))}]
                ).to_csv(out_dir / "tables" / "monte_carlo_type1_error.csv", index=False)
    print(f"\nOutputs in {out_dir}/tables/monte_carlo_validation.csv and monte_carlo_type1_error.csv")
    return pd.DataFrame(rows_out)


# =========================================================================
# SECTION 9 ? DIAGRAMS
# =========================================================================
# TASK: "add the diagrams (10+ figures requested)". Implements the subset
# that is genuinely DATA-DRIVEN from what this pipeline actually computes
# (a coverage heatmap, cooperation trends, coefficient/IRR plots, marginal
# effects, lead-lag plot, a cross-specification robustness plot, and a
# cooperation network map). NOT implemented: the two purely conceptual/
# schematic diagrams (a hand-drawn "conceptual framework" box-and-arrow
# figure, and a "data construction pipeline" flow chart) -- these aren't
# generated from data and are better made directly in a document/slide
# tool than reverse-engineered as a matplotlib figure.

def _save_fig(fig, out_dir, name):
    fig_dir = out_dir / "figures"; fig_dir.mkdir(parents=True, exist_ok=True)
    path = fig_dir / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"    [figure] saved {path}")
    return path


def plot_data_coverage_heatmap(club_key, df, out_dir):
    """Figure 3: rows=countries, columns=years, coverage (1=real ES score
    present) for ES; a second panel for cooperation outcome coverage."""
    members = sorted(set(df.iso_i) | set(df.iso_j))
    years = sorted(df.year.unique())
    es_cov = pd.DataFrame(0, index=members, columns=years)
    for iso in members:
        sub_i = df[df.iso_i == iso][["year", "es_i"]].drop_duplicates()
        sub_j = df[df.iso_j == iso][["year", "es_j"]].rename(columns={"es_j": "es_i"}).drop_duplicates()
        both = pd.concat([sub_i, sub_j])
        for _, r in both.iterrows():
            if r.year in es_cov.columns and pd.notna(r.es_i):
                es_cov.loc[iso, r.year] = 1
    coop_col = "energy_events_sim" if "energy_events_sim" in df.columns else "coop_count"
    coop_cov = pd.DataFrame(0, index=members, columns=years)
    for iso in members:
        sub = df[(df.iso_i == iso) | (df.iso_j == iso)][["year", coop_col]]
        for yr, g in sub.groupby("year"):
            if yr in coop_cov.columns and g[coop_col].notna().any():
                coop_cov.loc[iso, yr] = 1

    fig, axes = plt.subplots(1, 2, figsize=(14, max(4, 0.28 * len(members))))
    for ax, (mat, title) in zip(axes, [(es_cov, "ES index coverage"), (coop_cov, "Cooperation outcome coverage")]):
        ax.imshow(mat.values, aspect="auto", cmap="Greens", vmin=0, vmax=1)
        ax.set_yticks(range(len(members))); ax.set_yticklabels(members, fontsize=6)
        ax.set_xticks(range(len(years))); ax.set_xticklabels(years, rotation=90, fontsize=6)
        ax.set_title(f"{club_key.upper()}: {title}", fontsize=10)
    plt.tight_layout()
    return _save_fig(fig, out_dir, "fig3_data_coverage_heatmap")


def plot_cooperation_trends(club_key, df, out_dir):
    """Figure 5: annual mean/total cooperation count and zero-share."""
    coop_col = "energy_events_sim" if "energy_events_sim" in df.columns else "coop_count"
    # FIX: title used to print the raw column name ("energy_events_sim"),
    # which is misleading once real data is active -- the column name
    # itself contains "_sim" as a naming artifact (see build_panel's
    # aliasing) even when every value in it is real. Uses the same
    # corrected real/simulated check as plot_cooperation_network.
    is_sim = ("energy_events_sim" in df.columns and "formal_agreements_sim" in df.columns
             and not df["energy_events_sim"].equals(df["formal_agreements_sim"]))
    coop_label = "SIMULATED outcome" if is_sim else "real/mixed outcome"
    annual = df.groupby("year").agg(mean_count=(coop_col, "mean"), total_count=(coop_col, "sum"),
                                    zero_share=(coop_col, lambda s: (s == 0).mean()))
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
    ax1.plot(annual.index, annual.mean_count, "-o", color="#1F3864", label="Mean count")
    ax1b = ax1.twinx()
    ax1b.plot(annual.index, annual.total_count, "--s", color="#C00000", label="Total count")
    ax1.set_ylabel("Mean count per dyad"); ax1b.set_ylabel("Total count")
    ax1.set_title(f"{club_key.upper()}: cooperation trends ({coop_label})")
    lines1, labels1 = ax1.get_legend_handles_labels(); lines2, labels2 = ax1b.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=8)
    ax2.plot(annual.index, annual.zero_share * 100, "-o", color="#888888")
    ax2.set_ylabel("% dyad-years with zero cooperation"); ax2.set_xlabel("Year")
    ax2.set_title("Zero share over time")
    plt.tight_layout()
    return _save_fig(fig, out_dir, "fig5_cooperation_trends")


def plot_baseline_coefficients(club_key, models, out_dir):
    """Figure 6: IRR + 95% CI forest plot for the key baseline coefficients.
    ES level, Innovation, ES x Innovation and ES x Volatility are all read
    from the SINGLE correctly-specified m_full_ppml model (matching the
    DGP's exact functional form) -- not mixed across m1_ppml/m3_innov/
    m4_vol, which are each separately misspecified relative to each other
    (each omits at least one interaction the others include) and would
    give an internally inconsistent combined picture if blended here."""
    targets = [("m_full_ppml", "es_mean_l1_z", "ES level (at avg innov/vol)"),
              ("m11_es_improvement", "es_impr_mean_l1_z", "ES improvement (ln MLI)"),
              ("m_full_ppml", "innov_mean_l1_z", "Innovation"),
              ("m_full_ppml", "es_mean_l1_z:innov_mean_l1_z", "ES x Innovation"),
              ("m_full_ppml", "es_mean_l1_z:vol_z", "ES x Brent turbulence"),
              ("m7_diff", "es_diff_l1_z", "ES gap (asymmetry)")]
    rows = []
    for model_key, term, label in targets:
        if model_key not in models:
            continue
        mm = models[model_key]
        if term not in mm.coef().index:
            continue
        b, se = float(mm.coef()[term]), float(mm.se()[term])
        rows.append((label, b, se))
    if not rows:
        return None
    fig, ax = plt.subplots(figsize=(8, 0.6 * len(rows) + 1.5))
    for i, (label, b, se) in enumerate(rows):
        irr, lo, hi = math.exp(b), math.exp(b - 1.96 * se), math.exp(b + 1.96 * se)
        ax.plot([lo, hi], [i, i], color="#1F3864", lw=2)
        ax.scatter([irr], [i], color="#C00000", s=50, zorder=5)
    ax.axvline(1.0, color="grey", ls="--", lw=1)
    ax.set_yticks(range(len(rows))); ax.set_yticklabels([r[0] for r in rows])
    ax.set_xlabel("Incidence rate ratio (95% CI)")
    ax.set_title(f"{club_key.upper()}: baseline coefficients (IRR scale)")
    plt.tight_layout()
    return _save_fig(fig, out_dir, "fig6_baseline_irr")


def plot_marginal_effects_innovation(club_key, models, out_dir):
    """Figure 7: marginal ES effect (dE(Y)/dES = beta1 + beta3*Innovation)
    across Innovation from -2 to +2 SD, with a 95% CI band. Uses
    m_full_ppml (the model matching the DGP's exact functional form,
    ES x Innovation AND ES x Volatility both included) rather than
    m3_innov, which omits ES x Volatility and would give a marginal-effect
    curve subject to the same omitted-moderation distortion the reviewer
    identified for the headline ES coefficient."""
    model_key = "m_full_ppml" if "m_full_ppml" in models else "m3_innov"
    if model_key not in models:
        return None
    mm = models[model_key]
    names = list(mm.coef().index)
    if "es_mean_l1_z" not in names or "es_mean_l1_z:innov_mean_l1_z" not in names:
        return None
    b1 = float(mm.coef()["es_mean_l1_z"]); b3 = float(mm.coef()["es_mean_l1_z:innov_mean_l1_z"])
    V = mm._vcov
    i1, i3 = names.index("es_mean_l1_z"), names.index("es_mean_l1_z:innov_mean_l1_z")
    grid = np.linspace(-2, 2, 41)
    me = b1 + b3 * grid
    var = V[i1, i1] + grid**2 * V[i3, i3] + 2 * grid * V[i1, i3]
    se = np.sqrt(np.clip(var, 0, None))
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(grid, me, color="#1F3864", lw=2)
    ax.fill_between(grid, me - 1.96 * se, me + 1.96 * se, color="#2E5A88", alpha=0.2)
    ax.axhline(0, color="grey", ls="--", lw=1)
    ax.set_xlabel("Innovation (SD)"); ax.set_ylabel("Marginal effect of ES (log scale)")
    ax.set_title(f"{club_key.upper()}: marginal ES effect across Innovation ({model_key})")
    plt.tight_layout()
    return _save_fig(fig, out_dir, "fig7_marginal_effects_innovation")


def plot_lead_lag(club_key, models, out_dir):
    """Figure 9: lead-lag coefficient plot -- t-2, t-1, t+1 (this pipeline
    does not compute t-3/t+2; plotted range is exactly what's available,
    not padded to imply a wider window than actually estimated)."""
    specs = [("m6_lag2", "es_mean_l2_z", -2), ("m1_ppml", "es_mean_l1_z", -1),
            ("m6_lead1", "es_mean_f1_z", 1)]
    rows = []
    for model_key, term, t in specs:
        if model_key in models and term in models[model_key].coef().index:
            mm = models[model_key]
            rows.append((t, float(mm.coef()[term]), float(mm.se()[term])))
    if not rows:
        return None
    rows.sort()
    ts = [r[0] for r in rows]; bs = [r[1] for r in rows]; ses = [r[2] for r in rows]
    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.errorbar(ts, bs, yerr=[1.96 * s for s in ses], fmt="o-", color="#1F3864", capsize=4)
    ax.axhline(0, color="grey", ls="--", lw=1); ax.axvline(0, color="#C00000", ls=":", lw=1)
    # TASK (reviewer): "the red reference line should ideally... have a
    # visible t=0 marker or explanation that the contemporaneous
    # coefficient is intentionally omitted." The line was already at t=0;
    # what was missing was the annotation explaining WHY there is no data
    # point there (t=0 is the cooperation year itself, not a separate
    # lag/lead to be estimated -- there is no "0-year lag" regression).
    y_top = max(b + 1.96 * s for b, s in zip(bs, ses))
    y_bottom = min(b - 1.96 * s for b, s in zip(bs, ses))
    y_range = y_top - y_bottom if y_top > y_bottom else 1.0
    ax.annotate("t=0: cooperation year itself\n(not a separate lag/lead estimate)",
               xy=(0, y_top), xytext=(0.3, y_top - 0.05 * y_range),
               fontsize=8, color="#C00000", ha="left", va="top")
    ax.set_xticks(ts); ax.set_xlabel("Relative year (0 = cooperation year)")
    ax.set_ylabel("ES coefficient"); ax.set_title(f"{club_key.upper()}: lead-lag pattern (available range)")
    plt.tight_layout()
    return _save_fig(fig, out_dir, "fig9_lead_lag")


def plot_robustness_specs(club_key, models, out_dir):
    """Figure 10: ES (or closest analogue) coefficient across every
    available specification, as a single forest plot."""
    candidates = [("m1_ppml", "es_mean_l1_z", "Baseline (PPML)"),
                 ("m2_ols", "es_mean_l1_z", "FE-OLS benchmark (FE-OLS)"),
                 ("m3_innov", "es_mean_l1_z", "+ Innovation interaction (PPML)"),
                 ("m4_vol", "es_mean_l1_z", "+ Volatility interaction (PPML)"),
                 ("m6_lag2", "es_mean_l2_z", "t-2 lag (PPML)"),
                 ("m6_lead1", "es_mean_f1_z", "t+1 placebo (PPML)"),
                 ("m7_min", "es_min_l1_z", "Pair minimum (PPML)"),
                 ("m7_diff", "es_diff_l1_z", "Pair gap (PPML)"),
                 ("m8_joint_lags", "es_mean_l1_z", "Joint lags, t-1 term (PPML)"),
                 ("m9_conditional_placebo", "es_mean_l1_z", "Conditional placebo, lag term (PPML)"),
                 ("m10_mean_gap", "es_mean_l1_z", "Mean+gap joint (PPML)"),
                 ("m11_es_improvement", "es_impr_mean_l1_z", "ES improvement, ln MLI (PPML)"),
                 ("m12_rd_patent_separate", "es_mean_l1_z", "R&D/patent-stock separate (PPML)"),
                 ("mA_gravity_feols", "es_mean_l1_z", "Gravity, dyad FE, real RTA + real trade (FE-OLS)"),
                 ("mA_gravity_ppml", "es_mean_l1_z", "Gravity, dyad FE, real RTA + real trade (PPML)"),
                 ("m_full_ppml", "es_mean_l1_z", "Full/preferred: ES+Innov+ESxInnov+ESxVol (PPML)")]
    rows = []
    for model_key, term, label in candidates:
        if model_key in models and term in models[model_key].coef().index:
            mm = models[model_key]
            rows.append((label, float(mm.coef()[term]), float(mm.se()[term])))
    if not rows:
        return None
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(rows) + 1.5))
    for i, (label, b, se) in enumerate(rows):
        ax.plot([b - 1.96 * se, b + 1.96 * se], [i, i], color="#1F3864", lw=2)
        ax.scatter([b], [i], color="#C00000", s=40, zorder=5)
    ax.axvline(0, color="grey", ls="--", lw=1)
    ax.set_yticks(range(len(rows))); ax.set_yticklabels([r[0] for r in rows], fontsize=8)
    ax.set_xlabel("ES coefficient (log scale; note: not all rows use the identical term, see labels)")
    ax.set_title(f"{club_key.upper()}: ES coefficient across specifications")
    plt.tight_layout()
    return _save_fig(fig, out_dir, "fig10_robustness_specs")


def plot_cooperation_network(club_key, df, out_dir):
    """Figure 4: cooperation network map -- nodes=countries, edge width=
    mean annual cooperation. Uses whatever outcome is actually active
    (real if present, simulated otherwise -- the title states which).

    FIX (found via a direct user report -- a network figure was showing
    "SIMULATED outcome" for a run independently confirmed, at the
    regression level, to be using ~70-86% real CORDIS data): the OLD
    is_sim check compared energy_events_sim to coop_count. That
    comparison can NEVER be informative -- build_panel aliases
    energy_events_sim = coop_count in BOTH branches (real: "f[\"energy_
    events_sim\"] = f[\"coop_count\"]" right after the real merge;
    simulated: "f[\"coop_count\"] = f[\"energy_events_sim\"]" in the
    fallback) -- so the two columns are identical by construction
    regardless of which branch ran, and the figure reported SIMULATED
    unconditionally. Confirmed directly, not assumed: coop_count in an
    EU run with real CORDIS active took values up to 685 (a real project
    count; the simulated negative-binomial baseline does not reach that
    high) while still tripping the old is_sim check. Fixed by reusing
    the comparison plot_event_agreement_distributions already uses
    correctly: energy_events_sim vs formal_agreements_sim, which ARE
    genuinely different simulated draws when no real data is active, but
    are aliases of the SAME real series (hence identical) when it is.
    """
    import networkx as nx
    coop_col = "energy_events_sim" if "energy_events_sim" in df.columns and df["energy_events_sim"].notna().any() else "coop_count"
    is_sim = ("energy_events_sim" in df.columns and "formal_agreements_sim" in df.columns
             and not df["energy_events_sim"].equals(df["formal_agreements_sim"]))
    edge_data = df.groupby(["iso_i", "iso_j"])[coop_col].mean().reset_index()
    edge_data = edge_data[edge_data[coop_col] > 0]
    if len(edge_data) == 0:
        return None
    # Dense event-like outcomes (real or simulated) can make an unfiltered
    # network a near-complete, illegible graph -- show the strongest third
    # of edges by weight, which is enough to reveal the backbone structure
    # (e.g. which countries cluster together) without visual clutter, and
    # say so explicitly in the title rather than silently dropping data.
    threshold = edge_data[coop_col].quantile(2 / 3)
    edge_data_shown = edge_data[edge_data[coop_col] >= threshold]
    G = nx.Graph()
    for _, r in edge_data_shown.iterrows():
        G.add_edge(r.iso_i, r.iso_j, weight=r[coop_col])
    if G.number_of_edges() == 0:
        return None
    pos = nx.circular_layout(G)
    fig, ax = plt.subplots(figsize=(8, 8))
    weights = [G[u][v]["weight"] for u, v in G.edges()]
    max_w = max(weights) if weights else 1
    nx.draw_networkx_nodes(G, pos, node_size=300, node_color="#1F3864", ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=7, font_color="white", ax=ax)
    nx.draw_networkx_edges(G, pos, width=[1 + 4 * w / max_w for w in weights], alpha=0.5, edge_color="#2E5A88", ax=ax)
    tag = "SIMULATED outcome" if is_sim else "real/mixed outcome"
    ax.set_title(f"{club_key.upper()}: cooperation network ({tag})\n"
                f"top third of dyads by mean annual count shown ({len(edge_data_shown)}/{len(edge_data)} edges)")
    ax.axis("off")
    plt.tight_layout()
    return _save_fig(fig, out_dir, "fig4_cooperation_network")


def plot_event_agreement_distributions(club_key, df, out_dir):
    """New figure: distributions of energy_events_sim and
    formal_agreements_sim, with zero shares annotated -- shows directly
    why the two outcomes behave so differently in estimation (dense vs
    sparse), rather than only stating their zero-share numbers in text.
    Skipped when real cooperation data is active: in that case both
    columns are aliases of the SAME real series (by design, see
    build_panel), so plotting them side by side would show two identical,
    uninformative panels rather than genuinely different simulated DGPs."""
    if "energy_events_sim" not in df.columns or "formal_agreements_sim" not in df.columns:
        return None
    if df["energy_events_sim"].equals(df["formal_agreements_sim"]):
        print(f"    [fig_event_agreement_distributions] skipped -- real cooperation data is "
             f"active, so both columns are the same real series, not distinct simulated DGPs.")
        return None
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for ax, col, title in zip(axes, ["energy_events_sim", "formal_agreements_sim"],
                              ["Energy events (primary, NB)", "Formal agreements (robustness, sparse)"]):
        vals = df[col].dropna()
        zero_share = (vals == 0).mean()
        max_v = int(vals.max()) if len(vals) else 0
        # FIX: one bin per integer becomes invisible once the range is
        # wide (e.g. real data can range into the hundreds) -- capped at
        # a sensible number of bins instead, still integer-aligned for a
        # count outcome.
        n_bins = min(max_v + 1, 30) if max_v > 0 else 2
        bins = np.linspace(-0.5, max_v + 0.5, n_bins + 1)
        ax.hist(vals, bins=bins, color="#1F3864", edgecolor="white")
        ax.set_title(f"{title}\nzero share={zero_share:.1%}, mean={vals.mean():.2f}, max={max_v}", fontsize=9)
        ax.set_xlabel("Count"); ax.set_ylabel("Dyad-years")
    fig.suptitle(f"{club_key.upper()}: simulated cooperation outcome distributions", y=1.03)
    plt.tight_layout()
    return _save_fig(fig, out_dir, "fig_event_agreement_distributions")


def plot_marginal_effects_volatility(club_key, models, out_dir, prov=None):
    """Figure 8: marginal ES effect across price volatility (structurally
    identical construction to Figure 7). Uses m_full_ppml for the same
    reason as Figure 7 -- m4_vol omits ES x Innovation, which would
    distort this marginal-effect curve the same way an omitted moderator
    distorts a headline coefficient.

    TASK (reviewer point 4): the axis label and title used to hardcode
    "SIMULATED, not real Brent data" unconditionally -- stale as soon as
    real EIA Brent data was integrated (see load_real_brent_annual), and
    exactly the kind of claim the reviewer flagged as no longer accurate.
    This now checks prov for the ACTUAL source used in this run rather
    than asserting either way, and -- whether real or simulated -- uses
    "Brent price turbulence" rather than "volatility": the real series is
    a rolling 3-year trailing standard deviation of ANNUAL log returns
    (see that function's docstring), a genuine but structurally different
    statistic from intra-year/monthly volatility, so calling it
    "volatility" unqualified overstates precision it does not have.
    """
    model_key = "m_full_ppml" if "m_full_ppml" in models else "m4_vol"
    if model_key not in models:
        return None
    mm = models[model_key]
    names = list(mm.coef().index)
    if "es_mean_l1_z" not in names or "es_mean_l1_z:vol_z" not in names:
        return None
    b1 = float(mm.coef()["es_mean_l1_z"]); b3 = float(mm.coef()["es_mean_l1_z:vol_z"])
    V = mm._vcov
    i1, i3 = names.index("es_mean_l1_z"), names.index("es_mean_l1_z:vol_z")
    grid = np.linspace(-2, 2, 41)
    me = b1 + b3 * grid
    var = V[i1, i1] + grid**2 * V[i3, i3] + 2 * grid * V[i1, i3]
    se = np.sqrt(np.clip(var, 0, None))
    vol_status = ""
    if prov is not None:
        for row in prov:
            if row.get("variable") == "price volatility":
                vol_status = row.get("status", "")
    is_real_vol = "SIMULATED" not in vol_status.upper() and vol_status != ""
    if is_real_vol:
        axis_label = "Brent price turbulence (SD) -- real EIA data, annual-frequency rolling measure"
        title_tag = "real Brent price turbulence"
    else:
        axis_label = "Brent price turbulence (SD) -- SIMULATED, not real Brent data"
        title_tag = "SIMULATED price turbulence"
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(grid, me, color="#C00000", lw=2)
    ax.fill_between(grid, me - 1.96 * se, me + 1.96 * se, color="#C00000", alpha=0.15)
    ax.axhline(0, color="grey", ls="--", lw=1)
    ax.set_xlabel(axis_label)
    ax.set_ylabel("Marginal effect of ES (log scale)")
    ax.set_title(f"{club_key.upper()}: marginal ES effect across {title_tag} ({model_key})")
    plt.tight_layout()
    return _save_fig(fig, out_dir, "fig8_marginal_effects_volatility")


def plot_combined_trends(results_by_df, out_dir):
    """Combined G20 + EU-22 annual cooperation trends in one figure (the
    per-club fig5 exists separately; this is the explicit side-by-side
    comparison requested). TASK (reviewer): "the analysis is not 'Europe.'
    It is an EU-22 subset excluding Croatia, Cyprus, Estonia, Malta and
    Romania... use 'EU-22' everywhere until coverage is expanded" -- fixed
    here and in the chart title below (was "G20 vs Europe")."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
    colors = {"g20": "#1F3864", "eu": "#C00000"}
    for club_key, df in results_by_df.items():
        coop_col = "energy_events_sim" if "energy_events_sim" in df.columns else "coop_count"
        annual = df.groupby("year").agg(mean_count=(coop_col, "mean"), zero_share=(coop_col, lambda s: (s == 0).mean()))
        ax1.plot(annual.index, annual.mean_count, "-o", color=colors.get(club_key, "grey"), label=club_key.upper())
        ax2.plot(annual.index, annual.zero_share * 100, "-o", color=colors.get(club_key, "grey"), label=club_key.upper())
    ax1.set_ylabel("Mean count per dyad"); ax1.set_title("Annual cooperation trends: G20 vs EU-22")
    ax1.legend(fontsize=9)
    ax2.set_ylabel("% dyad-years with zero cooperation"); ax2.set_xlabel("Year")
    ax2.legend(fontsize=9)
    plt.tight_layout()
    return _save_fig(fig, out_dir, "fig5b_combined_trends_g20_eu")


def generate_diagrams(club_key, df, models, out_dir, prov=None):
    print(f"\n  [{club_key}] Generating diagrams...")
    plot_data_coverage_heatmap(club_key, df, out_dir)
    plot_cooperation_trends(club_key, df, out_dir)
    plot_event_agreement_distributions(club_key, df, out_dir)
    plot_baseline_coefficients(club_key, models, out_dir)
    plot_marginal_effects_innovation(club_key, models, out_dir)
    plot_marginal_effects_volatility(club_key, models, out_dir, prov)
    plot_lead_lag(club_key, models, out_dir)
    plot_robustness_specs(club_key, models, out_dir)
    plot_cooperation_network(club_key, df, out_dir)


def compare_clubs(results_by_club):
    if len(results_by_club) < 2:
        return
    out_dir = OUT_BASE / "comparison"; (out_dir / "tables").mkdir(parents=True, exist_ok=True)

    rows = []
    for key, r in results_by_club.items():
        rows.append({
            "club": r["label"], "members": r["n_members"], "dyads": r["n_dyads"],
            "estimation_obs": r["n_obs"], "zero_cooperation_share": round(r["zero_share"], 3),
            "ppml_ES_baseline": round(r["coefs"].get("m1_ppml", float("nan")), 4),
            "ppml_ES_x_Innov": round(r["coefs"].get("m3_innov", float("nan")), 4),
            "ppml_ES_x_Vol": round(r["coefs"].get("m4_vol", float("nan")), 4),
        })
    summary = pd.DataFrame(rows)
    summary.to_csv(out_dir / "tables" / "comparison_summary.csv", index=False)

    print("=" * 70)
    print("CROSS-CLUB COMPARISON  (NOTE: this is the NAIVE side-by-side of two")
    print("SEPARATELY estimated models -- it CANNOT support a claim that the")
    print("two clubs' effects differ. See Section 8B / run_pooled_comparison()")
    print("above for the actual EU-G20 difference test (delta_1, delta_3,")
    print("Wald tests). Cooperation outcomes are simulated for both clubs")
    print("unless data/cooperation.csv / data/eu_cooperation.csv were populated.")
    print("=" * 70)
    print(summary.to_string(index=False))
    print(f"\nComparison table written to {out_dir}/tables/comparison_summary.csv")
    return summary


# =========================================================================
# SECTION 9 ? CLI / MAIN / SELF-TESTS
# =========================================================================
def write_templates(club_keys):
    for ck in club_keys:
        club = CLUBS[ck]
        members = list(club["members_map"].keys())
        dyads = list(combinations(members, 2))
        grid = pd.DataFrame([(i, j, y) for (i, j) in dyads for y in YEARS],
                            columns=["iso_i", "iso_j", "year"])
        grid.assign(coop_count=np.nan, coop_depth=np.nan).to_csv(DATA / club["coop_file"], index=False)
        grid.assign(trade=np.nan).to_csv(DATA / club["trade_file"], index=False)
        if club["fdi_file"]:
            grid.assign(fdi_count=np.nan).to_csv(DATA / club["fdi_file"], index=False)
        print(f"[{club['label']}] templates written to {DATA}/ ({len(grid)} rows each)")


def _parse_club_arg():
    if "--club" in sys.argv:
        i = sys.argv.index("--club")
        val = sys.argv[i + 1].lower()
        if val not in ("g20", "eu", "both"):
            raise SystemExit(f"--club must be g20, eu, or both (got {val!r})")
        return ["g20", "eu"] if val == "both" else [val]
    return ["g20", "eu"]  # default: run both


def main():
    from pprint import pprint

    if "--montecarlo" in sys.argv:
        print("in montecarlo")
        run_monte_carlo_validation()
        return
    if "--pooled" in sys.argv and "--club" not in sys.argv:
        print("in pooled")
        run_pooled_comparison()
        print()
        run_monte_carlo_validation()
        return
    club_keys = _parse_club_arg()
    results = {}
    print("club keys", club_keys)
    for ck in club_keys:
        print("run_club",ck)
        results[ck] = run_club(ck)

        print("\n\n After run_club:")
        print("\n\n>> results[ck][club]: ",results[ck]["club"] )
        print("\n\n>> results[ck][label]: ",results[ck]["label"] )
        print("\n\n>> results[ck][n_members]: ",results[ck]["n_members"] )
        print("\n\n>> results[ck][n_dyads]: ",results[ck]["n_dyads"] )
        print("\n\n>> results[ck][n_obs]: ",results[ck]["n_obs"] )
        print("\n\n>> results[ck][zero_share]: ",results[ck]["zero_share"] )
        print("\n\n>> results[ck][coefs]: ",results[ck]["coefs"] )
        print("\n\n>> results[ck][provenance]: ",results[ck]["provenance"].to_string() )
        print("\n\n>> results[ck][df]: ",results[ck]["df"] )

        print("\n\n>> results[ck][provenance] type: ",type(results[ck]["provenance"]) )
        print("\n\n>> Provenance Columns: ", results[ck]["provenance"].columns)
        print("\n\n>> results[ck][df] type: ",type(results[ck]["df"]) )
        print("\n\n>> df Columns: ",results[ck]["df"].columns)
        #pprint("\n\n>> df Columns: ", str(results[ck]["df"].columns.tolist()))


    print("\n\n>> Before compare_clubs")
    compare_clubs(results)
    if len(club_keys) == 2:
        plot_combined_trends({ck: r["df"] for ck, r in results.items()}, OUT_BASE / "comparison")
        print()
        run_pooled_comparison()
        print()
        # Default (both-clubs) run uses a faster B here to keep total
        # runtime practical -- run `--montecarlo` or `--pooled` directly
        # for the full B=1000 battery (that path is unchanged above).
        run_monte_carlo_validation(B=150)


    print("\n\n>> main routine run successfully")


def selftest():
    # DEA engine
    X = np.array([[1.0], [1.0]]); Y = np.array([[2.0], [1.0]]); B = np.array([[1.0], [1.0]])
    assert _ddf(X[0], Y[0], B[0], X, Y, B) < 1e-6 and _ddf(X[1], Y[1], B[1], X, Y, B) > 1e-6
    assert haversine((0, 0), (0, 0)) < 1e-9

    # World Bank parser
    s = [{"page": 1}, [{"countryiso3code": "USA", "date": "2020", "value": 3.45}]]
    assert parse_wb_payload(s, "rd_gdp").iloc[0]["rd_gdp"] == 3.45
    assert parse_wb_payload([{}, None], "x").empty

    # IMF SDMX parser
    imf = {"CompactData": {"DataSet": {"Series": [
        {"@REF_AREA": "US", "@COUNTERPART_AREA": "CN",
         "Obs": [{"@TIME_PERIOD": "2015", "@OBS_VALUE": "12.3"}]}]}}}
    assert parse_imf_compactdata(imf, "fdi_value").iloc[0]["fdi_value"] == 12.3

    # Eurostat JSON-stat 2.0 parser
    js = {"id": ["geo", "time"], "size": [2, 2],
          "dimension": {"geo": {"category": {"index": {"DE": 0, "FR": 1}}},
                        "time": {"category": {"index": {"2020": 0, "2021": 1}}}},
          "value": {"0": 3.1, "1": 3.2, "2": 2.1, "3": 2.3}}
    df = _jsonstat_to_long(js, "x")
    assert abs(float(df[(df.geo == "DE") & (df.year == 2021)]["x"].iloc[0]) - 3.2) < 1e-9
    assert abs(float(df[(df.geo == "FR") & (df.year == 2020)]["x"].iloc[0]) - 2.1) < 1e-9

    # club configs sanity
    assert len(CLUBS["g20"]["members_map"]) == 19  # 19 sovereigns; EU aggregate removed
    assert len(CLUBS["eu"]["members_map"]) == 22  # 22 real-data-covered (HRV/CYP/EST/MLT/ROU excluded)
    for ck, club in CLUBS.items():
        for m in club["members_map"]:
            assert m in club["capitals_fallback"], f"{ck}: {m} missing capital"
            assert m in club["langs"], f"{ck}: {m} missing language"

    print("self-tests: OK")


if __name__ == "__main__":
    if "--selftest" in sys.argv:
        selftest()
    elif "--templates" in sys.argv:
        write_templates(_parse_club_arg())
    else:
        main()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
club keys ['g20', 'eu']
run_club g20
====================================================================== in run_club
G20 ? DEA + connectors + PPML
Before compute_es_des dict_keys(['ARG', 'AUS', 'BRA', 'CAN', 'CHN', 'DEU', 'FRA', 'GBR', 'IDN', 'IND', 'ITA', 'JPN', 'KOR', 'MEX', 'RUS', 'SAU', 'TUR', 'USA', 'ZAF']) /content/drive/MyDrive/ppml/output_from_claude/g20 [] g20
in compute_es_dea


>> In get_common_es_frontier
in load_ch2_dea_mpi
  [Chapter 2 DEA/MPI] 38/38 union members covered (2000-2023, bias-corrected).
after load_ch2_dea_mpi: Returns:      iso  year        es  ml_index  ml_effch  ml_techch
0    ARG  2000  1.000000       NaN       NaN        NaN
1    ARG  2001  1.000000  1.055790  1.078111   0.979296
2    ARG  2002  1.000000  0.974019  0.980574   0.993316
3    ARG  2003  1.000000  1.007865  1.051088   0.958878
4    ARG  2004  1.000000  1.013095 